## Libraries

In [ ]:
"""
SWOT Urban Hydrology Analysis Pipeline
============================================
Purpose: Build DataFrame with Near/Far ISA geometry analysis
- Near/Far ISA: Only use segment directly connected to Node
- If MultiLineString, ignore distant segments
"""

import xarray as xr
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
from datetime import timedelta
from shapely.geometry import Point, LineString
import rasterio
import re
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

import xarray as xr
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
from datetime import timedelta
from shapely.geometry import Point, LineString
import rasterio
import re
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

## Calculate near-far ISA and building height at Node Level (Fctimp + BuildingHeight) Version2

In [ ]:
## This method takes too many memories, so divided as batches and saved as parquet

In [ ]:
# #!/usr/bin/env python
# """
# SWOT Urban Hydrology Analysis Pipeline v6.2 (Optimized)
# - Fixed Rasterio I/O bottleneck (Global Worker State)
# - Vectorized sampling using src.sample()
# - Per-reach processing with memory optimization
# """
# import pandas as pd
# import numpy as np
# import geopandas as gpd
# from pathlib import Path
# from shapely.geometry import Point, LineString
# from shapely import wkt
# import rasterio
# from concurrent.futures import ProcessPoolExecutor
# import multiprocessing as mp
# from collections import defaultdict
# import os
# import time
# import gc
# import warnings
# warnings.filterwarnings('ignore')

# # =============================================================================
# # Configuration
# # =============================================================================
# BASE_PATH = Path('/work/pi_kandread_umass_edu/swot-urban')
# PARQUET_PATH = BASE_PATH / '5_fctimp'
# TVP_PATH = BASE_PATH / '6_pixc_tvp' / 'SWOT_TVP_Data'
# POLYGON_PATH = BASE_PATH / '2_urban_reaches' / 'all_reaches_vD.parquet'
# NLCD_FCTIMP_PATH = str(BASE_PATH / '0_data_collection/1_nlcd/Annual_NLCD_FctImp_2024_CU_C1V1_EPSG4326.tif')
# BUILDING_HEIGHT_PATH = str(BASE_PATH / '0_data_collection/3_150m_building/GBH2020_150m_GEDI_4326.tif')
# OUTPUT_PATH = BASE_PATH / '3_analysis_results'
# TEMP_PATH = OUTPUT_PATH / 'temp_reaches'

# # Core counts managed by Slurm
# N_WORKERS = int(os.environ.get('SLURM_CPUS_PER_TASK', 48))


# # =============================================================================
# # Global Worker State (Initialize once per worker)
# # =============================================================================
# src_isa = None
# src_bld = None


# def init_worker(isa_path, bld_path):
#     """
#     Worker initializer: Opens raster files once and keeps them open.
#     """
#     global src_isa, src_bld
#     try:
#         src_isa = rasterio.open(isa_path)
#     except Exception as e:
#         print(f"Worker Error opening ISA: {e}")
#         src_isa = None
        
#     if Path(bld_path).exists():
#         try:
#             src_bld = rasterio.open(bld_path)
#         except Exception as e:
#             src_bld = None
#     else:
#         src_bld = None


# # =============================================================================
# # Sampling Functions (Optimized)
# # =============================================================================
# def sample_raster_along_line_optimized(line, node_pt, dataset_type='isa', n_samples=100, valid_range=(0, 100)):
#     """
#     Optimized sampling using src.sample() instead of repeated read()
#     """
#     empty_result = {
#         'mean_val': 0.0, 'nonzero_ratio': 0.0,
#         'dw_mean_val': 0.0, 'dw_ratio': 0.0,
#     }
    
#     # Select the correct global dataset handle
#     if dataset_type == 'isa':
#         src = src_isa
#     else:
#         src = src_bld
    
#     if src is None or line is None or line.is_empty:
#         return empty_result
    
#     # 1. Geometry Prep
#     target_line = None
#     if line.geom_type == 'MultiLineString':
#         segments = list(line.geoms)
#         if not segments:
#             return empty_result
#         target_line = min(segments, key=lambda s: s.distance(node_pt))
#         if target_line.length == 0:
#             return empty_result
#     elif line.geom_type == 'LineString':
#         if line.length == 0:
#             return empty_result
#         target_line = line
#     else:
#         return empty_result
    
#     # 2. Coordinate Generation
#     start_pt = Point(target_line.coords[0])
#     end_pt = Point(target_line.coords[-1])
#     reverse = node_pt.distance(start_pt) > node_pt.distance(end_pt)
    
#     coords = []
#     indices = []
    
#     for i in range(n_samples + 1):
#         t = 1.0 - (i / n_samples) if reverse else i / n_samples
#         pt = target_line.interpolate(t, normalized=True)
#         coords.append((pt.x, pt.y))
#         indices.append(i)
    
#     # 3. Fast Sampling (Vectorized)
#     values = []
#     try:
#         sampled_gen = src.sample(coords, indexes=1)
        
#         for idx, val in zip(indices, sampled_gen):
#             v = float(val[0])  # val is array, get first element
            
#             if dataset_type == 'building':
#                 if v >= 0 and not np.isnan(v):
#                     values.append((idx, v))
#             else:  # ISA
#                 if valid_range[0] <= v <= valid_range[1]:
#                     values.append((idx, v))
#     except Exception:
#         return empty_result
    
#     if not values:
#         return empty_result
    
#     # 4. Statistics Calculation
#     all_vals = [v for _, v in values]
#     mean_val = np.mean(all_vals)
#     nonzero_ratio = sum(1 for v in all_vals if v > 0) / (n_samples + 1)
    
#     weighted_sum = 0.0
#     weight_sum = 0.0
#     weighted_nonzero_sum = 0.0
    
#     for idx, val in values:
#         weight = (n_samples - idx) / n_samples
#         weighted_sum += weight * val
#         weight_sum += weight
#         if val > 0:
#             weighted_nonzero_sum += weight
    
#     dw_mean_val = weighted_sum / weight_sum if weight_sum > 0 else 0.0
#     dw_ratio = weighted_nonzero_sum / weight_sum if weight_sum > 0 else 0.0
    
#     return {
#         'mean_val': mean_val, 'nonzero_ratio': nonzero_ratio,
#         'dw_mean_val': dw_mean_val, 'dw_ratio': dw_ratio,
#     }


# def calculate_near_far_metrics(node_lon, node_lat, sat_lon, sat_lat, polygon):
#     """Calculate Near/Far metrics using optimized sampling."""
#     empty_result = {
#         'near_isa': 0.0, 'far_isa': 0.0, 'near_ratio': 0.0, 'far_ratio': 0.0,
#         'dw_near_isa': 0.0, 'dw_far_isa': 0.0, 'dw_near_ratio': 0.0, 'dw_far_ratio': 0.0,
#         'near_building': 0.0, 'far_building': 0.0, 'near_ratio_building': 0.0, 'far_ratio_building': 0.0,
#         'dw_near_building': 0.0, 'dw_far_building': 0.0, 'dw_near_ratio_building': 0.0, 'dw_far_ratio_building': 0.0,
#     }
    
#     node_pt = Point(node_lon, node_lat)
#     dx = sat_lon - node_lon
#     dy = sat_lat - node_lat
#     length = np.sqrt(dx**2 + dy**2)
    
#     if length == 0:
#         return empty_result
    
#     ux, uy = dx / length, dy / length
#     extend = 0.2
    
#     near_line = LineString([node_pt, Point(node_lon + ux * extend, node_lat + uy * extend)])
#     far_line = LineString([node_pt, Point(node_lon - ux * extend, node_lat - uy * extend)])
    
#     try:
#         near_in_poly = near_line.intersection(polygon) if near_line.intersects(polygon) else None
#         far_in_poly = far_line.intersection(polygon) if far_line.intersects(polygon) else None
#     except:
#         return empty_result
    
#     result = empty_result.copy()
    
#     # ISA sampling (optimized)
#     near_isa = sample_raster_along_line_optimized(near_in_poly, node_pt, dataset_type='isa')
#     far_isa = sample_raster_along_line_optimized(far_in_poly, node_pt, dataset_type='isa')
    
#     result.update({
#         'near_isa': near_isa['mean_val'], 'far_isa': far_isa['mean_val'],
#         'near_ratio': near_isa['nonzero_ratio'], 'far_ratio': far_isa['nonzero_ratio'],
#         'dw_near_isa': near_isa['dw_mean_val'], 'dw_far_isa': far_isa['dw_mean_val'],
#         'dw_near_ratio': near_isa['dw_ratio'], 'dw_far_ratio': far_isa['dw_ratio'],
#     })
    
#     # Building sampling (optimized)
#     if src_bld is not None:
#         near_bld = sample_raster_along_line_optimized(near_in_poly, node_pt, dataset_type='building')
#         far_bld = sample_raster_along_line_optimized(far_in_poly, node_pt, dataset_type='building')
        
#         result.update({
#             'near_building': near_bld['mean_val'], 'far_building': far_bld['mean_val'],
#             'near_ratio_building': near_bld['nonzero_ratio'], 'far_ratio_building': far_bld['nonzero_ratio'],
#             'dw_near_building': near_bld['dw_mean_val'], 'dw_far_building': far_bld['dw_mean_val'],
#             'dw_near_ratio_building': near_bld['dw_ratio'], 'dw_far_ratio_building': far_bld['dw_ratio'],
#         })
    
#     return result


# def get_geometry_category(near_val, far_val):
#     has_near, has_far = near_val > 0, far_val > 0
#     if has_near and has_far:
#         return 'Near/Far'
#     elif has_near:
#         return 'Near'
#     elif has_far:
#         return 'Far'
#     else:
#         return 'NA'


# def get_sample_geometry_category(near_val, far_val, near_ratio, far_ratio, min_ratio=0.5):
#     has_near = near_val > 0 and near_ratio >= min_ratio
#     has_far = far_val > 0 and far_ratio >= min_ratio
#     if has_near and has_far:
#         return 'Near/Far'
#     elif has_near:
#         return 'Near'
#     elif has_far:
#         return 'Far'
#     else:
#         return 'NA'


# def process_single_task(task):
#     """Process single observation using global raster handles."""
#     idx, node_lon, node_lat, sat_lon, sat_lat, polygon_wkt = task
    
#     result = {
#         'idx': idx,
#         'near_isa': 0.0, 'far_isa': 0.0, 'near_ratio': 0.0, 'far_ratio': 0.0,
#         'dw_near_isa': 0.0, 'dw_far_isa': 0.0, 'dw_near_ratio': 0.0, 'dw_far_ratio': 0.0,
#         'near_building': 0.0, 'far_building': 0.0, 'near_ratio_building': 0.0, 'far_ratio_building': 0.0,
#         'dw_near_building': 0.0, 'dw_far_building': 0.0, 'dw_near_ratio_building': 0.0, 'dw_far_ratio_building': 0.0,
#         'geometry_category': 'NA', 'sample_geometry_category': 'NA',
#         'geometry_category_building': 'NA', 'sample_geometry_category_building': 'NA',
#     }
    
#     if sat_lon is None or sat_lat is None or np.isnan(sat_lon) or np.isnan(sat_lat):
#         result['geometry_category'] = 'No_Sat_Data'
#         return result
    
#     if not (np.isfinite(node_lon) and np.isfinite(sat_lon)):
#         return result
    
#     try:
#         polygon = wkt.loads(polygon_wkt)
#     except:
#         return result
    
#     metrics = calculate_near_far_metrics(node_lon, node_lat, sat_lon, sat_lat, polygon)
#     result.update(metrics)
    
#     # Categories
#     result['geometry_category'] = get_geometry_category(result['near_isa'], result['far_isa'])
#     result['sample_geometry_category'] = get_sample_geometry_category(
#         result['near_isa'], result['far_isa'], result['near_ratio'], result['far_ratio']
#     )
#     result['geometry_category_building'] = get_geometry_category(result['near_building'], result['far_building'])
#     result['sample_geometry_category_building'] = get_sample_geometry_category(
#         result['near_building'], result['far_building'], result['near_ratio_building'], result['far_ratio_building']
#     )
    
#     return result


# # =============================================================================
# # Process Single Reach
# # =============================================================================
# def process_single_reach(reach_id, node_files_list, tvp_file, polygon_wkt, temp_dir):
#     """Process a single reach with optimized parallel workers."""
#     out_file = temp_dir / f'reach_{reach_id}.parquet'
    
#     # Skip if already processed
#     if out_file.exists():
#         return out_file
    
#     # make sure node_files_list is a list
#     if not isinstance(node_files_list, list):
#         node_files_list = [node_files_list]
    
#     try:
#         # Load ALL node files for this reach
#         node_dfs = []
#         for nf in node_files_list:
#             try:
#                 df = pd.read_parquet(nf)
#                 node_dfs.append(df)
#             except:
#                 pass
        
#         if not node_dfs:
#             return None
        
#         node_df = pd.concat(node_dfs, ignore_index=True)
#         del node_dfs
        
#         # Load TVP data if exists
#         if tvp_file and tvp_file.exists():
#             tvp_df = pd.read_parquet(tvp_file)
            
#             # Merge by time - remove timezone to match types
#             node_df['time_dt'] = pd.to_datetime(node_df['time_str']).dt.tz_localize(None)
#             tvp_df['time_dt'] = pd.to_datetime(tvp_df['time'], unit='s')
            
#             node_df = node_df.sort_values('time_dt').reset_index(drop=True)
#             tvp_df = tvp_df.sort_values('time_dt')
            
#             node_df = pd.merge_asof(
#                 node_df,
#                 tvp_df[['time_dt', 'latitude', 'longitude', 'altitude']],
#                 on='time_dt',
#                 direction='nearest',
#                 tolerance=pd.Timedelta('30s')
#             ).rename(columns={
#                 'latitude': 'sat_lat',
#                 'longitude': 'sat_lon',
#                 'altitude': 'sat_altitude'
#             })
            
#             del tvp_df
#         else:
#             node_df['sat_lon'] = np.nan
#             node_df['sat_lat'] = np.nan
#             node_df['sat_altitude'] = np.nan
        
#         # Prepare tasks
#         tasks = []
#         for idx, row in node_df.iterrows():
#             task = (idx, row['lon'], row['lat'], row['sat_lon'], row['sat_lat'], polygon_wkt)
#             tasks.append(task)
        
#         # Process with parallel workers (with initializer for global raster handles)
#         if len(tasks) > 0:
#             with ProcessPoolExecutor(
#                 max_workers=N_WORKERS,
#                 initializer=init_worker,
#                 initargs=(NLCD_FCTIMP_PATH, BUILDING_HEIGHT_PATH)
#             ) as executor:
#                 results = list(executor.map(process_single_task, tasks, chunksize=100))
            
#             # Merge results
#             results_df = pd.DataFrame(results).set_index('idx')
            
#             result_cols = [
#                 'near_isa', 'far_isa', 'near_ratio', 'far_ratio',
#                 'dw_near_isa', 'dw_far_isa', 'dw_near_ratio', 'dw_far_ratio',
#                 'near_building', 'far_building', 'near_ratio_building', 'far_ratio_building',
#                 'dw_near_building', 'dw_far_building', 'dw_near_ratio_building', 'dw_far_ratio_building',
#                 'geometry_category', 'sample_geometry_category',
#                 'geometry_category_building', 'sample_geometry_category_building',
#             ]
            
#             for col in result_cols:
#                 if col in results_df.columns:
#                     node_df[col] = results_df[col]
            
#             del results, results_df
        
#         # Save
#         node_df.to_parquet(out_file)
        
#         n_rows = len(node_df)
#         del node_df
#         gc.collect()
        
#         return out_file
        
#     except Exception as e:
#         print(f"\n      Error processing reach {reach_id}: {e}")
#         return None


# # =============================================================================
# # Main Pipeline
# # =============================================================================
# def run_pipeline():
#     """Run per-reach pipeline with optimized workers."""
    
#     OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
#     TEMP_PATH.mkdir(parents=True, exist_ok=True)
    
#     print("=" * 70)
#     print("SWOT Urban Hydrology Analysis Pipeline v6.2 (Optimized)")
#     print(f"  Workers per reach: {N_WORKERS}")
#     print(f"  Rasterio I/O: Global worker state (open once per worker)")
#     print("=" * 70)
    
#     total_start = time.time()
    
#     # ==========================================================================
#     # PHASE 1: Build lookup tables
#     # ==========================================================================
#     print("\n[PHASE 1] Building lookup tables...")
    
#     # Load polygons -> dict
#     print("  Loading polygons...")
#     polygons_gdf = gpd.read_parquet(POLYGON_PATH)
#     if polygons_gdf.crs != 'EPSG:4326':
#         polygons_gdf = polygons_gdf.to_crs('EPSG:4326')
    
#     polygon_wkt_dict = {}
#     for _, row in polygons_gdf.iterrows():
#         rid = row['reach_id']
#         polygon_wkt_dict[int(rid)] = row.geometry.wkt
    
#     print(f"    {len(polygon_wkt_dict)} polygons")
#     del polygons_gdf
#     gc.collect()
    
#     # Build node file lookup: reach_id -> LIST of file paths
#     print("  Scanning node files...")
#     node_files = sorted(PARQUET_PATH.glob('fctimp_mean_*_node_*.parquet'))
    
#     node_file_dict = defaultdict(list)
    
#     for f in node_files:
#         try:
#             df_sample = pd.read_parquet(f, columns=['reach_id'])
#             if len(df_sample) > 0:
#                 reach_id = int(df_sample['reach_id'].iloc[0])
#                 node_file_dict[reach_id].append(f)
#         except Exception as e:
#             print(f"    Warning: Could not read {f.name}: {e}")
    
#     node_file_dict = dict(node_file_dict)
    
#     total_files = sum(len(files) for files in node_file_dict.values())
#     print(f"    {total_files} node files mapped to {len(node_file_dict)} reaches")
    
#     # Build TVP file lookup: reach_id -> file path
#     print("  Scanning TVP files...")
#     tvp_files = sorted(TVP_PATH.glob('tvp_reach_*.parquet'))
    
#     tvp_file_dict = {}
#     for f in tvp_files:
#         try:
#             reach_id = int(f.stem.split('_')[-1])
#             tvp_file_dict[reach_id] = f
#         except:
#             pass
    
#     print(f"    {len(tvp_file_dict)} TVP files")
    
#     # ==========================================================================
#     # PHASE 2: Process each reach
#     # ==========================================================================
#     valid_reaches = set(polygon_wkt_dict.keys()) & set(node_file_dict.keys())
#     valid_reaches = sorted(valid_reaches)
    
#     print(f"\n[PHASE 2] Processing {len(valid_reaches)} reaches...")
    
#     processed_files = []
#     total_rows = 0
    
#     for i, reach_id in enumerate(valid_reaches, 1):
#         node_files_list = node_file_dict[reach_id]
#         tvp_file = tvp_file_dict.get(reach_id)
#         polygon_wkt = polygon_wkt_dict[reach_id]
        
#         print(f"  [{i}/{len(valid_reaches)}] Reach {reach_id} ({len(node_files_list)} node files)...", end=" ", flush=True)
#         start = time.time()
        
#         result_file = process_single_reach(reach_id, node_files_list, tvp_file, polygon_wkt, TEMP_PATH)
        
#         if result_file:
#             processed_files.append(result_file)
#             try:
#                 n_rows = len(pd.read_parquet(result_file, columns=['reach_id']))
#                 total_rows += n_rows
#                 print(f"done ({n_rows} rows, {time.time()-start:.1f}s)")
#             except:
#                 print(f"done ({time.time()-start:.1f}s)")
#         else:
#             print("failed")
        
#         gc.collect()
    
#     # ==========================================================================
#     # PHASE 3: Merge all results
#     # ==========================================================================
#     print(f"\n[PHASE 3] Merging {len(processed_files)} reach files...")
    
#     all_dfs = []
#     for f in processed_files:
#         try:
#             all_dfs.append(pd.read_parquet(f))
#         except Exception as e:
#             print(f"  Error reading {f}: {e}")
    
#     if all_dfs:
#         final_df = pd.concat(all_dfs, ignore_index=True)
#         del all_dfs
#         gc.collect()
        
#         # Save final results
#         out_parquet = OUTPUT_PATH / 'swot_urban_nearfar_v6.2_results.parquet'
#         final_df.to_parquet(out_parquet)
#         print(f"  Saved: {out_parquet}")
        
#         out_csv = OUTPUT_PATH / 'swot_urban_nearfar_v6.2_results.csv'
#         final_df.to_csv(out_csv, index=False)
#         print(f"  Saved: {out_csv}")
        
#         # Clean up temp files
#         print("  Cleaning up temp files...")
#         for f in processed_files:
#             try:
#                 f.unlink()
#             except:
#                 pass
#         try:
#             TEMP_PATH.rmdir()
#         except:
#             pass
        
#         # ==========================================================================
#         # Summary
#         # ==========================================================================
#         print("\n" + "=" * 70)
#         print("SUMMARY")
#         print("=" * 70)
#         print(f"Total rows: {len(final_df)}")
#         print(f"Total reaches: {len(valid_reaches)}")
#         print(f"Total time: {(time.time()-total_start)/60:.1f} minutes")
        
#         print(f"\n[ISA Geometry Category]")
#         print(final_df['geometry_category'].value_counts())
        
#         print(f"\n[Building Geometry Category]")
#         print(final_df['geometry_category_building'].value_counts())
        
#         valid_mask = final_df['geometry_category'] != 'No_Sat_Data'
#         n_valid = valid_mask.sum()
#         print(f"\n[Valid observations: {n_valid}/{len(final_df)} ({100*n_valid/len(final_df):.1f}%)]")
        
#         if valid_mask.any():
#             print(f"\n[ISA Stats (valid only)]")
#             print(f"  near_isa:    mean={final_df.loc[valid_mask, 'near_isa'].mean():.2f}, "
#                   f"median={final_df.loc[valid_mask, 'near_isa'].median():.2f}")
#             print(f"  far_isa:     mean={final_df.loc[valid_mask, 'far_isa'].mean():.2f}, "
#                   f"median={final_df.loc[valid_mask, 'far_isa'].median():.2f}")
#             print(f"  dw_near_isa: mean={final_df.loc[valid_mask, 'dw_near_isa'].mean():.2f}, "
#                   f"median={final_df.loc[valid_mask, 'dw_near_isa'].median():.2f}")
#             print(f"  dw_far_isa:  mean={final_df.loc[valid_mask, 'dw_far_isa'].mean():.2f}, "
#                   f"median={final_df.loc[valid_mask, 'dw_far_isa'].median():.2f}")
            
#             print(f"\n[Building Stats (valid only)]")
#             print(f"  near_building:    mean={final_df.loc[valid_mask, 'near_building'].mean():.2f}, "
#                   f"median={final_df.loc[valid_mask, 'near_building'].median():.2f}")
#             print(f"  far_building:     mean={final_df.loc[valid_mask, 'far_building'].mean():.2f}, "
#                   f"median={final_df.loc[valid_mask, 'far_building'].median():.2f}")
#             print(f"  dw_near_building: mean={final_df.loc[valid_mask, 'dw_near_building'].mean():.2f}, "
#                   f"median={final_df.loc[valid_mask, 'dw_near_building'].median():.2f}")
#             print(f"  dw_far_building:  mean={final_df.loc[valid_mask, 'dw_far_building'].mean():.2f}, "
#                   f"median={final_df.loc[valid_mask, 'dw_far_building'].median():.2f}")
        
#         return final_df
#     else:
#         print("  Error: No data to merge")
#         return None


# # =============================================================================
# # Entry Point
# # =============================================================================
# if __name__ == '__main__':
#     mp.set_start_method('spawn', force=True)
#     run_pipeline()

In [ ]:
import pandas as pd
from pathlib import Path

BASE_PATH = Path('/work/pi_kandread_umass_edu/swot-urban')
PARQUET_PATH = BASE_PATH / '5_fctimp'
TVP_PATH = BASE_PATH / '6_pixc_tvp' / 'SWOT_TVP_Data'

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# 1. fctimp (Node) 파일 확인
print("FCTIMP (Node) 파일")

node_files = sorted(PARQUET_PATH.glob('fctimp_mean_*_node_*.parquet'))
print(f"총 파일 수: {len(node_files)}")
print(f"파일명 예시: {node_files[0].name}")

node_df = pd.read_parquet(node_files[0])
print(f"\nShape: {node_df.shape}")
print(f"\nColumns ({len(node_df.columns)}):")
print(node_df.columns.tolist())
print(f"\nDtypes:")
print(node_df.dtypes)
print(f"\n첫 3행:")
print(node_df.head(3))

# 2. TVP 파일 확인
print("TVP 파일")

tvp_files = sorted(TVP_PATH.glob('tvp_reach_*.parquet'))
print(f"총 파일 수: {len(tvp_files)}")
print(f"파일명 예시: {tvp_files[0].name}")

tvp_df = pd.read_parquet(tvp_files[0])
print(f"\nShape: {tvp_df.shape}")
print(f"\nColumns ({len(tvp_df.columns)}):")
print(tvp_df.columns.tolist())
print(f"\nDtypes:")
print(tvp_df.dtypes)
print(f"\n첫 3행:")
print(tvp_df.head(3))

# 3. 같은 reach_id로 매칭해서 비교
print("같은 Reach 매칭 비교")

# Node에서 reach_id 추출
node_reach_id = int(node_df['reach_id'].iloc[0])
print(f"Node reach_id: {node_reach_id}")

# 해당 reach의 TVP 파일 찾기
tvp_file_matched = TVP_PATH / f'tvp_reach_{node_reach_id}.parquet'
if tvp_file_matched.exists():
    tvp_matched = pd.read_parquet(tvp_file_matched)
    print(f"TVP 파일 찾음: {tvp_file_matched.name}")
    print(f"TVP shape: {tvp_matched.shape}")
    print(f"\nTVP 첫 3행:")
    print(tvp_matched.head(3))
else:
    print(f"TVP 파일 없음: tvp_reach_{node_reach_id}.parquet")
    
    # 다른 reach 시도
    print("\n다른 reach로 시도...")
    for nf in node_files[:20]:
        df = pd.read_parquet(nf, columns=['reach_id'])
        rid = int(df['reach_id'].iloc[0])
        tvp_check = TVP_PATH / f'tvp_reach_{rid}.parquet'
        if tvp_check.exists():
            print(f"  Found: reach_id={rid}")
            node_df = pd.read_parquet(nf)
            tvp_matched = pd.read_parquet(tvp_check)
            print(f"\n  Node shape: {node_df.shape}")
            print(f"  TVP shape: {tvp_matched.shape}")
            break

In [ ]:
import pandas as pd
import numpy as np

J2000_EPOCH = pd.Timestamp('2000-01-01 00:00:00')

# Node 시간
node_time = pd.Timestamp('2023-10-29 22:27:48')
print(f"Node 시간: {node_time}")

# Node 시간을 J2000 초로 변환
node_j2000_seconds = (node_time - J2000_EPOCH).total_seconds()
print(f"Node J2000 초: {node_j2000_seconds:.1f}")

# TVP에서 해당 시간대 찾기
print(f"\nTVP time 범위:")
print(f"  min: {tvp_matched['time'].min():.1f} -> {J2000_EPOCH + pd.to_timedelta(tvp_matched['time'].min(), unit='s')}")
print(f"  max: {tvp_matched['time'].max():.1f} -> {J2000_EPOCH + pd.to_timedelta(tvp_matched['time'].max(), unit='s')}")

# Node 시간 근처의 TVP 찾기
time_diff = abs(tvp_matched['time'] - node_j2000_seconds)
closest_idx = time_diff.idxmin()
closest_row = tvp_matched.loc[closest_idx]

print(f"\nNode 시간과 가장 가까운 TVP:")
print(f"  TVP time: {closest_row['time']:.1f}")
print(f"  TVP datetime: {J2000_EPOCH + pd.to_timedelta(closest_row['time'], unit='s')}")
print(f"  차이: {time_diff.loc[closest_idx]:.1f} 초")
print(f"  위성 위치: ({closest_row['longitude']:.4f}, {closest_row['latitude']:.4f})")
print(f"  source_file: {closest_row['source_file']}")

# 해당 시간대의 TVP 데이터 보기 (±1초 범위)
mask = abs(tvp_matched['time'] - node_j2000_seconds) < 1.0
nearby = tvp_matched[mask]
print(f"\n±1초 범위 TVP 데이터: {len(nearby)} 행")
if len(nearby) > 0:
    print(nearby.head())

In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

BASE_PATH = Path('/work/pi_kandread_umass_edu/swot-urban')
PARQUET_PATH = BASE_PATH / '5_fctimp'
TVP_PATH = BASE_PATH / '6_pixc_tvp' / 'SWOT_TVP_Data'
POLYGON_PATH = BASE_PATH / '2_urban_reaches' / 'all_reaches_vD.parquet'

# 1. Polygon reach_ids
print("1. Polygon reach_ids...")
polygons_gdf = gpd.read_parquet(POLYGON_PATH)
polygon_reach_ids = set(polygons_gdf['reach_id'].astype(int).values)
print(f"   Polygon reaches: {len(polygon_reach_ids)}")

# 2. TVP reach_ids
print("\n2. TVP reach_ids...")
tvp_files = list(TVP_PATH.glob('tvp_reach_*.parquet'))
tvp_reach_ids = set()
for f in tvp_files:
    try:
        rid = int(f.stem.split('_')[-1])
        tvp_reach_ids.add(rid)
    except:
        pass
print(f"   TVP reaches: {len(tvp_reach_ids)}")

# 3. Node 파일 확인
print("\n3. Node 파일 샘플 확인...")
node_files = sorted(PARQUET_PATH.glob('fctimp_mean_*_node_*.parquet'))
print(f"   Node files: {len(node_files)}")

# 처음 10개 파일 확인
for i, nf in enumerate(node_files[:10]):
    try:
        df = pd.read_parquet(nf)
        rid = int(df['reach_id'].iloc[0])
        fctimp = df['fctimp_mean'].iloc[0]
        in_tvp = rid in tvp_reach_ids
        in_poly = rid in polygon_reach_ids
        print(f"   [{i}] reach={rid}, fctimp={fctimp:.3f}, TVP={in_tvp}, Poly={in_poly}")
    except Exception as e:
        print(f"   [{i}] Error: {e}")

# 4. 조건 맞는 reach 찾기
print("\n4. 조건 맞는 reach 찾기 (fctimp > 0, TVP 있음, Polygon 있음)...")
found = []
for nf in node_files[:500]:  # 더 많이 스캔
    try:
        df = pd.read_parquet(nf)
        if len(df) > 0:
            rid = int(df['reach_id'].iloc[0])
            fctimp = df['fctimp_mean'].iloc[0]
            if fctimp > 0 and rid in tvp_reach_ids and rid in polygon_reach_ids:
                found.append((rid, fctimp, nf.name))
                if len(found) >= 5:
                    break
    except:
        pass

if found:
    print(f"   찾음! {len(found)}개")
    for rid, fctimp, fname in found:
        print(f"     reach={rid}, fctimp={fctimp:.3f}, file={fname}")
else:
    print("   못 찾음")
    
    # fctimp > 0인 것만 찾기
    print("\n5. fctimp > 0인 reach 확인...")
    for nf in node_files[:100]:
        try:
            df = pd.read_parquet(nf)
            if len(df) > 0 and df['fctimp_mean'].iloc[0] > 0:
                rid = int(df['reach_id'].iloc[0])
                fctimp = df['fctimp_mean'].iloc[0]
                print(f"     reach={rid}, fctimp={fctimp:.3f}, TVP={rid in tvp_reach_ids}, Poly={rid in polygon_reach_ids}")
        except:
            pass

In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
import re

BASE_PATH = Path('/work/pi_kandread_umass_edu/swot-urban')
PARQUET_PATH = BASE_PATH / '5_fctimp'
TVP_PATH = BASE_PATH / '6_pixc_tvp' / 'SWOT_TVP_Data'
POLYGON_PATH = BASE_PATH / '2_urban_reaches' / 'all_reaches_vD.parquet'

# Polygon & TVP reach_ids
polygons_gdf = gpd.read_parquet(POLYGON_PATH)
polygon_reach_ids = set(polygons_gdf['reach_id'].astype(int).values)

tvp_files = list(TVP_PATH.glob('tvp_reach_*.parquet'))
tvp_reach_ids = {int(f.stem.split('_')[-1]) for f in tvp_files}

# 파일명에서 fctimp 값 추출
print("파일명에서 fctimp 값 분포 확인...")
node_files = sorted(PARQUET_PATH.glob('fctimp_mean_*_node_*.parquet'))

fctimp_values = set()
for nf in node_files:
    # fctimp_mean_0.123456_node_xxx.parquet
    match = re.search(r'fctimp_mean_([0-9.]+)_node', nf.name)
    if match:
        fctimp_values.add(float(match.group(1)))

print(f"Unique fctimp values: {len(fctimp_values)}")
print(f"Values: {sorted(fctimp_values)[:20]}...")
print(f"Max: {max(fctimp_values)}")

# 높은 fctimp 값을 가진 파일 찾기
print("\n높은 fctimp (> 0.1) 파일 찾기...")
high_fctimp_files = []
for nf in node_files:
    match = re.search(r'fctimp_mean_([0-9.]+)_node', nf.name)
    if match:
        fctimp = float(match.group(1))
        if fctimp > 0.1:  # 10% 이상
            high_fctimp_files.append((fctimp, nf))

print(f"fctimp > 0.1인 파일: {len(high_fctimp_files)}")

if high_fctimp_files:
    # 가장 높은 fctimp 파일들
    high_fctimp_files.sort(reverse=True)
    print("\n가장 높은 fctimp 파일들:")
    for fctimp, nf in high_fctimp_files[:10]:
        try:
            df = pd.read_parquet(nf)
            rid = int(df['reach_id'].iloc[0])
            in_tvp = rid in tvp_reach_ids
            in_poly = rid in polygon_reach_ids
            print(f"  fctimp={fctimp:.3f}, reach={rid}, TVP={in_tvp}, Poly={in_poly}, file={nf.name}")
        except Exception as e:
            print(f"  Error: {e}")

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
from shapely.geometry import Point, LineString
import rasterio
import matplotlib.pyplot as plt

BASE_PATH = Path('/work/pi_kandread_umass_edu/swot-urban')
PARQUET_PATH = BASE_PATH / '5_fctimp'
TVP_PATH = BASE_PATH / '6_pixc_tvp' / 'SWOT_TVP_Data'
POLYGON_PATH = BASE_PATH / '2_urban_reaches' / 'all_reaches_vD.parquet'
NLCD_FCTIMP_PATH = str(BASE_PATH / '0_data_collection/1_nlcd/Annual_NLCD_FctImp_2024_CU_C1V1_EPSG4326.tif')

J2000_EPOCH = pd.Timestamp('2000-01-01 00:00:00')
TARGET_REACH_ID = 74282700231

# 파일 직접 찾기
print("파일명 확인...")
target_file = list(PARQUET_PATH.glob('fctimp_mean_0.605247_node_*.parquet'))[0]
print(f"Target file: {target_file.name}")

# 로드
node_df = pd.read_parquet(target_file)
print(f"reach_id in file: {node_df['reach_id'].iloc[0]}")
print(f"Rows: {len(node_df)}")

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
from shapely.geometry import Point, LineString
import rasterio
import matplotlib.pyplot as plt

BASE_PATH = Path('/work/pi_kandread_umass_edu/swot-urban')
PARQUET_PATH = BASE_PATH / '5_fctimp'
TVP_PATH = BASE_PATH / '6_pixc_tvp' / 'SWOT_TVP_Data'
POLYGON_PATH = BASE_PATH / '2_urban_reaches' / 'all_reaches_vD.parquet'
NLCD_FCTIMP_PATH = str(BASE_PATH / '0_data_collection/1_nlcd/Annual_NLCD_FctImp_2024_CU_C1V1_EPSG4326.tif')
BUILDING_HEIGHT_PATH = str(BASE_PATH / '0_data_collection/3_150m_building/GBH2020_150m_GEDI_4326.tif')

J2000_EPOCH = pd.Timestamp('2000-01-01 00:00:00')
TARGET_REACH_ID = 74282700231

# Sampling Function
def sample_raster_along_line(line, node_pt, raster_path, n_samples=100, is_building=False):
    empty_result = {'mean_val': 0.0, 'nonzero_ratio': 0.0, 'dw_mean_val': 0.0, 'dw_ratio': 0.0, 'values': [], 'coords': []}
    
    if line is None or line.is_empty:
        return empty_result
    
    if line.geom_type == 'MultiLineString':
        segments = list(line.geoms)
        if not segments:
            return empty_result
        target_line = min(segments, key=lambda s: s.distance(node_pt))
        if target_line.length == 0:
            return empty_result
    elif line.geom_type == 'LineString':
        if line.length == 0:
            return empty_result
        target_line = line
    else:
        return empty_result
    
    start_pt = Point(target_line.coords[0])
    end_pt = Point(target_line.coords[-1])
    reverse = node_pt.distance(start_pt) > node_pt.distance(end_pt)
    
    values = []
    coords_sampled = []
    
    with rasterio.open(raster_path) as src:
        for i in range(n_samples + 1):
            t = 1.0 - (i / n_samples) if reverse else i / n_samples
            pt = target_line.interpolate(t, normalized=True)
            coords_sampled.append((pt.x, pt.y))
            
            try:
                row, col = src.index(pt.x, pt.y)
                if 0 <= row < src.height and 0 <= col < src.width:
                    val = src.read(1, window=((row, row+1), (col, col+1)))[0, 0]
                    
                    if is_building:
                        if val >= 0 and not np.isnan(val):
                            values.append((i, float(val)))
                    else:
                        if 0 <= val <= 100:
                            values.append((i, float(val)))
            except:
                pass
    
    if not values:
        return empty_result
    
    all_vals = [v for _, v in values]
    mean_val = np.mean(all_vals)
    nonzero_ratio = sum(1 for v in all_vals if v > 0) / (n_samples + 1)
    
    weighted_sum = 0.0
    weight_sum = 0.0
    weighted_nonzero_sum = 0.0
    
    for idx, val in values:
        weight = (n_samples - idx) / n_samples
        weighted_sum += weight * val
        weight_sum += weight
        if val > 0:
            weighted_nonzero_sum += weight
    
    dw_mean_val = weighted_sum / weight_sum if weight_sum > 0 else 0.0
    dw_ratio = weighted_nonzero_sum / weight_sum if weight_sum > 0 else 0.0
    
    return {
        'mean_val': mean_val,
        'nonzero_ratio': nonzero_ratio,
        'dw_mean_val': dw_mean_val,
        'dw_ratio': dw_ratio,
        'values': values,
        'coords': coords_sampled,
    }


# 1. 데이터 로드
print(f"v6.4 Near/Far ISA 검증 - Reach {TARGET_REACH_ID}")

# 높은 fctimp 파일들 모두 로드
print("\n[1] Node 파일 로드...")
all_node_files = list(PARQUET_PATH.glob('fctimp_mean_0.605247_node_*.parquet'))
print(f"  Found {len(all_node_files)} files with fctimp=0.605")

node_dfs = []
for nf in all_node_files:
    df = pd.read_parquet(nf)
    if int(df['reach_id'].iloc[0]) == TARGET_REACH_ID:
        node_dfs.append(df)

node_df = pd.concat(node_dfs, ignore_index=True)
print(f"  Total rows: {len(node_df)}")
print(f"  fctimp_mean: {node_df['fctimp_mean'].iloc[0]:.4f}")

# Polygon
print("\n[2] Polygon 로드...")
polygons_gdf = gpd.read_parquet(POLYGON_PATH)
if polygons_gdf.crs != 'EPSG:4326':
    polygons_gdf = polygons_gdf.to_crs('EPSG:4326')
polygon = polygons_gdf[polygons_gdf['reach_id'] == TARGET_REACH_ID].geometry.iloc[0]
print(f"  Polygon bounds: {[round(x, 4) for x in polygon.bounds]}")

# TVP
print("\n[3] TVP 로드...")
tvp_file = TVP_PATH / f'tvp_reach_{TARGET_REACH_ID}.parquet'
tvp_df = pd.read_parquet(tvp_file)
print(f"  TVP rows: {len(tvp_df)}")

# 2. 시간 매칭
print("\n[4] TVP 매칭...")
node_df['time_dt'] = pd.to_datetime(node_df['time_str']).dt.tz_localize(None)
tvp_df['time_dt'] = J2000_EPOCH + pd.to_timedelta(tvp_df['time'], unit='s')

node_df = node_df.sort_values('time_dt').reset_index(drop=True)
tvp_df = tvp_df.sort_values('time_dt')

merged_df = pd.merge_asof(
    node_df,
    tvp_df[['time_dt', 'latitude', 'longitude', 'altitude']],
    on='time_dt',
    direction='nearest',
    tolerance=pd.Timedelta('30s')
).rename(columns={'latitude': 'sat_lat', 'longitude': 'sat_lon', 'altitude': 'sat_altitude'})

matched = merged_df['sat_lon'].notna().sum()
print(f"  TVP 매칭: {matched}/{len(merged_df)} ({100*matched/len(merged_df):.1f}%)")

if matched == 0:
    print("\n  매칭 안됨! 시간 범위 확인:")
    print(f"    Node: {node_df['time_dt'].min()} ~ {node_df['time_dt'].max()}")
    print(f"    TVP:  {tvp_df['time_dt'].min()} ~ {tvp_df['time_dt'].max()}")
else:
    # 3. Near/Far ISA 계산
    print("\n[5] Near/Far ISA 계산...")
    
    test_row = merged_df[merged_df['sat_lon'].notna()].iloc[0]
    print(f"  Node: ({test_row['lon']:.6f}, {test_row['lat']:.6f})")
    print(f"  Sat:  ({test_row['sat_lon']:.6f}, {test_row['sat_lat']:.6f})")
    print(f"  Time: {test_row['time_dt']}")
    
    node_pt = Point(test_row['lon'], test_row['lat'])
    sat_lon, sat_lat = test_row['sat_lon'], test_row['sat_lat']
    
    dx = sat_lon - test_row['lon']
    dy = sat_lat - test_row['lat']
    length = np.sqrt(dx**2 + dy**2)
    ux, uy = dx / length, dy / length
    
    extend = 0.2
    
    near_line = LineString([node_pt, Point(test_row['lon'] + ux * extend, test_row['lat'] + uy * extend)])
    far_line = LineString([node_pt, Point(test_row['lon'] - ux * extend, test_row['lat'] - uy * extend)])
    
    near_in_poly = near_line.intersection(polygon) if near_line.intersects(polygon) else None
    far_in_poly = far_line.intersection(polygon) if far_line.intersects(polygon) else None
    
    print(f"\n  Near line: {near_in_poly.geom_type if near_in_poly and not near_in_poly.is_empty else 'None'}")
    print(f"  Far line: {far_in_poly.geom_type if far_in_poly and not far_in_poly.is_empty else 'None'}")
    
    # ISA 샘플링
    near_isa = sample_raster_along_line(near_in_poly, node_pt, NLCD_FCTIMP_PATH)
    far_isa = sample_raster_along_line(far_in_poly, node_pt, NLCD_FCTIMP_PATH)
    
    print(f"\n  === Near ISA ===")
    print(f"    mean_val: {near_isa['mean_val']:.2f}")
    print(f"    nonzero_ratio: {near_isa['nonzero_ratio']:.2f}")
    print(f"    dw_mean_val: {near_isa['dw_mean_val']:.2f}")
    print(f"    샘플 수: {len(near_isa['values'])}")
    if near_isa['values']:
        vals = [v for _, v in near_isa['values'][:10]]
        print(f"    처음 10개 값: {vals}")
    
    print(f"\n  === Far ISA ===")
    print(f"    mean_val: {far_isa['mean_val']:.2f}")
    print(f"    nonzero_ratio: {far_isa['nonzero_ratio']:.2f}")
    print(f"    dw_mean_val: {far_isa['dw_mean_val']:.2f}")
    print(f"    샘플 수: {len(far_isa['values'])}")
    if far_isa['values']:
        vals = [v for _, v in far_isa['values'][:10]]
        print(f"    처음 10개 값: {vals}")
    
    # 4. 시각화
    print("\n[6] 시각화...")
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    ax1 = axes[0]
    if polygon.geom_type == 'MultiPolygon':
        for poly in polygon.geoms:
            x, y = poly.exterior.xy
            ax1.fill(x, y, alpha=0.3, color='blue')
            ax1.plot(x, y, color='blue', linewidth=1)
    else:
        x, y = polygon.exterior.xy
        ax1.fill(x, y, alpha=0.3, color='blue')
        ax1.plot(x, y, color='blue', linewidth=1)
    
    ax1.scatter(test_row['lon'], test_row['lat'], color='green', s=100, zorder=5, label='Node')
    ax1.scatter(sat_lon, sat_lat, color='red', s=50, marker='^', label='Satellite')
    ax1.annotate('', xy=(sat_lon, sat_lat), xytext=(test_row['lon'], test_row['lat']),
                 arrowprops=dict(arrowstyle='->', color='red', lw=2))
    
    if near_in_poly and not near_in_poly.is_empty:
        if near_in_poly.geom_type == 'LineString':
            x, y = near_in_poly.xy
            ax1.plot(x, y, color='orange', linewidth=3, label=f'Near (ISA={near_isa["mean_val"]:.1f})')
        elif near_in_poly.geom_type == 'MultiLineString':
            for seg in near_in_poly.geoms:
                x, y = seg.xy
                ax1.plot(x, y, color='orange', linewidth=3)
            ax1.plot([], [], color='orange', linewidth=3, label=f'Near (ISA={near_isa["mean_val"]:.1f})')
    
    if far_in_poly and not far_in_poly.is_empty:
        if far_in_poly.geom_type == 'LineString':
            x, y = far_in_poly.xy
            ax1.plot(x, y, color='purple', linewidth=3, label=f'Far (ISA={far_isa["mean_val"]:.1f})')
        elif far_in_poly.geom_type == 'MultiLineString':
            for seg in far_in_poly.geoms:
                x, y = seg.xy
                ax1.plot(x, y, color='purple', linewidth=3)
            ax1.plot([], [], color='purple', linewidth=3, label=f'Far (ISA={far_isa["mean_val"]:.1f})')
    
    ax1.set_title(f'Reach {TARGET_REACH_ID} (fctimp={test_row["fctimp_mean"]:.1%})')
    ax1.set_xlabel('Longitude')
    ax1.set_ylabel('Latitude')
    ax1.legend(loc='best')
    ax1.set_aspect('equal')
    
    ax2 = axes[1]
    if near_isa['values']:
        near_idx = [i for i, _ in near_isa['values']]
        near_vals = [v for _, v in near_isa['values']]
        ax2.plot(near_idx, near_vals, 'o-', color='orange', label=f'Near (mean={near_isa["mean_val"]:.1f})')
    
    if far_isa['values']:
        far_idx = [i for i, _ in far_isa['values']]
        far_vals = [v for _, v in far_isa['values']]
        ax2.plot(far_idx, far_vals, 's-', color='purple', label=f'Far (mean={far_isa["mean_val"]:.1f})')
    
    ax2.set_xlabel('Sample Index (0 = closest to node)')
    ax2.set_ylabel('ISA Value (%)')
    ax2.set_title('ISA Values Along Near/Far Lines')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # 결론
    print("검증 결과")
    print(f"  fctimp_mean (원본): {test_row['fctimp_mean']:.1%}")
    print(f"  Near ISA: {near_isa['mean_val']:.2f} (dw: {near_isa['dw_mean_val']:.2f})")
    print(f"  Far ISA:  {far_isa['mean_val']:.2f} (dw: {far_isa['dw_mean_val']:.2f})")
    
    if near_isa['mean_val'] > 0 or far_isa['mean_val'] > 0:
        print("\nISA 값이 정상적으로 계산됨!")
    else:
        print("\nISA 값이 0 - 확인 필요")

In [ ]:
"""
v6.5 파이프라인과 동일한 방식으로 검증
- node_file_dict 빌드 방식 동일
- TVP 매칭 방식 동일
- Near/Far ISA 계산 방식 동일
"""
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
from shapely.geometry import Point, LineString
from shapely import wkt
import rasterio
from collections import defaultdict
import matplotlib.pyplot as plt

# Configuration (v6.5와 동일)
BASE_PATH = Path('/work/pi_kandread_umass_edu/swot-urban')
PARQUET_PATH = BASE_PATH / '5_fctimp'
TVP_PATH = BASE_PATH / '6_pixc_tvp' / 'SWOT_TVP_Data'
POLYGON_PATH = BASE_PATH / '2_urban_reaches' / 'all_reaches_vD.parquet'
NLCD_FCTIMP_PATH = str(BASE_PATH / '0_data_collection/1_nlcd/Annual_NLCD_FctImp_2024_CU_C1V1_EPSG4326.tif')
BUILDING_HEIGHT_PATH = str(BASE_PATH / '0_data_collection/3_150m_building/GBH2020_150m_GEDI_4326.tif')

# J2000 epoch (v6.5)
J2000_EPOCH = pd.Timestamp('2000-01-01 00:00:00')

print("v6.5 파이프라인 검증 (동일한 로직)")

# PHASE 1: Build lookup tables (v6.5와 동일)
print("\n[PHASE 1] Building lookup tables (v6.5 방식)...")

# Load polygons -> dict
print("  Loading polygons...")
polygons_gdf = gpd.read_parquet(POLYGON_PATH)
if polygons_gdf.crs != 'EPSG:4326':
    polygons_gdf = polygons_gdf.to_crs('EPSG:4326')

polygon_wkt_dict = {}
for _, row in polygons_gdf.iterrows():
    rid = row['reach_id']
    polygon_wkt_dict[int(rid)] = row.geometry.wkt

print(f"    {len(polygon_wkt_dict)} polygons")

# Build node file lookup: reach_id -> LIST of file paths (v6.5와 동일)
print("  Scanning node files...")
node_files = sorted(PARQUET_PATH.glob('fctimp_mean_*_node_*.parquet'))

node_file_dict = defaultdict(list)

for f in node_files:
    try:
        df_sample = pd.read_parquet(f, columns=['reach_id'])
        if len(df_sample) > 0:
            reach_id = int(df_sample['reach_id'].iloc[0])
            node_file_dict[reach_id].append(f)
    except:
        pass

node_file_dict = dict(node_file_dict)

total_files = sum(len(files) for files in node_file_dict.values())
print(f"    {total_files} node files mapped to {len(node_file_dict)} reaches")

# Build TVP file lookup (v6.5와 동일)
print("  Scanning TVP files...")
tvp_files = sorted(TVP_PATH.glob('tvp_reach_*.parquet'))

tvp_file_dict = {}
for f in tvp_files:
    try:
        reach_id = int(f.stem.split('_')[-1])
        tvp_file_dict[reach_id] = f
    except:
        pass

print(f"    {len(tvp_file_dict)} TVP files")

# PHASE 2: 테스트할 Reach 선택 (높은 fctimp)
print("\n[PHASE 2] 테스트할 Reach 선택...")

# 높은 fctimp를 가진 reach 찾기
valid_reaches = set(polygon_wkt_dict.keys()) & set(node_file_dict.keys()) & set(tvp_file_dict.keys())
print(f"  Valid reaches (polygon + node + TVP): {len(valid_reaches)}")

# fctimp가 높은 reach 찾기
best_reach = None
best_fctimp = 0

for reach_id in list(valid_reaches)[:100]:  # 처음 100개만 스캔
    files = node_file_dict[reach_id]
    try:
        df = pd.read_parquet(files[0], columns=['fctimp_mean'])
        fctimp = df['fctimp_mean'].iloc[0]
        if fctimp > best_fctimp:
            best_fctimp = fctimp
            best_reach = reach_id
    except:
        pass

TARGET_REACH_ID = best_reach
print(f"  선택된 Reach: {TARGET_REACH_ID} (fctimp={best_fctimp:.4f})")

# PHASE 3: process_single_reach 로직 재현
print(f"\n[PHASE 3] process_single_reach 로직 (Reach {TARGET_REACH_ID})...")

node_files_list = node_file_dict[TARGET_REACH_ID]
tvp_file = tvp_file_dict[TARGET_REACH_ID]
polygon_wkt = polygon_wkt_dict[TARGET_REACH_ID]

print(f"  Node files: {len(node_files_list)}")
print(f"  TVP file: {tvp_file.name}")

# Load ALL node files for this reach (v6.5와 동일)
node_dfs = []
for nf in node_files_list:
    try:
        df = pd.read_parquet(nf)
        node_dfs.append(df)
    except:
        pass

node_df = pd.concat(node_dfs, ignore_index=True)
print(f"  Node rows: {len(node_df)}")

# Load TVP data (v6.5와 동일)
tvp_df = pd.read_parquet(tvp_file)
print(f"  TVP rows: {len(tvp_df)}")

# Time conversion - J2000 epoch (v6.5와 동일)
node_df['time_dt'] = pd.to_datetime(node_df['time_str']).dt.tz_localize(None)
tvp_df['time_dt'] = J2000_EPOCH + pd.to_timedelta(tvp_df['time'], unit='s')

node_df = node_df.sort_values('time_dt').reset_index(drop=True)
tvp_df = tvp_df.sort_values('time_dt')

# Merge (v6.5와 동일)
node_df = pd.merge_asof(
    node_df,
    tvp_df[['time_dt', 'latitude', 'longitude', 'altitude']],
    on='time_dt',
    direction='nearest',
    tolerance=pd.Timedelta('30s')
).rename(columns={
    'latitude': 'sat_lat',
    'longitude': 'sat_lon',
    'altitude': 'sat_altitude'
})

matched = node_df['sat_lon'].notna().sum()
print(f"\n  TVP 매칭: {matched}/{len(node_df)} ({100*matched/len(node_df):.1f}%)")

# PHASE 4: Near/Far ISA 계산 (v6.5와 동일한 로직)
print("\n[PHASE 4] Near/Far ISA 계산...")

def sample_raster_along_line(line, node_pt, raster_path, n_samples=100, is_building=False):
    """v6.5 sample_raster_along_line_optimized와 동일한 로직 (단일 테스트용)"""
    empty_result = {'mean_val': 0.0, 'nonzero_ratio': 0.0, 'dw_mean_val': 0.0, 'dw_ratio': 0.0, 'values': []}
    
    if line is None or line.is_empty:
        return empty_result
    
    if line.geom_type == 'MultiLineString':
        segments = list(line.geoms)
        if not segments:
            return empty_result
        target_line = min(segments, key=lambda s: s.distance(node_pt))
        if target_line.length == 0:
            return empty_result
    elif line.geom_type == 'LineString':
        if line.length == 0:
            return empty_result
        target_line = line
    else:
        return empty_result
    
    start_pt = Point(target_line.coords[0])
    end_pt = Point(target_line.coords[-1])
    reverse = node_pt.distance(start_pt) > node_pt.distance(end_pt)
    
    values = []
    
    with rasterio.open(raster_path) as src:
        for i in range(n_samples + 1):
            t = 1.0 - (i / n_samples) if reverse else i / n_samples
            pt = target_line.interpolate(t, normalized=True)
            
            try:
                row, col = src.index(pt.x, pt.y)
                if 0 <= row < src.height and 0 <= col < src.width:
                    val = src.read(1, window=((row, row+1), (col, col+1)))[0, 0]
                    
                    if is_building:
                        if val >= 0 and not np.isnan(val):
                            values.append((i, float(val)))
                    else:
                        if 0 <= val <= 100:
                            values.append((i, float(val)))
            except:
                pass
    
    if not values:
        return empty_result
    
    all_vals = [v for _, v in values]
    mean_val = np.mean(all_vals)
    nonzero_ratio = sum(1 for v in all_vals if v > 0) / (n_samples + 1)
    
    weighted_sum = 0.0
    weight_sum = 0.0
    weighted_nonzero_sum = 0.0
    
    for idx, val in values:
        weight = (n_samples - idx) / n_samples
        weighted_sum += weight * val
        weight_sum += weight
        if val > 0:
            weighted_nonzero_sum += weight
    
    dw_mean_val = weighted_sum / weight_sum if weight_sum > 0 else 0.0
    dw_ratio = weighted_nonzero_sum / weight_sum if weight_sum > 0 else 0.0
    
    return {'mean_val': mean_val, 'nonzero_ratio': nonzero_ratio, 'dw_mean_val': dw_mean_val, 'dw_ratio': dw_ratio, 'values': values}


def get_geometry_category(near_val, far_val):
    has_near, has_far = near_val > 0, far_val > 0
    if has_near and has_far:
        return 'Near/Far'
    elif has_near:
        return 'Near'
    elif has_far:
        return 'Far'
    else:
        return 'NA'


# 테스트할 row 선택
test_row = node_df[node_df['sat_lon'].notna()].iloc[0]
polygon = wkt.loads(polygon_wkt)

print(f"  Node: ({test_row['lon']:.6f}, {test_row['lat']:.6f})")
print(f"  Sat:  ({test_row['sat_lon']:.6f}, {test_row['sat_lat']:.6f})")
print(f"  Time: {test_row['time_dt']}")
print(f"  fctimp_mean: {test_row['fctimp_mean']:.4f}")

# Near/Far 라인 계산 (v6.5와 동일)
node_pt = Point(test_row['lon'], test_row['lat'])
sat_lon, sat_lat = test_row['sat_lon'], test_row['sat_lat']

dx = sat_lon - test_row['lon']
dy = sat_lat - test_row['lat']
length = np.sqrt(dx**2 + dy**2)
ux, uy = dx / length, dy / length

extend = 0.2

near_line = LineString([node_pt, Point(test_row['lon'] + ux * extend, test_row['lat'] + uy * extend)])
far_line = LineString([node_pt, Point(test_row['lon'] - ux * extend, test_row['lat'] - uy * extend)])

near_in_poly = near_line.intersection(polygon) if near_line.intersects(polygon) else None
far_in_poly = far_line.intersection(polygon) if far_line.intersects(polygon) else None

# ISA 샘플링
near_isa = sample_raster_along_line(near_in_poly, node_pt, NLCD_FCTIMP_PATH)
far_isa = sample_raster_along_line(far_in_poly, node_pt, NLCD_FCTIMP_PATH)

# Building 샘플링
near_bld = sample_raster_along_line(near_in_poly, node_pt, BUILDING_HEIGHT_PATH, is_building=True)
far_bld = sample_raster_along_line(far_in_poly, node_pt, BUILDING_HEIGHT_PATH, is_building=True)

# Category 계산 (v6.5와 동일)
geometry_category = get_geometry_category(near_isa['mean_val'], far_isa['mean_val'])
geometry_category_building = get_geometry_category(near_bld['mean_val'], far_bld['mean_val'])

# PHASE 5: 결과 출력
print("검증 결과 (v6.5 로직)")

print(f"\n[입력]")
print(f"  Reach ID: {TARGET_REACH_ID}")
print(f"  fctimp_mean: {test_row['fctimp_mean']:.4f} ({test_row['fctimp_mean']*100:.2f}%)")
print(f"  Node files: {len(node_files_list)}")
print(f"  Total observations: {len(node_df)}")

print(f"\n[TVP 매칭]")
print(f"  J2000 Epoch: {J2000_EPOCH}")
print(f"  매칭률: {matched}/{len(node_df)} ({100*matched/len(node_df):.1f}%)")

print(f"\n[ISA 결과]")
print(f"  near_isa: {near_isa['mean_val']:.2f}")
print(f"  far_isa: {far_isa['mean_val']:.2f}")
print(f"  dw_near_isa: {near_isa['dw_mean_val']:.2f}")
print(f"  dw_far_isa: {far_isa['dw_mean_val']:.2f}")
print(f"  near_ratio: {near_isa['nonzero_ratio']:.2f}")
print(f"  far_ratio: {far_isa['nonzero_ratio']:.2f}")
print(f"  geometry_category: {geometry_category}")

print(f"\n[Building 결과]")
print(f"  near_building: {near_bld['mean_val']:.2f}")
print(f"  far_building: {far_bld['mean_val']:.2f}")
print(f"  geometry_category_building: {geometry_category_building}")

# 검증
if matched > 0 and (near_isa['mean_val'] > 0 or far_isa['mean_val'] > 0):
    print("v6.5 로직 검증 성공!")
    print("   - TVP 매칭 정상")
    print("   - Near/Far ISA 계산 정상")
    print("   - 프로덕션 실행 준비 완료")
else:
    print(" 실패 - 확인 필요")


In [ ]:
import pandas as pd
from pathlib import Path

# 경로 설정
RESULT_PATH = Path('/project/pi_kandread_umass_edu/swot-urban/3_analysis_results')

# pandas 옵션 - 모든 컬럼/행 표시
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

# 폴더 내용 확인
print("=== Files in directory ===")
for f in sorted(RESULT_PATH.glob('*')):
    if f.is_file():
        size_mb = f.stat().st_size / 1024 / 1024
        print(f"{f.name}: {size_mb:.1f} MB")
    else:
        print(f"{f.name}/ (folder)")

# temp_reaches 폴더 확인
temp_path = RESULT_PATH / 'temp_reaches'
if temp_path.exists():
    reach_files = sorted(temp_path.glob('*.parquet'))
    print(f"\ntemp_reaches: {len(reach_files)} files")
    
    if reach_files:
        first_file = reach_files[0]
        print(f"First file: {first_file.name}")
        
        df = pd.read_parquet(first_file)
        print(f"Shape: {df.shape}")
        
        # 모든 컬럼 정보
        print(f"\n=== All Columns ({len(df.columns)}) ===")
        for i, col in enumerate(df.columns):
            dtype = df[col].dtype
            n_null = df[col].isna().sum()
            n_total = len(df)
            null_pct = 100 * n_null / n_total
            print(f"{i+1:3d}. {col:40s} | {str(dtype):15s} | NaN: {n_null:5d}/{n_total} ({null_pct:5.1f}%)")
        
        # NaN 있는 컬럼만 요약
        print(f"\n=== Columns with NaN ===")
        nan_cols = df.columns[df.isna().any()].tolist()
        if nan_cols:
            for col in nan_cols:
                n_null = df[col].isna().sum()
                print(f"  {col}: {n_null} NaN ({100*n_null/len(df):.1f}%)")
        else:
            print("  None!")
        
        # 첫 5행 (스크롤 가능)
        print(f"\n=== First 5 rows (all columns) ===")
        display(df.head())
        
        # geometry_category 분포
        if 'geometry_category' in df.columns:
            print(f"\n=== geometry_category ===")
            print(df['geometry_category'].value_counts())

# 최종 결과 파일
final_parquet = RESULT_PATH / 'swot_urban_nearfar_v6.2_results.parquet'
if final_parquet.exists():
    print(f"Final Result: {final_parquet.name}")
    
    df_final = pd.read_parquet(final_parquet)
    print(f"Shape: {df_final.shape}")
    
    # NaN 체크
    print(f"\n=== Columns with NaN (Final) ===")
    for col in df_final.columns:
        n_null = df_final[col].isna().sum()
        if n_null > 0:
            print(f"  {col}: {n_null} NaN ({100*n_null/len(df_final):.1f}%)")
    
    print(f"\n=== geometry_category ===")
    print(df_final['geometry_category'].value_counts())

## Calculate near-far ISA and building height at Node Level (Fctimp + BuildingHeight)

In [ ]:
"""
SWOT Urban Hydrology Analysis Pipeline v6
============================================
Purpose: Build DataFrame with Near/Far ISA geometry analysis
- Near/Far ISA: Only use segment directly connected to Node
- If MultiLineString, ignore distant segments

v6 Updates:
- Distance-weighted Near/Far ISA (closer to node = higher weight)
- Building height (GBH2020) sampling with same logic
"""
import xarray as xr
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
from datetime import timedelta
from shapely.geometry import Point, LineString
import rasterio
import re
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Configuration
ISA_VALUES = [0.01, 0.06, 0.12, 0.17, 0.22, 0.28, 0.32, 0.40, 0.45, 0.50, 0.55, 0.61]
SPIKE_THRESHOLDS = {
    0.01: 0.1832,
    0.06: 0.1832,
    0.12: 0.1832,
    0.17: 0.1832,
    0.22: 0.1832,
    0.28: 0.1832,
    0.32: 0.1832,
    0.40: 0.1832,
    0.45: 0.1832,
    0.50: 0.1832,
    0.55: 0.1832,
    0.61: 0.1832,
}

# Paths
BASE_PATH = Path('/work/pi_kandread_umass_edu/swot-urban')
PARQUET_PATH = BASE_PATH / '1_data_processing'
PIXC_BASE = Path('/project/pi_kandread_umass_edu/swot-urban/0_data_collection/SWOT_PIXC')
PIXCVEC_BASE = Path('/project/pi_kandread_umass_edu/swot-urban/0_data_collection/SWOT_PIXCVec')
POLYGON_PATH = BASE_PATH / '2_urban_reaches' / 'selected_12_vD.shp'
NLCD_FCTIMP_PATH = BASE_PATH / '0_data_collection/1_nlcd/Annual_NLCD_FctImp_2024_CU_C1V1_EPSG4326.tif'
BUILDING_HEIGHT_PATH = BASE_PATH / '0_data_collection/3_150m_building/GBH2020_150m_GEDI_4326.tif'
OUTPUT_PATH = BASE_PATH / '3_analysis_results'


# Step 1: Load Parquet Files
def load_all_parquet_files():
    """Load and combine all ISA parquet files"""
    
    all_dfs = []
    
    for isa in ISA_VALUES:
        parquet_file = PARQUET_PATH / f'fctimp_mean_{isa:.2f}_node.parquet'
        
        if parquet_file.exists():
            df = pd.read_parquet(parquet_file)
            df['isa_value'] = isa
            df['isa_str'] = f'{isa:.2f}'
            all_dfs.append(df)
            print(f"Loaded ISA {isa:.2f}: {len(df)} rows")
        else:
            print(f"File not found: {parquet_file}")
    
    combined_df = pd.concat(all_dfs, ignore_index=True)
    print(f"\nTotal: {len(combined_df)} rows")
    
    return combined_df


def classify_spikes(df, threshold_dict=None):
    """Classify spikes based on error_reach"""
    
    if threshold_dict is None:
        threshold_dict = SPIKE_THRESHOLDS
    
    df = df.copy()
    df['spike_threshold'] = df['isa_value'].map(threshold_dict)
    df['is_spike'] = df['error_reach'].abs() >= df['spike_threshold']  # Fixed bug
    
    return df


# Step 2: PIXC File Matching
def filter_best_pixc_files(files):
    """Latest version selection: PGD0 > PID0, higher counter"""
    file_dict = {}
    for f in files:
        match = re.match(r'(.+)_(P[IG]D\d)_(\d+)\.nc', f.name)
        if not match:
            continue
        base, processor, counter = match.group(1), match.group(2), int(match.group(3))
        if base not in file_dict:
            file_dict[base] = (f, processor, counter)
        else:
            _, old_proc, old_cnt = file_dict[base]
            if (processor == 'PGD0' and old_proc == 'PID0') or (processor == old_proc and counter > old_cnt):
                file_dict[base] = (f, processor, counter)
    return [f for f, _, _ in file_dict.values()]


def find_pixc_file(time_str, crid, isa_str, window_minutes=5):
    """Find PIXC file for observation"""
    
    pixc_dir = PIXC_BASE / f'fctimp_{isa_str}'
    
    if not pixc_dir.exists():
        return None
    
    pixc_files = filter_best_pixc_files(list(pixc_dir.glob('SWOT_L2_HR_PIXC_*.nc')))
    
    dt = pd.to_datetime(time_str)
    if dt.tzinfo:
        dt = dt.tz_localize(None)
    start = dt - timedelta(minutes=window_minutes)
    end = dt + timedelta(minutes=window_minutes)
    
    for pf in pixc_files:
        parts = pf.name.split('_')
        if len(parts) >= 10:
            try:
                file_start = pd.to_datetime(parts[7], format='%Y%m%dT%H%M%S')
                file_end = pd.to_datetime(parts[8], format='%Y%m%dT%H%M%S')
                file_crid = parts[9]
                
                if not (file_end < start or file_start > end) and file_crid == crid:
                    return pf
            except:
                continue
    
    return None


# Step 3: Extract Satellite Position
def extract_satellite_position(pixc_file):
    """Extract satellite position from PIXC tvp group"""
    
    try:
        ds_tvp = xr.open_dataset(pixc_file, group='tvp')
        result = {
            'sat_lon': np.nanmedian(ds_tvp['longitude'].values),
            'sat_lat': np.nanmedian(ds_tvp['latitude'].values),
            'sat_altitude': np.nanmedian(ds_tvp['altitude'].values),
            'sat_heading': np.nanmedian(ds_tvp['velocity_heading'].values),
        }
        ds_tvp.close()
        return result
    except:
        return None


# Step 4: Near/Far Sampling (v6 - with distance weighting & building)
def sample_raster_along_line(line, node_pt, raster_path, n_samples=100, 
                              valid_range=(0, 100), is_building=False):
    """
    Sample raster values along a line with distance weighting.
    
    v6: Returns both simple and distance-weighted results.
    
    Distance weighting logic:
    - Samples closer to node get HIGHER weight
    - Weight for sample i (0-indexed from node): (n_samples - i) / n_samples
    - First sample (closest to node): weight = 1.0
    - Last sample (farthest from node): weight = 1/n_samples
    
    Parameters:
    -----------
    line : shapely geometry - Intersection result
    node_pt : shapely.Point - Node position (used to find connected segment)
    raster_path : Path to raster file
    n_samples : Number of samples
    valid_range : (min, max) for valid values
    is_building : If True, use different valid range logic (>= 0)
    
    Returns:
    --------
    dict with keys:
        - mean_val: Simple mean of valid values
        - nonzero_ratio: Fraction of samples with value > 0
        - dw_mean_val: Distance-weighted mean
        - dw_ratio: Distance-weighted ratio
    """
    
    empty_result = {
        'mean_val': 0.0,
        'nonzero_ratio': 0.0,
        'dw_mean_val': 0.0,
        'dw_ratio': 0.0,
    }
    
    if line is None or line.is_empty:
        return empty_result
    
    # Handle different geometry types - find segment connected to node
    if line.geom_type == 'MultiLineString':
        segments = list(line.geoms)
        
        min_dist = float('inf')
        connected_segment = None
        
        for seg in segments:
            dist = seg.distance(node_pt)
            if dist < min_dist:
                min_dist = dist
                connected_segment = seg
        
        if connected_segment is None or connected_segment.length == 0:
            return empty_result
        
        target_line = connected_segment
        
    elif line.geom_type == 'LineString':
        if line.length == 0:
            return empty_result
        target_line = line
    else:
        return empty_result
    
    # Determine sampling direction: start from node
    # Check which endpoint is closer to node
    start_pt = Point(target_line.coords[0])
    end_pt = Point(target_line.coords[-1])
    
    if node_pt.distance(start_pt) <= node_pt.distance(end_pt):
        # Start is closer to node - sample from 0 to 1
        reverse = False
    else:
        # End is closer to node - sample from 1 to 0
        reverse = True
    
    # Sample raster values
    values = []  # (sample_index, value) - index 0 is closest to node
    
    try:
        with rasterio.open(raster_path) as src:
            for i in range(n_samples + 1):
                # Normalized position along line
                if reverse:
                    t = 1.0 - (i / n_samples)
                else:
                    t = i / n_samples
                
                pt = target_line.interpolate(t, normalized=True)
                
                try:
                    row, col = src.index(pt.x, pt.y)
                    if 0 <= row < src.height and 0 <= col < src.width:
                        val = src.read(1, window=((row, row+1), (col, col+1)))[0, 0]
                        
                        # Validate value
                        if is_building:
                            # Building height: >= 0 is valid
                            if val >= 0 and not np.isnan(val):
                                values.append((i, float(val)))
                        else:
                            # FctImp: 0-100 is valid
                            if valid_range[0] <= val <= valid_range[1]:
                                values.append((i, float(val)))
                except:
                    pass
    except Exception as e:
        return empty_result
    
    if not values:
        return empty_result
    
    # Calculate simple statistics
    all_vals = [v for _, v in values]
    mean_val = np.mean(all_vals)
    nonzero_count = sum(1 for v in all_vals if v > 0)
    nonzero_ratio = nonzero_count / (n_samples + 1)  # Based on total samples attempted
    
    # Calculate distance-weighted statistics using Weighted Average
    # Weight: closer to node (lower index) = higher weight
    # weight_i = (n_samples - i) / n_samples for i in [0, n_samples]
    # idx=0 (closest to node): weight = 1.0
    # idx=n_samples (farthest): weight = 0.0
    #
    # Weighted Average: Σ(w_i x v_i) / Σ(w_i)
    # This preserves the original scale of values
    
    weighted_sum = 0.0
    weight_sum = 0.0
    weighted_nonzero_sum = 0.0  # Sum of weights where value > 0
    
    for idx, val in values:
        weight = (n_samples - idx) / n_samples  # Higher weight for samples closer to node
        weighted_sum += weight * val
        weight_sum += weight
        if val > 0:
            weighted_nonzero_sum += weight
    
    # Weighted average (same scale as simple mean)
    dw_mean_val = weighted_sum / weight_sum if weight_sum > 0 else 0.0
    
    # Weighted ratio: proportion of weight from nonzero samples
    dw_ratio = weighted_nonzero_sum / weight_sum if weight_sum > 0 else 0.0
    
    return {
        'mean_val': mean_val,
        'nonzero_ratio': nonzero_ratio,
        'dw_mean_val': dw_mean_val,
        'dw_ratio': dw_ratio,
    }


def calculate_near_far_metrics(node_lon, node_lat, sat_lon, sat_lat, polygon,
                                nlcd_path=NLCD_FCTIMP_PATH, 
                                building_path=BUILDING_HEIGHT_PATH):
    """
    Calculate Near/Far metrics for both ISA and Building Height.
    
    v6: Returns metrics including distance-weighted values.
    
    Near: Node -> Satellite direction (inside polygon, connected to Node)
    Far: Node -> opposite direction (inside polygon, connected to Node)
    
    Returns: dict with all metrics
    """
    
    empty_result = {
        # ISA - simple
        'near_isa': 0.0,
        'far_isa': 0.0,
        'near_ratio': 0.0,
        'far_ratio': 0.0,
        # ISA - distance weighted
        'dw_near_isa': 0.0,
        'dw_far_isa': 0.0,
        'dw_near_ratio': 0.0,
        'dw_far_ratio': 0.0,
        # Building - simple
        'near_building': 0.0,
        'far_building': 0.0,
        'near_ratio_building': 0.0,
        'far_ratio_building': 0.0,
        # Building - distance weighted
        'dw_near_building': 0.0,
        'dw_far_building': 0.0,
        'dw_near_ratio_building': 0.0,
        'dw_far_ratio_building': 0.0,
    }
    
    node_pt = Point(node_lon, node_lat)
    
    # Direction vector: Node -> Satellite
    dx = sat_lon - node_lon
    dy = sat_lat - node_lat
    length = np.sqrt(dx**2 + dy**2)
    
    if length == 0:
        return empty_result
    
    ux, uy = dx / length, dy / length
    extend = 0.2  # ~20km
    
    # Near line: Node -> Satellite direction
    near_line = LineString([node_pt, Point(node_lon + ux * extend, node_lat + uy * extend)])
    
    # Far line: Node -> opposite of Satellite
    far_line = LineString([node_pt, Point(node_lon - ux * extend, node_lat - uy * extend)])
    
    try:
        near_in_poly = near_line.intersection(polygon) if near_line.intersects(polygon) else None
        far_in_poly = far_line.intersection(polygon) if far_line.intersects(polygon) else None
    except:
        return empty_result
    
    result = empty_result.copy()
    
    # Sample ISA (FctImp)
    near_isa_data = sample_raster_along_line(near_in_poly, node_pt, nlcd_path, is_building=False)
    far_isa_data = sample_raster_along_line(far_in_poly, node_pt, nlcd_path, is_building=False)
    
    result['near_isa'] = near_isa_data['mean_val']
    result['far_isa'] = far_isa_data['mean_val']
    result['near_ratio'] = near_isa_data['nonzero_ratio']
    result['far_ratio'] = far_isa_data['nonzero_ratio']
    result['dw_near_isa'] = near_isa_data['dw_mean_val']
    result['dw_far_isa'] = far_isa_data['dw_mean_val']
    result['dw_near_ratio'] = near_isa_data['dw_ratio']
    result['dw_far_ratio'] = far_isa_data['dw_ratio']
    
    # Sample Building Height
    if Path(building_path).exists():
        near_bld_data = sample_raster_along_line(near_in_poly, node_pt, building_path, is_building=True)
        far_bld_data = sample_raster_along_line(far_in_poly, node_pt, building_path, is_building=True)
        
        result['near_building'] = near_bld_data['mean_val']
        result['far_building'] = far_bld_data['mean_val']
        result['near_ratio_building'] = near_bld_data['nonzero_ratio']
        result['far_ratio_building'] = far_bld_data['nonzero_ratio']
        result['dw_near_building'] = near_bld_data['dw_mean_val']
        result['dw_far_building'] = far_bld_data['dw_mean_val']
        result['dw_near_ratio_building'] = near_bld_data['dw_ratio']
        result['dw_far_ratio_building'] = far_bld_data['dw_ratio']
    
    return result


def get_geometry_category(near_val, far_val):
    """Categorize: Near/Far, Near, Far, NA"""
    
    has_near = near_val > 0
    has_far = far_val > 0
    
    if has_near and has_far:
        return 'Near/Far'
    elif has_near:
        return 'Near'
    elif has_far:
        return 'Far'
    else:
        return 'NA'


def get_sample_geometry_category(near_val, far_val, near_ratio, far_ratio, min_ratio=0.5):
    """Stricter categorization: at least 50% of samples must be non-zero"""
    
    has_near = near_val > 0 and near_ratio >= min_ratio
    has_far = far_val > 0 and far_ratio >= min_ratio
    
    if has_near and has_far:
        return 'Near/Far'
    elif has_near:
        return 'Near'
    elif has_far:
        return 'Far'
    else:
        return 'NA'


# Step 5: Process Single Observation
def process_single_observation(row, polygon):
    """Process single observation - v6 with all metrics"""
    
    result = {
        'sat_lon': np.nan,
        'sat_lat': np.nan,
        'sat_altitude': np.nan,
        'sat_heading': np.nan,
        # ISA - simple
        'near_isa': 0.0,
        'far_isa': 0.0,
        'near_ratio': 0.0,
        'far_ratio': 0.0,
        # ISA - distance weighted
        'dw_near_isa': 0.0,
        'dw_far_isa': 0.0,
        'dw_near_ratio': 0.0,
        'dw_far_ratio': 0.0,
        # Building - simple
        'near_building': 0.0,
        'far_building': 0.0,
        'near_ratio_building': 0.0,
        'far_ratio_building': 0.0,
        # Building - distance weighted
        'dw_near_building': 0.0,
        'dw_far_building': 0.0,
        'dw_near_ratio_building': 0.0,
        'dw_far_ratio_building': 0.0,
        # Categories
        'geometry_category': 'NA',
        'sample_geometry_category': 'NA',
        'geometry_category_building': 'NA',
        'sample_geometry_category_building': 'NA',
    }
    
    # Find PIXC file
    pixc_file = find_pixc_file(row['time_str'], row['crid'], row['isa_str'])
    
    if pixc_file is None:
        return result
    
    # Extract satellite position
    sat_data = extract_satellite_position(pixc_file)
    
    if sat_data is None:
        return result
    
    result['sat_lon'] = sat_data['sat_lon']
    result['sat_lat'] = sat_data['sat_lat']
    result['sat_altitude'] = sat_data['sat_altitude']
    result['sat_heading'] = sat_data['sat_heading']
    
    # Calculate all Near/Far metrics
    node_lon, node_lat = row['lon'], row['lat']
    
    if np.isfinite(result['sat_lon']) and np.isfinite(node_lon):
        metrics = calculate_near_far_metrics(
            node_lon, node_lat,
            result['sat_lon'], result['sat_lat'],
            polygon
        )
        
        # Update result with all metrics
        for key, value in metrics.items():
            result[key] = value
        
        # ISA categories
        result['geometry_category'] = get_geometry_category(
            result['near_isa'], result['far_isa']
        )
        result['sample_geometry_category'] = get_sample_geometry_category(
            result['near_isa'], result['far_isa'],
            result['near_ratio'], result['far_ratio']
        )
        
        # Building categories
        result['geometry_category_building'] = get_geometry_category(
            result['near_building'], result['far_building']
        )
        result['sample_geometry_category_building'] = get_sample_geometry_category(
            result['near_building'], result['far_building'],
            result['near_ratio_building'], result['far_ratio_building']
        )
    
    return result


# Main Pipeline
def run_pipeline(output_dir=None, spike_thresholds=None):
    """Run full pipeline - v6"""
    
    if output_dir is None:
        output_dir = OUTPUT_PATH
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    if spike_thresholds is None:
        spike_thresholds = SPIKE_THRESHOLDS
    
    print("SWOT Urban Hydrology Analysis Pipeline (v6)")
    print("  - Distance-weighted Near/Far ISA")
    print("  - Building Height (GBH2020) sampling")
    
    # [1] Load polygons
    print("\n[1] Loading polygons...")
    polygons_gdf = gpd.read_file(POLYGON_PATH).to_crs('EPSG:4326')
    print(f"    Loaded {len(polygons_gdf)} polygons")
    
    # [2] Load parquet files
    print("\n[2] Loading parquet files...")
    base_df = load_all_parquet_files()
    
    # [3] Classify spikes
    print("\n[3] Classifying spikes (error_reach >= threshold)...")
    base_df = classify_spikes(base_df, spike_thresholds)
    spike_count = base_df['is_spike'].sum()
    print(f"    Spikes: {spike_count} / {len(base_df)} ({100*spike_count/len(base_df):.1f}%)")
    
    # [4] Process
    print("\n[4] Processing satellite positions, Near/Far ISA & Building...")
    
    all_new_data = []
    
    for isa in ISA_VALUES:
        isa_str = f'{isa:.2f}'
        print(f"\n    ISA = {isa_str}...")
        
        polygon_row = polygons_gdf[np.isclose(polygons_gdf['fctimp_mea'], isa, atol=0.01)]
        if polygon_row.empty:
            print(f"      No polygon found")
            continue
        
        polygon = polygon_row.geometry.iloc[0]
        
        isa_mask = base_df['isa_str'] == isa_str
        isa_indices = base_df[isa_mask].index.tolist()
        print(f"      {len(isa_indices)} observations")
        
        for idx in tqdm(isa_indices, desc=f"      ISA {isa_str}"):
            row = base_df.loc[idx]
            result = process_single_observation(row, polygon)
            all_new_data.append({'idx': idx, **result})
    
    # [5] Merge and save
    print("\n[5] Merging and saving...")
    new_cols_df = pd.DataFrame(all_new_data).set_index('idx')
    
    # All columns to merge
    merge_cols = [
        'sat_lon', 'sat_lat', 'sat_altitude', 'sat_heading',
        # ISA simple
        'near_isa', 'far_isa', 'near_ratio', 'far_ratio',
        # ISA distance weighted
        'dw_near_isa', 'dw_far_isa', 'dw_near_ratio', 'dw_far_ratio',
        # Building simple
        'near_building', 'far_building', 'near_ratio_building', 'far_ratio_building',
        # Building distance weighted
        'dw_near_building', 'dw_far_building', 'dw_near_ratio_building', 'dw_far_ratio_building',
        # Categories
        'geometry_category', 'sample_geometry_category',
        'geometry_category_building', 'sample_geometry_category_building',
    ]
    
    for col in merge_cols:
        base_df[col] = new_cols_df[col]
    
    output_file = Path(output_dir) / 'swot_urban_nearfar_v6_results.parquet'
    base_df.to_parquet(output_file)
    print(f"    Saved: {output_file}")
    
    csv_file = Path(output_dir) / 'swot_urban_nearfar_v6_results.csv'
    base_df.to_csv(csv_file, index=False)
    print(f"    Saved: {csv_file}")
    
    # Summary
    print("Summary (v6)")
    print(f"Total: {len(base_df)}")
    print(f"Spikes: {base_df['is_spike'].sum()}, Normal: {(~base_df['is_spike']).sum()}")
    
    print(f"\n[ISA Geometry Category]")
    print(base_df['geometry_category'].value_counts())
    
    print(f"\n[Building Geometry Category]")
    print(base_df['geometry_category_building'].value_counts())
    
    print(f"\n[ISA Stats]")
    print(f"  Simple:   near={base_df['near_isa'].mean():.2f}, far={base_df['far_isa'].mean():.2f}")
    print(f"  DW:       near={base_df['dw_near_isa'].mean():.2f}, far={base_df['dw_far_isa'].mean():.2f}")
    
    print(f"\n[Building Stats]")
    print(f"  Simple:   near={base_df['near_building'].mean():.2f}, far={base_df['far_building'].mean():.2f}")
    print(f"  DW:       near={base_df['dw_near_building'].mean():.2f}, far={base_df['dw_far_building'].mean():.2f}")
    
    return base_df


def run_test(isa_value=0.12, n_rows=10, spike_threshold=None):
    """Test with single ISA - v6"""
    
    isa_str = f'{isa_value:.2f}'
    
    if spike_threshold is None:
        spike_threshold = SPIKE_THRESHOLDS.get(isa_value, 0.3)
    
    print(f"TEST (v6): ISA={isa_str}, rows={n_rows if n_rows else 'ALL'}, threshold={spike_threshold}")
    print("  - Distance-weighted Near/Far ISA")
    print("  - Building Height (GBH2020) sampling")
    
    # Load polygon
    print("\n[1] Loading polygon...")
    polygons_gdf = gpd.read_file(POLYGON_PATH).to_crs('EPSG:4326')
    polygon_row = polygons_gdf[np.isclose(polygons_gdf['fctimp_mea'], isa_value, atol=0.01)]
    
    if polygon_row.empty:
        print(f"    ERROR: No polygon for ISA {isa_str}")
        return None
    
    polygon = polygon_row.geometry.iloc[0]
    print(f"    Bounds: {polygon.bounds}")
    
    # Load parquet
    print("\n[2] Loading parquet...")
    parquet_file = PARQUET_PATH / f'fctimp_mean_{isa_str}_node.parquet'
    
    if not parquet_file.exists():
        print(f"    ERROR: {parquet_file} not found")
        return None
    
    df = pd.read_parquet(parquet_file)
    df['isa_value'] = isa_value
    df['isa_str'] = isa_str
    print(f"    Loaded: {len(df)} rows")
    
    # Classify spikes
    df['spike_threshold'] = spike_threshold
    df['is_spike'] = df['error_reach'] >= spike_threshold
    print(f"    Spikes: {df['is_spike'].sum()} / {len(df)}")
    
    # Limit rows
    if n_rows is not None:
        df = df.head(n_rows)
        print(f"    Testing: {n_rows} rows")
    
    # Process
    print("\n[3] Processing...")
    results = []
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="    "):
        result = process_single_observation(row, polygon)
        results.append(result)
        
        if len(results) <= 3:
            print(f"\n    Row {len(results)}: {row['time_str']}")
            print(f"      node=({row['lon']:.4f}, {row['lat']:.4f})")
            if np.isfinite(result['sat_lon']):
                print(f"      sat=({result['sat_lon']:.4f}, {result['sat_lat']:.4f})")
            print(f"      error_reach={row['error_reach']:.3f}, is_spike={row['is_spike']}")
            print(f"      [ISA]")
            print(f"        Simple: near={result['near_isa']:.2f}, far={result['far_isa']:.2f}")
            print(f"        DW:     near={result['dw_near_isa']:.2f}, far={result['dw_far_isa']:.2f}")
            print(f"      [Building]")
            print(f"        Simple: near={result['near_building']:.2f}, far={result['far_building']:.2f}")
            print(f"        DW:     near={result['dw_near_building']:.2f}, far={result['dw_far_building']:.2f}")
            print(f"      cat={result['geometry_category']}, cat_bld={result['geometry_category_building']}")
    
    # Merge
    results_df = pd.DataFrame(results)
    for col in results_df.columns:
        df[col] = results_df[col].values
    
    # Summary
    print("TEST SUMMARY (v6)")
    print(f"Total: {len(df)}")
    print(f"Sat found: {df['sat_lon'].notna().sum()}")
    print(f"Spikes: {df['is_spike'].sum()}, Normal: {(~df['is_spike']).sum()}")
    
    print(f"\n[ISA Geometry]")
    print(df['geometry_category'].value_counts())
    
    print(f"\n[Building Geometry]")
    print(df['geometry_category_building'].value_counts())
    
    print(f"\n[ISA Stats]")
    print(f"  near_isa:    {df['near_isa'].min():.1f} ~ {df['near_isa'].max():.1f} (mean={df['near_isa'].mean():.1f})")
    print(f"  far_isa:     {df['far_isa'].min():.1f} ~ {df['far_isa'].max():.1f} (mean={df['far_isa'].mean():.1f})")
    print(f"  dw_near_isa: {df['dw_near_isa'].min():.1f} ~ {df['dw_near_isa'].max():.1f} (mean={df['dw_near_isa'].mean():.1f})")
    print(f"  dw_far_isa:  {df['dw_far_isa'].min():.1f} ~ {df['dw_far_isa'].max():.1f} (mean={df['dw_far_isa'].mean():.1f})")
    
    print(f"\n[Building Stats]")
    print(f"  near_building:    {df['near_building'].min():.1f} ~ {df['near_building'].max():.1f} (mean={df['near_building'].mean():.1f})")
    print(f"  far_building:     {df['far_building'].min():.1f} ~ {df['far_building'].max():.1f} (mean={df['far_building'].mean():.1f})")
    print(f"  dw_near_building: {df['dw_near_building'].min():.1f} ~ {df['dw_near_building'].max():.1f} (mean={df['dw_near_building'].mean():.1f})")
    print(f"  dw_far_building:  {df['dw_far_building'].min():.1f} ~ {df['dw_far_building'].max():.1f} (mean={df['dw_far_building'].mean():.1f})")
    
    return df

In [ ]:
df_full = run_pipeline()
df_full.to_parquet("node_level_fctimp_buildingheight_withDW.parquet")

## Calculate near-far ISA only at Reach Level 

In [ ]:
"""
SWOT Urban Analysis - Reach Level Aggregation
==============================================
Convert node-level final_df to reach-level final_reach_df

Aggregation:
- error_reach: keep (same for all nodes in reach)
- flow_angle: mean per ISA
- near_isa: mean per ISA
- far_isa: mean per ISA
- geometry_category: recalculate using MAX near_isa/far_isa >= 30% threshold
"""

import pandas as pd
import numpy as np
from pathlib import Path

# Threshold for Near/Far classification (30%)
NEAR_FAR_THRESHOLD = 30.0


def get_geometry_category(near_isa_max, far_isa_max, threshold=NEAR_FAR_THRESHOLD):
    """
    Categorize based on MAX near_isa and far_isa with threshold
    
    Only count as Near/Far if max value >= 30%
    """
    has_near = near_isa_max >= threshold
    has_far = far_isa_max >= threshold
    
    if has_near and has_far:
        return 'Near/Far'
    elif has_near:
        return 'Near'
    elif has_far:
        return 'Far'
    else:
        return 'NA'


def aggregate_to_reach_level(final_df):
    """
    Aggregate node-level DataFrame to reach-level
    """
    
    print("Aggregating Node-level to Reach-level")
    
    print(f"\nInput: {len(final_df)} node-level rows")
    print(f"Near/Far threshold: {NEAR_FAR_THRESHOLD}%")
    
    # Group by ISA (isa_value or isa_str) + reach_id + time_str
    # Each group = one observation of one reach at one time
    group_cols = ['isa_value', 'isa_str', 'reach_id', 'time_str', 'crid']
    
    # Aggregation rules
    agg_dict = {
        # Keep first (same for all nodes in reach)
        'error_reach': 'first',
        'wse_reach_relative_m': 'first',
        'wse_usgs_relative_m': 'first',
        'is_spike': 'first',
        'spike_threshold': 'first',
        
        # Average across nodes
        'flow_angle': 'mean',
        'near_isa': 'mean',
        'far_isa': 'mean',
        'error_node': 'mean',
        'cor_error_node_relative_m': 'mean',
        'del_wse': 'mean',
        
        # Satellite info (same for all nodes, take first)
        'sat_lon': 'first',
        'sat_lat': 'first',
        'sat_altitude': 'first',
        'sat_heading': 'first',
        
        # Count nodes
        'node_id': 'count',
    }
    
    # Filter to only existing columns
    agg_dict = {k: v for k, v in agg_dict.items() if k in final_df.columns}
    
    # Group and aggregate
    final_reach_df = final_df.groupby(group_cols, as_index=False).agg(agg_dict)
    
    # Rename node_id count
    if 'node_id' in final_reach_df.columns:
        final_reach_df = final_reach_df.rename(columns={'node_id': 'n_nodes'})
    
    # Get MAX near_isa and far_isa for geometry_category decision
    if 'near_isa' in final_df.columns and 'far_isa' in final_df.columns:
        max_agg = final_df.groupby(group_cols, as_index=False).agg({
            'near_isa': 'max',
            'far_isa': 'max'
        })
        max_agg = max_agg.rename(columns={
            'near_isa': 'near_isa_max',
            'far_isa': 'far_isa_max'
        })
        
        # Merge max values
        final_reach_df = final_reach_df.merge(max_agg, on=group_cols, how='left')
        
        # Calculate geometry_category using MAX values with threshold
        final_reach_df['geometry_category'] = final_reach_df.apply(
            lambda row: get_geometry_category(row['near_isa_max'], row['far_isa_max']),
            axis=1
        )
    
    print(f"Output: {len(final_reach_df)} reach-level rows")
    
    # Summary
    print("Summary")
    print(f"ISA values: {sorted(final_reach_df['isa_value'].unique())}")
    print(f"Unique reaches: {final_reach_df['reach_id'].nunique()}")
    
    if 'is_spike' in final_reach_df.columns:
        print(f"\nSpike classification:")
        print(f"  Spikes: {final_reach_df['is_spike'].sum()}")
        print(f"  Normal: {(~final_reach_df['is_spike']).sum()}")
    
    if 'geometry_category' in final_reach_df.columns:
        print(f"\nGeometry Category (threshold={NEAR_FAR_THRESHOLD}%):")
        print(final_reach_df['geometry_category'].value_counts().to_string())
    
    if 'near_isa_max' in final_reach_df.columns:
        print(f"\nNear/Far ISA max stats:")
        print(f"  near_isa_max: min={final_reach_df['near_isa_max'].min():.1f}, max={final_reach_df['near_isa_max'].max():.1f}, mean={final_reach_df['near_isa_max'].mean():.1f}")
        print(f"  far_isa_max:  min={final_reach_df['far_isa_max'].min():.1f}, max={final_reach_df['far_isa_max'].max():.1f}, mean={final_reach_df['far_isa_max'].mean():.1f}")
    
    if 'n_nodes' in final_reach_df.columns:
        print(f"\nNodes per reach: min={final_reach_df['n_nodes'].min()}, max={final_reach_df['n_nodes'].max()}, mean={final_reach_df['n_nodes'].mean():.1f}")
    
    return final_reach_df


def main():
    """Load final_df from parquet and aggregate to reach level"""
    
    # Path
    input_path = Path('/work/pi_kandread_umass_edu/swot-urban/3_analysis_results/swot_urban_nearfar_results.parquet')
    output_path = Path('/work/pi_kandread_umass_edu/swot-urban/3_analysis_results/swot_urban_nearfar_reach_results.parquet')
    
    print(f"Loading: {input_path}")
    final_df = pd.read_parquet(input_path)
    
    # Aggregate
    final_reach_df = aggregate_to_reach_level(final_df)
    
    # Save
    final_reach_df.to_parquet(output_path)
    print(f"\nSaved: {output_path}")
    
    # Also save CSV
    csv_path = output_path.with_suffix('.csv')
    final_reach_df.to_csv(csv_path, index=False)
    print(f"Saved: {csv_path}")
    
    return final_reach_df

In [ ]:
final_reach_df = main()
final_reach_df.to_parquet("reach_level_near_far.parquet")

## Near-Far ISA and Building Height Analysis

### Load Datasets

In [ ]:
# final_reach_df = pd.read_parquet("reach_level_near_far.parquet")
final_df = pd.read_parquet("/project/pi_kandread_umass_edu/swot-urban/3_analysis_results/swot_urban_nearfar_v6.5_results.parquet")
# analysis_final_df = pd.read_parquet("pixc_pixels_combined.parquet")
# final_df = pd.read_parquet("cor_node_level_near_far_ver3.parquet")

# Node-level
threshold = 0.1832
final_df['is_spike'] = final_df['error_reach'].abs() >= threshold
final_df['abs_error_node'] = final_df['error_node'].abs()

# # Reach-level
# final_reach_df['is_spike'] = final_reach_df['error_reach'].abs() >= threshold

In [ ]:
"""
Masking optimization v2: geometry-split analysis
======================================================================
(a) Flow angle: All/Far/Near/Both/Neither error reduction + All removed
    Optimal threshold = max sum of slopes across categories -> 뒤에점
(b) Building height: parallel < opt_angle, far_building >= 5 m
(c) ISA: parallel < opt_angle, far_isa > 0
    Optimal threshold = max marginal efficiency (Δred / Δrm) -> 뒤에점

y1 = Cumulative 68th %ile error reduction (%)
y2 = Observations removed (%)

Run on server:  python optimize_mask_individual_v2.py
"""
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.transforms import blended_transform_factory

# Publication style
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'stix',
    'font.size': 14,
    'axes.labelsize': 14,
    'axes.titlesize': 14,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'axes.linewidth': 1.0,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.major.size': 6,
    'ytick.major.size': 6,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
})

# Data loading
DATA_PATH = "/project/pi_kandread_umass_edu/swot-urban/3_analysis_results/swot_urban_nearfar_v6.5_results.parquet"
COLS_NEEDED = ['flow_angle', 'far_building', 'far_isa', 'error_node',
               'near_isa', 'near_ratio', 'far_ratio']
MIN_RATIO = 0.50

print("Loading data ...")
df = pd.read_parquet(DATA_PATH, columns=COLS_NEEDED, engine='pyarrow')
df['abs_error'] = df['error_node'].abs()
df['flow_angle_parallel'] = df['flow_angle'].apply(
    lambda a: min(a % 180, 180 - a % 180))

if df['far_isa'].max() <= 1.0:
    df['far_isa_pct'] = df['far_isa'] * 100
else:
    df['far_isa_pct'] = df['far_isa']

# Geometry categorisation (mutually exclusive)
has_far  = (df['far_isa'].fillna(0) > 0) & (df['far_ratio'].fillna(0) >= MIN_RATIO)
has_near = (df['near_isa'].fillna(0) > 0) & (df['near_ratio'].fillna(0) >= MIN_RATIO)

abs_err  = df['abs_error'].values
flow_par = df['flow_angle_parallel'].values
far_bldg = df['far_building'].values
far_isa  = df['far_isa_pct'].values

is_far_only  = (has_far & ~has_near).values
is_near_only = (has_near & ~has_far).values
is_both      = (has_far & has_near).values
is_neither   = (~has_far & ~has_near).values
is_all       = np.ones(len(df), dtype=bool)

n_total = len(df)
p68_total = np.quantile(abs_err, 0.68)

CAT_ORDER = ['All', 'Far', 'Near', 'Both', 'Neither']
CAT = {
    'All': is_all, 'Far': is_far_only, 'Near': is_near_only,
    'Both': is_both, 'Neither': is_neither,
}
COLORS = {
    'All': '#000000', 'Far': '#D62728', 'Near': '#2CA02C',
    'Both': '#1F77B4', 'Neither': '#7F7F7F',
}
LABELS = {
    'All': 'All Observations', 'Far': 'Far-Nadir Only',
    'Near': 'Near-Nadir Only', 'Both': 'Both Geometries',
    'Neither': 'Neither Geometry',
}

# Baselines per category
baselines = {}
print(f"\nTotal: {n_total:,}   68th %ile = {p68_total:.4f} m")
for name in CAT_ORDER:
    n = CAT[name].sum()
    p = np.quantile(abs_err[CAT[name]], 0.68) if n > 0 else np.nan
    baselines[name] = p
    print(f"  {name:<10s}: {n:>9,} ({n/n_total*100:>5.1f}%)  68th %ile = {p:.4f} m")
print()

# 1. Flow angle sweep
print("1. FLOW ANGLE SWEEP")

angle_thresholds = np.arange(1, 32, 2)
results_angle = {name: [] for name in CAT_ORDER}

for angle_t in angle_thresholds:
    remove = flow_par < angle_t
    for name in CAT_ORDER:
        mask = CAT[name]
        retained = mask & ~remove
        n_rm  = (mask & remove).sum()
        n_cat = mask.sum()
        if retained.sum() > 0:
            p68_ret = np.quantile(abs_err[retained], 0.68)
            pct_red = (baselines[name] - p68_ret) / baselines[name] * 100
        else:
            pct_red = np.nan
        results_angle[name].append({
            'threshold': angle_t,
            'pct_reduction': pct_red,
            'pct_removed': n_rm / n_cat * 100 if n_cat > 0 else 0,
        })
    vals = "  ".join(f"{name}={results_angle[name][-1]['pct_reduction']:>+5.1f}%"
                     for name in CAT_ORDER)
    print(f"  <{angle_t:>2.0f}°: {vals}")

print()

# Optimal angle threshold (needed for sections 2 & 3)
slopes_a = []
for i in range(len(angle_thresholds) - 1):
    dx = float(angle_thresholds[i+1] - angle_thresholds[i])
    s = sum((results_angle[n][i+1]['pct_reduction'] - results_angle[n][i]['pct_reduction']) / dx
            for n in CAT_ORDER)
    slopes_a.append(s)
opt_step_a = int(np.argmax(slopes_a))
opt_idx_a  = opt_step_a + 1          # 뒤에점
opt_thresh_a = float(angle_thresholds[opt_idx_a])
print(f">>> Optimal angle threshold: {opt_thresh_a:.0f}°  "
      f"(max slope sum at {angle_thresholds[opt_step_a]:.0f}°->{opt_thresh_a:.0f}°)\n")

# 2. BUILDING HEIGHT SWEEP (parallel < opt_angle, far_bldg >= 5 m)
print(f"2. BUILDING HEIGHT SWEEP  (parallel < {opt_thresh_a:.0f}°, bldg ≥ 5 m)")

has_valid_bldg = far_bldg >= 0
is_par_bldg    = (flow_par < opt_thresh_a) & has_valid_bldg
n_sub_bldg     = is_par_bldg.sum()
p68_sub_bldg   = np.quantile(abs_err[is_par_bldg], 0.68)
print(f"   Subset: {n_sub_bldg:,},  68th %ile = {p68_sub_bldg:.4f} m\n")

bldg_thresholds = np.arange(20, 4.5, -0.5)
results_bldg = []

for h_t in bldg_thresholds:
    remove = is_par_bldg & (far_bldg > h_t)
    n_rm   = remove.sum()
    ret_sub = is_par_bldg & ~remove
    if ret_sub.sum() > 0:
        p68s = np.quantile(abs_err[ret_sub], 0.68)
        red_sub = (p68_sub_bldg - p68s) / p68_sub_bldg * 100
    else:
        red_sub = np.nan
    rm_sub = n_rm / n_sub_bldg * 100
    p68t = np.quantile(abs_err[~remove], 0.68)
    red_tot = (p68_total - p68t) / p68_total * 100
    rm_tot  = n_rm / n_total * 100

    results_bldg.append(dict(threshold=h_t, n_removed=n_rm,
                             red_sub=red_sub, rm_sub=rm_sub,
                             red_tot=red_tot, rm_tot=rm_tot))
    print(f"  >{h_t:>5.1f}m: rm={n_rm:>6,d}  sub[{red_sub:>+5.1f}% | {rm_sub:>5.1f}%]  "
          f"tot[{red_tot:>+5.2f}% | {rm_tot:>5.2f}%]")

print()

# 3. ISA SWEEP (parallel < opt_angle, far_isa > 0)
print(f"3. FAR ISA SWEEP  (parallel < {opt_thresh_a:.0f}°, ISA > 0)")

has_valid_isa = far_isa > 0
is_par_isa    = (flow_par < opt_thresh_a) & has_valid_isa
n_sub_isa     = is_par_isa.sum()
p68_sub_isa   = np.quantile(abs_err[is_par_isa], 0.68)
print(f"   Subset: {n_sub_isa:,},  68th %ile = {p68_sub_isa:.4f} m\n")

isa_thresholds = np.arange(63, 4, -2)
results_isa = []

for isa_t in isa_thresholds:
    remove = is_par_isa & (far_isa > isa_t)
    n_rm   = remove.sum()
    ret_sub = is_par_isa & ~remove
    if ret_sub.sum() > 0:
        p68s = np.quantile(abs_err[ret_sub], 0.68)
        red_sub = (p68_sub_isa - p68s) / p68_sub_isa * 100
    else:
        red_sub = np.nan
    rm_sub = n_rm / n_sub_isa * 100
    p68t    = np.quantile(abs_err[~remove], 0.68)
    red_tot = (p68_total - p68t) / p68_total * 100
    rm_tot  = n_rm / n_total * 100

    results_isa.append(dict(threshold=isa_t, n_removed=n_rm,
                            red_sub=red_sub, rm_sub=rm_sub,
                            red_tot=red_tot, rm_tot=rm_tot))
    print(f"  >{isa_t:>4.0f}%: rm={n_rm:>6,d}  sub[{red_sub:>+5.1f}% | {rm_sub:>5.1f}%]  "
          f"tot[{red_tot:>+5.2f}% | {rm_tot:>5.2f}%]")

print()

# 4. Combined masks
print("4. COMBINED MASKS")

combos = [
    ("No masking",           np.zeros(n_total, dtype=bool)),
    ("Bldg > 13 m",         is_par_bldg & (far_bldg > 13)),
    ("Bldg > 8 m",          is_par_bldg & (far_bldg > 8)),
    ("ISA > 40%",            is_par_isa  & (far_isa > 40)),
    ("ISA > 30%",            is_par_isa  & (far_isa > 30)),
    ("Bldg>8 OR ISA>30%",   (is_par_bldg & (far_bldg > 8)) | (is_par_isa & (far_isa > 30))),
    ("Bldg>13 OR ISA>30%",  (is_par_bldg & (far_bldg > 13)) | (is_par_isa & (far_isa > 30))),
    (f"All parallel <{opt_thresh_a:.0f}°", flow_par < opt_thresh_a),
]

print(f"\n{'Mask':<28s} {'Removed':>8s} {'%Tot':>7s} {'68th%ile':>9s} {'Red%':>7s}")
for label, mask in combos:
    n_rm = mask.sum()
    p68r = np.quantile(abs_err[~mask], 0.68)
    red  = (p68_total - p68r) / p68_total * 100
    print(f"{label:<28s} {n_rm:>8,d} {n_rm/n_total*100:>6.2f}% {p68r:>9.4f} {red:>+6.2f}%")

# OPTIMAL THRESHOLDS (b, c)
print("OPTIMAL THRESHOLDS")
print(f"  (a) Flow angle : {opt_thresh_a:.0f}°")

# (b) Knee point: normalize red_sub & rm_sub to [0,1], find max(red_norm - rm_norm)
#     = point where error reduction leads sample removal the most (sweet spot)
rm_b  = np.array([r['rm_sub']  for r in results_bldg])
red_b = np.array([r['red_sub'] for r in results_bldg])
rm_b_norm  = rm_b  / rm_b.max()       if rm_b.max()        > 0 else rm_b
red_b_norm = red_b / np.nanmax(red_b)  if np.nanmax(red_b) > 0 else red_b
knee_b = red_b_norm - rm_b_norm
opt_idx_b = int(np.argmax(knee_b))
opt_thresh_b = results_bldg[opt_idx_b]['threshold']
print(f"  (b) Building ht: {opt_thresh_b:.1f} m  (knee score={knee_b[opt_idx_b]:.3f},  "
      f"red_sub={results_bldg[opt_idx_b]['red_sub']:.1f}% / "
      f"rm_sub={results_bldg[opt_idx_b]['rm_sub']:.1f}%)")

# (c) Knee point: same approach for ISA
rm_c  = np.array([r['rm_sub']  for r in results_isa])
red_c = np.array([r['red_sub'] for r in results_isa])
rm_c_norm  = rm_c  / rm_c.max()       if rm_c.max()        > 0 else rm_c
red_c_norm = red_c / np.nanmax(red_c)  if np.nanmax(red_c) > 0 else red_c
knee_c = red_c_norm - rm_c_norm
opt_idx_c = int(np.argmax(knee_c))
opt_thresh_c = results_isa[opt_idx_c]['threshold']
print(f"  (c) ISA        : {opt_thresh_c:.0f}%  (knee score={knee_c[opt_idx_c]:.3f},  "
      f"red_sub={results_isa[opt_idx_c]['red_sub']:.1f}% / "
      f"rm_sub={results_isa[opt_idx_c]['rm_sub']:.1f}%)")

# Plotting
print("\nMaking plots ...")

ANNO_FS = 13  # match tick label size

fig, axes = plt.subplots(1, 3, figsize=(19, 6.5))

# (a) Flow Angle
ax = axes[0]
xa = [r['threshold'] for r in results_angle['All']]

for name in CAT_ORDER:
    vals = [r['pct_reduction'] for r in results_angle[name]]
    ax.plot(xa, vals, color=COLORS[name], linestyle='-', marker='o',
            markersize=4, linewidth=1.8, label=LABELS[name])

ax.set_xlabel('Flow Angle Threshold (°)')
ax.set_ylabel('Cumul. 68th %ile Error Reduction (%)')
ax.set_title('(a) Flow Angle Masking')
ax.grid(True, alpha=0.25, linestyle='-', linewidth=0.5)

ax2 = ax.twinx()
vals_rm = [r['pct_removed'] for r in results_angle['All']]
ax2.plot(xa, vals_rm, color='black', linestyle='--', linewidth=1.2,
         alpha=0.5, label='All Obs. Removed (%)')
ax2.set_ylabel('Observations Removed (%)', color='gray')
ax2.tick_params(axis='y', labelcolor='gray')

# Optimal threshold annotation
ax.axvline(x=opt_thresh_a, color='#444444', linestyle='--',
           linewidth=1.5, alpha=0.7, zorder=3)

for name in CAT_ORDER:
    val = results_angle[name][opt_idx_a]['pct_reduction']
    ax.plot(opt_thresh_a, val, 'o', color=COLORS[name], markersize=8,
            markeredgecolor='white', markeredgewidth=1.0, zorder=5)

rm_all = results_angle['All'][opt_idx_a]['pct_removed']
ax2.plot(opt_thresh_a, rm_all, 's', color='black', markersize=7,
         markeredgecolor='white', markeredgewidth=1.0, zorder=5)

# Text box
anno = [f'θ* = {opt_thresh_a:.0f}°']
for name in CAT_ORDER:
    v = results_angle[name][opt_idx_a]['pct_reduction']
    abs_cm = baselines[name] * v
    anno.append(f'  {LABELS[name]}: {v:.1f}% ({abs_cm:.1f} cm)')
anno.append(f'  Removed (All): {rm_all:.1f}%')
bbox_a = dict(boxstyle='round,pad=0.35', facecolor='#FFFFF0',
              edgecolor='gray', alpha=0.92)
trans_a = blended_transform_factory(ax.transData, ax.transAxes)
ax.text(opt_thresh_a + 1.2, 0.62, '\n'.join(anno), transform=trans_a,
        fontsize=9, va='top', family='monospace', bbox=bbox_a, zorder=6)

lines1, labs1 = ax.get_legend_handles_labels()
lines2, labs2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labs1 + labs2,
          loc='upper left', fontsize=14, ncol=1,
          framealpha=0.9, edgecolor='gray')

# Helper: annotate optimal on (b)/(c)
def annotate_bc(ax_main, ax_right, opt_x, r, panel_label, p68_sub, p68_tot):
    """Draw vertical line, θ* label inside plot, and annotate intersections."""
    ax_main.axvline(x=opt_x, color='#444444', linestyle='--',
                    linewidth=1.5, alpha=0.7, zorder=3)

    abs_sub_cm = p68_sub * r['red_sub']
    abs_tot_cm = p68_tot * r['red_tot']

    # Intersection dots
    ax_main.plot(opt_x, r['red_sub'], 'o', color='blue', markersize=8,
                 markeredgecolor='white', markeredgewidth=1.0, zorder=5)
    ax_main.plot(opt_x, r['red_tot'], 'o', color='blue', markersize=6,
                 markeredgecolor='white', markeredgewidth=0.8, alpha=0.6, zorder=5)
    ax_right.plot(opt_x, r['rm_sub'], 's', color='red', markersize=7,
                  markeredgecolor='white', markeredgewidth=1.0, zorder=5)
    ax_right.plot(opt_x, r['rm_tot'], 's', color='red', markersize=5,
                  markeredgecolor='white', markeredgewidth=0.8, alpha=0.6, zorder=5)

    # Horizontal reference lines
    xlims = ax_main.get_xlim()
    ax_main.plot([xlims[0], opt_x], [r['red_sub'], r['red_sub']],
                 color='blue', linestyle=':', linewidth=0.8, alpha=0.4, zorder=2)
    ax_main.plot([xlims[0], opt_x], [r['red_tot'], r['red_tot']],
                 color='blue', linestyle=':', linewidth=0.6, alpha=0.3, zorder=2)
    ax_right.plot([opt_x, xlims[1]], [r['rm_sub'], r['rm_sub']],
                  color='red', linestyle=':', linewidth=0.8, alpha=0.4, zorder=2)
    ax_right.plot([opt_x, xlims[1]], [r['rm_tot'], r['rm_tot']],
                  color='red', linestyle=':', linewidth=0.6, alpha=0.3, zorder=2)

    # Value labels on y-axes
    ax_main.text(xlims[0], r['red_sub'],
                 f' {r["red_sub"]:.1f}%\n ({abs_sub_cm:.1f} cm)',
                 fontsize=ANNO_FS, color='blue', fontweight='bold', va='center')
    ax_main.text(xlims[0]-15, r['red_tot'],
                 f' {r["red_tot"]:.2f}%\n ({abs_tot_cm:.1f} cm)',
                 fontsize=ANNO_FS - 1, color='blue', alpha=0.7, va='center')
    ax_right.text(xlims[1], r['rm_sub'], f'{r["rm_sub"]:.1f}% ',
                  fontsize=ANNO_FS, color='red', fontweight='bold',
                  va='center', ha='right')
    ax_right.text(xlims[1], r['rm_tot'], f'{r["rm_tot"]:.2f}% ',
                  fontsize=ANNO_FS - 1, color='red', alpha=0.7,
                  va='center', ha='right')

    print(xlims[0] + 10)

    # θ* label inside plot, right of dashed line
    unit = 'm' if 'uilding' in panel_label else '%'
    x_range = abs(xlims[1] - xlims[0])
    text_x = opt_x - 0.03 * x_range   # inverted axis: lower x = right
    trans = blended_transform_factory(ax_main.transData, ax_main.transAxes)
    ax_main.text(text_x, 0.95, f'θ* = {opt_x:.0f} {unit}', fontsize=ANNO_FS,
                 fontweight='bold', color='#444444', ha='right', va='top',
                 transform=trans, zorder=6)

# (b) Building Height
ax = axes[1]
xb = [r['threshold'] for r in results_bldg]

ax.plot(xb, [r['red_sub'] for r in results_bldg],
        'b-o', markersize=4, linewidth=1.8,
        label=f'Parallel subset (n={n_sub_bldg:,})')
ax.plot(xb, [r['red_tot'] for r in results_bldg],
        color='blue', linestyle='--', linewidth=1.2, alpha=0.5,
        label=f'Total dataset (n={n_total:,})')
ax.set_xlabel('Far-Nadir Building Height Threshold (m)')
ax.set_ylabel('Cumul. 68th %ile Error Reduction (%)', color='blue')
ax.tick_params(axis='y', labelcolor='blue')
ax.set_title(f'(b) Building Height Masking (Parallel < {opt_thresh_a:.0f}°)')
ax.grid(True, alpha=0.25, linestyle='-', linewidth=0.5)
ax.invert_xaxis()

ax2 = ax.twinx()
ax2.plot(xb, [r['rm_sub'] for r in results_bldg],
         'r-s', markersize=3, linewidth=1.4, label='Subset removed (%)')
ax2.plot(xb, [r['rm_tot'] for r in results_bldg],
         color='red', linestyle='--', linewidth=1.0, alpha=0.5,
         label='Total removed (%)')
ax2.set_ylabel('Observations Removed (%)', color='red')
ax2.tick_params(axis='y', labelcolor='red')

lines1, labs1 = ax.get_legend_handles_labels()
lines2, labs2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labs1 + labs2,
          loc='upper left', fontsize=13, framealpha=0.9, edgecolor='gray')

annotate_bc(ax, ax2, opt_thresh_b, results_bldg[opt_idx_b],
            'Building', p68_sub_bldg, p68_total)

# (c) ISA
ax = axes[2]
xc = [r['threshold'] for r in results_isa]

ax.plot(xc, [r['red_sub'] for r in results_isa],
        'b-o', markersize=4, linewidth=1.8,
        label=f'Parallel subset (n={n_sub_isa:,})')
ax.plot(xc, [r['red_tot'] for r in results_isa],
        color='blue', linestyle='--', linewidth=1.2, alpha=0.5,
        label=f'Total dataset (n={n_total:,})')
ax.set_xlabel('Far-Nadir ISA Threshold (%)')
ax.set_ylabel('Cumul. 68th %ile Error Reduction (%)', color='blue')
ax.tick_params(axis='y', labelcolor='blue')
ax.set_title(f'(c) ISA Masking (Parallel < {opt_thresh_a:.0f}°)')
ax.grid(True, alpha=0.25, linestyle='-', linewidth=0.5)
ax.invert_xaxis()

ax2 = ax.twinx()
ax2.plot(xc, [r['rm_sub'] for r in results_isa],
         'r-s', markersize=3, linewidth=1.4, label='Subset removed (%)')
ax2.plot(xc, [r['rm_tot'] for r in results_isa],
         color='red', linestyle='--', linewidth=1.0, alpha=0.5,
         label='Total removed (%)')
ax2.set_ylabel('Observations Removed (%)', color='red')
ax2.tick_params(axis='y', labelcolor='red')

lines1, labs1 = ax.get_legend_handles_labels()
lines2, labs2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labs1 + labs2,
          loc='upper left', fontsize=14, framealpha=0.9, edgecolor='gray')

annotate_bc(ax, ax2, opt_thresh_c, results_isa[opt_idx_c],
            'ISA%', p68_sub_isa, p68_total)

plt.tight_layout()
plt.savefig('mask_individual_sweeps_v2.png', dpi=300, bbox_inches='tight')
plt.savefig('mask_individual_sweeps_v2.pdf', bbox_inches='tight')
print("Saved: mask_individual_sweeps_v2.png / .pdf")
print("\nDone!")


In [ ]:
"""
Step 1: Evaluate each masking variable INDEPENDENTLY.
  - Flow angle threshold sweep
  - Far building height threshold sweep (within parallel)
  - Far ISA threshold sweep (within parallel)

For each, show: P68 of total retained, P68 of parallel retained,
error reduction, and sample counts.

Then combine the best thresholds.

Run on server:
  python optimize_mask_individual.py
"""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Config
DATA_PATH = "/project/pi_kandread_umass_edu/swot-urban/3_analysis_results/swot_urban_nearfar_v6.5_results.parquet"
COLS_NEEDED = ['flow_angle', 'far_building', 'far_isa', 'error_node', 'reach_id']

print("Loading data...")
df = pd.read_parquet(DATA_PATH, columns=COLS_NEEDED, engine='pyarrow')
df['abs_error'] = df['error_node'].abs()

# Flow angle -> continuous variable for sweeping
df['flow_angle_parallel'] = df['flow_angle'].apply(lambda a: min(a % 180, 180 - a % 180))

# Baseline
n_total = len(df)
p68_total = np.quantile(df['abs_error'].values, 0.68)

# ISA in percent
if df['far_isa'].max() <= 1.0:
    df['far_isa_pct'] = df['far_isa'] * 100
else:
    df['far_isa_pct'] = df['far_isa']

abs_err = df['abs_error'].values
flow_par = df['flow_angle_parallel'].values
far_bldg = df['far_building'].values
far_isa = df['far_isa_pct'].values

print(f"Total: {n_total:,}")
print(f"P68 total: {p68_total:.4f} m")
print()

# 1. Flow angle sweep
#    Remove all observations with flow_angle_parallel < threshold
print("1. FLOW ANGLE SWEEP: remove parallel obs (angle < threshold)")

angle_thresholds = np.arange(2, 32, 1)  # 2° to 31°
results_angle = []

for angle_t in angle_thresholds:
    mask = flow_par < angle_t
    n_removed = mask.sum()
    retained = abs_err[~mask]
    p68_ret = np.quantile(retained, 0.68)
    pct_red = (p68_total - p68_ret) / p68_total * 100
    pct_retained = len(retained) / n_total * 100
    results_angle.append({
        'threshold': angle_t, 'n_removed': n_removed,
        'p68_retained': p68_ret, 'pct_reduction': pct_red,
        'pct_retained': pct_retained
    })
    print(f"  <{angle_t:>3.0f}°: removed {n_removed:>7,d} ({100-pct_retained:>5.1f}%)  "
          f"P68={p68_ret:.4f}m  reduction={pct_red:>5.1f}%")

print()

# 2. BUILDING HEIGHT SWEEP (within parallel <10°)
#    Remove parallel obs with far_building > threshold
print("2. BUILDING HEIGHT SWEEP (parallel <10° only)")

is_par10 = flow_par < 10
n_par10 = is_par10.sum()
p68_par10 = np.quantile(abs_err[is_par10], 0.68)
print(f"   Parallel <10°: {n_par10:,} obs, P68={p68_par10:.4f}m")
print()

bldg_thresholds = np.arange(3, 21, 0.5)
results_bldg = []

for h_t in bldg_thresholds:
    mask = is_par10 & (far_bldg > h_t)
    n_removed = mask.sum()
    # P68 of retained TOTAL
    retained_total = abs_err[~mask]
    p68_ret_total = np.quantile(retained_total, 0.68)
    pct_red_total = (p68_total - p68_ret_total) / p68_total * 100
    # P68 of retained PARALLEL (what's left of parallel after mask)
    par_retained = is_par10 & (~mask)
    n_par_retained = par_retained.sum()
    if n_par_retained > 0:
        p68_ret_par = np.quantile(abs_err[par_retained], 0.68)
        pct_red_par = (p68_par10 - p68_ret_par) / p68_par10 * 100
    else:
        p68_ret_par = np.nan
        pct_red_par = np.nan
    results_bldg.append({
        'threshold': h_t, 'n_removed': n_removed,
        'p68_ret_total': p68_ret_total, 'pct_red_total': pct_red_total,
        'p68_ret_par': p68_ret_par, 'pct_red_par': pct_red_par,
        'pct_retained_total': len(retained_total) / n_total * 100,
        'pct_par_retained': n_par_retained / n_par10 * 100
    })
    print(f"  >{h_t:>5.1f}m: removed {n_removed:>6,d}  "
          f"P68_total={p68_ret_total:.4f}m ({pct_red_total:>+5.1f}%)  "
          f"P68_parallel={p68_ret_par:.4f}m ({pct_red_par:>+5.1f}%)  "
          f"par_retained={n_par_retained/n_par10*100:.1f}%")

print()

# 3. ISA SWEEP (within parallel <10°)
#    Remove parallel obs with far_isa > threshold
print("3. FAR ISA SWEEP (parallel <10° only)")

isa_thresholds = np.arange(5, 65, 2)
results_isa = []

for isa_t in isa_thresholds:
    mask = is_par10 & (far_isa > isa_t)
    n_removed = mask.sum()
    retained_total = abs_err[~mask]
    p68_ret_total = np.quantile(retained_total, 0.68)
    pct_red_total = (p68_total - p68_ret_total) / p68_total * 100
    par_retained = is_par10 & (~mask)
    n_par_retained = par_retained.sum()
    if n_par_retained > 0:
        p68_ret_par = np.quantile(abs_err[par_retained], 0.68)
        pct_red_par = (p68_par10 - p68_ret_par) / p68_par10 * 100
    else:
        p68_ret_par = np.nan
        pct_red_par = np.nan
    results_isa.append({
        'threshold': isa_t, 'n_removed': n_removed,
        'p68_ret_total': p68_ret_total, 'pct_red_total': pct_red_total,
        'p68_ret_par': p68_ret_par, 'pct_red_par': pct_red_par,
        'pct_retained_total': len(retained_total) / n_total * 100,
        'pct_par_retained': n_par_retained / n_par10 * 100
    })
    print(f"  >{isa_t:>4.0f}%: removed {n_removed:>6,d}  "
          f"P68_total={p68_ret_total:.4f}m ({pct_red_total:>+5.1f}%)  "
          f"P68_parallel={p68_ret_par:.4f}m ({pct_red_par:>+5.1f}%)  "
          f"par_retained={n_par_retained/n_par10*100:.1f}%")

print()

# 4. COMBINED: best thresholds from individual sweeps
print("4. COMBINED MASKS")

combos = [
    ("Angle<10°", flow_par < 10, None, None),
    ("Angle<10° + Bldg>8m", is_par10 & (far_bldg > 8), None, None),
    ("Angle<10° + ISA>30%", is_par10 & (far_isa > 30), None, None),
    ("Angle<10° + ISA>40%", is_par10 & (far_isa > 40), None, None),
    ("Angle<10° + (Bldg>8m OR ISA>30%)", is_par10 & ((far_bldg > 8) | (far_isa > 30)), None, None),
    ("Angle<10° + (Bldg>8m OR ISA>40%)", is_par10 & ((far_bldg > 8) | (far_isa > 40)), None, None),
    ("Angle<10° + (Bldg>8m AND ISA>20%)", is_par10 & ((far_bldg > 8) & (far_isa > 20)), None, None),
    ("Angle<10° + (Bldg>13m OR ISA>30%)", is_par10 & ((far_bldg > 13) | (far_isa > 30)), None, None),
    ("Remove ALL parallel", is_par10, None, None),
]

print(f"\n{'Mask':<45s} {'Removed':>8s} {'%Rem':>6s} {'P68_tot':>8s} {'Red%':>6s} "
      f"{'P68_par':>8s} {'RedP%':>6s} {'Par_ret%':>8s}")

for label, mask, _, _ in combos:
    n_removed = mask.sum()
    retained = abs_err[~mask]
    p68_ret = np.quantile(retained, 0.68)
    pct_red = (p68_total - p68_ret) / p68_total * 100

    par_retained = is_par10 & (~mask)
    n_par_ret = par_retained.sum()
    if n_par_ret > 0:
        p68_par_ret = np.quantile(abs_err[par_retained], 0.68)
        pct_red_par = (p68_par10 - p68_par_ret) / p68_par10 * 100
    else:
        p68_par_ret = np.nan
        pct_red_par = np.nan

    print(f"{label:<45s} {n_removed:>8,d} {n_removed/n_total*100:>5.1f}% "
          f"{p68_ret:>8.4f} {pct_red:>5.1f}% "
          f"{p68_par_ret:>8.4f} {pct_red_par:>5.1f}% "
          f"{n_par_ret/n_par10*100:>7.1f}%")

    # Extra: median and P68 of REMOVED observations
    if n_removed > 0:
        removed_err = abs_err[mask]
        print(f"{'':45s}   -> removed obs: median={np.median(removed_err):.4f}m  "
              f"P68={np.quantile(removed_err, 0.68):.4f}m  "
              f"P90={np.quantile(removed_err, 0.90):.4f}m")

# 5. REACH-LEVEL analysis (P68 per reach, then median across reaches)
#    This may show larger improvements
print("5. REACH-LEVEL P68 (median P68 across reaches)")

if 'reach_id' in df.columns:
    # Baseline: P68 per reach
    reach_p68_baseline = df.groupby('reach_id')['abs_error'].quantile(0.68)
    median_reach_p68_baseline = reach_p68_baseline.median()
    mean_reach_p68_baseline = reach_p68_baseline.mean()
    print(f"  Baseline: median reach P68 = {median_reach_p68_baseline:.4f}m, "
          f"mean reach P68 = {mean_reach_p68_baseline:.4f}m")
    print()

    for label, mask, _, _ in combos[:6]:
        df_ret = df[~mask]
        reach_p68_ret = df_ret.groupby('reach_id')['abs_error'].quantile(0.68)
        # Only compare reaches that still have data
        common = reach_p68_baseline.index.intersection(reach_p68_ret.index)
        med_ret = reach_p68_ret[common].median()
        mean_ret = reach_p68_ret[common].mean()
        pct_red_med = (median_reach_p68_baseline - med_ret) / median_reach_p68_baseline * 100
        pct_red_mean = (mean_reach_p68_baseline - mean_ret) / mean_reach_p68_baseline * 100
        n_removed_reaches = len(reach_p68_baseline) - len(reach_p68_ret)
        print(f"  {label:<45s}")
        print(f"    median reach P68: {med_ret:.4f}m ({pct_red_med:+.1f}%)  "
              f"mean reach P68: {mean_ret:.4f}m ({pct_red_mean:+.1f}%)  "
              f"reaches lost: {n_removed_reaches}")

# PLOTTING: 3 panels - individual sweeps
print("\nGenerating plots...")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) Flow angle sweep
ax = axes[0]
ra = results_angle
x = [r['threshold'] for r in ra]
ax.plot(x, [r['pct_reduction'] for r in ra], 'b-o', markersize=4, label='Error reduction (%)')
ax.set_xlabel('Flow angle threshold (°)')
ax.set_ylabel('P68 error reduction (%)', color='b')
ax.tick_params(axis='y', labelcolor='b')
ax.set_title('(a) Flow Angle: remove obs < threshold')
ax2 = ax.twinx()
ax2.plot(x, [100-r['pct_retained'] for r in ra], 'r--s', markersize=3, label='Samples removed (%)')
ax2.set_ylabel('Samples removed (%)', color='r')
ax2.tick_params(axis='y', labelcolor='r')
ax.axvline(x=10, color='gray', linestyle=':', label='10° (current)')
ax.legend(loc='upper left', fontsize=8)
ax2.legend(loc='upper right', fontsize=8)

# (b) Building height sweep (parallel P68)
ax = axes[1]
rb = results_bldg
x = [r['threshold'] for r in rb]
ax.plot(x, [r['pct_red_total'] for r in rb], 'b--', markersize=3, label='Total P68 reduction', alpha=0.5)
ax.plot(x, [r['pct_red_par'] for r in rb], 'b-o', markersize=4, label='Parallel P68 reduction')
ax.set_xlabel('Building height threshold (m)')
ax.set_ylabel('P68 error reduction (%)', color='b')
ax.tick_params(axis='y', labelcolor='b')
ax.set_title('(b) Building Height (parallel <10°)')
ax2 = ax.twinx()
ax2.plot(x, [100-r['pct_par_retained'] for r in rb], 'r--s', markersize=3, label='Parallel removed (%)')
ax2.set_ylabel('Parallel samples removed (%)', color='r')
ax2.tick_params(axis='y', labelcolor='r')
ax.axvline(x=8, color='gray', linestyle=':', label='8m (current)')
ax.legend(loc='upper right', fontsize=7)
ax2.legend(loc='right', fontsize=7)

# (c) ISA sweep (parallel P68)
ax = axes[2]
ri = results_isa
x = [r['threshold'] for r in ri]
ax.plot(x, [r['pct_red_total'] for r in ri], 'b--', markersize=3, label='Total P68 reduction', alpha=0.5)
ax.plot(x, [r['pct_red_par'] for r in ri], 'b-o', markersize=4, label='Parallel P68 reduction')
ax.set_xlabel('ISA threshold (%)')
ax.set_ylabel('P68 error reduction (%)', color='b')
ax.tick_params(axis='y', labelcolor='b')
ax.set_title('(c) Far ISA (parallel <10°)')
ax2 = ax.twinx()
ax2.plot(x, [100-r['pct_par_retained'] for r in ri], 'r--s', markersize=3, label='Parallel removed (%)')
ax2.set_ylabel('Parallel samples removed (%)', color='r')
ax2.tick_params(axis='y', labelcolor='r')
ax.axvline(x=40, color='gray', linestyle=':', label='40% (current)')
ax.legend(loc='upper right', fontsize=7)
ax2.legend(loc='right', fontsize=7)

plt.tight_layout()
plt.savefig('mask_individual_sweeps.png', dpi=200, bbox_inches='tight')
plt.savefig('mask_individual_sweeps.pdf', bbox_inches='tight')
print("Saved: mask_individual_sweeps.png/pdf")

print("\nDone!")


In [ ]:
"""
Step 1: Evaluate each masking variable INDEPENDENTLY.
  - Flow angle threshold sweep
  - Far building height threshold sweep (within parallel)
  - Far ISA threshold sweep (within parallel)

For each, show: P68 of total retained, P68 of parallel retained,
error reduction, and sample counts.

Then combine the best thresholds.

Plots: y1 = P68 error (m), y2 = error reduction (%), embedded table

Run on server:
  python optimize_mask_individual_v2.py
"""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Config
DATA_PATH = "/project/pi_kandread_umass_edu/swot-urban/3_analysis_results/swot_urban_nearfar_v6.5_results.parquet"
COLS_NEEDED = ['flow_angle', 'far_building', 'far_isa', 'error_node', 'reach_id']

print("Loading data...")
df = pd.read_parquet(DATA_PATH, columns=COLS_NEEDED, engine='pyarrow')
df['abs_error'] = df['error_node'].abs()

# Flow angle -> continuous variable for sweeping
df['flow_angle_parallel'] = df['flow_angle'].apply(lambda a: min(a % 180, 180 - a % 180))

# Baseline
n_total = len(df)
p68_total = np.quantile(df['abs_error'].values, 0.68)

# ISA in percent
if df['far_isa'].max() <= 1.0:
    df['far_isa_pct'] = df['far_isa'] * 100
else:
    df['far_isa_pct'] = df['far_isa']

abs_err = df['abs_error'].values
flow_par = df['flow_angle_parallel'].values
far_bldg = df['far_building'].values
far_isa = df['far_isa_pct'].values

print(f"Total: {n_total:,}")
print(f"P68 total: {p68_total:.4f} m")
print()

# 1. Flow angle sweep
#    Remove all observations with flow_angle_parallel < threshold
print("1. FLOW ANGLE SWEEP: remove parallel obs (angle < threshold)")

angle_thresholds = np.arange(2, 32, 1)  # 2° to 31°
results_angle = []

for angle_t in angle_thresholds:
    mask = flow_par < angle_t
    n_removed = mask.sum()
    retained = abs_err[~mask]
    p68_ret = np.quantile(retained, 0.68)
    pct_red = (p68_total - p68_ret) / p68_total * 100
    pct_retained = len(retained) / n_total * 100
    results_angle.append({
        'threshold': angle_t, 'n_removed': n_removed,
        'p68_retained': p68_ret, 'pct_reduction': pct_red,
        'pct_retained': pct_retained
    })
    print(f"  <{angle_t:>3.0f}°: removed {n_removed:>7,d} ({100-pct_retained:>5.1f}%)  "
          f"P68={p68_ret:.4f}m  reduction={pct_red:>5.1f}%")

print()

# 2. BUILDING HEIGHT SWEEP (within parallel <10°)
#    Remove parallel obs with far_building > threshold
print("2. BUILDING HEIGHT SWEEP (parallel <10° only)")

is_par10 = flow_par < 10
n_par10 = is_par10.sum()
p68_par10 = np.quantile(abs_err[is_par10], 0.68)
print(f"   Parallel <10°: {n_par10:,} obs, P68={p68_par10:.4f}m")
print()

bldg_thresholds = np.arange(20, 2.5, -0.5)  # high -> low: remove worst first
results_bldg = []

for h_t in bldg_thresholds:
    mask = is_par10 & (far_bldg > h_t)
    n_removed = mask.sum()
    # P68 of retained TOTAL
    retained_total = abs_err[~mask]
    p68_ret_total = np.quantile(retained_total, 0.68)
    pct_red_total = (p68_total - p68_ret_total) / p68_total * 100
    # P68 of retained PARALLEL (what's left of parallel after mask)
    par_retained = is_par10 & (~mask)
    n_par_retained = par_retained.sum()
    if n_par_retained > 0:
        p68_ret_par = np.quantile(abs_err[par_retained], 0.68)
        pct_red_par = (p68_par10 - p68_ret_par) / p68_par10 * 100
    else:
        p68_ret_par = np.nan
        pct_red_par = np.nan
    results_bldg.append({
        'threshold': h_t, 'n_removed': n_removed,
        'p68_ret_total': p68_ret_total, 'pct_red_total': pct_red_total,
        'p68_ret_par': p68_ret_par, 'pct_red_par': pct_red_par,
        'pct_retained_total': len(retained_total) / n_total * 100,
        'pct_par_retained': n_par_retained / n_par10 * 100
    })
    print(f"  >{h_t:>5.1f}m: removed {n_removed:>6,d}  "
          f"P68_total={p68_ret_total:.4f}m ({pct_red_total:>+5.1f}%)  "
          f"P68_parallel={p68_ret_par:.4f}m ({pct_red_par:>+5.1f}%)  "
          f"par_retained={n_par_retained/n_par10*100:.1f}%")

print()

# 3. ISA SWEEP (within parallel <10°)
#    Remove parallel obs with far_isa > threshold
print("3. FAR ISA SWEEP (parallel <10° only)")

isa_thresholds = np.arange(63, 4, -2)  # high -> low: remove worst first
results_isa = []

for isa_t in isa_thresholds:
    mask = is_par10 & (far_isa > isa_t)
    n_removed = mask.sum()
    retained_total = abs_err[~mask]
    p68_ret_total = np.quantile(retained_total, 0.68)
    pct_red_total = (p68_total - p68_ret_total) / p68_total * 100
    par_retained = is_par10 & (~mask)
    n_par_retained = par_retained.sum()
    if n_par_retained > 0:
        p68_ret_par = np.quantile(abs_err[par_retained], 0.68)
        pct_red_par = (p68_par10 - p68_ret_par) / p68_par10 * 100
    else:
        p68_ret_par = np.nan
        pct_red_par = np.nan
    results_isa.append({
        'threshold': isa_t, 'n_removed': n_removed,
        'p68_ret_total': p68_ret_total, 'pct_red_total': pct_red_total,
        'p68_ret_par': p68_ret_par, 'pct_red_par': pct_red_par,
        'pct_retained_total': len(retained_total) / n_total * 100,
        'pct_par_retained': n_par_retained / n_par10 * 100
    })
    print(f"  >{isa_t:>4.0f}%: removed {n_removed:>6,d}  "
          f"P68_total={p68_ret_total:.4f}m ({pct_red_total:>+5.1f}%)  "
          f"P68_parallel={p68_ret_par:.4f}m ({pct_red_par:>+5.1f}%)  "
          f"par_retained={n_par_retained/n_par10*100:.1f}%")

print()

# 4. COMBINED: best thresholds from individual sweeps
print("4. COMBINED MASKS")

combos = [
    ("Angle<10°", flow_par < 10, None, None),
    ("Angle<10° + Bldg>8m", is_par10 & (far_bldg > 8), None, None),
    ("Angle<10° + ISA>30%", is_par10 & (far_isa > 30), None, None),
    ("Angle<10° + ISA>40%", is_par10 & (far_isa > 40), None, None),
    ("Angle<10° + (Bldg>8m OR ISA>30%)", is_par10 & ((far_bldg > 8) | (far_isa > 30)), None, None),
    ("Angle<10° + (Bldg>8m OR ISA>40%)", is_par10 & ((far_bldg > 8) | (far_isa > 40)), None, None),
    ("Angle<10° + (Bldg>8m AND ISA>20%)", is_par10 & ((far_bldg > 8) & (far_isa > 20)), None, None),
    ("Angle<10° + (Bldg>13m OR ISA>30%)", is_par10 & ((far_bldg > 13) | (far_isa > 30)), None, None),
    ("Remove ALL parallel", is_par10, None, None),
]

print(f"\n{'Mask':<45s} {'Removed':>8s} {'%Rem':>6s} {'P68_tot':>8s} {'Red%':>6s} "
      f"{'P68_par':>8s} {'RedP%':>6s} {'Par_ret%':>8s}")

for label, mask, _, _ in combos:
    n_removed = mask.sum()
    retained = abs_err[~mask]
    p68_ret = np.quantile(retained, 0.68)
    pct_red = (p68_total - p68_ret) / p68_total * 100

    par_retained = is_par10 & (~mask)
    n_par_ret = par_retained.sum()
    if n_par_ret > 0:
        p68_par_ret = np.quantile(abs_err[par_retained], 0.68)
        pct_red_par = (p68_par10 - p68_par_ret) / p68_par10 * 100
    else:
        p68_par_ret = np.nan
        pct_red_par = np.nan

    print(f"{label:<45s} {n_removed:>8,d} {n_removed/n_total*100:>5.1f}% "
          f"{p68_ret:>8.4f} {pct_red:>5.1f}% "
          f"{p68_par_ret:>8.4f} {pct_red_par:>5.1f}% "
          f"{n_par_ret/n_par10*100:>7.1f}%")

    # Extra: median and P68 of REMOVED observations
    if n_removed > 0:
        removed_err = abs_err[mask]
        print(f"{'':45s}   -> removed obs: median={np.median(removed_err):.4f}m  "
              f"P68={np.quantile(removed_err, 0.68):.4f}m  "
              f"P90={np.quantile(removed_err, 0.90):.4f}m")

# 5. REACH-LEVEL analysis (P68 per reach, then median across reaches)
#    This may show larger improvements
print("5. REACH-LEVEL P68 (median P68 across reaches)")

if 'reach_id' in df.columns:
    # Baseline: P68 per reach
    reach_p68_baseline = df.groupby('reach_id')['abs_error'].quantile(0.68)
    median_reach_p68_baseline = reach_p68_baseline.median()
    mean_reach_p68_baseline = reach_p68_baseline.mean()
    print(f"  Baseline: median reach P68 = {median_reach_p68_baseline:.4f}m, "
          f"mean reach P68 = {mean_reach_p68_baseline:.4f}m")
    print()

    for label, mask, _, _ in combos[:6]:
        df_ret = df[~mask]
        reach_p68_ret = df_ret.groupby('reach_id')['abs_error'].quantile(0.68)
        # Only compare reaches that still have data
        common = reach_p68_baseline.index.intersection(reach_p68_ret.index)
        med_ret = reach_p68_ret[common].median()
        mean_ret = reach_p68_ret[common].mean()
        pct_red_med = (median_reach_p68_baseline - med_ret) / median_reach_p68_baseline * 100
        pct_red_mean = (mean_reach_p68_baseline - mean_ret) / mean_reach_p68_baseline * 100
        n_removed_reaches = len(reach_p68_baseline) - len(reach_p68_ret)
        print(f"  {label:<45s}")
        print(f"    median reach P68: {med_ret:.4f}m ({pct_red_med:+.1f}%)  "
              f"mean reach P68: {mean_ret:.4f}m ({pct_red_mean:+.1f}%)  "
              f"reaches lost: {n_removed_reaches}")

# PLOTTING: 3 panels - individual sweeps
#   y1 (left)  = P68 error (m)        [blue]
#   y2 (right) = error reduction (%)   [red]
#   embedded table in bottom-right of each panel:
#     columns = threshold, P68(m), reduction(%), samples removed(%)
print("\nGenerating plots...")

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

def add_embedded_table(ax, col_labels, table_data):
    """Add a small table inside the axes at bottom-right."""
    n_rows = len(table_data)
    row_h = 0.038
    tbl_h = min(row_h * (n_rows + 1), 0.50)
    tbl = ax.table(
        cellText=table_data,
        colLabels=col_labels,
        loc='bottom right',
        cellLoc='center',
        bbox=[0.35, 0.02, 0.63, tbl_h]
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(6.5)
    # Header style
    for j in range(len(col_labels)):
        cell = tbl[0, j]
        cell.set_facecolor('#d4e6f1')
        cell.set_text_props(weight='bold', fontsize=6.5)
    # Alternating row colors
    for i in range(1, n_rows + 1):
        for j in range(len(col_labels)):
            cell = tbl[i, j]
            cell.set_facecolor('#f8f9fa' if i % 2 == 0 else 'white')
            cell.set_edgecolor('#cccccc')

# (a) Flow angle sweep
ax = axes[0]
ra = results_angle
xa = [r['threshold'] for r in ra]
y1a = [r['p68_retained'] for r in ra]
y2a = [r['pct_reduction'] for r in ra]
samp_rm_a = [100 - r['pct_retained'] for r in ra]

ax.plot(xa, y1a, 'b-o', markersize=4, label='P68 error (m)')
ax.set_xlabel('Flow angle threshold (°)', fontsize=10)
ax.set_ylabel('P68 error (m)', color='b', fontsize=10)
ax.tick_params(axis='y', labelcolor='b')
ax.set_title('(a) Flow Angle: remove obs with angle < threshold', fontsize=10)
ax.axhline(y=p68_total, color='blue', linestyle=':', alpha=0.4, linewidth=0.8)
ax.text(xa[0], p68_total, f'  baseline={p68_total:.3f}m', fontsize=7,
        color='blue', alpha=0.6, va='bottom')

ax2a = ax.twinx()
ax2a.plot(xa, y2a, 'r--s', markersize=3, label='Error reduction (%)')
ax2a.set_ylabel('Error reduction (%)', color='r', fontsize=10)
ax2a.tick_params(axis='y', labelcolor='r')
ax.axvline(x=10, color='gray', linestyle=':', alpha=0.5, label='10° (current)')
ax.legend(loc='upper left', fontsize=7)
ax2a.legend(loc='upper right', fontsize=7)

# Table for (a): subsample every 3 steps
step_a = 3
idx_a = list(range(0, len(ra), step_a))
if (len(ra) - 1) not in idx_a:
    idx_a.append(len(ra) - 1)
tbl_data_a = [[f"{xa[i]:.0f}°", f"{y1a[i]:.4f}", f"{y2a[i]:.1f}", f"{samp_rm_a[i]:.1f}"]
              for i in idx_a]
add_embedded_table(ax, ['Angle(°)', 'P68(m)', 'Red.(%)', 'Rm.(%)'], tbl_data_a)

# (b) Building height sweep (parallel zone)
ax = axes[1]
rb = results_bldg
xb = [r['threshold'] for r in rb]
y1b_tot = [r['p68_ret_total'] for r in rb]
y1b_par = [r['p68_ret_par'] for r in rb]
y2b_par = [r['pct_red_par'] for r in rb]
samp_rm_b = [100 - r['pct_par_retained'] for r in rb]

ax.plot(xb, y1b_tot, 'b--', markersize=3, alpha=0.4, label='Total P68 (m)')
ax.plot(xb, y1b_par, 'b-o', markersize=4, label='Parallel P68 (m)')
ax.set_xlabel('Building height threshold (m)', fontsize=10)
ax.set_ylabel('P68 error (m)', color='b', fontsize=10)
ax.tick_params(axis='y', labelcolor='b')
ax.set_title('(b) Building Height (parallel <10°): remove obs > threshold', fontsize=9)
ax.axhline(y=p68_par10, color='blue', linestyle=':', alpha=0.4, linewidth=0.8)
ax.text(xb[0], p68_par10, f'  par baseline={p68_par10:.3f}m', fontsize=7,
        color='blue', alpha=0.6, va='bottom')

ax2b = ax.twinx()
ax2b.plot(xb, y2b_par, 'r--s', markersize=3, label='Parallel reduction (%)')
ax2b.set_ylabel('Parallel error reduction (%)', color='r', fontsize=10)
ax2b.tick_params(axis='y', labelcolor='r')
ax.axvline(x=8, color='gray', linestyle=':', alpha=0.5, label='8m (current)')
ax.legend(loc='upper left', fontsize=7)
ax2b.legend(loc='upper right', fontsize=7)

# Table for (b): subsample every 4 steps (0.5m grid -> every 2m)
step_b = 4
idx_b = list(range(0, len(rb), step_b))
if (len(rb) - 1) not in idx_b:
    idx_b.append(len(rb) - 1)
tbl_data_b = [[f"{xb[i]:.1f}", f"{y1b_par[i]:.4f}", f"{y2b_par[i]:.1f}", f"{samp_rm_b[i]:.1f}"]
              for i in idx_b]
add_embedded_table(ax, ['Ht(m)', 'P68(m)', 'Red.(%)', 'Rm.(%)'], tbl_data_b)

# (c) ISA sweep (parallel zone)
ax = axes[2]
ri = results_isa
xc = [r['threshold'] for r in ri]
y1c_tot = [r['p68_ret_total'] for r in ri]
y1c_par = [r['p68_ret_par'] for r in ri]
y2c_par = [r['pct_red_par'] for r in ri]
samp_rm_c = [100 - r['pct_par_retained'] for r in ri]

ax.plot(xc, y1c_tot, 'b--', markersize=3, alpha=0.4, label='Total P68 (m)')
ax.plot(xc, y1c_par, 'b-o', markersize=4, label='Parallel P68 (m)')
ax.set_xlabel('ISA threshold (%)', fontsize=10)
ax.set_ylabel('P68 error (m)', color='b', fontsize=10)
ax.tick_params(axis='y', labelcolor='b')
ax.set_title('(c) Far ISA (parallel <10°): remove obs > threshold', fontsize=10)
ax.axhline(y=p68_par10, color='blue', linestyle=':', alpha=0.4, linewidth=0.8)
ax.text(xc[0], p68_par10, f'  par baseline={p68_par10:.3f}m', fontsize=7,
        color='blue', alpha=0.6, va='bottom')

ax2c = ax.twinx()
ax2c.plot(xc, y2c_par, 'r--s', markersize=3, label='Parallel reduction (%)')
ax2c.set_ylabel('Parallel error reduction (%)', color='r', fontsize=10)
ax2c.tick_params(axis='y', labelcolor='r')
ax.axvline(x=40, color='gray', linestyle=':', alpha=0.5, label='40% (current)')
ax.legend(loc='upper left', fontsize=7)
ax2c.legend(loc='upper right', fontsize=7)

# Table for (c): subsample every 3 steps (2% grid -> every 6%)
step_c = 3
idx_c = list(range(0, len(ri), step_c))
if (len(ri) - 1) not in idx_c:
    idx_c.append(len(ri) - 1)
tbl_data_c = [[f"{xc[i]:.0f}", f"{y1c_par[i]:.4f}", f"{y2c_par[i]:.1f}", f"{samp_rm_c[i]:.1f}"]
              for i in idx_c]
add_embedded_table(ax, ['ISA(%)', 'P68(m)', 'Red.(%)', 'Rm.(%)'], tbl_data_c)

plt.tight_layout()
plt.savefig('mask_individual_sweeps_v2.png', dpi=200, bbox_inches='tight')
plt.savefig('mask_individual_sweeps_v2.pdf', bbox_inches='tight')
print("Saved: mask_individual_sweeps_v2.png/pdf")

print("\nDone!")


In [ ]:
"""
Step 1: Evaluate each masking variable INDEPENDENTLY.
  - Flow angle threshold sweep
  - Far building height threshold sweep (within parallel)
  - Far ISA threshold sweep (within parallel)

For each, show: P68 of total retained, P68 of parallel retained,
error reduction, and sample counts.

Then combine the best thresholds.

Plots: y1 = P68 error (m), y2 = error reduction (%), embedded table

Run on server:
  python optimize_mask_individual_v2.py
"""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Config
DATA_PATH = "/project/pi_kandread_umass_edu/swot-urban/3_analysis_results/swot_urban_nearfar_v6.5_results.parquet"
COLS_NEEDED = ['flow_angle', 'far_building', 'far_isa', 'error_node', 'reach_id']

print("Loading data...")
df = pd.read_parquet(DATA_PATH, columns=COLS_NEEDED, engine='pyarrow')
df['abs_error'] = df['error_node'].abs()

# Flow angle -> continuous variable for sweeping
df['flow_angle_parallel'] = df['flow_angle'].apply(lambda a: min(a % 180, 180 - a % 180))

# Baseline
n_total = len(df)
p68_total = np.quantile(df['abs_error'].values, 0.68)

# ISA in percent
if df['far_isa'].max() <= 1.0:
    df['far_isa_pct'] = df['far_isa'] * 100
else:
    df['far_isa_pct'] = df['far_isa']

abs_err = df['abs_error'].values
flow_par = df['flow_angle_parallel'].values
far_bldg = df['far_building'].values
far_isa = df['far_isa_pct'].values

print(f"Total: {n_total:,}")
print(f"P68 total: {p68_total:.4f} m")
print()

# 1. Flow angle sweep
#    Remove all observations with flow_angle_parallel < threshold
print("1. FLOW ANGLE SWEEP: remove parallel obs (angle < threshold)")

angle_thresholds = np.arange(2, 32, 1)  # 2° to 31°
results_angle = []

for angle_t in angle_thresholds:
    mask = flow_par < angle_t
    n_removed = mask.sum()
    retained = abs_err[~mask]
    p68_ret = np.quantile(retained, 0.68)
    pct_red = (p68_total - p68_ret) / p68_total * 100
    pct_retained = len(retained) / n_total * 100
    results_angle.append({
        'threshold': angle_t, 'n_removed': n_removed,
        'p68_retained': p68_ret, 'pct_reduction': pct_red,
        'pct_retained': pct_retained
    })
    print(f"  <{angle_t:>3.0f}°: removed {n_removed:>7,d} ({100-pct_retained:>5.1f}%)  "
          f"P68={p68_ret:.4f}m  reduction={pct_red:>5.1f}%")

print()

# 2. BUILDING HEIGHT SWEEP (within parallel <10°)
#    Remove parallel obs with far_building > threshold
print("2. BUILDING HEIGHT SWEEP (parallel <10° only)")

is_par10 = flow_par < 10
n_par10 = is_par10.sum()
p68_par10 = np.quantile(abs_err[is_par10], 0.68)
print(f"   Parallel <10°: {n_par10:,} obs, P68={p68_par10:.4f}m")
print()

bldg_thresholds = np.arange(20, 2.5, -0.5)  # high -> low: remove worst first
results_bldg = []

for h_t in bldg_thresholds:
    mask = is_par10 & (far_bldg > h_t)
    n_removed = mask.sum()
    # P68 of retained TOTAL
    retained_total = abs_err[~mask]
    p68_ret_total = np.quantile(retained_total, 0.68)
    pct_red_total = (p68_total - p68_ret_total) / p68_total * 100
    # P68 of retained PARALLEL (what's left of parallel after mask)
    par_retained = is_par10 & (~mask)
    n_par_retained = par_retained.sum()
    if n_par_retained > 0:
        p68_ret_par = np.quantile(abs_err[par_retained], 0.68)
        pct_red_par = (p68_par10 - p68_ret_par) / p68_par10 * 100
    else:
        p68_ret_par = np.nan
        pct_red_par = np.nan
    results_bldg.append({
        'threshold': h_t, 'n_removed': n_removed,
        'p68_ret_total': p68_ret_total, 'pct_red_total': pct_red_total,
        'p68_ret_par': p68_ret_par, 'pct_red_par': pct_red_par,
        'pct_retained_total': len(retained_total) / n_total * 100,
        'pct_par_retained': n_par_retained / n_par10 * 100
    })
    print(f"  >{h_t:>5.1f}m: removed {n_removed:>6,d}  "
          f"P68_total={p68_ret_total:.4f}m ({pct_red_total:>+5.1f}%)  "
          f"P68_parallel={p68_ret_par:.4f}m ({pct_red_par:>+5.1f}%)  "
          f"par_retained={n_par_retained/n_par10*100:.1f}%")

print()

# 3. ISA SWEEP (within parallel <10°)
#    Remove parallel obs with far_isa > threshold
print("3. FAR ISA SWEEP (parallel <10° only)")

isa_thresholds = np.arange(63, 4, -2)  # high -> low: remove worst first
results_isa = []

for isa_t in isa_thresholds:
    mask = is_par10 & (far_isa > isa_t)
    n_removed = mask.sum()
    retained_total = abs_err[~mask]
    p68_ret_total = np.quantile(retained_total, 0.68)
    pct_red_total = (p68_total - p68_ret_total) / p68_total * 100
    par_retained = is_par10 & (~mask)
    n_par_retained = par_retained.sum()
    if n_par_retained > 0:
        p68_ret_par = np.quantile(abs_err[par_retained], 0.68)
        pct_red_par = (p68_par10 - p68_ret_par) / p68_par10 * 100
    else:
        p68_ret_par = np.nan
        pct_red_par = np.nan
    results_isa.append({
        'threshold': isa_t, 'n_removed': n_removed,
        'p68_ret_total': p68_ret_total, 'pct_red_total': pct_red_total,
        'p68_ret_par': p68_ret_par, 'pct_red_par': pct_red_par,
        'pct_retained_total': len(retained_total) / n_total * 100,
        'pct_par_retained': n_par_retained / n_par10 * 100
    })
    print(f"  >{isa_t:>4.0f}%: removed {n_removed:>6,d}  "
          f"P68_total={p68_ret_total:.4f}m ({pct_red_total:>+5.1f}%)  "
          f"P68_parallel={p68_ret_par:.4f}m ({pct_red_par:>+5.1f}%)  "
          f"par_retained={n_par_retained/n_par10*100:.1f}%")

print()

# 4. COMBINED: best thresholds from individual sweeps
print("4. COMBINED MASKS")

combos = [
    ("Angle<10°", flow_par < 10, None, None),
    ("Angle<10° + Bldg>8m", is_par10 & (far_bldg > 8), None, None),
    ("Angle<10° + ISA>30%", is_par10 & (far_isa > 30), None, None),
    ("Angle<10° + ISA>40%", is_par10 & (far_isa > 40), None, None),
    ("Angle<10° + (Bldg>8m OR ISA>30%)", is_par10 & ((far_bldg > 8) | (far_isa > 30)), None, None),
    ("Angle<10° + (Bldg>8m OR ISA>40%)", is_par10 & ((far_bldg > 8) | (far_isa > 40)), None, None),
    ("Angle<10° + (Bldg>8m AND ISA>20%)", is_par10 & ((far_bldg > 8) & (far_isa > 20)), None, None),
    ("Angle<10° + (Bldg>13m OR ISA>30%)", is_par10 & ((far_bldg > 13) | (far_isa > 30)), None, None),
    ("Remove ALL parallel", is_par10, None, None),
]

print(f"\n{'Mask':<45s} {'Removed':>8s} {'%Rem':>6s} {'P68_tot':>8s} {'Red%':>6s} "
      f"{'P68_par':>8s} {'RedP%':>6s} {'Par_ret%':>8s}")

for label, mask, _, _ in combos:
    n_removed = mask.sum()
    retained = abs_err[~mask]
    p68_ret = np.quantile(retained, 0.68)
    pct_red = (p68_total - p68_ret) / p68_total * 100

    par_retained = is_par10 & (~mask)
    n_par_ret = par_retained.sum()
    if n_par_ret > 0:
        p68_par_ret = np.quantile(abs_err[par_retained], 0.68)
        pct_red_par = (p68_par10 - p68_par_ret) / p68_par10 * 100
    else:
        p68_par_ret = np.nan
        pct_red_par = np.nan

    print(f"{label:<45s} {n_removed:>8,d} {n_removed/n_total*100:>5.1f}% "
          f"{p68_ret:>8.4f} {pct_red:>5.1f}% "
          f"{p68_par_ret:>8.4f} {pct_red_par:>5.1f}% "
          f"{n_par_ret/n_par10*100:>7.1f}%")

    # Extra: median and P68 of REMOVED observations
    if n_removed > 0:
        removed_err = abs_err[mask]
        print(f"{'':45s}   -> removed obs: median={np.median(removed_err):.4f}m  "
              f"P68={np.quantile(removed_err, 0.68):.4f}m  "
              f"P90={np.quantile(removed_err, 0.90):.4f}m")

# 5. REACH-LEVEL analysis (P68 per reach, then median across reaches)
#    This may show larger improvements
print("5. REACH-LEVEL P68 (median P68 across reaches)")

if 'reach_id' in df.columns:
    # Baseline: P68 per reach
    reach_p68_baseline = df.groupby('reach_id')['abs_error'].quantile(0.68)
    median_reach_p68_baseline = reach_p68_baseline.median()
    mean_reach_p68_baseline = reach_p68_baseline.mean()
    print(f"  Baseline: median reach P68 = {median_reach_p68_baseline:.4f}m, "
          f"mean reach P68 = {mean_reach_p68_baseline:.4f}m")
    print()

    for label, mask, _, _ in combos[:6]:
        df_ret = df[~mask]
        reach_p68_ret = df_ret.groupby('reach_id')['abs_error'].quantile(0.68)
        # Only compare reaches that still have data
        common = reach_p68_baseline.index.intersection(reach_p68_ret.index)
        med_ret = reach_p68_ret[common].median()
        mean_ret = reach_p68_ret[common].mean()
        pct_red_med = (median_reach_p68_baseline - med_ret) / median_reach_p68_baseline * 100
        pct_red_mean = (mean_reach_p68_baseline - mean_ret) / mean_reach_p68_baseline * 100
        n_removed_reaches = len(reach_p68_baseline) - len(reach_p68_ret)
        print(f"  {label:<45s}")
        print(f"    median reach P68: {med_ret:.4f}m ({pct_red_med:+.1f}%)  "
              f"mean reach P68: {mean_ret:.4f}m ({pct_red_mean:+.1f}%)  "
              f"reaches lost: {n_removed_reaches}")

# PLOTTING: 3 panels - individual sweeps
#   y1 (left)  = P68 error (m)        [blue]
#   y2 (right) = error reduction (%)   [red]
#   embedded table in bottom-right of each panel:
#     columns = threshold, P68(m), reduction(%), samples removed(%)
print("\nGenerating plots...")

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

def add_embedded_table(ax, col_labels, table_data):
    """Add a small table inside the axes at bottom-right."""
    n_rows = len(table_data)
    row_h = 0.038
    tbl_h = min(row_h * (n_rows + 1), 0.50)
    tbl = ax.table(
        cellText=table_data,
        colLabels=col_labels,
        loc='bottom right',
        cellLoc='center',
        bbox=[0.35, 0.02, 0.63, tbl_h]
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(6.5)
    # Header style
    for j in range(len(col_labels)):
        cell = tbl[0, j]
        cell.set_facecolor('#d4e6f1')
        cell.set_text_props(weight='bold', fontsize=6.5)
    # Alternating row colors
    for i in range(1, n_rows + 1):
        for j in range(len(col_labels)):
            cell = tbl[i, j]
            cell.set_facecolor('#f8f9fa' if i % 2 == 0 else 'white')
            cell.set_edgecolor('#cccccc')

# (a) Flow angle sweep
ax = axes[0]
ra = results_angle
xa = [r['threshold'] for r in ra]
y1a = [r['p68_retained'] for r in ra]
y2a = [r['pct_reduction'] for r in ra]
samp_rm_a = [100 - r['pct_retained'] for r in ra]

ax.plot(xa, y1a, 'b-o', markersize=4, label='P68 error (m)')
ax.set_xlabel('Flow angle threshold (°)', fontsize=10)
ax.set_ylabel('P68 error (m)', color='b', fontsize=10)
ax.tick_params(axis='y', labelcolor='b')
ax.set_title('(a) Flow Angle: remove obs with angle < threshold', fontsize=10)
ax.axhline(y=p68_total, color='blue', linestyle=':', alpha=0.4, linewidth=0.8)
ax.text(xa[0], p68_total, f'  baseline={p68_total:.3f}m', fontsize=7,
        color='blue', alpha=0.6, va='bottom')

ax2a = ax.twinx()
ax2a.plot(xa, y2a, 'r--s', markersize=3, label='Error reduction (%)')
ax2a.set_ylabel('Error reduction (%)', color='r', fontsize=10)
ax2a.tick_params(axis='y', labelcolor='r')
ax.axvline(x=10, color='gray', linestyle=':', alpha=0.5, label='10° (current)')
ax.legend(loc='upper left', fontsize=7)
ax2a.legend(loc='upper right', fontsize=7)

# Table for (a): subsample every 3 steps
step_a = 3
idx_a = list(range(0, len(ra), step_a))
if (len(ra) - 1) not in idx_a:
    idx_a.append(len(ra) - 1)
tbl_data_a = [[f"{xa[i]:.0f}°", f"{y1a[i]:.4f}", f"{y2a[i]:.1f}", f"{samp_rm_a[i]:.1f}"]
              for i in idx_a]
add_embedded_table(ax, ['Angle(°)', 'P68(m)', 'Red.(%)', 'Rm.(%)'], tbl_data_a)

# (b) Building height sweep (parallel zone)
ax = axes[1]
rb = results_bldg
xb = [r['threshold'] for r in rb]
y1b_tot = [r['p68_ret_total'] for r in rb]
y1b_par = [r['p68_ret_par'] for r in rb]
y2b_par = [r['pct_red_par'] for r in rb]
samp_rm_b = [100 - r['pct_par_retained'] for r in rb]

ax.plot(xb, y1b_tot, 'b--', markersize=3, alpha=0.4, label='Total P68 (m)')
ax.plot(xb, y1b_par, 'b-o', markersize=4, label='Parallel P68 (m)')
ax.set_xlabel('Building height threshold (m)', fontsize=10)
ax.set_ylabel('P68 error (m)', color='b', fontsize=10)
ax.tick_params(axis='y', labelcolor='b')
ax.set_title('(b) Building Height (parallel <10°): remove obs > threshold', fontsize=9)
ax.axhline(y=p68_par10, color='blue', linestyle=':', alpha=0.4, linewidth=0.8)
ax.text(xb[0], p68_par10, f'  par baseline={p68_par10:.3f}m', fontsize=7,
        color='blue', alpha=0.6, va='bottom')

ax2b = ax.twinx()
ax2b.plot(xb, y2b_par, 'r--s', markersize=3, label='Parallel reduction (%)')
ax2b.set_ylabel('Parallel error reduction (%)', color='r', fontsize=10)
ax2b.tick_params(axis='y', labelcolor='r')
ax.axvline(x=8, color='gray', linestyle=':', alpha=0.5, label='8m (current)')
ax.legend(loc='upper left', fontsize=7)
ax2b.legend(loc='upper right', fontsize=7)

# Table for (b): subsample every 4 steps (0.5m grid -> every 2m)
step_b = 4
idx_b = list(range(0, len(rb), step_b))
if (len(rb) - 1) not in idx_b:
    idx_b.append(len(rb) - 1)
tbl_data_b = [[f"{xb[i]:.1f}", f"{y1b_par[i]:.4f}", f"{y2b_par[i]:.1f}", f"{samp_rm_b[i]:.1f}"]
              for i in idx_b]
add_embedded_table(ax, ['Ht(m)', 'P68(m)', 'Red.(%)', 'Rm.(%)'], tbl_data_b)

# (c) ISA sweep (parallel zone)
ax = axes[2]
ri = results_isa
xc = [r['threshold'] for r in ri]
y1c_tot = [r['p68_ret_total'] for r in ri]
y1c_par = [r['p68_ret_par'] for r in ri]
y2c_par = [r['pct_red_par'] for r in ri]
samp_rm_c = [100 - r['pct_par_retained'] for r in ri]

ax.plot(xc, y1c_tot, 'b--', markersize=3, alpha=0.4, label='Total P68 (m)')
ax.plot(xc, y1c_par, 'b-o', markersize=4, label='Parallel P68 (m)')
ax.set_xlabel('ISA threshold (%)', fontsize=10)
ax.set_ylabel('P68 error (m)', color='b', fontsize=10)
ax.tick_params(axis='y', labelcolor='b')
ax.set_title('(c) Far ISA (parallel <10°): remove obs > threshold', fontsize=10)
ax.axhline(y=p68_par10, color='blue', linestyle=':', alpha=0.4, linewidth=0.8)
ax.text(xc[0], p68_par10, f'  par baseline={p68_par10:.3f}m', fontsize=7,
        color='blue', alpha=0.6, va='bottom')

ax2c = ax.twinx()
ax2c.plot(xc, y2c_par, 'r--s', markersize=3, label='Parallel reduction (%)')
ax2c.set_ylabel('Parallel error reduction (%)', color='r', fontsize=10)
ax2c.tick_params(axis='y', labelcolor='r')
ax.axvline(x=40, color='gray', linestyle=':', alpha=0.5, label='40% (current)')
ax.legend(loc='upper left', fontsize=7)
ax2c.legend(loc='upper right', fontsize=7)

# Table for (c): subsample every 3 steps (2% grid -> every 6%)
step_c = 3
idx_c = list(range(0, len(ri), step_c))
if (len(ri) - 1) not in idx_c:
    idx_c.append(len(ri) - 1)
tbl_data_c = [[f"{xc[i]:.0f}", f"{y1c_par[i]:.4f}", f"{y2c_par[i]:.1f}", f"{samp_rm_c[i]:.1f}"]
              for i in idx_c]
add_embedded_table(ax, ['ISA(%)', 'P68(m)', 'Red.(%)', 'Rm.(%)'], tbl_data_c)

plt.tight_layout()
plt.savefig('mask_individual_sweeps_v2.png', dpi=200, bbox_inches='tight')
plt.savefig('mask_individual_sweeps_v2.pdf', bbox_inches='tight')
print("Saved: mask_individual_sweeps_v2.png/pdf")

print("\nDone!")


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr

# 알려주신 구역 분류 함수 (결측치 방어 코드만 한 줄 추가)
def get_flow_angle_zone(flow_angle):
    if pd.isna(flow_angle):
        return 'Unknown'
    fa = flow_angle % 360
    if (fa >= 350 or fa <= 10) or (170 <= fa <= 190):
        return 'Parallel'
    elif (80 <= fa <= 100) or (260 <= fa <= 280):
        return 'Perpendicular'
    else:
        return 'Other'

def extract_near_far_statistics_with_angle(final_df):
    df = final_df.copy()
    
    # Node-level absolute error 계산
    df['abs_error_node'] = df['error_node'].abs()
    
    # 1. Flow Angle을 기반으로 구역 할당 및 Parallel 필터링
    df['angle_zone'] = df['flow_angle'].apply(get_flow_angle_zone)
    df_parallel = df[df['angle_zone'] == 'Parallel'].copy()
    
    print(f"Total Parallel nodes (based on flow_angle): {len(df_parallel):,}")
    if len(df_parallel) == 0:
        print("경고: 'Parallel' 조건을 만족하는 데이터가 없습니다.")
        return
        
    # 2. ISA 컬럼 설정 및 단위 맞추기 (0~1 비율이라면 *100 유지, 이미 0~100이면 *100 제거)
    col_isa_far = 'far_isa'
    col_isa_near = 'near_isa'
    
    df_parallel['isa_far_pct'] = df_parallel[col_isa_far] * 100
    df_parallel['isa_near_pct'] = df_parallel[col_isa_near] * 100

    # 커스텀 Bins
    custom_bins = [0, 5, 10, 20, 30, 40, 50, 60, 70, 80, 100]
    custom_labels = ['0-5', '5-10', '10-20', '20-30', '30-40', '40-50', '50-60', '60-70', '70-80', '80-100']

    # 1. Far-nadir
    print("1. FAR-NADIR ISA vs ERROR (Parallel Reaches Only)")
    
    df_far = df_parallel.dropna(subset=['isa_far_pct', 'abs_error_node'])
    df_far['isa_bin'] = pd.cut(df_far['isa_far_pct'], bins=custom_bins, labels=custom_labels, include_lowest=True, right=True)

    stats_far = df_far.groupby('isa_bin')['abs_error_node'].agg(
        N='count',
        P68=lambda x: np.percentile(x, 68) if len(x) > 0 else np.nan
    ).reset_index()
    stats_far['P68'] = stats_far['P68'].round(3)
    print(stats_far.to_string(index=False))

    rho_far, p_far = spearmanr(df_far['isa_far_pct'], df_far['abs_error_node'])
    p_far_str = "< 10^{-12}" if p_far < 1e-12 else f"= {p_far:.3e}"
    print(f"\nFar-nadir Spearman correlation: rho = {rho_far:.3f} (p {p_far_str})")

    try:
        p68_far_low = stats_far.loc[stats_far['isa_bin'] == '0-5', 'P68'].values[0]
        p68_far_high = stats_far.loc[stats_far['isa_bin'] == '80-100', 'P68'].values[0]
        if p68_far_low > 0 and not np.isnan(p68_far_high):
            ratio_far = p68_far_high / p68_far_low
            print(f"Far-nadir 68th-percentile ratio (80-100% / 0-5%): {ratio_far:.1f}")
    except IndexError:
        print("Far-nadir 80-100% 구간에 데이터가 없습니다.")

    # 2. Near-nadir
    print("2. NEAR-NADIR ISA vs ERROR (Parallel Reaches Only)")
    
    df_near = df_parallel.dropna(subset=['isa_near_pct', 'abs_error_node'])
    df_near['isa_bin'] = pd.cut(df_near['isa_near_pct'], bins=custom_bins, labels=custom_labels, include_lowest=True, right=True)

    stats_near = df_near.groupby('isa_bin')['abs_error_node'].agg(
        N='count',
        P68=lambda x: np.percentile(x, 68) if len(x) > 0 else np.nan
    ).reset_index()
    stats_near['P68'] = stats_near['P68'].round(3)
    print(stats_near.to_string(index=False))

    rho_near, p_near = spearmanr(df_near['isa_near_pct'], df_near['abs_error_node'])
    p_near_str = "< 0.001" if p_near < 0.001 else f"= {p_near:.3f}"
    print(f"\nNear-nadir Spearman correlation: rho = {rho_near:.3f} (p {p_near_str})")

    try:
        p68_near_low = stats_near.loc[stats_near['isa_bin'] == '0-5', 'P68'].values[0]
        p68_near_high = stats_near.loc[stats_near['isa_bin'] == '80-100', 'P68'].values[0]
        if p68_near_low > 0 and not np.isnan(p68_near_high):
            ratio_near = p68_near_high / p68_near_low
            print(f"Near-nadir 68th-percentile ratio (80-100% / 0-5%): {ratio_near:.1f}")
    except IndexError:
        print("Near-nadir 80-100% 구간에 데이터가 없습니다.")

# 함수 실행 
extract_near_far_statistics_with_angle(final_df)

In [ ]:
"""
Compute all statistics needed for Section 4.4.2 rewrite.
Uses the same data structure and bin definitions as the Figure 8 plotting code.
"""
import pandas as pd
import numpy as np
from scipy import stats

# Load data
DATA_PATH = "/project/pi_kandread_umass_edu/swot-urban/3_analysis_results/swot_urban_nearfar_v6.5_results.parquet"
df = final_df.copy()
df['abs_error_node'] = df['error_node'].abs()
THRESHOLD = 0.1832

# Flow angle zone classification
def classify_flow_angle(angle):
    a = angle % 180
    if a <= 10 or a >= 170:
        return 'Parallel'
    elif 80 <= a <= 100:
        return 'Perpendicular'
    else:
        return 'Other'

df['flow_zone'] = df['flow_angle'].apply(classify_flow_angle)

# ISA bin function (same as plotting code)
def get_isa_bin(isa):
    if isa < 5:
        return '0-5'
    elif isa < 10:
        return '5-10'
    elif isa < 20:
        return '10-20'
    elif isa < 30:
        return '20-30'
    elif isa < 45:
        return '30-45'
    elif isa < 60:
        return '45-60'
    else:
        return '60+'

ISA_ORDER = ['0-5', '5-10', '10-20', '20-30', '30-45', '45-60', '60+']

# Building height bin function (same as plotting code)
def get_building_bins_dynamic(series, interval=2, min_samples=500, start_from=5):
    bins = [0, start_from]
    current = start_from
    while current < series.max():
        bins.append(current + interval)
        current += interval

    labels = []
    for i in range(len(bins) - 1):
        if i == len(bins) - 2:
            labels.append(f'{bins[i]}+')
        else:
            labels.append(f'{bins[i]}-{bins[i+1]}')

    result = pd.cut(series, bins=bins + [np.inf] if bins[-1] < series.max() else bins,
                     labels=labels[:len(bins)-1] if bins[-1] >= series.max() else labels,
                     right=False)

    # Merge small bins from right
    value_counts = result.value_counts()
    merge_from = None
    for label in reversed(labels):
        if label in value_counts.index and value_counts[label] < min_samples:
            merge_from = label
        else:
            break

    if merge_from is not None:
        merge_idx = labels.index(merge_from)
        merge_label = f'{labels[merge_idx].split("-")[0]}+'
        # Convert to string first, then replace (avoids duplicate category error)
        result = result.astype(str)
        for label in labels[merge_idx:]:
            result = result.replace(label, merge_label)
    else:
        result = result.astype(str)

    return result

# Filter to PARALLEL zone (same as Figure 8)
df_par = df[df['flow_zone'] == 'Parallel'].copy()
print(f"Total samples: {len(df)}")
print(f"Parallel zone samples: {len(df_par)}")
print()

# Apply bins
df_par['isa_bin'] = df_par['far_isa'].apply(get_isa_bin)
df_par['bldg_bin'] = get_building_bins_dynamic(df_par['far_building'])

# Get the actual building height bin order
bldg_bins_unique = df_par['bldg_bin'].unique()
def sort_key(x):
    return float(x.replace('+', '').split('-')[0])
BLDG_ORDER = sorted(bldg_bins_unique, key=sort_key)

print(f"Building height bins: {BLDG_ORDER}")
print(f"ISA bins: {ISA_ORDER}")
print()

# 1. BUILDING HEIGHT: Overall 68th percentile per bin
print("1. BUILDING HEIGHT — 68th percentile error per bin (Parallel zone)")
for b in BLDG_ORDER:
    subset = df_par[df_par['bldg_bin'] == b]['abs_error_node']
    p68 = subset.quantile(0.68)
    med = subset.median()
    p25 = subset.quantile(0.25)
    p75 = subset.quantile(0.75)
    exceed = (subset > THRESHOLD).mean() * 100
    print(f"  {b:>8s}:  n={len(subset):>6d}  median={med:.3f}m  68th={p68:.3f}m  "
          f"IQR=[{p25:.3f}, {p75:.3f}]  exceed>{THRESHOLD}m: {exceed:.1f}%")
print()

# 2. ISA: Overall 68th percentile per bin (Parallel zone)
print("2. ISA — 68th percentile error per bin (Parallel zone)")
for isa in ISA_ORDER:
    subset = df_par[df_par['isa_bin'] == isa]['abs_error_node']
    if len(subset) == 0:
        continue
    p68 = subset.quantile(0.68)
    med = subset.median()
    p25 = subset.quantile(0.25)
    p75 = subset.quantile(0.75)
    exceed = (subset > THRESHOLD).mean() * 100
    print(f"  {isa:>8s}:  n={len(subset):>6d}  median={med:.3f}m  68th={p68:.3f}m  "
          f"IQR=[{p25:.3f}, {p75:.3f}]  exceed>{THRESHOLD}m: {exceed:.1f}%")
print()

# 3. CROSS-TAB: Building height x ISA, 68th percentile
#    (This is the key Figure 8 top panel data)
print("3. CROSS-TAB: Building Height × ISA — 68th percentile (Parallel)")
print(f"{'Bldg\\ISA':>10s}", end="")
for isa in ISA_ORDER:
    print(f"  {isa:>8s}", end="")
print("      n_total")

for b in BLDG_ORDER:
    print(f"{b:>10s}", end="")
    row_n = 0
    for isa in ISA_ORDER:
        subset = df_par[(df_par['bldg_bin'] == b) & (df_par['isa_bin'] == isa)]['abs_error_node']
        row_n += len(subset)
        if len(subset) >= 30:
            p68 = subset.quantile(0.68)
            print(f"  {p68:>7.3f}m", end="")
        else:
            print(f"  {'n<30':>8s}", end="")
    print(f"  {row_n:>8d}")
print()

# Same table but with sample sizes
print("3b. CROSS-TAB: Building Height × ISA — SAMPLE SIZES")
print(f"{'Bldg\\ISA':>10s}", end="")
for isa in ISA_ORDER:
    print(f"  {isa:>8s}", end="")
print()

for b in BLDG_ORDER:
    print(f"{b:>10s}", end="")
    for isa in ISA_ORDER:
        subset = df_par[(df_par['bldg_bin'] == b) & (df_par['isa_bin'] == isa)]
        print(f"  {len(subset):>8d}", end="")
    print()
print()

# 4. CROSS-TAB: ISA x Building height, 68th percentile
#    (This is the key Figure 8 bottom panel data)
print("4. CROSS-TAB: ISA × Building Height — 68th percentile (Parallel)")
print(f"{'ISA\\Bldg':>10s}", end="")
for b in BLDG_ORDER:
    print(f"  {b:>8s}", end="")
print("      n_total")

for isa in ISA_ORDER:
    print(f"{isa:>10s}", end="")
    row_n = 0
    for b in BLDG_ORDER:
        subset = df_par[(df_par['isa_bin'] == isa) & (df_par['bldg_bin'] == b)]['abs_error_node']
        row_n += len(subset)
        if len(subset) >= 30:
            p68 = subset.quantile(0.68)
            print(f"  {p68:>7.3f}m", end="")
        else:
            print(f"  {'n<30':>8s}", end="")
    print(f"  {row_n:>8d}")
print()

# 5. Non-monotonic analysis
print("5. NON-MONOTONIC BUILDING HEIGHT ANALYSIS")

# Find peak bin and reversal
bldg_p68 = {}
for b in BLDG_ORDER:
    subset = df_par[df_par['bldg_bin'] == b]['abs_error_node']
    bldg_p68[b] = subset.quantile(0.68)

peak_bin = max(bldg_p68, key=bldg_p68.get)
peak_val = bldg_p68[peak_bin]
print(f"  Peak error bin: {peak_bin} -> 68th percentile = {peak_val:.3f}m")
print()

# Compare consecutive bins for reversal
for i in range(len(BLDG_ORDER) - 1):
    b1, b2 = BLDG_ORDER[i], BLDG_ORDER[i + 1]
    v1, v2 = bldg_p68[b1], bldg_p68[b2]
    direction = "↑" if v2 > v1 else "↓" if v2 < v1 else "="
    pct_change = ((v2 - v1) / v1) * 100
    print(f"  {b1} -> {b2}: {v1:.3f}m -> {v2:.3f}m  {direction}  ({pct_change:+.1f}%)")

print()

# KS test between peak bin and next higher bin
peak_idx = BLDG_ORDER.index(peak_bin)
if peak_idx < len(BLDG_ORDER) - 1:
    next_bin = BLDG_ORDER[peak_idx + 1]
    s1 = df_par[df_par['bldg_bin'] == peak_bin]['abs_error_node']
    s2 = df_par[df_par['bldg_bin'] == next_bin]['abs_error_node']
    ks_stat, ks_p = stats.ks_2samp(s1, s2)
    print(f"  KS test ({peak_bin} vs {next_bin}): statistic={ks_stat:.4f}, p-value={ks_p:.2e}")

# Also check if last bin drops below peak
last_bin = BLDG_ORDER[-1]
if last_bin != peak_bin:
    s1 = df_par[df_par['bldg_bin'] == peak_bin]['abs_error_node']
    s2 = df_par[df_par['bldg_bin'] == last_bin]['abs_error_node']
    ks_stat, ks_p = stats.ks_2samp(s1, s2)
    print(f"  KS test ({peak_bin} vs {last_bin}): statistic={ks_stat:.4f}, p-value={ks_p:.2e}")
print()

# 6. Multiplicative interaction
#    Within specific building height bins, how much does ISA matter?
print("6. MULTIPLICATIVE INTERACTION: ISA effect within each height bin")
for b in BLDG_ORDER:
    isa_vals = {}
    for isa in ISA_ORDER:
        subset = df_par[(df_par['bldg_bin'] == b) & (df_par['isa_bin'] == isa)]['abs_error_node']
        if len(subset) >= 30:
            isa_vals[isa] = subset.quantile(0.68)

    if len(isa_vals) >= 2:
        min_isa = min(isa_vals, key=isa_vals.get)
        max_isa = max(isa_vals, key=isa_vals.get)
        ratio = isa_vals[max_isa] / isa_vals[min_isa] if isa_vals[min_isa] > 0 else float('inf')
        print(f"  {b:>8s}: lowest ISA={min_isa} ({isa_vals[min_isa]:.3f}m) -> "
              f"highest ISA={max_isa} ({isa_vals[max_isa]:.3f}m)  ratio={ratio:.2f}x")
    else:
        print(f"  {b:>8s}: insufficient data")
print()

# 7. Spearman correlations
print("7. SPEARMAN CORRELATIONS (Parallel zone)")

# Overall: building height vs error
rho_bldg, p_bldg = stats.spearmanr(df_par['far_building'], df_par['abs_error_node'])
print(f"  Building height vs error (all): ρ={rho_bldg:.4f}, p={p_bldg:.2e}")

# Overall: ISA vs error
rho_isa, p_isa = stats.spearmanr(df_par['far_isa'], df_par['abs_error_node'])
print(f"  ISA vs error (all):             ρ={rho_isa:.4f}, p={p_isa:.2e}")

# Partial: ISA vs error conditioned on height bins
print()
print("  ISA vs error, conditioned on building height bin:")
for b in BLDG_ORDER:
    subset = df_par[df_par['bldg_bin'] == b]
    if len(subset) >= 30:
        rho, p = stats.spearmanr(subset['far_isa'], subset['abs_error_node'])
        print(f"    {b:>8s}: ρ={rho:.4f}, p={p:.2e}, n={len(subset)}")

# Partial: Building vs error conditioned on ISA bins
print()
print("  Building height vs error, conditioned on ISA bin:")
for isa in ISA_ORDER:
    subset = df_par[df_par['isa_bin'] == isa]
    if len(subset) >= 30:
        rho, p = stats.spearmanr(subset['far_building'], subset['abs_error_node'])
        print(f"    {isa:>8s}: ρ={rho:.4f}, p={p:.2e}, n={len(subset)}")

print()

# 8. THRESHOLD EXCEEDANCE RATES (cross-tab)
print("8. THRESHOLD EXCEEDANCE RATES (>{:.4f}m) — Building × ISA".format(THRESHOLD))
print(f"{'Bldg\\ISA':>10s}", end="")
for isa in ISA_ORDER:
    print(f"  {isa:>8s}", end="")
print()

for b in BLDG_ORDER:
    print(f"{b:>10s}", end="")
    for isa in ISA_ORDER:
        subset = df_par[(df_par['bldg_bin'] == b) & (df_par['isa_bin'] == isa)]['abs_error_node']
        if len(subset) >= 30:
            exceed = (subset > THRESHOLD).mean() * 100
            print(f"  {exceed:>7.1f}%", end="")
        else:
            print(f"  {'n<30':>8s}", end="")
    print()
print()

# 9. SUMMARY: Key numbers for the narrative
print("9. KEY NARRATIVE NUMBERS")

# Lowest height bin error
lowest_b = BLDG_ORDER[0]
lowest_err = bldg_p68[lowest_b]
print(f"  Lowest building bin ({lowest_b}): 68th = {lowest_err:.3f}m")

# Peak bin
print(f"  Peak building bin ({peak_bin}): 68th = {peak_val:.3f}m")
print(f"  Peak / Lowest ratio: {peak_val/lowest_err:.2f}x")

# Reversal
if peak_idx < len(BLDG_ORDER) - 1:
    last_val = bldg_p68[BLDG_ORDER[-1]]
    print(f"  Last building bin ({BLDG_ORDER[-1]}): 68th = {last_val:.3f}m")
    print(f"  Reversal from peak: {((peak_val - last_val)/peak_val)*100:.1f}% reduction")

# ISA range in peak building bin
peak_isa_vals = {}
for isa in ISA_ORDER:
    subset = df_par[(df_par['bldg_bin'] == peak_bin) & (df_par['isa_bin'] == isa)]['abs_error_node']
    if len(subset) >= 30:
        peak_isa_vals[isa] = subset.quantile(0.68)

if peak_isa_vals:
    min_isa_k = min(peak_isa_vals, key=peak_isa_vals.get)
    max_isa_k = max(peak_isa_vals, key=peak_isa_vals.get)
    print(f"  In peak bin ({peak_bin}), ISA range: {peak_isa_vals[min_isa_k]:.3f}m ({min_isa_k}) "
          f"-> {peak_isa_vals[max_isa_k]:.3f}m ({max_isa_k}), "
          f"ratio={peak_isa_vals[max_isa_k]/peak_isa_vals[min_isa_k]:.2f}x")

print()
print("DONE. Copy everything above and paste into the chat.")


In [ ]:
print(final_df.columns.tolist())

In [ ]:
def get_sample_geometry_category_vectorized(near_val, far_val, near_ratio, far_ratio, min_ratio=0.5):
    has_near = (near_val > 0) & (near_ratio >= min_ratio)
    has_far = (far_val > 0) & (far_ratio >= min_ratio)

    conditions = [
        has_near & has_far,  
        has_near,           
        has_far             
    ]

    choices = [
        'Both',   
        'Near',   
        'Far'     
    ]

    return np.select(conditions, choices, default='Neither')

In [ ]:
final_df['50_geometry_category'] = get_sample_geometry_category_vectorized(
    final_df['near_isa'], 
    final_df['far_isa'],
    final_df['near_ratio'], 
    final_df['far_ratio'], 
    min_ratio=0.50
)

In [ ]:
final_df['bdg_50_geometry_category'] = get_sample_geometry_category_vectorized(
    final_df['near_isa'], 
    final_df['far_isa'],
    final_df['near_ratio_building'], 
    final_df['far_ratio_building'], 
    min_ratio=0.50
)

In [ ]:
final_df['0_geometry_category'] = get_sample_geometry_category_vectorized(
    final_df['near_isa'], 
    final_df['far_isa'],
    final_df['near_ratio'], 
    final_df['far_ratio'], 
    min_ratio=0
)

#### whole for angle with box

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap
from pathlib import Path

plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

OUTPUT_DIR = Path("figures")
OUTPUT_DIR.mkdir(exist_ok=True)


def get_parallel_bin(flow_angle, bin_width=2, n_bins=15):
    """Parallel 각도(0°, 180°) 기준 deviation에 따른 bin 할당."""
    fa = flow_angle % 360
    dev_from_0 = min(fa, 360 - fa)
    dev_from_180 = abs(fa - 180)
    deviation = min(dev_from_0, dev_from_180)
    
    for i in range(n_bins):
        lower = i * bin_width
        upper = (i + 1) * bin_width
        if lower <= deviation < upper:
            return i + 1
    return None


def plot_boxplot_panel(ax, data, y_col, geom_cat, geom_col='50_geometry_category',
                       threshold=0.1832, bin_width=2, n_bins=15, ylim=(0, 2)):
    """Single panel boxplot with angle bins"""
    ax.set_facecolor('white')
    
    # Colormap - Oranges
    cmap = get_cmap('Oranges', n_bins + 4)
    
    df_cat = data[data[geom_col] == geom_cat].copy()
    df_cat['parallel_bin'] = df_cat['flow_angle'].apply(
        lambda x: get_parallel_bin(x, bin_width, n_bins)
    )
    df_cat = df_cat.dropna(subset=['parallel_bin'])
    df_cat['parallel_bin'] = df_cat['parallel_bin'].astype(int)
    
    # Prepare data for boxplot
    box_data = []
    positions = []
    colors = []
    p68_values = []
    
    for bin_num in range(1, n_bins + 1):
        bin_data = df_cat[df_cat['parallel_bin'] == bin_num][y_col].dropna()
        
        if len(bin_data) >= 10:
            box_data.append(bin_data.values)
            positions.append(bin_num)
            colors.append(cmap(bin_num + 2))
            p68_values.append((bin_num, np.percentile(bin_data, 68)))
    
    if not box_data:
        ax.text(0.5, 0.5, 'No data', transform=ax.transAxes, ha='center', va='center')
        return
    
    # Draw boxplots
    bp = ax.boxplot(box_data, positions=positions, widths=0.7, patch_artist=True,
                    showfliers=False, whis=[5, 95])
    
    # Color each box
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_edgecolor('black')
        patch.set_linewidth(1)
    
    for element in ['whiskers', 'caps']:
        for item in bp[element]:
            item.set_color('black')
            item.set_linewidth(1)
    
    for median in bp['medians']:
        median.set_color('black')
        median.set_linewidth(1.5)
    
    # 68th percentile markers (black X)
    for bin_num, p68 in p68_values:
        ax.scatter([bin_num], [p68], marker='x', color='black', s=80, zorder=5, linewidths=2)
    
    # Threshold line
    ax.axhline(y=threshold, color='black', linestyle='--', linewidth=2, alpha=0.8)
    ax.text(n_bins + 0.5, threshold, f'T={threshold}m', fontsize=11, fontweight='bold',
            va='center', ha='left')
    
    # X-axis labels
    bin_labels = [f'{i*bin_width}-{(i+1)*bin_width}°' for i in range(n_bins)]
    ax.set_xticks(range(1, n_bins + 1))
    ax.set_xticklabels(bin_labels, rotation=45, ha='right', fontsize=9)
    
    ax.set_xlim(0.5, n_bins + 0.5)
    ax.set_ylim(ylim)
    ax.set_xlabel('Deviation from Parallel', fontsize=12)
    ax.set_ylabel(r'$|Relative\ WSE\ Error|$ (m)', fontsize=12)
    ax.set_title(f'{geom_cat} (n={len(df_cat):,})', fontsize=14, fontweight='bold', pad=10)
    ax.tick_params(labelsize=10)
    ax.grid(True, alpha=0.3, axis='y')
    
    for spine in ax.spines.values():
        spine.set_color('black')


def plot_boxplot_by_angle_bins(df, y_col, geom_col='50_geometry_category', threshold=0.1832,
                                bin_width=2, n_bins=15, figsize=(18, 12), ylim=(0, 2),
                                save=True, show=True):
    """4-panel boxplot (Neither, Near, Far, Both) with angle bins."""
    df_plot = df.dropna(subset=[y_col, 'flow_angle', geom_col]).copy()
    if len(df_plot) == 0:
        print("No data available!")
        return None
    
    fig, axes = plt.subplots(2, 2, figsize=figsize, facecolor='white')
    axes = axes.flatten()
    
    geometry_categories = ['Neither', 'Near', 'Far', 'Both']
    
    for idx, cat in enumerate(geometry_categories):
        plot_boxplot_panel(axes[idx], df_plot, y_col, cat, geom_col,
                           threshold, bin_width, n_bins, ylim)
    
    fig.suptitle(r'$|Relative\ WSE\ Error|$ by Deviation from Parallel Angle',
                 fontsize=18, fontweight='bold', y=0.98)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    
    if save:
        filename = f"figure_boxplot_parallel_bins_{y_col}_{geom_col}.png"
        fig.savefig(OUTPUT_DIR / filename, dpi=300, facecolor='white', bbox_inches='tight')
        filename_pdf = f"figure_boxplot_parallel_bins_{y_col}_{geom_col}.pdf"
        fig.savefig(OUTPUT_DIR / filename_pdf, dpi=300, facecolor='white', bbox_inches='tight')
        print(f"Saved: {OUTPUT_DIR / filename}")
    
    if show:
        plt.show()
    
    return fig

plot_boxplot_by_angle_bins(final_df, y_col='abs_error_node', geom_col='50_geometry_category', threshold=0.1832,
                                bin_width=5, n_bins=15, figsize=(18, 12), ylim=(0, 2),
                                save=True)

#### whole for angle (used)

In [ ]:
"""
Parallel Angle Conditional Effect - 5 Panel (Reorganized Layout)
================================================================
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.gridspec import GridSpec
from pathlib import Path

plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

OUTPUT_DIR = Path("figures")
OUTPUT_DIR.mkdir(exist_ok=True)


def get_parallel_bin(flow_angle, bin_width=5, n_bins=15):
    fa = flow_angle % 360
    dev_from_0 = min(fa, 360 - fa)
    dev_from_180 = abs(fa - 180)
    deviation = min(dev_from_0, dev_from_180)
    
    for i in range(n_bins):
        if i * bin_width <= deviation < (i + 1) * bin_width:
            return i + 1
    return None


def _format_axis(ax, offset, ylim_max, bin_width, n_bins):
    ax.set_yscale('log')
    ax.set_ylim(offset * 0.9, ylim_max + offset)
    
    def format_func(value, tick_number):
        original = value - offset
        if abs(original) < 0.01: return '0'
        elif abs(original - 1) < 0.01: return '1'
        elif abs(original - 2) < 0.01: return '2'
        elif abs(original - 4) < 0.01: return '4'
        else: return ''  # Hide all other ticks
    
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(format_func))
    ax.yaxis.set_minor_formatter(ticker.NullFormatter())  # Hide minor ticks
    tick_values = [0 + offset, 1 + offset, 2 + offset, 4 + offset]
    ax.set_yticks(tick_values)
    ax.minorticks_off()  # Disable minor ticks completely
    ax.set_xlim(0, bin_width * n_bins)
    ax.tick_params(colors='black', labelsize=11)
    ax.grid(True, alpha=0.3, color='gray')
    for spine in ax.spines.values():
        spine.set_color('black')


def plot_neither_panel(ax, data, y_col, geom_col,
                       threshold=0.1832, offset=1, ylim_max=5,
                       bin_width=5, n_bins=15):
    ax.set_facecolor('white')
    df_cat = data[data[geom_col] == 'Neither'].copy()
    
    if len(df_cat) < 100:
        ax.text(0.5, 0.5, 'Insufficient data', transform=ax.transAxes,
                ha='center', va='center', fontsize=12)
        return
    
    df_cat['parallel_bin'] = df_cat['flow_angle'].apply(
        lambda x: get_parallel_bin(x, bin_width, n_bins))
    df_cat = df_cat.dropna(subset=['parallel_bin'])
    df_cat['parallel_bin'] = df_cat['parallel_bin'].astype(int)
    
    bin_centers = [(i - 0.5) * bin_width for i in range(1, n_bins + 1)]
    
    y_vals, n_per_bin = [], []
    for bin_num in range(1, n_bins + 1):
        bin_data = df_cat[df_cat['parallel_bin'] == bin_num][y_col].dropna()
        n_per_bin.append(len(bin_data))
        y_vals.append(np.percentile(bin_data, 68) + offset if len(bin_data) >= 10 else np.nan)
    
    y_vals, n_per_bin = np.array(y_vals), np.array(n_per_bin)
    valid_mask = ~np.isnan(y_vals)
    x_valid = np.array(bin_centers)[valid_mask]
    
    ax.plot(x_valid, y_vals[valid_mask], 'o-', color='dimgray',
            linewidth=2.5, markersize=8, label='Median')
    
    for x, y, n in zip(x_valid, y_vals[valid_mask], n_per_bin[valid_mask]):
        ax.text(x, y * 1.1, f'n={n:,}', ha='center', va='bottom',
                fontsize=11, color='black', fontweight='bold', rotation=90)
    
    ax.axhline(y=threshold + offset, color='black', linestyle='--', linewidth=2, alpha=0.8)
    ax.text(bin_width * n_bins * 0.5, (threshold + offset) * 0.95, f'T={threshold}m',
            fontsize=12, fontweight='bold', va='top', ha='center')
    
    _format_axis(ax, offset, ylim_max, bin_width, n_bins)
    ax.set_title(f'Neither Zone (n={len(df_cat):,})', fontsize=14, fontweight='bold')
    ax.legend(loc='upper right', fontsize=10, framealpha=0.9)


def plot_conditional_panel(ax, data, y_col, geom_cat, geom_col, isa_col, title,
                            threshold=0.1832, offset=1, ylim_max=5,
                            bin_width=5, n_bins=15,
                            quantiles=[0, 5, 25, 50, 75, 95, 100], tolerance_pct=5):
    ax.set_facecolor('white')
    df_cat = data[data[geom_col] == geom_cat].copy()
    
    if len(df_cat) < 100:
        ax.text(0.5, 0.5, 'Insufficient data', transform=ax.transAxes,
                ha='center', va='center', fontsize=12)
        return
    
    df_cat['parallel_bin'] = df_cat['flow_angle'].apply(
        lambda x: get_parallel_bin(x, bin_width, n_bins))
    df_cat = df_cat.dropna(subset=['parallel_bin', isa_col])
    df_cat['parallel_bin'] = df_cat['parallel_bin'].astype(int)
    
    bin_centers = [(i - 0.5) * bin_width for i in range(1, n_bins + 1)]
    quantile_values = np.percentile(df_cat[isa_col].dropna(), quantiles)
    
    all_lines = []
    for q_val in quantile_values:
        lower, upper = max(0, q_val - tolerance_pct), min(100, q_val + tolerance_pct)
        df_subset = df_cat[(df_cat[isa_col] >= lower) & (df_cat[isa_col] <= upper)]
        if len(df_subset) < 50:
            continue
        y_vals = []
        for bin_num in range(1, n_bins + 1):
            bin_data = df_subset[df_subset['parallel_bin'] == bin_num][y_col].dropna()
            y_vals.append(np.percentile(bin_data, 68) if len(bin_data) >= 10 else np.nan)
        all_lines.append(y_vals)
    
    if len(all_lines) == 0:
        ax.text(0.5, 0.5, 'Insufficient data', transform=ax.transAxes,
                ha='center', va='center', fontsize=12)
        return
    
    all_lines = np.array(all_lines)
    with np.errstate(all='ignore'):
        median_line = np.nanmedian(all_lines, axis=0) + offset
        q25_line = np.nanpercentile(all_lines, 25, axis=0) + offset
        q75_line = np.nanpercentile(all_lines, 75, axis=0) + offset
        q5_line = np.nanpercentile(all_lines, 5, axis=0) + offset
        q95_line = np.nanpercentile(all_lines, 95, axis=0) + offset
    
    valid_mask = ~np.isnan(median_line - offset)
    x_valid = np.array(bin_centers)[valid_mask]
    
    n_per_bin = np.array([len(df_cat[df_cat['parallel_bin'] == b][y_col].dropna()) 
                          for b in range(1, n_bins + 1)])
    
    fill_color, line_color = ('peru', 'darkorange') if isa_col == 'far_isa' else ('green', 'darkgreen')
    isa_label = isa_col.replace('_', ' ').title()
    
    ax.fill_between(x_valid, q5_line[valid_mask], q95_line[valid_mask],
                    alpha=0.2, color=fill_color, label=f'{isa_label} 5-95%')
    ax.fill_between(x_valid, q25_line[valid_mask], q75_line[valid_mask],
                    alpha=0.4, color=fill_color, label=f'{isa_label} 25-75%')
    ax.plot(x_valid, median_line[valid_mask], 'o-', color=line_color,
            linewidth=2.5, markersize=8, label=f'{isa_label} Median')
    
    for x, y, n in zip(x_valid, median_line[valid_mask], n_per_bin[valid_mask]):
        ax.text(x, y * 1.1, f'n={n:,}', ha='center', va='bottom',
                fontsize=11, color='black', fontweight='bold', rotation=90)
    
    ax.axhline(y=threshold + offset, color='black', linestyle='--', linewidth=2, alpha=0.8)
    ax.text(bin_width * n_bins * 0.5, (threshold + offset) * 0.95, f'T={threshold}m',
            fontsize=12, fontweight='bold', va='top', ha='center')
    
    _format_axis(ax, offset, ylim_max, bin_width, n_bins)
    ax.set_title(f'{title} (n={len(df_cat):,})', fontsize=14, fontweight='bold')
    ax.legend(loc='upper right', fontsize=10, framealpha=0.9)


def plot_angle_conditional_effect_5panel(df, y_col, geom_col='50_geometry_category',
                                          threshold=0.1832, offset=1, ylim_max=5,
                                          bin_width=5, n_bins=15,
                                          quantiles=[0, 5, 25, 50, 75, 95, 100],
                                          tolerance_pct=5,
                                          figsize=(18, 10), save=True, show=True):
    print("Angle Conditional Effect Analysis - 5 Panel")
    
    df_plot = df.dropna(subset=[y_col, 'flow_angle', geom_col, 'near_isa', 'far_isa']).copy()
    print(f"\n[Data] Total: {len(df_plot):,}")
    
    fig = plt.figure(figsize=figsize, facecolor='white')
    gs = GridSpec(2, 3, figure=fig, width_ratios=[1, 1, 1], hspace=0.2, wspace=0.15)
    
    # Col 1: Neither (spans 2 rows)
    ax_neither = fig.add_subplot(gs[:, 0])
    plot_neither_panel(ax_neither, df_plot, y_col, geom_col,
                       threshold, offset, ylim_max, bin_width, n_bins)
    
    # Col 2: Near ISA conditioning
    ax_near = fig.add_subplot(gs[0, 1])
    plot_conditional_panel(ax_near, df_plot, y_col, 'Near', geom_col, 'near_isa',
                           'Near Zone by Near ISA', threshold, offset, ylim_max,
                           bin_width, n_bins, quantiles, tolerance_pct)
    
    ax_both_near = fig.add_subplot(gs[1, 1], sharex=ax_near, sharey=ax_near)
    plot_conditional_panel(ax_both_near, df_plot, y_col, 'Both', geom_col, 'near_isa',
                           'Both Zone by Near ISA', threshold, offset, ylim_max,
                           bin_width, n_bins, quantiles, tolerance_pct)
    
    # Col 3: Far ISA conditioning
    ax_far = fig.add_subplot(gs[0, 2], sharey=ax_near)
    plot_conditional_panel(ax_far, df_plot, y_col, 'Far', geom_col, 'far_isa',
                           'Far Zone by Far ISA', threshold, offset, ylim_max,
                           bin_width, n_bins, quantiles, tolerance_pct)
    
    ax_both_far = fig.add_subplot(gs[1, 2], sharex=ax_far, sharey=ax_near)
    plot_conditional_panel(ax_both_far, df_plot, y_col, 'Both', geom_col, 'far_isa',
                           'Both Zone by Far ISA', threshold, offset, ylim_max,
                           bin_width, n_bins, quantiles, tolerance_pct)
    
    fig.text(0.5, 0.02, 'Deviation from Parallel (°)', ha='center', fontsize=14, fontweight='bold')
    # fig.text(0.02, 0.5, r'68th Percentile $|Relative\ WSE\ Error|$ (m)', 
    #          va='center', rotation='vertical', fontsize=14, fontweight='bold')
    fig.supylabel(r'68th Percentile $|Relative\ WSE\ Error|$ (m)', 
              fontsize=14, fontweight='bold', x=0.03)
    
    plt.tight_layout()
    fig.subplots_adjust(left=0.06, bottom=0.08)
    
    if save:
        filename = f"figure_angle_conditional_5panel_{y_col}.png"
        fig.savefig(OUTPUT_DIR / filename, dpi=300, facecolor='white', bbox_inches='tight')
        filename_pdf = f"figure_angle_conditional_5panel_{y_col}.pdf"
        fig.savefig(OUTPUT_DIR / filename_pdf, dpi=300, facecolor='white', bbox_inches='tight')
        print(f"\nSaved: {OUTPUT_DIR / filename}")
    
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    return fig

In [ ]:
fig = plot_angle_conditional_effect_5panel(
    final_df, y_col='abs_error_node',
    geom_col='50_geometry_category',
    ylim_max=4, bin_width=2, n_bins=30
)

#### whole for isa

In [ ]:
"""
ISA Conditional Effect - 2x2 Panel
==================================
Row 1: All data
Row 2: Parallel angle only
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path

plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

OUTPUT_DIR = Path("figures")
OUTPUT_DIR.mkdir(exist_ok=True)


def get_isa_bin(isa_value):
    if isa_value <= 5: return '0-5'
    elif isa_value <= 10: return '5-10'
    elif isa_value <= 20: return '10-20'
    elif isa_value <= 30: return '20-30'
    elif isa_value <= 40: return '30-40'
    elif isa_value <= 50: return '40-50'
    elif isa_value <= 60: return '50-60'
    elif isa_value <= 70: return '60-70'
    elif isa_value <= 80: return '70-80'
    else: return '80-100'


def get_flow_angle_zone(flow_angle):
    fa = flow_angle % 360
    if (fa >= 350 or fa <= 10) or (170 <= fa <= 190):
        return 'Parallel'
    elif (80 <= fa <= 100) or (260 <= fa <= 280):
        return 'Perpendicular'
    else:
        return 'Other'


def plot_isa_panel(ax, df, y_col, main_isa_col, condition_isa_col,
                   threshold=0.1832, offset=1, ylim_max=5,
                   quantiles=[0, 5, 25, 50, 75, 95, 100], tolerance_pct=5,
                   show_xlabel=False, show_ylabel=False, title_suffix=''):
    """Single panel ISA conditional effect."""
    ax.set_facecolor('white')
    
    bin_order = ['0-5', '5-10', '10-20', '20-30', '30-40', '40-50', 
                 '50-60', '60-70', '70-80', '80-100']
    bin_centers = [2.5, 7.5, 15, 25, 35, 45, 55, 65, 75, 90]
    
    quantile_values = np.percentile(df[condition_isa_col].dropna(), quantiles)
    
    all_lines = []
    for q_val in quantile_values:
        lower = max(0, q_val - tolerance_pct)
        upper = min(100, q_val + tolerance_pct)
        df_subset = df[(df[condition_isa_col] >= lower) & (df[condition_isa_col] <= upper)].copy()
        
        if len(df_subset) < 50:
            continue
        
        df_subset['isa_bin'] = df_subset[main_isa_col].apply(get_isa_bin)
        y_vals = []
        for bin_label in bin_order:
            bin_data = df_subset[df_subset['isa_bin'] == bin_label][y_col].dropna()
            if len(bin_data) >= 10:
                y_vals.append(np.percentile(bin_data, 68))
            else:
                y_vals.append(np.nan)
        all_lines.append(y_vals)
    
    if len(all_lines) == 0:
        ax.text(0.5, 0.5, 'Insufficient data', transform=ax.transAxes, ha='center')
        return
    
    all_lines = np.array(all_lines)
    
    with np.errstate(all='ignore'):
        median_line = np.nanmedian(all_lines, axis=0) + offset
        q25_line = np.nanpercentile(all_lines, 25, axis=0) + offset
        q75_line = np.nanpercentile(all_lines, 75, axis=0) + offset
        q5_line = np.nanpercentile(all_lines, 5, axis=0) + offset
        q95_line = np.nanpercentile(all_lines, 95, axis=0) + offset
    
    valid_mask = ~np.isnan(median_line - offset)
    x_valid = np.array(bin_centers)[valid_mask]
    
    cond_label = condition_isa_col.replace('_', ' ').title()
    main_label = main_isa_col.replace('_', ' ').title()
    
    # Colors: darkorange/peru for Far, green for Near
    if main_isa_col == 'far_isa':
        fill_color = 'peru'
        line_color = 'darkorange'
    else:
        fill_color = 'mediumseagreen'
        line_color = 'darkgreen'
    
    # n per bin
    n_per_bin = []
    for bin_label in bin_order:
        df_bin = df[df[main_isa_col].apply(get_isa_bin) == bin_label]
        n_per_bin.append(len(df_bin[y_col].dropna()))
    n_per_bin = np.array(n_per_bin)
    
    ax.fill_between(x_valid, q5_line[valid_mask], q95_line[valid_mask],
                    alpha=0.2, color=fill_color, label=f'{cond_label} 5-95%')
    ax.fill_between(x_valid, q25_line[valid_mask], q75_line[valid_mask],
                    alpha=0.4, color=fill_color, label=f'{cond_label} 25-75%')
    ax.plot(x_valid, median_line[valid_mask], 'o-', color=line_color,
            linewidth=2.5, markersize=8, label=f'{cond_label} Median')
    
    # n= labels
    n_valid = n_per_bin[valid_mask]
    for x, y, n in zip(x_valid, median_line[valid_mask], n_valid):
        ax.text(x, y * 1.1, f'n={n:,}', ha='center', va='bottom',
                fontsize=11, color='black', fontweight='bold', rotation=90)
    
    # Threshold
    ax.axhline(y=threshold + offset, color='black', linestyle='--', linewidth=2, alpha=0.8)
    ax.text(50, (threshold + offset) * 0.85, f'T={threshold}m',
            fontsize=12, fontweight='bold', va='top', ha='center')
    
    # Y-axis
    ax.set_yscale('log')
    ax.set_ylim(offset * 0.9, ylim_max + offset)
    
    def format_func(value, tick_number):
        original = value - offset
        if abs(original) < 0.01: return '0'
        elif abs(original) >= 10: return f'{original:.0f}'
        elif abs(original) >= 1: return f'{original:.1f}'
        else: return f'{original:.2f}'
    
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(format_func))
    tick_values_orig = [0, 0.2, 0.5, 1, 2, 5, 10, 20]
    tick_values = [v + offset for v in tick_values_orig if v <= ylim_max]
    ax.set_yticks(tick_values)
    
    ax.set_xlim(0, 100)
    ax.set_title(f'68%tile Error by {main_label} (Conditioned by {cond_label}{title_suffix})',
                 color='black', fontsize=13, fontweight='bold')
    ax.tick_params(colors='black', labelsize=11)
    ax.grid(True, alpha=0.3, color='gray')
    ax.legend(loc='upper left', fontsize=10, framealpha=0.9)
    
    for spine in ax.spines.values():
        spine.set_color('black')


def plot_isa_conditional_2x2(df, y_col, geom_col='50_geometry_category',
                              threshold=0.1832, offset=1, ylim_max=5,
                              quantiles=[0, 5, 25, 50, 75, 95, 100], tolerance_pct=5,
                              figsize=(14, 12), save=True, show=True):
    """
    2x2: Row 1 = All, Row 2 = Parallel only
    """
    print("ISA Conditional Effect - 2x2")
    
    df_plot = df.dropna(subset=[y_col, 'near_isa', 'far_isa', 'flow_angle']).copy()
    df_plot['flow_zone'] = df_plot['flow_angle'].apply(get_flow_angle_zone)
    df_parallel = df_plot[df_plot['flow_zone'] == 'Parallel'].copy()
    
    print(f"\n[Data] All: {len(df_plot):,}, Parallel: {len(df_parallel):,}")
    
    fig, axes = plt.subplots(2, 2, figsize=figsize, facecolor='white',
                              sharex=True, sharey=True)
    fig.subplots_adjust(hspace=0.2, wspace=0.08)
    
    # Row 1: All data
    plot_isa_panel(axes[0, 0], df_plot, y_col, 'near_isa', 'far_isa',
                   threshold, offset, ylim_max, quantiles, tolerance_pct,
                   title_suffix=f', n={len(df_plot):,}')
    
    plot_isa_panel(axes[0, 1], df_plot, y_col, 'far_isa', 'near_isa',
                   threshold, offset, ylim_max, quantiles, tolerance_pct,
                   title_suffix=f', n={len(df_plot):,}')
    
    # Row 2: Parallel only
    plot_isa_panel(axes[1, 0], df_parallel, y_col, 'near_isa', 'far_isa',
                   threshold, offset, ylim_max, quantiles, tolerance_pct,
                   title_suffix=f', Parallel, n={len(df_parallel):,}')
    
    plot_isa_panel(axes[1, 1], df_parallel, y_col, 'far_isa', 'near_isa',
                   threshold, offset, ylim_max, quantiles, tolerance_pct,
                   title_suffix=f', Parallel, n={len(df_parallel):,}')
    
    # Shared labels
    fig.text(0.5, 0.02, 'ISA (%)', ha='center', fontsize=14, fontweight='bold')
    fig.text(0.02, 0.5, r'68th Percentile $|Relative\ WSE\ Error|$ (m)',
             va='center', rotation='vertical', fontsize=14, fontweight='bold')
    
    plt.tight_layout(rect=[0.04, 0.04, 1, 1])
    
    if save:
        filename = f"figure_isa_conditional_2x2_{y_col}.png"
        fig.savefig(OUTPUT_DIR / filename, dpi=300, facecolor='white', bbox_inches='tight')
        filename_pdf = f"figure_isa_conditional_2x2_{y_col}.pdf"
        fig.savefig(OUTPUT_DIR / filename_pdf, dpi=300, facecolor='white', bbox_inches='tight')
        print(f"\nSaved: {OUTPUT_DIR / filename}")
    
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    return fig

In [ ]:
fig = plot_isa_conditional_2x2(final_df, y_col='abs_error_node', ylim_max=20)

#### parallel only for isa (used)

In [ ]:
"""
ISA Conditional Effect - Parallel Angle Only (2 Panel)
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path

plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

OUTPUT_DIR = Path("figures")
OUTPUT_DIR.mkdir(exist_ok=True)


def get_isa_bin(isa_value):
    if isa_value <= 5: return '0-5'
    elif isa_value <= 10: return '5-10'
    elif isa_value <= 20: return '10-20'
    elif isa_value <= 30: return '20-30'
    elif isa_value <= 40: return '30-40'
    elif isa_value <= 50: return '40-50'
    elif isa_value <= 60: return '50-60'
    elif isa_value <= 70: return '60-70'
    elif isa_value <= 80: return '70-80'
    else: return '80-100'


def get_flow_angle_zone(flow_angle):
    fa = flow_angle % 360
    if (fa >= 350 or fa <= 10) or (170 <= fa <= 190):
        return 'Parallel'
    elif (80 <= fa <= 100) or (260 <= fa <= 280):
        return 'Perpendicular'
    else:
        return 'Other'


def plot_isa_panel(ax, df, y_col, main_isa_col, condition_isa_col,
                   threshold=0.1832, offset=1, ylim_max=5,
                   quantiles=[0, 5, 25, 50, 75, 95, 100], tolerance_pct=5,
                   min_n=30):
    """Single panel ISA conditional effect."""
    ax.set_facecolor('white')
    
    bin_order = ['0-5', '5-10', '10-20', '20-30', '30-40', '40-50', 
                 '50-60', '60-70', '70-80', '80-100']
    bin_centers = [2.5, 7.5, 15, 25, 35, 45, 55, 65, 75, 90]
    
    # Calculate n per bin first (for filtering)
    n_per_bin = []
    for bin_label in bin_order:
        df_bin = df[df[main_isa_col].apply(get_isa_bin) == bin_label]
        n_per_bin.append(len(df_bin[y_col].dropna()))
    n_per_bin = np.array(n_per_bin)
    
    # Filter bins with n >= min_n
    valid_bins_mask = n_per_bin >= min_n
    
    quantile_values = np.percentile(df[condition_isa_col].dropna(), quantiles)
    
    all_lines = []
    for q_val in quantile_values:
        lower = max(0, q_val - tolerance_pct)
        upper = min(100, q_val + tolerance_pct)
        df_subset = df[(df[condition_isa_col] >= lower) & (df[condition_isa_col] <= upper)].copy()
        
        if len(df_subset) < 50:
            continue
        
        df_subset['isa_bin'] = df_subset[main_isa_col].apply(get_isa_bin)
        y_vals = []
        for i, bin_label in enumerate(bin_order):
            if not valid_bins_mask[i]:
                y_vals.append(np.nan)
                continue
            bin_data = df_subset[df_subset['isa_bin'] == bin_label][y_col].dropna()
            if len(bin_data) >= 10:
                y_vals.append(np.percentile(bin_data, 68))
            else:
                y_vals.append(np.nan)
        all_lines.append(y_vals)
    
    if len(all_lines) == 0:
        ax.text(0.5, 0.5, 'Insufficient data', transform=ax.transAxes, ha='center')
        return 0
    
    all_lines = np.array(all_lines)
    
    with np.errstate(all='ignore'):
        median_line = np.nanmedian(all_lines, axis=0) + offset
        q25_line = np.nanpercentile(all_lines, 25, axis=0) + offset
        q75_line = np.nanpercentile(all_lines, 75, axis=0) + offset
        q5_line = np.nanpercentile(all_lines, 5, axis=0) + offset
        q95_line = np.nanpercentile(all_lines, 95, axis=0) + offset
    
    valid_mask = ~np.isnan(median_line - offset)
    x_valid = np.array(bin_centers)[valid_mask]
    
    cond_label = condition_isa_col.replace('_', ' ').title()
    main_label = main_isa_col.replace('_', ' ').title()
    
    if main_isa_col == 'far_isa':
        fill_color = 'peru'
        line_color = 'darkorange'
    else:
        fill_color = 'mediumseagreen'
        line_color = 'darkgreen'
    
    ax.fill_between(x_valid, q5_line[valid_mask], q95_line[valid_mask],
                    alpha=0.2, color=fill_color, label=f'{cond_label} 5-95%')
    ax.fill_between(x_valid, q25_line[valid_mask], q75_line[valid_mask],
                    alpha=0.4, color=fill_color, label=f'{cond_label} 25-75%')
    ax.plot(x_valid, median_line[valid_mask], 'o-', color=line_color,
            linewidth=2.5, markersize=8, label=f'{cond_label} Median')
    
    # n= labels
    n_valid = n_per_bin[valid_mask]
    for x, y, n in zip(x_valid, median_line[valid_mask], n_valid):
        ax.text(x, y * 1.1, f'n={n:,}', ha='center', va='bottom',
                fontsize=11, color='black', fontweight='bold', rotation=90)
    
    # Threshold
    ax.axhline(y=threshold + offset, color='black', linestyle='--', linewidth=2, alpha=0.8)
    ax.text(50, (threshold + offset) * 0.85, f'T={threshold}m',
            fontsize=12, fontweight='bold', va='top', ha='center')
    
    ax.set_yscale('log')
    ax.set_ylim(offset * 0.9, ylim_max + offset)
    
    def format_func(value, tick_number):
        original = value - offset
        if abs(original) < 0.01: return '0'
        elif abs(original) >= 10: return f'{original:.0f}'
        elif abs(original) >= 1: return f'{original:.1f}'
        else: return f'{original:.2f}'
    
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(format_func))
    tick_values_orig = [0, 0.2, 0.5, 1, 2, 5, 10, 20]
    tick_values = [v + offset for v in tick_values_orig if v <= ylim_max]
    ax.set_yticks(tick_values)
    
    ax.set_xlim(0, 80)
    ax.set_xlabel(f'{main_label} (%)', color='black', fontsize=13)
    ax.set_ylabel(r'68th Percentile $|Relative\ WSE\ Error|$ (m)', color='black', fontsize=13)
    ax.tick_params(colors='black', labelsize=11)
    ax.grid(True, alpha=0.3, color='gray')
    ax.legend(loc='upper left', fontsize=10, framealpha=0.9)
    
    for spine in ax.spines.values():
        spine.set_color('black')
    
    return len(df)


def plot_isa_conditional_parallel(df, y_col, threshold=0.1832, offset=1, ylim_max=5,
                                   quantiles=[0, 5, 25, 50, 75, 95, 100], tolerance_pct=5,
                                   min_n=30, figsize=(14, 6), save=True, show=True):
    """
    2-panel: Parallel only - Near ISA / Far ISA conditional effect
    """
    print("ISA Conditional Effect - Parallel Angle")
    
    df_plot = df.dropna(subset=[y_col, 'near_isa', 'far_isa', 'flow_angle']).copy()
    df_plot['flow_zone'] = df_plot['flow_angle'].apply(get_flow_angle_zone)
    df_parallel = df_plot[df_plot['flow_zone'] == 'Parallel'].copy()
    
    print(f"\n[Data] Parallel: {len(df_parallel):,}")
    
    fig, axes = plt.subplots(1, 2, figsize=figsize, facecolor='white')
    
    # Left: Near ISA (by Far ISA)
    n_near = plot_isa_panel(axes[0], df_parallel, y_col, 'near_isa', 'far_isa',
                            threshold, offset, ylim_max, quantiles, tolerance_pct, min_n)
    axes[0].set_title(f'Near ISA (by Far ISA, n={len(df_parallel):,})',
                      fontsize=14, fontweight='bold')
    
    # Right: Far ISA (by Near ISA)
    n_far = plot_isa_panel(axes[1], df_parallel, y_col, 'far_isa', 'near_isa',
                           threshold, offset, ylim_max, quantiles, tolerance_pct, min_n)
    axes[1].set_title(f'Far ISA (by Near ISA, n={len(df_parallel):,})',
                      fontsize=14, fontweight='bold')
    
    fig.suptitle('Parallel Angle Analysis - Conditional Effect of Far & Near ISA',
                 fontsize=16, fontweight='bold', y=0.98)
    
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    
    if save:
        filename = f"figure_isa_conditional_parallel_{y_col}.png"
        fig.savefig(OUTPUT_DIR / filename, dpi=300, facecolor='white', bbox_inches='tight')
        filename_pdf = f"figure_isa_conditional_parallel_{y_col}.pdf"
        fig.savefig(OUTPUT_DIR / filename_pdf, dpi=300, facecolor='white', bbox_inches='tight')
        print(f"\nSaved: {OUTPUT_DIR / filename}")
    
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    return fig

In [ ]:
fig = plot_isa_conditional_parallel(final_df, y_col='abs_error_node', ylim_max=20, min_n=30)

#### paralle and far zone for building height (used)

In [ ]:
"""
Building Height Conditional Effect (Conditioned by Far ISA)
===========================================================
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path

plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

OUTPUT_DIR = Path("figures")
OUTPUT_DIR.mkdir(exist_ok=True)


def get_flow_angle_zone(flow_angle):
    fa = flow_angle % 360
    if (fa >= 350 or fa <= 10) or (170 <= fa <= 190):
        return 'Parallel'
    elif (80 <= fa <= 100) or (260 <= fa <= 280):
        return 'Perpendicular'
    else:
        return 'Other'


def get_building_bin(value, edges, labels):
    """Assign building height to bin based on edges."""
    for i in range(len(edges) - 1):
        if edges[i] <= value < edges[i + 1]:
            return labels[i]
    return None


def plot_building_conditional_panel(ax, df, y_col, building_col, isa_col,
                                     building_edges, building_labels,
                                     threshold=0.1832, offset=1, ylim_max=5,
                                     quantiles=[0, 5, 25, 50, 75, 95, 100], 
                                     tolerance_pct=5, min_n=30):
    """Building height conditional effect panel."""
    ax.set_facecolor('white')
    
    df = df.copy()
    df['building_bin'] = df[building_col].apply(
        lambda x: get_building_bin(x, building_edges, building_labels)
    )
    df = df.dropna(subset=['building_bin', isa_col])
    
    bin_centers = []
    for i in range(len(building_edges) - 1):
        if building_edges[i + 1] == np.inf:
            bin_centers.append(building_edges[i] + 1)  # +1 for inf bin
        else:
            bin_centers.append((building_edges[i] + building_edges[i + 1]) / 2)
    
    n_per_bin = []
    for label in building_labels:
        n = len(df[df['building_bin'] == label][y_col].dropna())
        n_per_bin.append(n)
    n_per_bin = np.array(n_per_bin)
    valid_bins_mask = n_per_bin >= min_n
    
    quantile_values = np.percentile(df[isa_col].dropna(), quantiles)
    
    all_lines = []
    for q_val in quantile_values:
        lower = max(0, q_val - tolerance_pct)
        upper = min(100, q_val + tolerance_pct)
        df_subset = df[(df[isa_col] >= lower) & (df[isa_col] <= upper)].copy()
        
        if len(df_subset) < 50:
            continue
        
        y_vals = []
        for i, label in enumerate(building_labels):
            if not valid_bins_mask[i]:
                y_vals.append(np.nan)
                continue
            bin_data = df_subset[df_subset['building_bin'] == label][y_col].dropna()
            if len(bin_data) >= 10:
                y_vals.append(np.percentile(bin_data, 68))
            else:
                y_vals.append(np.nan)
        all_lines.append(y_vals)
    
    if len(all_lines) == 0:
        ax.text(0.5, 0.5, 'Insufficient data', transform=ax.transAxes, ha='center')
        return 0
    
    all_lines = np.array(all_lines)
    
    with np.errstate(all='ignore'):
        median_line = np.nanmedian(all_lines, axis=0) + offset
        q25_line = np.nanpercentile(all_lines, 25, axis=0) + offset
        q75_line = np.nanpercentile(all_lines, 75, axis=0) + offset
        q5_line = np.nanpercentile(all_lines, 5, axis=0) + offset
        q95_line = np.nanpercentile(all_lines, 95, axis=0) + offset
        q0_line = np.nanmin(all_lines, axis=0) + offset
        q100_line = np.nanmax(all_lines, axis=0) + offset
    
    valid_mask = ~np.isnan(median_line - offset)
    x_valid = np.array(bin_centers)[valid_mask]
    
    isa_label = isa_col.replace('_', ' ').title()
    
    fill_color = 'peru'
    line_color = 'darkorange'
    
    ax.fill_between(x_valid, q0_line[valid_mask], q100_line[valid_mask],
                    alpha=0.15, color=fill_color, label=f'{isa_label} 0-100%')
    ax.fill_between(x_valid, q5_line[valid_mask], q95_line[valid_mask],
                    alpha=0.25, color=fill_color, label=f'{isa_label} 5-95%')
    ax.fill_between(x_valid, q25_line[valid_mask], q75_line[valid_mask],
                    alpha=0.45, color=fill_color, label=f'{isa_label} 25-75%')
    ax.plot(x_valid, median_line[valid_mask], 'o-', color=line_color,
            linewidth=2.5, markersize=8, label=f'{isa_label} Median')
    
    # Mark last point (inf bin) with black "+" below
    last_x = bin_centers[-1]
    if valid_bins_mask[-1]:
        ax.text(last_x, offset * 1.5, '13+', ha='center', va='top',
                fontsize=14, color='black', fontweight='bold')
    
    n_valid = n_per_bin[valid_mask]
    for x, y, n in zip(x_valid, median_line[valid_mask], n_valid):
        ax.text(x, y * 1.1, f'n={n:,}', ha='center', va='bottom',
                fontsize=11, color='black', fontweight='bold', rotation=90)
    
    ax.axhline(y=threshold + offset, color='black', linestyle='--', linewidth=2, alpha=0.8)
    x_mid = (building_edges[0] + building_edges[-2]) / 2
    ax.text(x_mid, (threshold + offset) * 0.95, f'T={threshold}m',
            fontsize=12, fontweight='bold', va='top', ha='center')
    
    ax.set_yscale('log')
    ax.set_ylim(offset * 0.9, ylim_max + offset)
    
    # Force only 0, 1, 2, 4 ticks
    def format_func(value, tick_number):
        original = value - offset
        if abs(original) < 0.01: return '0'
        elif abs(original - 1) < 0.01: return '1'
        elif abs(original - 2) < 0.01: return '2'
        elif abs(original - 4) < 0.01: return '4'
        else: return ''
    
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(format_func))
    ax.yaxis.set_minor_formatter(ticker.NullFormatter())
    tick_values = [0 + offset, 1 + offset, 2 + offset, 4 + offset]
    ax.set_yticks(tick_values)
    ax.minorticks_off()
    
    ax.set_xlim(building_edges[0] - 1, building_edges[-2] + 3)
    ax.set_xlabel('Far Building Height (m)', color='black', fontsize=13)
    ax.set_ylabel(r'68th Percentile $|Relative\ WSE\ Error|$ (m)', color='black', fontsize=13)
    ax.tick_params(colors='black', labelsize=11)
    ax.grid(True, alpha=0.3, color='gray')
    ax.legend(loc='upper left', fontsize=10, framealpha=0.9)
    
    for spine in ax.spines.values():
        spine.set_color('black')
    
    return len(df)


def plot_building_conditional(df, y_col='abs_error_node',
                               building_col='far_building',
                               isa_col='far_isa',
                               building_edges=[5, 7, 9, 11, 13, 15, 17, 19, np.inf],
                               threshold=0.1832, offset=1, ylim_max=5,
                               quantiles=[0, 5, 25, 50, 75, 95, 100],
                               tolerance_pct=5, min_n=30,
                               figsize=(10, 6), save=True, show=True):
    print("Building Height Conditional Effect")
    
    df_plot = df.dropna(subset=[y_col, building_col, isa_col, 'flow_angle']).copy()
    df_plot['flow_zone'] = df_plot['flow_angle'].apply(get_flow_angle_zone)
    df_parallel = df_plot[df_plot['flow_zone'] == 'Parallel'].copy()
    df_parallel = df_parallel[df_parallel[building_col] >= building_edges[0]]
    
    print(f"\n[Data] Parallel (building >= {building_edges[0]}m): {len(df_parallel):,}")
    
    building_labels = []
    for i in range(len(building_edges) - 1):
        if building_edges[i + 1] == np.inf:
            building_labels.append(f'{building_edges[i]}+')
        else:
            building_labels.append(f'{int(building_edges[i])}-{int(building_edges[i+1])}')
    
    print(f"[Building bins]: {building_labels}")
    
    fig, ax = plt.subplots(1, 1, figsize=figsize, facecolor='white')
    
    n_data = plot_building_conditional_panel(
        ax, df_parallel, y_col, building_col, isa_col,
        building_edges, building_labels,
        threshold, offset, ylim_max, quantiles, tolerance_pct, min_n
    )
    
    isa_label = isa_col.replace('_', ' ').title()
    ax.set_title(f'Far Building Height (by {isa_label}, n={len(df_parallel):,})',
                 fontsize=14, fontweight='bold', pad=5)
    
    plt.tight_layout()
    
    if save:
        filename = f"figure_building_conditional_{building_col}_{y_col}.png"
        fig.savefig(OUTPUT_DIR / filename, dpi=300, facecolor='white', bbox_inches='tight')
        filename_pdf = f"figure_building_conditional_{building_col}_{y_col}.pdf"
        fig.savefig(OUTPUT_DIR / filename_pdf, dpi=300, facecolor='white', bbox_inches='tight')
        print(f"\nSaved: {OUTPUT_DIR / filename}")
    
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    return fig

In [ ]:
fig = plot_building_conditional(
    df = final_df, 
    y_col='abs_error_node',
    building_col='far_building',
    isa_col='far_isa',
    building_edges=[5,6,7,8,9,10,11,12,13, np.inf],  # 직접 설정
    ylim_max=4, min_n=30
)

In [ ]:
final_df['tan'] = np.abs(final_df.xtrk_dist_node) / final_df.sat_altitude

In [ ]:
final_df['tan'].describe()

In [ ]:
final_df['layover_shooting_m'] = final_df.far_building / final_df['tan']

In [ ]:
final_df['layover_shooting_m'].describe()

In [ ]:
final_df['layover_width_ratio'] = final_df['layover_shooting_m'] / final_df['width_node']

In [ ]:
"""
Boxplot: Layover & Width by Far Building Bins
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from pathlib import Path

OUTPUT_DIR = Path("figures")
OUTPUT_DIR.mkdir(exist_ok=True)


def get_flow_angle_zone(flow_angle):
    fa = flow_angle % 360
    if (fa >= 350 or fa <= 10) or (170 <= fa <= 190):
        return 'Parallel'
    return 'Other'


def get_building_bin(value, edges, labels):
    for i in range(len(edges) - 1):
        if edges[i] <= value < edges[i + 1]:
            return labels[i]
    return None


def plot_layover_width_boxplot(df, building_col='far_building',
                                building_edges=[5,6,7,8,9,10,11,12,13, np.inf],
                                figsize=(14, 7), save=True, show=True):
    # Filter parallel
    df_plot = df.dropna(subset=['layover_shooting_m', 'width_node', 'layover_width_ratio', building_col, 'flow_angle']).copy()
    df_plot['flow_zone'] = df_plot['flow_angle'].apply(get_flow_angle_zone)
    df_parallel = df_plot[df_plot['flow_zone'] == 'Parallel'].copy()
    df_parallel = df_parallel[df_parallel[building_col] >= building_edges[0]]
    
    labels = []
    for i in range(len(building_edges) - 1):
        if building_edges[i + 1] == np.inf:
            labels.append(f'{building_edges[i]}+')
        else:
            labels.append(f'{int(building_edges[i])}-{int(building_edges[i+1])}')
    
    df_parallel['building_bin'] = df_parallel[building_col].apply(
        lambda x: get_building_bin(x, building_edges, labels)
    )
    df_parallel = df_parallel.dropna(subset=['building_bin'])
    
    print(f"Data: {len(df_parallel):,}")
    
    fig, ax = plt.subplots(figsize=figsize, facecolor='white')
    ax2 = ax.twinx()
    
    box_data_layover, box_data_width, box_data_ratio = [], [], []
    valid_labels, n_counts = [], []
    positions_layover, positions_width, positions_ratio = [], [], []
    
    pos = 1
    for lbl in labels:
        subset = df_parallel[df_parallel['building_bin'] == lbl]
        if len(subset) >= 30:
            box_data_layover.append(subset['layover_shooting_m'].dropna().values)
            box_data_width.append(subset['width_node'].dropna().values)
            box_data_ratio.append(subset['layover_width_ratio'].dropna().values)
            valid_labels.append(lbl)
            n_counts.append(len(subset))
            positions_layover.append(pos - 0.25)
            positions_width.append(pos)
            positions_ratio.append(pos + 0.25)
            pos += 1
    
    # Layover boxplot
    bp1 = ax.boxplot(box_data_layover, positions=positions_layover, widths=0.22,
                     patch_artist=True, showfliers=False)
    for patch in bp1['boxes']:
        patch.set_facecolor('salmon')
        patch.set_alpha(0.7)
    for w in bp1['whiskers']: w.set_color('darkred')
    for c in bp1['caps']: c.set_color('darkred')
    for m in bp1['medians']: m.set_color('darkred'); m.set_linewidth(2)
    
    # Width boxplot
    bp2 = ax.boxplot(box_data_width, positions=positions_width, widths=0.22,
                     patch_artist=True, showfliers=False)
    for patch in bp2['boxes']:
        patch.set_facecolor('steelblue')
        patch.set_alpha(0.7)
    for w in bp2['whiskers']: w.set_color('navy')
    for c in bp2['caps']: c.set_color('navy')
    for m in bp2['medians']: m.set_color('navy'); m.set_linewidth(2)
    
    # Ratio boxplot
    bp3 = ax2.boxplot(box_data_ratio, positions=positions_ratio, widths=0.22,
                      patch_artist=True, showfliers=False)
    for patch in bp3['boxes']:
        patch.set_facecolor('lightgreen')
        patch.set_alpha(0.7)
    for w in bp3['whiskers']: w.set_color('darkgreen')
    for c in bp3['caps']: c.set_color('darkgreen')
    for m in bp3['medians']: m.set_color('darkgreen'); m.set_linewidth(2)
    
    # Reference line at ratio=2
    ax2.axhline(y=2, color='dimgray', linestyle='--', linewidth=2, alpha=0.8)
    
    # X-axis with n labels below
    ax.set_xticks(range(1, len(valid_labels) + 1))
    tick_labels = [f'{lbl}\n(n={n:,})' for lbl, n in zip(valid_labels, n_counts)]
    ax.set_xticklabels(tick_labels, fontsize=15)
    
    ax.set_xlabel('Far Building Height (m)', fontsize=15, fontweight='bold', labelpad=15)
    ax.set_ylabel('Distance (m)', fontsize=15, fontweight='bold')
    ax2.set_ylabel(r'Layover/Width Ratio ($\Delta r/w$)', fontsize=15, fontweight='bold', color='darkgreen')
    ax2.tick_params(axis='y', colors='darkgreen', labelsize=15)
    ax.tick_params(labelsize=15)
    ax.set_title('Layover, Width & Ratio by Building Height', fontsize=16, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    legend_elements = [Patch(facecolor='salmon', alpha=0.7, label=r'Layover ($\Delta r$)'),
                       Patch(facecolor='steelblue', alpha=0.7, label='River Width (w)'),
                       Patch(facecolor='lightgreen', alpha=0.7, label=r'Ratio ($\Delta r/w$)')]
    ax.legend(handles=legend_elements, loc='upper left', fontsize=15, framealpha=0.9)
    
    plt.tight_layout()
    
    if save:
        fig.savefig(OUTPUT_DIR / "boxplot_layover_width_by_building.png", dpi=300, bbox_inches='tight')
        fig.savefig(OUTPUT_DIR / "boxplot_layover_width_by_building.pdf", dpi=300, bbox_inches='tight')
        print(f"Saved: {OUTPUT_DIR / 'boxplot_layover_width_by_building.png'}")
    
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    return fig
fig = plot_layover_width_boxplot(final_df)

In [ ]:
final_df.columns[:50]

In [ ]:
"""
Boxplot: SWOT Quality Metrics by Far Building Bins
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

OUTPUT_DIR = Path("figures")
OUTPUT_DIR.mkdir(exist_ok=True)


def get_flow_angle_zone(flow_angle):
    fa = flow_angle % 360
    if (fa >= 350 or fa <= 10) or (170 <= fa <= 190):
        return 'Parallel'
    return 'Other'


def get_building_bin(value, edges, labels):
    for i in range(len(edges) - 1):
        if edges[i] <= value < edges[i + 1]:
            return labels[i]
    return None


def plot_swot_metrics_boxplot(df, building_col='far_building',
                               building_edges=[5,6,7,8,9,10,11,12,13,14, np.inf],
                               figsize=(18, 10), save=True, show=True):
    df = df.copy()
    df['area_ratio'] = df['area_detct'] / df['area_total']
    
    # Filter parallel
    df_plot = df.dropna(subset=[building_col, 'flow_angle']).copy()
    df_plot['flow_zone'] = df_plot['flow_angle'].apply(get_flow_angle_zone)
    df_parallel = df_plot[df_plot['flow_zone'] == 'Parallel'].copy()
    df_parallel = df_parallel[df_parallel[building_col] >= building_edges[0]]
    
    labels = []
    for i in range(len(building_edges) - 1):
        if building_edges[i + 1] == np.inf:
            labels.append(f'{building_edges[i]}+')
        else:
            labels.append(f'{int(building_edges[i])}-{int(building_edges[i+1])}')
    
    df_parallel['building_bin'] = df_parallel[building_col].apply(
        lambda x: get_building_bin(x, building_edges, labels)
    )
    df_parallel = df_parallel.dropna(subset=['building_bin'])
    
    print(f"Data: {len(df_parallel):,}")
    
    # 5 metrics
    metrics = [
        ('slope2', 'slope2', 'steelblue'),
        ('partial_f', 'partial_f', 'salmon'),
        ('ice_dyn_f', 'ice_dyn_f', 'lightgreen'),
        ('obs_frac_n', 'obs_frac_n', 'plum'),
        ('area_ratio', 'area_detct / area_total', 'moccasin')
    ]
    
    fig, axes = plt.subplots(2, 3, figsize=figsize, facecolor='white')
    axes = axes.flatten()
    
    for idx, (col, title, color) in enumerate(metrics):
        ax = axes[idx]
        
        box_data, valid_labels, n_counts, positions = [], [], [], []
        pos = 1
        for lbl in labels:
            subset = df_parallel[df_parallel['building_bin'] == lbl]
            vals = subset[col].dropna().values
            if len(vals) >= 30:
                box_data.append(vals)
                valid_labels.append(lbl)
                n_counts.append(len(vals))
                positions.append(pos)
                pos += 1
        
        if box_data:
            bp = ax.boxplot(box_data, positions=positions, widths=0.5,
                            patch_artist=True, showfliers=False)
            for patch in bp['boxes']:
                patch.set_facecolor(color)
                patch.set_alpha(0.7)
            for m in bp['medians']: m.set_color('darkred'); m.set_linewidth(2)
        
        ax.set_xticks(positions)
        tick_labels = [f'{lbl}\n(n={n:,})' for lbl, n in zip(valid_labels, n_counts)]
        ax.set_xticklabels(tick_labels, fontsize=11)
        ax.set_xlabel('Far Building Height (m)', fontsize=12, fontweight='bold')
        ax.set_ylabel(title, fontsize=12, fontweight='bold')
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.grid(True, alpha=0.3, axis='y')
    
    # Hide 6th panel
    axes[5].axis('off')
    
    fig.suptitle('SWOT Quality Metrics by Building Height (Parallel)', fontsize=16, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    
    if save:
        fig.savefig(OUTPUT_DIR / "boxplot_swot_metrics_by_building.png", dpi=300, bbox_inches='tight')
        fig.savefig(OUTPUT_DIR / "boxplot_swot_metrics_by_building.pdf", dpi=300, bbox_inches='tight')
        print(f"Saved: {OUTPUT_DIR / 'boxplot_swot_metrics_by_building.png'}")
    
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    return fig

fig = plot_swot_metrics_boxplot(final_df)

In [ ]:
"""
Boxplot: Near ISA & Near Building Height by Far Building Bins
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from pathlib import Path

OUTPUT_DIR = Path("figures")
OUTPUT_DIR.mkdir(exist_ok=True)


def get_flow_angle_zone(flow_angle):
    fa = flow_angle % 360
    if (fa >= 350 or fa <= 10) or (170 <= fa <= 190):
        return 'Parallel'
    return 'Other'


def get_building_bin(value, edges, labels):
    for i in range(len(edges) - 1):
        if edges[i] <= value < edges[i + 1]:
            return labels[i]
    return None


def plot_near_isa_building_boxplot(df, building_col='far_building',
                                    building_edges=[5,6,7,8,9,10,11,12,13,14, np.inf],
                                    figsize=(14, 7), save=True, show=True):
    # Filter parallel
    df_plot = df.dropna(subset=['near_isa', 'near_building', building_col, 'flow_angle']).copy()
    df_plot['flow_zone'] = df_plot['flow_angle'].apply(get_flow_angle_zone)
    df_parallel = df_plot[df_plot['flow_zone'] == 'Parallel'].copy()
    df_parallel = df_parallel[df_parallel[building_col] >= building_edges[0]]
    
    labels = []
    for i in range(len(building_edges) - 1):
        if building_edges[i + 1] == np.inf:
            labels.append(f'{building_edges[i]}+')
        else:
            labels.append(f'{int(building_edges[i])}-{int(building_edges[i+1])}')
    
    df_parallel['building_bin'] = df_parallel[building_col].apply(
        lambda x: get_building_bin(x, building_edges, labels)
    )
    df_parallel = df_parallel.dropna(subset=['building_bin'])
    
    print(f"Data: {len(df_parallel):,}")
    
    fig, ax = plt.subplots(figsize=figsize, facecolor='white')
    ax2 = ax.twinx()
    
    box_data_isa, box_data_building = [], []
    valid_labels, n_counts = [], []
    positions_isa, positions_building = [], []
    
    pos = 1
    for lbl in labels:
        subset = df_parallel[df_parallel['building_bin'] == lbl]
        if len(subset) >= 30:
            box_data_isa.append(subset['near_isa'].dropna().values)
            box_data_building.append(subset['near_building'].dropna().values)
            valid_labels.append(lbl)
            n_counts.append(len(subset))
            positions_isa.append(pos - 0.2)
            positions_building.append(pos + 0.2)
            pos += 1
    
    # Near ISA boxplot (green) - left y-axis
    bp1 = ax.boxplot(box_data_isa, positions=positions_isa, widths=0.35,
                     patch_artist=True, showfliers=False)
    for patch in bp1['boxes']:
        patch.set_facecolor('lightgreen')
        patch.set_alpha(0.7)
    for w in bp1['whiskers']: w.set_color('darkgreen')
    for c in bp1['caps']: c.set_color('darkgreen')
    for m in bp1['medians']: m.set_color('darkgreen'); m.set_linewidth(2)
    
    # Near Building boxplot (purple) - right y-axis
    bp2 = ax2.boxplot(box_data_building, positions=positions_building, widths=0.35,
                      patch_artist=True, showfliers=False)
    for patch in bp2['boxes']:
        patch.set_facecolor('plum')
        patch.set_alpha(0.7)
    for w in bp2['whiskers']: w.set_color('purple')
    for c in bp2['caps']: c.set_color('purple')
    for m in bp2['medians']: m.set_color('purple'); m.set_linewidth(2)
    
    # X-axis
    ax.set_xticks(range(1, len(valid_labels) + 1))
    tick_labels = [f'{lbl}\n(n={n:,})' for lbl, n in zip(valid_labels, n_counts)]
    ax.set_xticklabels(tick_labels, fontsize=15)
    
    ax.set_xlabel('Far Building Height (m)', fontsize=15, fontweight='bold', labelpad=15)
    ax.set_ylabel('Near ISA (%)', fontsize=15, fontweight='bold', color='darkgreen')
    ax2.set_ylabel('Near Building Height (m)', fontsize=15, fontweight='bold', color='purple')
    ax.tick_params(axis='y', colors='darkgreen', labelsize=15)
    ax2.tick_params(axis='y', colors='purple', labelsize=15)
    ax.tick_params(axis='x', labelsize=15)
    ax.set_title('Near ISA & Near Building Height by Far Building Height (Parallel)', fontsize=16, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    legend_elements = [Patch(facecolor='lightgreen', alpha=0.7, label='Near ISA (%)'),
                       Patch(facecolor='plum', alpha=0.7, label='Near Building Height (m)')]
    ax.legend(handles=legend_elements, loc='upper left', fontsize=15, framealpha=0.9)
    
    plt.tight_layout()
    
    if save:
        fig.savefig(OUTPUT_DIR / "boxplot_near_isa_building_by_far_building.png", dpi=300, bbox_inches='tight')
        fig.savefig(OUTPUT_DIR / "boxplot_near_isa_building_by_far_building.pdf", dpi=300, bbox_inches='tight')
        print(f"Saved: {OUTPUT_DIR / 'boxplot_near_isa_building_by_far_building.png'}")
    
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    return fig

fig = plot_near_isa_building_boxplot(final_df)

#### shadowing + near building (used)

In [ ]:
final_df['shadowing_dist'] = np.abs(final_df.xtrk_dist_node) * final_df.near_building / final_df.sat_altitude

In [ ]:
final_df['shadowing_dist'] = final_df.near_building * final_df.tan

In [ ]:
np.abs(final_df.xtrk_dist_node).describe()

In [ ]:
final_df.near_building.describe()

In [ ]:
final_df.sat_altitude.describe()

In [ ]:
"""
Boxplot: Shadow Length & Width by Far Building Bins
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from pathlib import Path

OUTPUT_DIR = Path("figures")
OUTPUT_DIR.mkdir(exist_ok=True)


def get_flow_angle_zone(flow_angle):
    fa = flow_angle % 360
    if (fa >= 350 or fa <= 10) or (170 <= fa <= 190):
        return 'Parallel'
    return 'Other'


def get_building_bin(value, edges, labels):
    for i in range(len(edges) - 1):
        if edges[i] <= value < edges[i + 1]:
            return labels[i]
    return None


def plot_shadow_width_boxplot(df, building_col='far_building',
                               building_edges=[5,6,7,8,9,10,11,12,13,14, np.inf],
                               figsize=(14, 7), save=True, show=True):
    # Calculate shadow length
    df = df.copy()
    df['shadow_length'] = np.abs(df['xtrk_dist_node']) * df['far_building'] / df['sat_altitude']
    df['shadow_width_ratio'] = df['shadow_length'] / df['width_node']
    
    # Filter parallel
    df_plot = df.dropna(subset=['shadow_length', 'width_node', 'shadow_width_ratio', building_col, 'flow_angle']).copy()
    df_plot['flow_zone'] = df_plot['flow_angle'].apply(get_flow_angle_zone)
    df_parallel = df_plot[df_plot['flow_zone'] == 'Parallel'].copy()
    df_parallel = df_parallel[df_parallel[building_col] >= building_edges[0]]
    
    labels = []
    for i in range(len(building_edges) - 1):
        if building_edges[i + 1] == np.inf:
            labels.append(f'{building_edges[i]}+')
        else:
            labels.append(f'{int(building_edges[i])}-{int(building_edges[i+1])}')
    
    df_parallel['building_bin'] = df_parallel[building_col].apply(
        lambda x: get_building_bin(x, building_edges, labels)
    )
    df_parallel = df_parallel.dropna(subset=['building_bin'])
    
    print(f"Data: {len(df_parallel):,}")
    
    fig, ax = plt.subplots(figsize=figsize, facecolor='white')
    ax2 = ax.twinx()
    
    box_data_shadow, box_data_width, box_data_ratio = [], [], []
    valid_labels, n_counts = [], []
    positions_shadow, positions_width, positions_ratio = [], [], []
    
    pos = 1
    for lbl in labels:
        subset = df_parallel[df_parallel['building_bin'] == lbl]
        if len(subset) >= 30:
            box_data_shadow.append(subset['shadow_length'].dropna().values)
            box_data_width.append(subset['width_node'].dropna().values)
            box_data_ratio.append(subset['shadow_width_ratio'].dropna().values)
            valid_labels.append(lbl)
            n_counts.append(len(subset))
            positions_shadow.append(pos - 0.25)
            positions_width.append(pos)
            positions_ratio.append(pos + 0.25)
            pos += 1
    
    # Shadow boxplot (purple)
    bp1 = ax.boxplot(box_data_shadow, positions=positions_shadow, widths=0.22,
                     patch_artist=True, showfliers=False)
    for patch in bp1['boxes']:
        patch.set_facecolor('plum')
        patch.set_alpha(0.7)
    for w in bp1['whiskers']: w.set_color('purple')
    for c in bp1['caps']: c.set_color('purple')
    for m in bp1['medians']: m.set_color('purple'); m.set_linewidth(2)
    
    # # Width boxplot (steelblue)
    # bp2 = ax.boxplot(box_data_width, positions=positions_width, widths=0.22,
    #                  patch_artist=True, showfliers=False)
    # for patch in bp2['boxes']:
    #     patch.set_facecolor('steelblue')
    #     patch.set_alpha(0.7)
    # for w in bp2['whiskers']: w.set_color('navy')
    # for c in bp2['caps']: c.set_color('navy')
    # for m in bp2['medians']: m.set_color('navy'); m.set_linewidth(2)
    
    # Ratio boxplot (orange)
    bp3 = ax2.boxplot(box_data_ratio, positions=positions_ratio, widths=0.22,
                      patch_artist=True, showfliers=False)
    for patch in bp3['boxes']:
        patch.set_facecolor('moccasin')
        patch.set_alpha(0.7)
    for w in bp3['whiskers']: w.set_color('darkorange')
    for c in bp3['caps']: c.set_color('darkorange')
    for m in bp3['medians']: m.set_color('darkorange'); m.set_linewidth(2)
    
    # Reference line at ratio=2
    ax2.axhline(y=2, color='dimgray', linestyle='--', linewidth=2, alpha=0.8)
    
    # X-axis with n labels below
    ax.set_xticks(range(1, len(valid_labels) + 1))
    tick_labels = [f'{lbl}\n(n={n:,})' for lbl, n in zip(valid_labels, n_counts)]
    ax.set_xticklabels(tick_labels, fontsize=15)
    
    ax.set_xlabel('Far Building Height (m)', fontsize=15, fontweight='bold', labelpad=15)
    ax.set_ylabel('Distance (m)', fontsize=15, fontweight='bold')
    ax2.set_ylabel(r'Shadow/Width Ratio ($L_{shadow}/w$)', fontsize=15, fontweight='bold', color='darkorange')
    ax2.tick_params(axis='y', colors='darkorange', labelsize=15)
    ax.tick_params(labelsize=15)
    ax.set_title('Shadow Length, Width & Ratio by Building Height', fontsize=16, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    legend_elements = [Patch(facecolor='plum', alpha=0.7, label=r'Shadow Length ($L_{shadow}$)'),
                       Patch(facecolor='steelblue', alpha=0.7, label='River Width (w)'),
                       Patch(facecolor='moccasin', alpha=0.7, label=r'Ratio ($L_{shadow}/w$)')]
    ax.legend(handles=legend_elements, loc='upper left', fontsize=15, framealpha=0.9)
    
    plt.tight_layout()
    
    if save:
        fig.savefig(OUTPUT_DIR / "boxplot_shadow_width_by_building.png", dpi=300, bbox_inches='tight')
        fig.savefig(OUTPUT_DIR / "boxplot_shadow_width_by_building.pdf", dpi=300, bbox_inches='tight')
        print(f"Saved: {OUTPUT_DIR / 'boxplot_shadow_width_by_building.png'}")
    
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    return fig

fig = plot_shadow_width_boxplot(final_df)

### Flow angle wise CDF analysis

In [ ]:
final_df.width_node.quantile(0.95)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# 스타일 초기화 및 설정
plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

OUTPUT_DIR = Path("figures")
OUTPUT_DIR.mkdir(exist_ok=True)

def get_flow_angle_zone(flow_angle):
    fa = flow_angle % 360
    if (fa >= 350 or fa <= 10) or (170 <= fa <= 190):
        return 'Parallel'
    elif (80 <= fa <= 100) or (260 <= fa <= 280):
        return 'Perpendicular'
    else:
        return 'Other'

def resolve_overlaps(values, min_dist):
    """
    겹치는 값들을 찾아 오른쪽(위)으로 밀어내어 위치를 조정하는 함수.
    values: (값, 원래인덱스) 튜플의 리스트 (값 기준 정렬됨)
    min_dist: 텍스트 간 최소 거리
    """
    n = len(values)
    if n == 0:
        return []
    
    # 1. 값만 추출하여 위치 배열 생성
    positions = np.array([v[0] for v in values], dtype=float)
    
    # 2. 겹침 해결 (오른쪽으로 밀기)
    # 여러 번 반복하여 연쇄적인 겹침을 해소합니다.
    for _ in range(3): # 3번 정도 반복하면 대부분 해소됨
        for i in range(1, n):
            dist = positions[i] - positions[i-1]
            if dist < min_dist:
                # 겹치면 뒤의 것을 밈
                push = min_dist - dist
                positions[i] += push
    
    return positions

def plot_cdf_single(ax, data, y_col, geom_col, geometry_categories, title, threshold=0.1832, 
                    show_xlabel=True, zone_key=None):
    """
    CDF plot with tight axis labels and no connecting lines.
    """
    ax.set_facecolor('white')
    
    # 색상 설정
    colors = {
        'Neither': 'dimgray',     
        'Near': 'teal',         
        'Far': 'coral',           
        'Both': 'olivedrab'          
    }
    
    label_map = {
        'Neither': 'Neither',
        'Near': 'Near-Nadir',
        'Far': 'Far-Nadir',
        'Both': 'Both'
    } if zone_key == 'Perpendicular' else {
        'Neither': 'Neither',
        'Near': 'Near',
        'Far': 'Far',
        'Both': 'Both'
    }
    
    stats_dict = {}
    
    # 라벨링을 위한 데이터 수집
    p68_items = []      # (값, 색상, 라벨텍스트)
    cdf_T_items = []    # (값, 색상, 라벨텍스트)
    
    for cat in geometry_categories:
        cat_data = data[data[geom_col] == cat][y_col].dropna()
        if len(cat_data) > 0:
            sorted_data = np.sort(cat_data)
            cdf = np.arange(1, len(sorted_data) + 1) / len(sorted_data)
            
            color = colors.get(cat, 'gray')
            ax.plot(sorted_data, cdf, color=color, 
                    label=f'{label_map[cat]} (n={len(cat_data)})', linewidth=2)
            
            p68 = np.percentile(cat_data, 68)
            pct_above_T = 100 * (cat_data > threshold).sum() / len(cat_data)
            cdf_at_T = (cat_data <= threshold).sum() / len(cat_data)
            
            stats_dict[cat] = {
                'n': len(cat_data), 'p68': p68, 'pct_above_T': pct_above_T,
                'cdf_at_T': cdf_at_T, 'color': color, 'label': label_map[cat]
            }
            
            # 마커 (교차점)
            ax.scatter([p68], [0.68], color=color, s=30, zorder=5)
            ax.scatter([threshold], [cdf_at_T], color=color, s=40, zorder=5, marker='s')
            
            # 점선 가이드
            ax.vlines(x=p68, ymin=0, ymax=0.68, colors=color, linestyles=':', linewidth=1, alpha=0.6)
            ax.hlines(y=cdf_at_T, xmin=0, xmax=threshold, colors=color, linestyles=':', linewidth=1, alpha=0.6)
            
            # 라벨 데이터 저장
            p68_items.append({'val': p68, 'color': color, 'text': f'{p68:.2f}'})
            cdf_T_items.append({'val': cdf_at_T, 'color': color, 'text': f'{cdf_at_T:.2f}'.lstrip('0')})

    # 기준선 (Target Error, 68%)
    ax.axvline(x=threshold, color='black', linestyle='--', linewidth=1.5, alpha=0.8)
    ax.axhline(y=0.68, color='black', linewidth=1.5, alpha=0.8)
    
    # 고정 텍스트 (68%, T값)
    ax.text(0.01, 0.69, '68%', transform=ax.get_yaxis_transform(), 
            color='black', fontsize=12, fontweight='bold', ha='left', va='bottom')
    ax.text(threshold, 1.01, f'T={threshold}m', transform=ax.get_xaxis_transform(),
            color='black', fontsize=11, fontweight='bold', ha='center', va='bottom')

    # 스마트 라벨 배치 (선 없음, 축 밀착, 겹침 방지)
    
    # 1. X축 라벨 (68%tile Error)
    # 정렬
    p68_items.sort(key=lambda x: x['val'])
    
    # 겹침 계산 (최소 간격 0.15)
    x_vals = [(item['val'], i) for i, item in enumerate(p68_items)]
    adj_x_vals = resolve_overlaps(x_vals, min_dist=0.15)
    
    # 그리기: y=0.01 (축 바로 위)
    y_base_x_label = 0.01 
    for i, item in enumerate(p68_items):
        new_x = adj_x_vals[i]
        
        # 화살표 없이 텍스트만 배치
        ax.text(new_x, y_base_x_label, item['text'], 
                transform=ax.get_xaxis_transform(), # x는 data좌표, y는 axis(0~1)좌표
                color=item['color'], fontsize=10.5, fontweight='bold', 
                ha='center', va='bottom')

    # 2. Y축 라벨 (CDF at T)
    # 정렬
    cdf_T_items.sort(key=lambda x: x['val'])
    
    # 겹침 계산 (최소 간격 0.06)
    y_vals = [(item['val'], i) for i, item in enumerate(cdf_T_items)]
    adj_y_vals = resolve_overlaps(y_vals, min_dist=0.06)
    
    # 그리기: x=0.01 (축 바로 옆)
    x_base_y_label = 0.01
    for i, item in enumerate(cdf_T_items):
        new_y = adj_y_vals[i]
        
        # 화살표 없이 텍스트만 배치
        ax.text(x_base_y_label, new_y, item['text'], 
                transform=ax.get_yaxis_transform(), # x는 axis(0~1)좌표, y는 data좌표
                color=item['color'], fontsize=10.5, fontweight='bold', 
                ha='left', va='center')

    ax.set_ylim(0, 1)
    
    if show_xlabel:
        ax.set_xlabel(r'$|Relative\ WSE\ Error|$ (m)', color='black', fontsize=13)
    ax.set_ylabel('CDF', color='black', fontsize=13)
    ax.set_title(title, color='black', fontsize=14, fontweight='bold', pad=10)
    ax.tick_params(colors='black', labelsize=11)
    
    # 범례 위치 변경: Right Center
    ax.legend(loc='center right', fontsize=11, facecolor='white', 
              edgecolor='gray', labelcolor='black', framealpha=0.9)
    
    ax.grid(True, alpha=0.3, color='gray')
    for spine in ax.spines.values():
        spine.set_color('black')
    
    return stats_dict

def plot_stats_panel(ax, all_stats, threshold=0.1832):
    # (기존 통계 패널 코드 유지)
    ax.set_facecolor('white')
    ax.axis('off')
    
    colors = {'Neither': 'dimgray', 'Near': 'teal', 'Far': 'coral', 'Both': 'olivedrab'}
    categories = ['Neither', 'Near', 'Far', 'Both']
    zones = ['Parallel', 'Perpendicular', 'Other']
    
    ax.text(0.5, 1, 'Statistics Summary', transform=ax.transAxes,
            fontsize=16, fontweight='bold', color='black', ha='center', va='top')
    ax.text(0.5, 0.92, '68th Percentile |Error| (m)', transform=ax.transAxes,
            fontsize=13, fontweight='bold', color='black', ha='center', va='top')
    
    header = f"{'Category':<10}{'Parallel':>14}{'Perpendicular':>18}{'Other':>12}"
    ax.text(0.5, 0.84, header, transform=ax.transAxes,
            fontsize=12, color='black', ha='center', va='top', family='monospace')
    ax.text(0.5, 0.79, "─" * 54, transform=ax.transAxes,
            fontsize=12, color='gray', ha='center', va='top', family='monospace')
    
    y_pos = 0.74
    for cat in categories:
        row = f"{cat:<10}"
        for zone in zones:
            if zone in all_stats and cat in all_stats[zone]:
                p68 = all_stats[zone][cat]['p68']
                row += f"{p68:>18.3f}" if zone == 'Perpendicular' else f"{p68:>14.3f}" if zone == 'Parallel' else f"{p68:>12.3f}"
            else:
                row += f"{'N/A':>18}" if zone == 'Perpendicular' else f"{'N/A':>14}" if zone == 'Parallel' else f"{'N/A':>12}"
        ax.text(0.5, y_pos, row, transform=ax.transAxes, fontsize=12, color=colors[cat], ha='center', va='top', family='monospace')
        y_pos -= 0.07
    
    y_pos -= 0.06
    ax.text(0.5, y_pos, f'% with |Error| > T (T={threshold}m)', transform=ax.transAxes,
            fontsize=13, fontweight='bold', color='black', ha='center', va='top')
    
    y_pos -= 0.08
    ax.text(0.5, y_pos, header, transform=ax.transAxes, fontsize=12, color='black', ha='center', va='top', family='monospace')
    y_pos -= 0.05
    ax.text(0.5, y_pos, "─" * 54, transform=ax.transAxes, fontsize=12, color='gray', ha='center', va='top', family='monospace')
    y_pos -= 0.05
    
    for cat in categories:
        row = f"{cat:<10}"
        for zone in zones:
            if zone in all_stats and cat in all_stats[zone]:
                pct = all_stats[zone][cat]['pct_above_T']
                row += f"{pct:>17.1f}%" if zone == 'Perpendicular' else f"{pct:>13.1f}%" if zone == 'Parallel' else f"{pct:>11.1f}%"
            else:
                row += f"{'N/A':>18}" if zone == 'Perpendicular' else f"{'N/A':>14}" if zone == 'Parallel' else f"{'N/A':>12}"
        ax.text(0.5, y_pos, row, transform=ax.transAxes, fontsize=12, color=colors[cat], ha='center', va='top', family='monospace')
        y_pos -= 0.07

def plot_cdf_2x2(df, y_col, geom_col='geometry_category', threshold=0.1832, 
                 figsize=(14, 12), xlim=(0, 2), save=True, show=True):
    df_plot = df.dropna(subset=[y_col, 'flow_angle', geom_col]).copy()
    if len(df_plot) == 0:
        return None
    
    df_plot['flow_zone'] = df_plot['flow_angle'].apply(get_flow_angle_zone)
    fig, axes = plt.subplots(2, 2, figsize=figsize, facecolor='white')
    
    zones = [
        ('Parallel', 'Parallel (350-10°, 170-190°)', axes[0, 0]),
        ('Perpendicular', 'Perpendicular (80-100°, 260-280°)', axes[0, 1]),
        ('Other', 'Other Angles', axes[1, 0]),
    ]
    
    all_stats = {}
    geometry_cats = ['Neither', 'Near', 'Far', 'Both']
    
    for zone_key, zone_title, ax in zones:
        df_zone = df_plot[df_plot['flow_zone'] == zone_key]
        title = f'{zone_title}\n(n={len(df_zone):,})'
        stats = plot_cdf_single(ax, df_zone, y_col, geom_col, geometry_cats, title, 
                                threshold, show_xlabel=True, zone_key=zone_key)
        ax.set_xlim(xlim)
        all_stats[zone_key] = stats
    
    plot_stats_panel(axes[1, 1], all_stats, threshold)
    
    fig.suptitle(r'CDF of $|Relative\ WSE\ Error|$ by ISA Location and Flow Angle', 
                 fontsize=18, fontweight='bold', color='black', y=0.98)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    
    if save:
        filename = f"figure5_cdf_2x2_{y_col}_{geom_col}.png"
        fig.savefig(OUTPUT_DIR / filename, dpi=300, facecolor='white', bbox_inches='tight')
        filename = f"figure5_cdf_2x2_{y_col}_{geom_col}.pdf"
        fig.savefig(OUTPUT_DIR / filename, dpi=300, facecolor='white', bbox_inches='tight')
        print(f"Saved: {OUTPUT_DIR / filename}")
    
    if show:
        plt.show()
    
    return fig

In [ ]:
plot_cdf_2x2(final_df, 'abs_error_node', geom_col='50_geometry_category', xlim=(0, 2), show=False)

### Parallel flow angle zone focus analysis (near-far isa)

#### CDF only

In [ ]:
"""
CDF Plot: Parallel Zone Analysis with ISA Bins
===============================================
Using log(x + offset) transformation
- Plot 1: Original Parallel zone CDF (by sample_geometry_category)
- Plot 2: Parallel zone - Near ISA bins (10% intervals)
- Plot 3: Parallel zone - Far ISA bins (10% intervals)
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path
import matplotlib.cm as cm

plt.style.use('dark_background')

OUTPUT_DIR = Path("unit_level_analysis")
OUTPUT_DIR.mkdir(exist_ok=True)


def get_flow_angle_zone(flow_angle):
    """
    Classify flow angle into zones
    """
    fa = flow_angle % 360
    if (fa >= 350 or fa <= 10) or (170 <= fa <= 190):
        return 'Parallel'
    elif (80 <= fa <= 100) or (260 <= fa <= 280):
        return 'Perpendicular'
    else:
        return 'Other'


def get_isa_bin(isa_value):
    """
    Classify ISA into 10% bins
    Assumes isa_value is percentage (0~100)
    """
    if isa_value <= 5:
        return '0-5'
    elif isa_value <= 10:
        return '5-10'
    elif isa_value <= 20:
        return '10-20'
    elif isa_value <= 30:
        return '20-30'
    elif isa_value <= 40:
        return '30-40'
    elif isa_value <= 50:
        return '40-50'
    elif isa_value <= 60:
        return '50-60'
    elif isa_value <= 70:
        return '60-70'
    elif isa_value <= 80:
        return '70-80'
    elif isa_value <= 90:
        return '80-90'
    else:
        return '90-100'


def plot_cdf_original(ax, data, y_col, threshold=0.1832, offset=1):
    """
    Original CDF plot by sample_geometry_category
    Using log(x + offset) transformation
    """
    ax.set_facecolor('black')
    
    colors = {
        'NA': 'white',
        'Near': '#3399ff',
        'Far': '#ff6666',
        'Near/Far': '#ffdd00'
    }
    
    geometry_cats = ['NA', 'Near', 'Far', 'Near/Far']
    pct_data = []
    p68_data = []
    
    all_errors = data[y_col].dropna()
    if len(all_errors) > 0:
        x_min_orig = all_errors.min()
        x_max_orig = all_errors.max()
    else:
        x_min_orig, x_max_orig = 0, 100
    
    for cat in geometry_cats:
        cat_data = data[data['sample_geometry_category'] == cat][y_col].dropna()
        if len(cat_data) > 0:
            sorted_data = np.sort(cat_data)
            cdf = np.arange(1, len(sorted_data) + 1) / len(sorted_data)
            
            # +offset 변환
            ax.plot(sorted_data + offset, cdf, color=colors[cat], 
                   label=f'{cat} (n={len(cat_data)})', linewidth=2)
            
            p68 = np.percentile(cat_data, 68)
            p68_data.append((cat, p68, colors[cat]))
            
            pct_below = 100 * (cat_data < threshold).sum() / len(cat_data)
            pct_above = 100 * (cat_data >= threshold).sum() / len(cat_data)
            pct_data.append((cat, pct_below, pct_above, colors[cat]))
    
    # Reference lines (+offset)
    ax.axvline(x=threshold + offset, color='orange', linestyle='-', linewidth=1.5, alpha=0.8)
    ax.axvline(x=0 + offset, color='white', linestyle='--', linewidth=0.8, alpha=0.5)
    ax.axhline(y=0.68, color='lime', linestyle='--', linewidth=1.5, alpha=0.8)
    ax.text(0.02, 0.70, '68%tile', transform=ax.transAxes, color='lime', fontsize=8, fontweight='bold')
    
    # 68th percentile markers
    y_offsets = [0.01, -0.04, 0.01, -0.04]
    va_aligns = ['bottom', 'top', 'bottom', 'top']
    
    for i, (cat, p68, color) in enumerate(p68_data):
        ax.axvline(x=p68 + offset, color=color, linestyle=':', linewidth=1, alpha=0.6)
        ax.scatter([p68 + offset], [0.68], color=color, s=30, zorder=5)
        
        y_pos = y_offsets[i % 4]
        va = va_aligns[i % 4]
        
        # 라벨은 원래 값
        ax.text(p68 + offset, y_pos, f'{p68:.2f}', transform=ax.get_xaxis_transform(),
                color=color, fontsize=9, fontweight='bold', ha='center', va=va)
    
    # Threshold percentage text
    if pct_data:
        ax.text(0.98, 0.62, f'{"T=0.1832m":8s} {"<T":>6s} {">T":>6s}', 
                transform=ax.transAxes, fontsize=8, color='lime', va='top', ha='right', 
                family='monospace')
        y_pos = 0.54
        for cat, below, above, color in pct_data:
            text = f'{cat:8s} {below:5.0f}% {above:5.0f}%'
            ax.text(0.98, y_pos, text, transform=ax.transAxes, 
                    fontsize=8, color=color, va='top', ha='right', family='monospace')
            y_pos -= 0.08
    
    # Log scale (symlog 대신)
    ax.set_xscale('log')
    ax.set_xlim((x_min_orig + offset) * 0.9, (x_max_orig + offset) * 1.1)
    ax.set_ylim(0, 1)
    
    # X축 라벨 (offset 빼서 원래 값으로 표시)
    def format_func(value, tick_number):
        original = value - offset
        if abs(original) < 0.01:
            return '0'
        elif abs(original) >= 10:
            return f'{original:.0f}'
        elif abs(original) >= 1:
            return f'{original:.1f}'
        else:
            return f'{original:.2f}'
    
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(format_func))
    
    tick_values_orig = [0, 0.2, 0.5, 1, 2, 5, 10, 20, 50, 100]
    tick_values = [v + offset for v in tick_values_orig 
                   if (x_min_orig + offset) * 0.9 <= v + offset <= (x_max_orig + offset) * 1.1]
    ax.set_xticks(tick_values)
    
    ax.set_xlabel('Error (m)', color='white', fontsize=10)
    ax.set_ylabel('CDF', color='white', fontsize=10)
    ax.set_title(f'Parallel (350-10°, 170-190°)\n(n={len(data):,})', 
                 color='white', fontsize=12, fontweight='bold')
    ax.tick_params(colors='white', labelsize=9)
    ax.legend(loc='lower right', fontsize=8, facecolor='black', edgecolor='white', labelcolor='white')
    ax.grid(True, alpha=0.2, color='white')
    for spine in ax.spines.values():
        spine.set_color('white')


def plot_cdf_by_isa_bin(ax, data, y_col, isa_col, title, threshold=0.1832, offset=1):
    """
    CDF plot by ISA bins (10% intervals)
    Using log(x + offset) transformation
    """
    ax.set_facecolor('black')
    
    bin_order = ['0-5', '5-10', '10-20', '20-30', '30-40', '40-50', 
                 '50-60', '60-70', '70-80', '80-90', '90-100']
    
    cmap = cm.get_cmap('cool', 11)
    bin_colors = {bin_order[i]: cmap(i) for i in range(11)}
    
    data = data.copy()
    data['isa_bin'] = data[isa_col].apply(get_isa_bin)
    
    all_errors = data[y_col].dropna()
    if len(all_errors) > 0:
        x_min_orig = all_errors.min()
        x_max_orig = all_errors.max()
    else:
        x_min_orig, x_max_orig = 0, 100
    
    # Plot CDF for each bin
    plotted_bins = []
    for bin_label in bin_order:
        bin_data = data[data['isa_bin'] == bin_label][y_col].dropna()
        if len(bin_data) > 0:
            sorted_data = np.sort(bin_data)
            cdf = np.arange(1, len(sorted_data) + 1) / len(sorted_data)
            color = bin_colors[bin_label]
            
            # +offset 변환
            ax.plot(sorted_data + offset, cdf, color=color, linewidth=1.5)
            
            p68 = np.percentile(bin_data, 68)
            plotted_bins.append((bin_label, p68, color, len(bin_data)))
    
    # 68th percentile line
    ax.axhline(y=0.68, color='lime', linestyle='--', linewidth=1.5, alpha=0.8)
    ax.text(0.02, 0.70, '68%tile', transform=ax.transAxes, color='lime', fontsize=8, fontweight='bold')
    
    # Reference lines (+offset)
    ax.axvline(x=threshold + offset, color='orange', linestyle='-', linewidth=1.5, alpha=0.8)
    ax.axvline(x=0 + offset, color='white', linestyle='--', linewidth=0.8, alpha=0.5)
    
    # Add text labels - alternate 4 positions
    y_offsets_text = [0.03, 0.08, -0.03, -0.08]
    va_aligns = ['bottom', 'bottom', 'top', 'top']
    
    for i, (bin_label, p68, color, n) in enumerate(plotted_bins):
        ax.scatter([p68 + offset], [0.68], color=color, s=30, zorder=5)
        ax.axvline(x=p68 + offset, color=color, linestyle=':', linewidth=0.8, alpha=0.4)
        
        y_offset = y_offsets_text[i % 4]
        va = va_aligns[i % 4]
        
        ax.text(p68 + offset, 0.68 + y_offset, f'{bin_label}%', color=color, fontsize=8, fontweight='bold',
                ha='center', va=va)
    
    # Add 68th percentile summary text
    if plotted_bins:
        ax.text(0.98, 0.48, '68%tile Error (m)', transform=ax.transAxes,
                fontsize=8, color='lime', va='top', ha='right', family='monospace', fontweight='bold')
        
        y_pos = 0.40
        for bin_label, p68, color, n in plotted_bins:
            text = f'{bin_label:>5}%: {p68:.2f}m (n={n})'
            ax.text(0.98, y_pos, text, transform=ax.transAxes,
                    fontsize=7, color=color, va='top', ha='right', family='monospace')
            y_pos -= 0.04
    
    # Log scale (symlog 대신)
    ax.set_xscale('log')
    ax.set_xlim((x_min_orig + offset) * 0.9, (x_max_orig + offset) * 1.1)
    ax.set_ylim(0, 1)
    
    # X축 라벨 (offset 빼서 원래 값으로 표시)
    def format_func(value, tick_number):
        original = value - offset
        if abs(original) < 0.01:
            return '0'
        elif abs(original) >= 10:
            return f'{original:.0f}'
        elif abs(original) >= 1:
            return f'{original:.1f}'
        else:
            return f'{original:.2f}'
    
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(format_func))
    
    tick_values_orig = [0, 0.2, 0.5, 1, 2, 5, 10, 20, 50, 100]
    tick_values = [v + offset for v in tick_values_orig 
                   if (x_min_orig + offset) * 0.9 <= v + offset <= (x_max_orig + offset) * 1.1]
    ax.set_xticks(tick_values)
    
    ax.set_xlabel('Error (m)', color='white', fontsize=10)
    ax.set_ylabel('CDF', color='white', fontsize=10)
    ax.set_title(title, color='white', fontsize=12, fontweight='bold')
    ax.tick_params(colors='white', labelsize=9)
    ax.grid(True, alpha=0.2, color='white')
    for spine in ax.spines.values():
        spine.set_color('white')


def plot_parallel_isa_analysis(df, y_col, threshold=0.1832, figsize=(18, 5), 
                                save=True, show=False, offset=1):
    """
    Create 1x3 plot for Parallel zone ISA analysis
    """
    
    df_plot = df.dropna(subset=[y_col, 'flow_angle', 'sample_geometry_category', 'near_isa', 'far_isa']).copy()
    df_plot['flow_zone'] = df_plot['flow_angle'].apply(get_flow_angle_zone)
    df_parallel = df_plot[df_plot['flow_zone'] == 'Parallel']
    
    if len(df_parallel) == 0:
        print("Warning: No data in Parallel zone")
        return None
    
    print(f"Parallel zone: {len(df_parallel):,} samples")
    
    fig, axes = plt.subplots(1, 3, figsize=figsize, facecolor='black')
    
    plot_cdf_original(axes[0], df_parallel, y_col, threshold, offset)
    plot_cdf_by_isa_bin(axes[1], df_parallel, y_col, 'near_isa', 'By Near ISA (%)', threshold, offset)
    plot_cdf_by_isa_bin(axes[2], df_parallel, y_col, 'far_isa', 'By Far ISA (%)', threshold, offset)
    
    fig.suptitle(f'Parallel Zone Analysis: {y_col}', 
                 fontsize=14, fontweight='bold', color='white', y=1.02)
    
    plt.tight_layout()
    
    if save:
        filename = f"parallel_isa_analysis_{y_col}_log.png"
        filepath = OUTPUT_DIR / filename
        fig.savefig(filepath, dpi=150, facecolor='black', bbox_inches='tight')
        print(f"Saved: {filepath}")
    
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    return fig

In [ ]:
plot_cdf_by_flow_angle(final_df, y_col='abs_error_node')#, offset=0.1)

#### Far ISA only (boxplot + CDF and Parallel zone all)

In [ ]:
"""
Parallel Zone Analysis: 1x3 Plot (Far ISA Only)
================================================
Plot 1: Original CDF by sample_geometry_category
Plot 2: CDF by Far ISA bins
Plot 3: Boxplot by Far ISA bins

Filtering:
- Far ISA: far_ratio >= isa_ratio
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from pathlib import Path

plt.style.use('dark_background')

OUTPUT_DIR = Path("figures")
OUTPUT_DIR.mkdir(exist_ok=True)


def get_flow_angle_zone(flow_angle):
    fa = flow_angle % 360
    if (fa >= 350 or fa <= 10) or (170 <= fa <= 190):
        return 'Parallel'
    elif (80 <= fa <= 100) or (260 <= fa <= 280):
        return 'Perpendicular'
    else:
        return 'Other'


def get_isa_bin(isa_value):
    if isa_value <= 5:
        return '0-5'
    elif isa_value <= 10:
        return '5-10'
    elif isa_value <= 20:
        return '10-20'
    elif isa_value <= 30:
        return '20-30'
    elif isa_value <= 40:
        return '30-40'
    elif isa_value <= 50:
        return '40-50'
    elif isa_value <= 60:
        return '50-60'
    elif isa_value <= 70:
        return '60-70'
    elif isa_value <= 80:
        return '70-80'
    elif isa_value <= 90:
        return '80-90'
    else:
        return '90-100'


def plot_cdf_original(ax, data, y_col, threshold=0.1832, offset=0.1):
    """
    Original CDF plot by sample_geometry_category
    """
    ax.set_facecolor('#1a1a1a')
    
    colors = {
        'NA': 'white',
        'Near': '#3399ff',
        'Far': '#ff6666',
        'Near/Far': '#ffdd00'
    }
    
    geometry_cats = ['NA', 'Near', 'Far', 'Near/Far']
    pct_data = []
    p68_data = []
    
    all_errors = data[y_col].dropna()
    if len(all_errors) > 0:
        x_min_orig = all_errors.min()
        x_max_orig = all_errors.max()
    else:
        x_min_orig, x_max_orig = 0, 100
    
    for cat in geometry_cats:
        cat_data = data[data['sample_geometry_category'] == cat][y_col].dropna()
        if len(cat_data) > 0:
            sorted_data = np.sort(cat_data)
            cdf = np.arange(1, len(sorted_data) + 1) / len(sorted_data)
            
            ax.plot(sorted_data + offset, cdf, color=colors[cat], 
                   label=f'{cat} (n={len(cat_data)})', linewidth=2)
            
            p68 = np.percentile(cat_data, 68)
            p68_data.append((cat, p68, colors[cat]))
            
            pct_below = 100 * (cat_data < threshold).sum() / len(cat_data)
            pct_above = 100 * (cat_data >= threshold).sum() / len(cat_data)
            pct_data.append((cat, pct_below, pct_above, colors[cat]))
    
    # Reference lines
    ax.axvline(x=threshold + offset, color='orange', linestyle='-', linewidth=1.5, alpha=0.8)
    ax.axvline(x=0 + offset, color='white', linestyle='--', linewidth=0.8, alpha=0.5)
    ax.axhline(y=0.68, color='lime', linestyle='--', linewidth=1.5, alpha=0.8)
    ax.text(0.02, 0.70, '68%tile', transform=ax.transAxes, color='lime', fontsize=8, fontweight='bold')
    
    # 68th percentile markers
    y_offsets = [0.01, -0.04, 0.01, -0.04]
    va_aligns = ['bottom', 'top', 'bottom', 'top']
    
    for i, (cat, p68, color) in enumerate(p68_data):
        ax.axvline(x=p68 + offset, color=color, linestyle=':', linewidth=1, alpha=0.6)
        ax.scatter([p68 + offset], [0.68], color=color, s=30, zorder=5)
        
        y_pos = y_offsets[i % 4]
        va = va_aligns[i % 4]
        
        ax.text(p68 + offset, y_pos, f'{p68:.2f}', transform=ax.get_xaxis_transform(),
                color=color, fontsize=9, fontweight='bold', ha='center', va=va)
    
    # Threshold percentage text
    if pct_data:
        ax.text(0.98, 0.62, f'{"T=0.1832m":8s} {"<T":>6s} {">T":>6s}', 
                transform=ax.transAxes, fontsize=8, color='lime', va='top', ha='right', 
                family='monospace')
        y_pos = 0.54
        for cat, below, above, color in pct_data:
            text = f'{cat:8s} {below:5.0f}% {above:5.0f}%'
            ax.text(0.98, y_pos, text, transform=ax.transAxes, 
                    fontsize=8, color=color, va='top', ha='right', family='monospace')
            y_pos -= 0.08
    
    ax.set_xscale('log')
    ax.set_xlim((x_min_orig + offset) * 0.9, (x_max_orig + offset) * 1.1)
    ax.set_ylim(0, 1)
    
    def format_func(value, tick_number):
        original = value - offset
        if abs(original) < 0.01:
            return '0'
        elif abs(original) >= 10:
            return f'{original:.0f}'
        elif abs(original) >= 1:
            return f'{original:.1f}'
        else:
            return f'{original:.2f}'
    
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(format_func))
    
    tick_values_orig = [0, 0.2, 0.5, 1, 2, 5, 10, 20, 50, 100]
    tick_values = [v + offset for v in tick_values_orig 
                   if (x_min_orig + offset) * 0.9 <= v + offset <= (x_max_orig + offset) * 1.1]
    ax.set_xticks(tick_values)
    
    ax.set_xlabel('abs_error_node (m)', color='white', fontsize=9)
    ax.set_ylabel('CDF', color='white', fontsize=9)
    ax.set_title(f'Parallel Zone (n={len(data):,})', 
                 color='white', fontsize=11, fontweight='bold')
    ax.tick_params(colors='white', labelsize=8)
    ax.legend(loc='lower right', fontsize=7, facecolor='black', edgecolor='white', labelcolor='white')
    ax.grid(True, alpha=0.2, color='white')
    
    for spine in ax.spines.values():
        spine.set_color('white')


def plot_cdf_by_isa_bin(ax, data, y_col, isa_col, title, threshold=0.1832, offset=1):
    """
    CDF plot by ISA bins
    """
    ax.set_facecolor('#1a1a1a')
    
    bin_order = ['0-5', '5-10', '10-20', '20-30', '30-40', '40-50', 
                 '50-60', '60-70', '70-80', '80-90', '90-100']
    
    cmap = cm.get_cmap('cool', 11)
    bin_colors = {bin_order[i]: cmap(i) for i in range(11)}
    
    data = data.copy()
    data['isa_bin'] = data[isa_col].apply(get_isa_bin)
    
    all_errors = data[y_col].dropna()
    if len(all_errors) > 0:
        x_min_orig = all_errors.min()
        x_max_orig = all_errors.max()
    else:
        x_min_orig, x_max_orig = 0, 100
    
    plotted_bins = []
    for bin_label in bin_order:
        bin_data = data[data['isa_bin'] == bin_label][y_col].dropna()
        if len(bin_data) > 0:
            sorted_data = np.sort(bin_data)
            cdf = np.arange(1, len(sorted_data) + 1) / len(sorted_data)
            color = bin_colors[bin_label]
            
            ax.plot(sorted_data + offset, cdf, color=color, linewidth=1.5)
            
            p68 = np.percentile(bin_data, 68)
            plotted_bins.append((bin_label, p68, color, len(bin_data)))
    
    ax.axhline(y=0.68, color='lime', linestyle='--', linewidth=1.5, alpha=0.8)
    ax.text(0.02, 0.70, '68%tile', transform=ax.transAxes, color='lime', fontsize=8, fontweight='bold')
    
    ax.axvline(x=threshold + offset, color='orange', linestyle='-', linewidth=1.5, alpha=0.8)
    ax.axvline(x=0 + offset, color='white', linestyle='--', linewidth=0.8, alpha=0.5)
    
    y_offsets_text = [0.03, 0.08, -0.03, -0.08]
    va_aligns = ['bottom', 'bottom', 'top', 'top']
    
    for i, (bin_label, p68, color, n) in enumerate(plotted_bins):
        ax.scatter([p68 + offset], [0.68], color=color, s=30, zorder=5)
        ax.axvline(x=p68 + offset, color=color, linestyle=':', linewidth=0.8, alpha=0.4)
        
        y_offset = y_offsets_text[i % 4]
        va = va_aligns[i % 4]
        
        ax.text(p68 + offset, 0.68 + y_offset, f'{bin_label}%', color=color, fontsize=7, fontweight='bold',
                ha='center', va=va)
    
    if plotted_bins:
        ax.text(0.98, 0.48, '68%tile Error (m)', transform=ax.transAxes,
                fontsize=7, color='lime', va='top', ha='right', family='monospace', fontweight='bold')
        
        y_pos = 0.40
        for bin_label, p68, color, n in plotted_bins:
            text = f'{bin_label:>5}%: {p68:.2f}m (n={n})'
            ax.text(0.98, y_pos, text, transform=ax.transAxes,
                    fontsize=6, color=color, va='top', ha='right', family='monospace')
            y_pos -= 0.035
    
    ax.set_xscale('log')
    ax.set_xlim((x_min_orig + offset) * 0.9, (x_max_orig + offset) * 1.1)
    ax.set_ylim(0, 1)
    
    def format_func(value, tick_number):
        original = value - offset
        if abs(original) < 0.01:
            return '0'
        elif abs(original) >= 10:
            return f'{original:.0f}'
        elif abs(original) >= 1:
            return f'{original:.1f}'
        else:
            return f'{original:.2f}'
    
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(format_func))
    
    tick_values_orig = [0, 0.2, 0.5, 1, 2, 5, 10, 20, 50, 100]
    tick_values = [v + offset for v in tick_values_orig 
                   if (x_min_orig + offset) * 0.9 <= v + offset <= (x_max_orig + offset) * 1.1]
    ax.set_xticks(tick_values)
    
    ax.set_xlabel('abs_error_node (m)', color='white', fontsize=9)
    ax.set_ylabel('CDF', color='white', fontsize=9)
    ax.set_title(title, color='white', fontsize=11, fontweight='bold')
    ax.tick_params(colors='white', labelsize=8)
    ax.grid(True, alpha=0.2, color='white')
    
    for spine in ax.spines.values():
        spine.set_color('white')


def plot_boxplot_by_isa_bin(ax, data, y_col, isa_col, title, highlight_bins=None):
    """
    Boxplot by ISA bins
    """
    ax.set_facecolor('#1a1a1a')
    
    if highlight_bins is None:
        highlight_bins = ['30-40', '50-60']
    
    bin_order = ['0-5', '5-10', '10-20', '20-30', '30-40', '40-50', 
                 '50-60', '60-70', '70-80', '80-90', '90-100']
    
    cmap = cm.get_cmap('cool', 11)
    bin_colors = {bin_order[i]: cmap(i) for i in range(11)}
    
    data = data.copy()
    data['isa_bin'] = data[isa_col].apply(get_isa_bin)
    
    existing_bins = [b for b in bin_order if b in data['isa_bin'].values]
    
    box_data = []
    positions = []
    colors = []
    bin_labels = []
    
    for i, bin_label in enumerate(existing_bins):
        bin_df = data[data['isa_bin'] == bin_label]
        if len(bin_df) > 0:
            error_vals = bin_df[y_col].dropna().values
            if len(error_vals) > 0:
                box_data.append(error_vals)
                positions.append(len(positions))
                colors.append(bin_colors[bin_label])
                bin_labels.append(bin_label)
    
    if len(positions) == 0:
        ax.text(0.5, 0.5, 'No Data', transform=ax.transAxes,
                ha='center', va='center', color='gray', fontsize=12)
        return
    
    bp = ax.boxplot(box_data, positions=positions, widths=0.6, patch_artist=True,
                    whis=1.5,
                    medianprops=dict(color='white', linewidth=2),
                    whiskerprops=dict(color='white'),
                    capprops=dict(color='white'),
                    flierprops=dict(marker='o', markersize=3, alpha=0.5))
    
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
        patch.set_edgecolor('white')
    
    for flier, color in zip(bp['fliers'], colors):
        flier.set_markerfacecolor(color)
        flier.set_markeredgecolor(color)
    
    for i, bin_label in enumerate(bin_labels):
        if bin_label in highlight_bins:
            ax.axvspan(i - 0.5, i + 0.5, alpha=0.15, color='yellow')
    
    ax.set_yscale('symlog', linthresh=1)
    
    error_ticks = [0.1, 0.5, 1, 2, 5, 10, 20]
    ax.set_yticks(error_ticks)
    ax.set_yticklabels([str(t) for t in error_ticks], fontsize=8, color='white')
    ax.set_ylim(0.05, 30)
    
    ax.set_xticks(positions)
    ax.set_xticklabels(bin_labels, rotation=45, ha='right', color='white', fontsize=8)
    ax.set_xlabel(f'{isa_col} Bin (%)', color='white', fontsize=9)
    ax.set_ylabel('abs_error_node (m)', color='white', fontsize=9)
    ax.set_title(title, color='white', fontsize=11, fontweight='bold')
    
    ax.grid(True, alpha=0.2, color='white')
    ax.set_xlim(-0.5, len(bin_labels) - 0.5)
    
    for spine in ax.spines.values():
        spine.set_color('white')
    
    ax.tick_params(colors='white', labelsize=8)


def plot_parallel_far_analysis(df, y_col='abs_error_node', 
                                isa_ratio=0.5,
                                threshold=0.1832, offset=1,
                                highlight_bins=None,
                                figsize=(16, 5),
                                save=True, show=True):
    """
    1x3 Parallel Zone Analysis (Far ISA Only)
    - Plot 1: Original CDF by sample_geometry_category
    - Plot 2: CDF by Far ISA bins
    - Plot 3: Boxplot by Far ISA bins
    
    Parameters
    ----------
    isa_ratio : float
        Ratio threshold for filtering (default: 0.5)
        - Far plots: far_ratio >= isa_ratio
    """
    
    print("Parallel Zone Analysis: 1x3 Plot (Far ISA Only)")
    print(f"[Config] isa_ratio threshold: {isa_ratio}")
    
    # Prepare data
    required_cols = [y_col, 'flow_angle', 'far_isa', 'far_ratio', 'sample_geometry_category']
    df_plot = df.dropna(subset=required_cols).copy()
    df_plot['flow_zone'] = df_plot['flow_angle'].apply(get_flow_angle_zone)
    df_parallel = df_plot[df_plot['flow_zone'] == 'Parallel'].copy()
    
    print(f"\n[Data] Total Parallel zone: {len(df_parallel):,}")
    
    # Filter by ratio for Far ISA plots
    df_far = df_parallel[df_parallel['far_ratio'] >= isa_ratio].copy()
    
    print(f"[Data] Far ISA (far_ratio >= {isa_ratio}): {len(df_far):,}")
    
    # Create figure
    fig, axes = plt.subplots(1, 3, figsize=figsize, facecolor='black')
    
    # Plot 1: Original CDF
    plot_cdf_original(axes[0], df_parallel, y_col, threshold, offset)
    
    # Plot 2: CDF by Far ISA
    plot_cdf_by_isa_bin(axes[1], df_far, y_col, 'far_isa', 
                        f'CDF by Far ISA (far_ratio ≥ {isa_ratio}, n={len(df_far):,})',
                        threshold, offset)
    
    # Plot 3: Boxplot by Far ISA
    plot_boxplot_by_isa_bin(axes[2], df_far, y_col, 'far_isa',
                            f'Boxplot by Far ISA (far_ratio ≥ {isa_ratio})',
                            highlight_bins)
    
    fig.suptitle(f'Parallel Zone Analysis - Far ISA (ratio ≥ {isa_ratio})', 
                 fontsize=14, fontweight='bold', color='white', y=0.98)
    
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    
    if save:
        filepath = OUTPUT_DIR / f"parallel_far_analysis_1x3_ratio{isa_ratio}.png"
        fig.savefig(filepath, dpi=150, facecolor='black', bbox_inches='tight')
        print(f"\nSaved: {filepath}")
    
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    return fig

In [ ]:
fig = plot_parallel_far_analysis(
    final_df, 
    y_col='abs_error_node',
    isa_ratio=0,
    threshold=0.1832,
    highlight_bins= None
)

#### Near-Far ISA simulataneously

In [ ]:
"""
Parallel Zone Analysis: 2x2 Plot (Far & Near ISA) - Clean Version
=================================================================
- Legend에 68%tile과 <T 퍼센티지 표시
- x축, y축 직접 표기 제거
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from pathlib import Path

# 흰색 바탕
plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

OUTPUT_DIR = Path("figures")
OUTPUT_DIR.mkdir(exist_ok=True)


def get_flow_angle_zone(flow_angle):
    fa = flow_angle % 360
    if (fa >= 350 or fa <= 10) or (170 <= fa <= 190):
        return 'Parallel'
    elif (80 <= fa <= 100) or (260 <= fa <= 280):
        return 'Perpendicular'
    else:
        return 'Other'


def get_isa_bin(isa_value):
    """80-100으로 합침"""
    if isa_value <= 5:
        return '0-5'
    elif isa_value <= 10:
        return '5-10'
    elif isa_value <= 20:
        return '10-20'
    elif isa_value <= 30:
        return '20-30'
    elif isa_value <= 40:
        return '30-40'
    elif isa_value <= 50:
        return '40-50'
    elif isa_value <= 60:
        return '50-60'
    elif isa_value <= 70:
        return '60-70'
    elif isa_value <= 80:
        return '70-80'
    else:
        return '80-100'


def plot_cdf_by_isa_bin(ax, data, y_col, isa_col, title, threshold=0.1832, offset=1, xlim_max=20):
    """CDF plot by ISA bins - Clean legend version"""
    ax.set_facecolor('white')
    
    bin_order = ['0-5', '5-10', '10-20', '20-30', '30-40', '40-50', 
                 '50-60', '60-70', '70-80', '80-100']
    
    # Far ISA -> Reds, Near ISA -> Blues
    if isa_col == 'far_isa':
        cmap = cm.get_cmap('Oranges', 12)
        bin_colors = {bin_order[i]: cmap(i + 2) for i in range(10)}
    else:
        cmap = cm.get_cmap('Greens', 12)
        bin_colors = {bin_order[i]: cmap(i + 2) for i in range(10)}
    
    data = data.copy()
    data['isa_bin'] = data[isa_col].apply(get_isa_bin)
    
    plotted_bins = []
    
    for bin_label in bin_order:
        bin_data = data[data['isa_bin'] == bin_label][y_col].dropna()
        if len(bin_data) > 0:
            sorted_data = np.sort(bin_data)
            cdf = np.arange(1, len(sorted_data) + 1) / len(sorted_data)
            color = bin_colors[bin_label]
            
            ax.plot(sorted_data + offset, cdf, color=color, linewidth=2)
            
            p68 = np.percentile(bin_data, 68)
            cdf_at_T = (bin_data <= threshold).sum() / len(bin_data)
            
            plotted_bins.append((bin_label, p68, cdf_at_T, color, len(bin_data)))
            
            # 68th percentile marker
            ax.scatter([p68 + offset], [0.68], color=color, s=40, zorder=5)
    
    # 68% horizontal line
    ax.axhline(y=0.68, color='black', linestyle='-', linewidth=2, alpha=0.8)
    ax.text(0.02, 0.70, '68%', transform=ax.transAxes, color='black', fontsize=12, fontweight='bold')
    
    # Threshold line
    ax.axvline(x=threshold + offset, color='black', linestyle='--', linewidth=2, alpha=0.8)
    ax.axvline(x=0 + offset, color='black', linestyle='--', linewidth=1, alpha=0.5)
    
    # T label at top
    ax.text(threshold + offset, 1.02, f'T={threshold}m', transform=ax.get_xaxis_transform(),
            color='black', fontsize=11, fontweight='bold', ha='center', va='bottom')
    
    # Legend table with proper alignment
    if plotted_bins:
        # Header
        header = f"{'ISA':>9}  {'68%tile':>9}  {'<T':>6}  {'n':>7}"
        ax.text(0.98, 0.54, header, transform=ax.transAxes,
                fontsize=11, color='black', va='top', ha='right', family='monospace', fontweight='bold')
        
        # Separator line
        ax.text(0.98, 0.50, "─" * 45, transform=ax.transAxes,
                fontsize=8, color='gray', va='top', ha='right', family='monospace')
        
        y_pos = 0.47
        for bin_label, p68, cdf_at_T, color, n in plotted_bins:
            pct_below_T = cdf_at_T * 100
            row = f"{bin_label + '%':>8}  {p68:>7.2f}m  {pct_below_T:>5.0f}%  {n:>7,}"
            ax.text(0.98, y_pos, row, transform=ax.transAxes,
                    fontsize=11, color=color, va='top', ha='right', family='monospace')
            y_pos -= 0.042
    
    ax.set_xscale('log')
    ax.set_xlim(offset * 0.9, xlim_max + offset)
    ax.set_ylim(0, 1)
    
    def format_func(value, tick_number):
        original = value - offset
        if abs(original) < 0.01:
            return '0'
        elif abs(original) >= 10:
            return f'{original:.0f}'
        elif abs(original) >= 1:
            return f'{original:.1f}'
        else:
            return f'{original:.2f}'
    
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(format_func))
    
    tick_values_orig = [0, 0.2, 0.5, 1, 2, 5, 10, 20]
    tick_values = [v + offset for v in tick_values_orig if v <= xlim_max]
    ax.set_xticks(tick_values)
    
    ax.set_xlabel(r'$|Relative\ WSE\ Error|$ (m)', color='black', fontsize=13)
    ax.set_ylabel('CDF', color='black', fontsize=13)
    ax.set_title(title, color='black', fontsize=14, fontweight='bold')
    ax.tick_params(colors='black', labelsize=11)
    ax.grid(True, alpha=0.3, color='gray')
    
    for spine in ax.spines.values():
        spine.set_color('black')


def plot_boxplot_by_isa_bin(ax, data, y_col, isa_col, title, xlabel_label):
    """Boxplot by ISA bins"""
    ax.set_facecolor('white')
    
    bin_order = ['0-5', '5-10', '10-20', '20-30', '30-40', '40-50', 
                 '50-60', '60-70', '70-80', '80-100']
    
    # Far ISA -> Reds, Near ISA -> Blues
    if isa_col == 'far_isa':
        cmap = cm.get_cmap('Oranges', 12)
        bin_colors = {bin_order[i]: cmap(i + 2) for i in range(10)}
    else:
        cmap = cm.get_cmap('Greens', 12)
        bin_colors = {bin_order[i]: cmap(i + 2) for i in range(10)}
    
    data = data.copy()
    data['isa_bin'] = data[isa_col].apply(get_isa_bin)
    
    existing_bins = [b for b in bin_order if b in data['isa_bin'].values]
    
    box_data = []
    positions = []
    colors = []
    bin_labels = []
    
    for i, bin_label in enumerate(existing_bins):
        bin_df = data[data['isa_bin'] == bin_label]
        if len(bin_df) > 0:
            error_vals = bin_df[y_col].dropna().values
            if len(error_vals) > 0:
                box_data.append(error_vals)
                positions.append(len(positions))
                colors.append(bin_colors[bin_label])
                bin_labels.append(bin_label)
    
    if len(positions) == 0:
        ax.text(0.5, 0.5, 'No Data', transform=ax.transAxes,
                ha='center', va='center', color='gray', fontsize=14)
        return
    
    bp = ax.boxplot(box_data, positions=positions, widths=0.6, patch_artist=True,
                    whis=1.5,
                    medianprops=dict(color='black', linewidth=2),
                    whiskerprops=dict(color='black'),
                    capprops=dict(color='black'),
                    flierprops=dict(marker='o', markersize=3, alpha=0.5))
    
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
        patch.set_edgecolor('black')
    
    for flier, color in zip(bp['fliers'], colors):
        flier.set_markerfacecolor(color)
        flier.set_markeredgecolor(color)
    
    ax.set_yscale('symlog', linthresh=1)
    
    error_ticks = [0.1, 0.5, 1, 2, 5, 10, 20]
    ax.set_yticks(error_ticks)
    ax.set_yticklabels([str(t) for t in error_ticks], fontsize=11, color='black')
    ax.set_ylim(0.05, 30)
    
    ax.set_xticks(positions)
    ax.set_xticklabels(bin_labels, rotation=45, ha='right', color='black', fontsize=11)
    ax.set_xlabel(xlabel_label, color='black', fontsize=13)
    ax.set_ylabel(r'$|Relative\ WSE\ Error|$ (m)', color='black', fontsize=13)
    ax.set_title(title, color='black', fontsize=14, fontweight='bold')
    
    ax.grid(True, alpha=0.3, color='gray')
    ax.set_xlim(-0.5, len(bin_labels) - 0.5)
    
    for spine in ax.spines.values():
        spine.set_color('black')
    
    ax.tick_params(colors='black', labelsize=11)


def plot_parallel_zone_2x2(df, y_col='abs_error_node', 
                            isa_ratio=0,
                            threshold=0.1832, offset=1,
                            xlim_max=20,
                            figsize=(14, 12),
                            save=True, show=True):
    """
    2x2 Parallel Zone Analysis (Far & Near ISA)
    """
    
    print("Parallel Angle Analysis: 2x2 Plot (Far & Near ISA)")
    
    # Prepare data
    required_cols = [y_col, 'flow_angle', 'far_isa', 'far_ratio', 'near_isa', 'near_ratio']
    df_plot = df.dropna(subset=required_cols).copy()
    df_plot['flow_zone'] = df_plot['flow_angle'].apply(get_flow_angle_zone)
    df_parallel = df_plot[df_plot['flow_zone'] == 'Parallel'].copy()
    
    print(f"\n[Data] Total Parallel zone: {len(df_parallel):,}")
    
    # Filter by ratio
    df_far = df_parallel[df_parallel['far_ratio'] >= isa_ratio].copy()
    df_near = df_parallel[df_parallel['near_ratio'] >= isa_ratio].copy()
    
    print(f"[Data] Far ISA: {len(df_far):,}")
    print(f"[Data] Near ISA: {len(df_near):,}")
    
    # Create figure
    fig, axes = plt.subplots(2, 2, figsize=figsize, facecolor='white')
    
    # Row 1: Far ISA
    plot_cdf_by_isa_bin(axes[0, 0], df_far, y_col, 'far_isa', 
                        f'CDF by Far ISA (n={len(df_far):,})',
                        threshold, offset, xlim_max)
    
    plot_boxplot_by_isa_bin(axes[0, 1], df_far, y_col, 'far_isa',
                            f'Boxplot by Far ISA', 'Far ISA (%)')
    
    # Row 2: Near ISA
    plot_cdf_by_isa_bin(axes[1, 0], df_near, y_col, 'near_isa', 
                        f'CDF by Near ISA (n={len(df_near):,})',
                        threshold, offset, xlim_max)
    
    plot_boxplot_by_isa_bin(axes[1, 1], df_near, y_col, 'near_isa',
                            f'Boxplot by Near ISA', 'Near ISA (%)')
    
    fig.suptitle('Parallel Angle Analysis - Far & Near ISA', 
                 fontsize=18, fontweight='bold', color='black', y=0.98)
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    
    if save:
        filepath = OUTPUT_DIR / f"figure6_parallel_zone_2x2.png"
        fig.savefig(filepath, dpi=300, facecolor='white', bbox_inches='tight')
        filepath = OUTPUT_DIR / f"figure6_parallel_zone_2x2.pdf"
        fig.savefig(filepath, dpi=300, facecolor='white', bbox_inches='tight')

        print(f"\nSaved: {filepath}")
    
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    return fig

In [ ]:
fig = plot_parallel_zone_2x2(
    final_df,
    y_col='abs_error_node',
    offset=1,
    isa_ratio=0,
    xlim_max=30,
    figsize=(14, 12),
    save=True, show=True
)

#### Why 50-60% so low compared to others?

In [ ]:
"""
Find variables with different patterns in ISA bin 50-60%
=========================================================
- Parallel zone only
- 0-5, 5-10 세분화된 ISA bins
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path

plt.style.use('dark_background')

OUTPUT_DIR = Path("unit_level_analysis")
OUTPUT_DIR.mkdir(exist_ok=True)


def get_flow_angle_zone(flow_angle):
    """Classify flow angle into zones"""
    fa = flow_angle % 360
    if (fa >= 350 or fa <= 10) or (170 <= fa <= 190):
        return 'Parallel'
    elif (80 <= fa <= 100) or (260 <= fa <= 280):
        return 'Perpendicular'
    else:
        return 'Other'


def get_isa_bin(isa_value):
    """0-5, 5-10 세분화 버전"""
    if isa_value <= 5:
        return '0-5'
    elif isa_value <= 10:
        return '5-10'
    elif isa_value <= 20:
        return '10-20'
    elif isa_value <= 30:
        return '20-30'
    elif isa_value <= 40:
        return '30-40'
    elif isa_value <= 50:
        return '40-50'
    elif isa_value <= 60:
        return '50-60'
    elif isa_value <= 70:
        return '60-70'
    elif isa_value <= 80:
        return '70-80'
    elif isa_value <= 90:
        return '80-90'
    else:
        return '90-100'


def prepare_data(df):
    """Add incidence angle and other derived variables + Parallel zone filter"""
    df = df.copy()
    
    # Parallel zone 필터링
    if 'flow_angle' in df.columns:
        df['flow_zone'] = df['flow_angle'].apply(get_flow_angle_zone)
        df = df[df['flow_zone'] == 'Parallel'].copy()
        print(f"[Filter] Parallel zone: {len(df):,} samples")
    
    # xtrk_dist 절대값
    if 'xtrk_dist' in df.columns:
        df['xtrk_dist_abs'] = np.abs(df['xtrk_dist'])
        print(f"[Derived] xtrk_dist_abs = |xtrk_dist|")
        print(f"          Range: {df['xtrk_dist_abs'].min():.1f} ~ {df['xtrk_dist_abs'].max():.1f}")
    
    # Incidence angle: arctan(sat_altitude / xtrk_dist_abs)
    if 'sat_altitude' in df.columns and 'xtrk_dist_abs' in df.columns:
        xtrk_abs = df['xtrk_dist_abs'].replace(0, np.nan)
        df['incidence_angle'] = np.degrees(np.arctan(df['sat_altitude'] / xtrk_abs))
        print(f"[Derived] incidence_angle = arctan(sat_altitude / |xtrk_dist|)")
        print(f"          Range: {df['incidence_angle'].min():.1f}° ~ {df['incidence_angle'].max():.1f}°")
    
    # Boolean을 numeric으로 변환
    for col in df.columns:
        if df[col].dtype == 'bool':
            df[col] = df[col].astype(int)
    
    return df


def analyze_anomalous_bin(df, target_bin='50-60', isa_col='far_isa', 
                          error_col='abs_error_node', variables=None,
                          isa_ratio=None):
    """
    Find variables that behave differently in the target ISA bin
    """
    
    print(f"ANOMALY ANALYSIS: Which variables explain {target_bin}% ISA bin behavior?")
    
    # Prepare data (includes Parallel zone filter)
    df = prepare_data(df)
    
    # isa_ratio 필터링
    if isa_ratio is not None:
        if isa_col == 'far_isa' and 'far_ratio' in df.columns:
            df = df[df['far_ratio'] >= isa_ratio].copy()
            print(f"[Filter] far_ratio >= {isa_ratio}: {len(df):,} samples")
        elif isa_col == 'near_isa' and 'near_ratio' in df.columns:
            df = df[df['near_ratio'] >= isa_ratio].copy()
            print(f"[Filter] near_ratio >= {isa_ratio}: {len(df):,} samples")
    
    df['isa_bin'] = df[isa_col].apply(get_isa_bin)
    
    # 세분화된 bin order
    bin_order = ['0-5', '5-10', '10-20', '20-30', '30-40', '40-50', 
                 '50-60', '60-70', '70-80', '80-90', '90-100']
    
    if variables is None:
        variables = ['width', 'xtrk_dist_abs', 'dark_frac',
                     'nlcd_21_mean', 'nlcd_22_mean', 'nlcd_23_mean', 'nlcd_24_mean',
                     'time_diff_hours', 'is_spike', 'sat_altitude',
                     'near_isa', 'near_ratio', 'far_ratio',
                     'near_building', 'far_building', 'incidence_angle']
    
    variables = [v for v in variables if v in df.columns]
    print(f"\n[Variables] Analyzing {len(variables)} variables:")
    print(f"  {variables}")
    
    results = []
    
    # Method 1: Median comparison
    print("METHOD 1: Median Deviation from Neighbors")
    
    target_idx = bin_order.index(target_bin)
    adjacent_bins = []
    if target_idx > 0:
        adjacent_bins.append(bin_order[target_idx - 1])
    if target_idx < len(bin_order) - 1:
        adjacent_bins.append(bin_order[target_idx + 1])
    
    print(f"\nTarget bin: {target_bin}")
    print(f"Adjacent bins: {adjacent_bins}")
    
    df_target = df[df['isa_bin'] == target_bin]
    df_adjacent = df[df['isa_bin'].isin(adjacent_bins)]
    df_other = df[df['isa_bin'] != target_bin]
    
    print(f"\n{'Variable':<20} {'Target Med':>12} {'Adjacent Med':>12} {'Deviation':>10} {'Direction':>12}")
    
    for var in variables:
        target_med = df_target[var].median()
        adjacent_med = df_adjacent[var].median()
        other_std = df_other[var].std()
        
        if other_std > 0 and not np.isnan(target_med) and not np.isnan(adjacent_med):
            deviation = (target_med - adjacent_med) / other_std
        else:
            deviation = 0
        
        direction = "↑ Higher" if deviation > 0.1 else ("↓ Lower" if deviation < -0.1 else "≈ Same")
        
        results.append({
            'variable': var,
            'target_median': target_med,
            'adjacent_median': adjacent_med,
            'deviation': deviation,
            'abs_deviation': abs(deviation)
        })
        
        print(f"{var:<20} {target_med:>12.3f} {adjacent_med:>12.3f} {deviation:>10.3f} {direction:>12}")
    
    results_df = pd.DataFrame(results).sort_values('abs_deviation', ascending=False)
    
    print("TOP 5 DEVIATING VARIABLES:")
    for _, row in results_df.head(5).iterrows():
        direction = "↑" if row['deviation'] > 0 else "↓"
        print(f"  {row['variable']:<20} {direction} deviation: {row['deviation']:+.3f}")
    
    # Method 2: Correlation with error by bin
    print("METHOD 2: Correlation with Error (Spearman)")
    
    corr_results = []
    
    for var in variables:
        bin_corrs = {}
        for bin_label in bin_order:
            bin_df = df[df['isa_bin'] == bin_label]
            if len(bin_df) > 10:
                try:
                    corr, pval = stats.spearmanr(bin_df[var].dropna(), 
                                              bin_df[error_col].dropna(), 
                                              nan_policy='omit')
                    if not np.isnan(corr):
                        bin_corrs[bin_label] = corr
                except:
                    pass
        
        if target_bin in bin_corrs:
            target_corr = bin_corrs[target_bin]
            other_corrs = [v for k, v in bin_corrs.items() if k != target_bin]
            
            if len(other_corrs) > 0:
                avg_other_corr = np.mean(other_corrs)
                corr_diff = target_corr - avg_other_corr
                sign_flip = (target_corr * avg_other_corr < 0)
                
                corr_results.append({
                    'variable': var,
                    'target_corr': target_corr,
                    'avg_other_corr': avg_other_corr,
                    'corr_diff': corr_diff,
                    'abs_corr_diff': abs(corr_diff),
                    'sign_flip': sign_flip
                })
    
    corr_df = pd.DataFrame(corr_results).sort_values('abs_corr_diff', ascending=False)
    
    print(f"\n{'Variable':<20} {'Target':>10} {'Others':>10} {'Diff':>10} {'Sign Flip':>10}")
    for _, row in corr_df.iterrows():
        flip = "⚠️ YES" if row['sign_flip'] else ""
        print(f"{row['variable']:<20} {row['target_corr']:>10.3f} {row['avg_other_corr']:>10.3f} {row['corr_diff']:>10.3f} {flip:>10}")
    
    sign_flip_vars = corr_df[corr_df['sign_flip'] == True]['variable'].tolist()
    if sign_flip_vars:
        print(f"\nSIGN FLIP DETECTED in: {sign_flip_vars}")
    
    # Method 3: Trend break analysis
    print("METHOD 3: Trend Break Analysis")
    
    trend_results = []
    
    for var in variables:
        bin_medians = {}
        for bin_label in bin_order:
            bin_df = df[df['isa_bin'] == bin_label]
            if len(bin_df) > 0:
                med = bin_df[var].median()
                if not np.isnan(med):
                    bin_medians[bin_label] = med
        
        if target_bin in bin_medians and len(adjacent_bins) == 2:
            prev_bin, next_bin = adjacent_bins[0], adjacent_bins[1]
            if prev_bin in bin_medians and next_bin in bin_medians:
                expected = (bin_medians[prev_bin] + bin_medians[next_bin]) / 2
                actual = bin_medians[target_bin]
                trend_break = actual - expected
                
                all_meds = list(bin_medians.values())
                med_range = max(all_meds) - min(all_meds)
                if med_range > 0:
                    normalized_break = trend_break / med_range
                else:
                    normalized_break = 0
                
                trend_results.append({
                    'variable': var,
                    'expected': expected,
                    'actual': actual,
                    'trend_break': trend_break,
                    'normalized_break': normalized_break,
                    'abs_normalized_break': abs(normalized_break)
                })
    
    trend_df = pd.DataFrame(trend_results).sort_values('abs_normalized_break', ascending=False)
    
    print(f"\n{'Variable':<20} {'Expected':>12} {'Actual':>12} {'Break':>10} {'Norm Break':>12}")
    for _, row in trend_df.iterrows():
        print(f"{row['variable']:<20} {row['expected']:>12.3f} {row['actual']:>12.3f} {row['trend_break']:>10.3f} {row['normalized_break']:>12.3f}")
    
    # SUMMARY
    print(f"SUMMARY: Most Anomalous Variables in {target_bin}% bin")
    
    results_df_ranked = results_df.reset_index(drop=True)
    corr_df_ranked = corr_df.reset_index(drop=True)
    trend_df_ranked = trend_df.reset_index(drop=True)
    
    summary = []
    for var in variables:
        rank_dev = results_df_ranked[results_df_ranked['variable'] == var].index.tolist()
        rank_dev = rank_dev[0] + 1 if rank_dev else len(variables)
        
        rank_corr = corr_df_ranked[corr_df_ranked['variable'] == var].index.tolist()
        rank_corr = rank_corr[0] + 1 if rank_corr else len(variables)
        
        rank_trend = trend_df_ranked[trend_df_ranked['variable'] == var].index.tolist()
        rank_trend = rank_trend[0] + 1 if rank_trend else len(variables)
        
        avg_rank = (rank_dev + rank_corr + rank_trend) / 3
        
        summary.append({
            'variable': var,
            'rank_deviation': rank_dev,
            'rank_correlation': rank_corr,
            'rank_trend': rank_trend,
            'avg_rank': avg_rank
        })
    
    summary_df = pd.DataFrame(summary).sort_values('avg_rank')
    
    print(f"\n{'Variable':<20} {'Rank Dev':>10} {'Rank Corr':>10} {'Rank Trend':>10} {'Avg Rank':>10}")
    for _, row in summary_df.iterrows():
        print(f"{row['variable']:<20} {row['rank_deviation']:>10.0f} {row['rank_correlation']:>10.0f} {row['rank_trend']:>10.0f} {row['avg_rank']:>10.1f}")
    
    return results_df, corr_df, trend_df, summary_df, df


def plot_variable_by_isa_bin(df, variables, target_bin='50-60', isa_col='far_isa',
                              error_col='abs_error_node', figsize=(20, 10),
                              ratio_vars=None,
                              isa_ratio=None,
                              save=True, show=True):
    """
    Visualize variables across ISA bins, highlighting target bin
    """
    
    # Prepare data (includes Parallel zone filter)
    df = prepare_data(df)
    
    # isa_ratio 필터링
    if isa_ratio is not None:
        if isa_col == 'far_isa' and 'far_ratio' in df.columns:
            df = df[df['far_ratio'] >= isa_ratio].copy()
            print(f"[Filter] far_ratio >= {isa_ratio}: {len(df):,} samples")
        elif isa_col == 'near_isa' and 'near_ratio' in df.columns:
            df = df[df['near_ratio'] >= isa_ratio].copy()
            print(f"[Filter] near_ratio >= {isa_ratio}: {len(df):,} samples")
    
    df['isa_bin'] = df[isa_col].apply(get_isa_bin)
    
    if ratio_vars is None:
        ratio_vars = ['is_spike']
    
    # 세분화된 bin order
    bin_order = ['0-5', '5-10', '10-20', '20-30', '30-40', '40-50', 
                 '50-60', '60-70', '70-80', '80-90', '90-100']
    existing_bins = [b for b in bin_order if b in df['isa_bin'].values]
    
    variables = [v for v in variables if v in df.columns]
    
    n_vars = len(variables)
    n_cols = 4
    n_rows = (n_vars + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize, facecolor='black')
    axes = axes.flatten()
    
    for idx, var in enumerate(variables):
        ax = axes[idx]
        ax.set_facecolor('#1a1a1a')
        
        if var not in df.columns:
            ax.text(0.5, 0.5, f'{var}\nNot Available', transform=ax.transAxes,
                    ha='center', va='center', color='gray')
            continue
        
        # 비율 변수: bar plot
        if var in ratio_vars:
            ratios = []
            colors = []
            bin_labels_plot = []
            
            for bin_label in existing_bins:
                bin_df = df[df['isa_bin'] == bin_label]
                if len(bin_df) > 0:
                    ratio = bin_df[var].sum() / len(bin_df)
                    ratios.append(ratio)
                    bin_labels_plot.append(bin_label)
                    if bin_label == target_bin:
                        colors.append('yellow')
                    else:
                        colors.append('cyan')
            
            if len(ratios) == 0:
                ax.text(0.5, 0.5, 'No Data', transform=ax.transAxes,
                        ha='center', va='center', color='gray')
                continue
            
            bars = ax.bar(range(len(ratios)), ratios, color=colors, alpha=0.7, edgecolor='white')
            
            for i, (ratio, bar) in enumerate(zip(ratios, bars)):
                ax.text(i, ratio + 0.01, f'{ratio:.2f}', ha='center', va='bottom',
                       color='white', fontsize=7)
            
            if target_bin in bin_labels_plot:
                target_idx = bin_labels_plot.index(target_bin)
                ax.axvspan(target_idx - 0.5, target_idx + 0.5, alpha=0.2, color='yellow')
            
            ax.set_xticks(range(len(bin_labels_plot)))
            ax.set_xticklabels(bin_labels_plot, rotation=45, ha='right', color='white', fontsize=7)
            ax.set_ylabel('Ratio', color='white', fontsize=8)
            ax.set_ylim(0, max(ratios) * 1.2 if ratios else 1)
            ax.set_title(f'{var} (ratio)', color='white', fontsize=10, fontweight='bold')
        
        # 일반 변수: boxplot
        else:
            box_data = []
            positions = []
            colors = []
            bin_labels_plot = []
            
            for i, bin_label in enumerate(existing_bins):
                bin_df = df[df['isa_bin'] == bin_label]
                if len(bin_df) > 0:
                    vals = bin_df[var].dropna().values.astype(float)
                    if len(vals) > 0:
                        box_data.append(vals)
                        positions.append(len(positions))
                        bin_labels_plot.append(bin_label)
                        if bin_label == target_bin:
                            colors.append('yellow')
                        else:
                            colors.append('cyan')
            
            if len(positions) == 0:
                ax.text(0.5, 0.5, 'No Data', transform=ax.transAxes,
                        ha='center', va='center', color='gray')
                continue
            
            bp = ax.boxplot(box_data, positions=positions, widths=0.6, patch_artist=True,
                            whis=1.5,
                            medianprops=dict(color='white', linewidth=2),
                            whiskerprops=dict(color='white'),
                            capprops=dict(color='white'),
                            flierprops=dict(marker='o', markersize=2, alpha=0.3))
            
            for patch, color in zip(bp['boxes'], colors):
                patch.set_facecolor(color)
                patch.set_alpha(0.7)
                patch.set_edgecolor('white')
            
            # Median line
            medians = [np.median(d) for d in box_data]
            ax.plot(positions, medians, 'w--', linewidth=1.5, alpha=0.7, marker='o', markersize=5)
            
            if target_bin in bin_labels_plot:
                target_idx = bin_labels_plot.index(target_bin)
                ax.axvspan(target_idx - 0.5, target_idx + 0.5, alpha=0.2, color='yellow')
            
            ax.set_xticks(range(len(bin_labels_plot)))
            ax.set_xticklabels(bin_labels_plot, rotation=45, ha='right', color='white', fontsize=7)
            ax.set_title(var, color='white', fontsize=10, fontweight='bold')
        
        ax.tick_params(colors='white', labelsize=8)
        ax.grid(True, alpha=0.2, color='white')
        
        for spine in ax.spines.values():
            spine.set_color('white')
    
    # Hide unused
    for idx in range(len(variables), len(axes)):
        axes[idx].set_visible(False)
    
    fig.suptitle(f'Parallel Zone: Variables by {isa_col} Bin (Yellow: {target_bin}%)',
                 fontsize=14, fontweight='bold', color='white', y=1.01)
    
    plt.tight_layout()
    
    if save:
        filepath = OUTPUT_DIR / f"variable_by_isa_bin_highlight_{target_bin.replace('-', '_')}.png"
        fig.savefig(filepath, dpi=150, facecolor='black', bbox_inches='tight')
        print(f"\nSaved: {filepath}")
    
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    return fig


# Run

# variables = [
#     'abs_error_node', 'xtrk_dist_abs', 'sat_altitude', 'incidence_angle',
#     'near_isa', 'near_ratio', 'near_building', 'dark_frac',
# ]

variables = [
    'width', 'xtrk_dist_abs', 'dark_frac',
    'nlcd_21_mean', 'nlcd_22_mean', 'nlcd_23_mean', 'nlcd_24_mean',
    'time_diff_hours', 'is_spike', 'sat_altitude',
    'near_isa', 'near_ratio', 'far_ratio',
    'near_building', 'far_building',
    'incidence_angle'
]

# 1. Statistical analysis
results_df, corr_df, trend_df, summary_df, df_prepared = analyze_anomalous_bin(
    final_df,
    target_bin='50-60',
    isa_col='far_isa',
    error_col='abs_error_node',
    variables=variables,
    isa_ratio=None  # ratio 필터 없음
)

# 2. Visualization
fig = plot_variable_by_isa_bin(
    final_df,
    variables=variables,
    target_bin='50-60',
    isa_col='far_isa',
    ratio_vars=['is_spike'],
    isa_ratio=None,  # ratio 필터 없음
    figsize=(20, 20),
    save=True, show=True
)

In [ ]:
variables = [
    'abs_error_node', 'dark_frac', 'xtrk_dist_abs', 'sat_altitude', 'incidence_angle', 'near_isa', 'near_ratio', 'near_building'
]

# 1. Statistical analysis
results_df, corr_df, trend_df, summary_df, df_prepared = analyze_anomalous_bin(
    final_df,
    target_bin='50-60',
    isa_col='far_isa',
    error_col='abs_error_node',
    variables=variables,
    isa_ratio=None  # ratio 필터 없음
)

# 2. Visualization
fig = plot_variable_by_isa_bin(
    final_df,
    variables=variables,
    target_bin='50-60',
    isa_col='far_isa',
    ratio_vars=['is_spike'],
    isa_ratio=None,  # ratio 필터 없음
    figsize=(20, 10),
    save=True, show=True
)

In [ ]:
analysis_final_df = pd.read_parquet("pixc_level_parquet_ver2/pixc_pixels_combined.parquet")

In [ ]:
"""
Classification & sig0 Distribution by ISA Bin
==============================================
- abs_error_node: 첫번째
- classification 2-7: 각 node별 비율
- sig0: node별 mean
- Parallel zone only
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use('dark_background')

OUTPUT_DIR = Path("unit_level_analysis")
OUTPUT_DIR.mkdir(exist_ok=True)


def get_flow_angle_zone(flow_angle):
    """Classify flow angle into zones"""
    fa = flow_angle % 360
    if (fa >= 350 or fa <= 10) or (170 <= fa <= 190):
        return 'Parallel'
    elif (80 <= fa <= 100) or (260 <= fa <= 280):
        return 'Perpendicular'
    else:
        return 'Other'


def get_isa_bin(isa_value):
    if isa_value <= 5:
        return '0-5'
    elif isa_value <= 10:
        return '5-10'
    elif isa_value <= 20:
        return '10-20'
    elif isa_value <= 30:
        return '20-30'
    elif isa_value <= 40:
        return '30-40'
    elif isa_value <= 50:
        return '40-50'
    elif isa_value <= 60:
        return '50-60'
    elif isa_value <= 70:
        return '60-70'
    elif isa_value <= 80:
        return '70-80'
    elif isa_value <= 90:
        return '80-90'
    else:
        return '90-100'


def prepare_classification_data(df):
    """
    Node별 classification 비율 및 sig0 mean 계산
    """
    print("Preparing classification data by node...")
    
    print(f"[Input] Total pixels: {len(df):,}")
    
    # Parallel zone 필터링
    if 'flow_angle' in df.columns:
        df = df.copy()
        df['flow_zone'] = df['flow_angle'].apply(get_flow_angle_zone)
        df = df[df['flow_zone'] == 'Parallel']
        print(f"[Filter] Parallel zone: {len(df):,} pixels")
    
    print(f"[Input] Unique nodes: {df.groupby(['node_id', 'time_str']).ngroups:,}")
    print(f"[Input] Classification values: {sorted(df['classification'].unique())}")
    
    # Node별 집계
    node_stats = df.groupby(['node_id', 'time_str']).agg({
        'classification': 'count',  # 전체 pixel 수
        'sig0': 'mean',             # sig0 평균
        'far_isa': 'first',         # ISA (node별 동일)
        'abs_error_node': 'first',  # error (node별 동일)
    }).rename(columns={'classification': 'total_pixels'})
    
    # Classification별 count
    class_counts = df.groupby(['node_id', 'time_str', 'classification']).size().unstack(fill_value=0)
    class_counts.columns = [f'class_{int(c)}_count' for c in class_counts.columns]
    
    # Merge
    node_df = node_stats.join(class_counts).reset_index()
    
    # 비율 계산
    for i in range(1, 8):
        count_col = f'class_{i}_count'
        ratio_col = f'class_{i}_ratio'
        if count_col in node_df.columns:
            node_df[ratio_col] = node_df[count_col] / node_df['total_pixels']
        else:
            node_df[ratio_col] = 0
    
    # ISA bin
    node_df['isa_bin'] = node_df['far_isa'].apply(get_isa_bin)
    
    print(f"\n[Output] Node-level data: {len(node_df):,} rows")
    
    return node_df


def plot_classification_by_isa_bin(df, target_bin='50-60',
                                    figsize=(20, 10), save=True, show=True):
    """
    abs_error_node + Classification 2-7 비율 + sig0를 ISA bin별로 boxplot
    """
    
    # Node-level 데이터 준비
    node_df = prepare_classification_data(df)
    
    bin_order = ['0-5', '5-10', '10-20', '20-30', '30-40', '40-50', 
                 '50-60', '60-70', '70-80', '80-90', '90-100']
    existing_bins = [b for b in bin_order if b in node_df['isa_bin'].values]
    
    # Variables: abs_error_node + class_2~7_ratio + sig0 (Class 1 제외)
    variables = ['abs_error_node'] + [f'class_{i}_ratio' for i in range(2, 8)] + ['sig0']
    
    # Title 매핑
    class_names = {
        'abs_error_node': 'abs_error_node (m)',
        'class_2_ratio': 'Class 2: Land near water',
        'class_3_ratio': 'Class 3: Water near land',
        'class_4_ratio': 'Class 4: Open water',
        'class_5_ratio': 'Class 5: Dark water',
        'class_6_ratio': 'Class 6: Low coh water near land',
        'class_7_ratio': 'Class 7: Low coh open water',
        'sig0': 'sig0 (mean)'
    }
    
    n_vars = len(variables)
    n_cols = 4
    n_rows = (n_vars + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize, facecolor='black')
    axes = axes.flatten()
    
    for idx, var in enumerate(variables):
        ax = axes[idx]
        ax.set_facecolor('#1a1a1a')
        
        if var not in node_df.columns:
            ax.text(0.5, 0.5, f'{var}\nNot Available', transform=ax.transAxes,
                    ha='center', va='center', color='gray')
            continue
        
        box_data = []
        positions = []
        colors = []
        bin_labels_plot = []
        
        for i, bin_label in enumerate(existing_bins):
            bin_df = node_df[node_df['isa_bin'] == bin_label]
            if len(bin_df) > 0:
                vals = bin_df[var].dropna().values.astype(float)
                if len(vals) > 0:
                    box_data.append(vals)
                    positions.append(len(positions))
                    bin_labels_plot.append(bin_label)
                    if bin_label == target_bin:
                        colors.append('yellow')
                    else:
                        colors.append('cyan')
        
        if len(positions) == 0:
            ax.text(0.5, 0.5, 'No Data', transform=ax.transAxes,
                    ha='center', va='center', color='gray')
            continue
        
        bp = ax.boxplot(box_data, positions=positions, widths=0.6, patch_artist=True,
                        whis=1.5,
                        medianprops=dict(color='white', linewidth=2),
                        whiskerprops=dict(color='white'),
                        capprops=dict(color='white'),
                        flierprops=dict(marker='o', markersize=2, alpha=0.3))
        
        for patch, color in zip(bp['boxes'], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
            patch.set_edgecolor('white')
        
        # Median line
        medians = [np.median(d) for d in box_data]
        ax.plot(positions, medians, 'w--', linewidth=1.5, alpha=0.7, marker='o', markersize=5)
        
        # Highlight target bin
        if target_bin in bin_labels_plot:
            target_idx = bin_labels_plot.index(target_bin)
            ax.axvspan(target_idx - 0.5, target_idx + 0.5, alpha=0.2, color='yellow')
        
        ax.set_xticks(range(len(bin_labels_plot)))
        ax.set_xticklabels(bin_labels_plot, rotation=45, ha='right', color='white', fontsize=7)
        
        title = class_names.get(var, var)
        ax.set_title(title, color='white', fontsize=10, fontweight='bold')
        
        ax.tick_params(colors='white', labelsize=8)
        ax.grid(True, alpha=0.2, color='white')
        
        for spine in ax.spines.values():
            spine.set_color('white')
    
    # Hide unused
    for idx in range(len(variables), len(axes)):
        axes[idx].set_visible(False)
    
    fig.suptitle(f'Parallel Zone: Classification Ratio & sig0 by far_isa Bin (Yellow: {target_bin}%)',
                 fontsize=14, fontweight='bold', color='white', y=1.01)
    
    plt.tight_layout()
    
    if save:
        filepath = OUTPUT_DIR / f"classification_by_isa_bin.png"
        fig.savefig(filepath, dpi=150, facecolor='black', bbox_inches='tight')
        print(f"\nSaved: {filepath}")
    
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    # Summary statistics
    print("SUMMARY: Median values by ISA bin")
    
    summary_data = []
    for bin_label in existing_bins:
        bin_df = node_df[node_df['isa_bin'] == bin_label]
        row = {'isa_bin': bin_label, 'n_nodes': len(bin_df)}
        for var in variables:
            if var in bin_df.columns:
                row[var] = bin_df[var].median()
        summary_data.append(row)
    
    summary_df = pd.DataFrame(summary_data)
    print(summary_df.to_string(index=False))
    
    return fig, node_df


# Run

# Load data
analysis_final_df = pd.read_parquet("pixc_level_parquet_ver2/pixc_pixels_combined.parquet")
print(f"Loaded: {len(analysis_final_df):,} pixels")
print(f"Columns: {list(analysis_final_df.columns)}")

# Plot
fig, node_df = plot_classification_by_isa_bin(
    analysis_final_df,
    target_bin='50-60',
    figsize=(20, 10),
    save=True, show=True
)

In [ ]:
"""
sig0 vs abs_error_node CDF Analysis
====================================
- CDF by sig0 bin
- Parallel zone only
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from pathlib import Path

plt.style.use('dark_background')

OUTPUT_DIR = Path("unit_level_analysis")
OUTPUT_DIR.mkdir(exist_ok=True)


def get_flow_angle_zone(flow_angle):
    fa = flow_angle % 360
    if (fa >= 350 or fa <= 10) or (170 <= fa <= 190):
        return 'Parallel'
    elif (80 <= fa <= 100) or (260 <= fa <= 280):
        return 'Perpendicular'
    else:
        return 'Other'


def get_sig0_bin(sig0_value):
    """sig0 binning"""
    if sig0_value < 0:
        return '<0'
    elif sig0_value <= 1:
        return '0-1'
    elif sig0_value <= 3:
        return '1-3'
    elif sig0_value <= 5:
        return '3-5'
    elif sig0_value <= 10:
        return '5-10'
    elif sig0_value <= 15:
        return '10-15'
    elif sig0_value <= 20:
        return '15-20'
    elif sig0_value <= 25:
        return '20-25'
    elif sig0_value <= 30:
        return '25-30'
    elif sig0_value <= 35:
        return '30-35'
    elif sig0_value <= 40:
        return '35-40'
    elif sig0_value <= 45:
        return '40-45'
    elif sig0_value <= 50:
        return '45-50'
    # elif sig0_value <= 55:
    #     return '50-55'
    # elif sig0_value <= 60:
    #     return '55-60'
    # elif sig0_value <= 65:
    #     return '60-65'
    # elif sig0_value <= 70:
    #     return '65-70'
    # elif sig0_value <= 75:
    #     return '70-75'
    # elif sig0_value <= 80:
    #     return '75-80'
    # elif sig0_value <= 85:
    #     return '80-85'
    elif sig0_value <= 90:
        return '50-90'
    else:
        return '>90'


def prepare_data(df):
    """Parallel zone filter"""
    df = df.copy()
    
    if 'flow_angle' in df.columns:
        df['flow_zone'] = df['flow_angle'].apply(get_flow_angle_zone)
        df = df[df['flow_zone'] == 'Parallel'].copy()
        print(f"[Filter] Parallel zone: {len(df):,} samples")
    
    return df


def plot_sig0_error_cdf(df, figsize=(10, 7), save=True, show=True):
    """
    CDF: abs_error_node by sig0 bin
    """
    
    df = prepare_data(df)
    df = df.dropna(subset=['sig0', 'abs_error_node']).copy()
    df['sig0_bin'] = df['sig0'].apply(get_sig0_bin)
    
    print(f"[Data] Total samples: {len(df):,}")
    print(f"[Data] sig0 range: {df['sig0'].min():.2f} ~ {df['sig0'].max():.2f}")
    
    bin_order = ['<0', '0-1', '1-3', '3-5', '5-10', '10-15', '15-20', '20-25', '25-30',
                 '30-35', '35-40', '40-45', '45-50', '50-90',\
                 # '50-55', '55-60', '60-65', '65-70', '70-75', '75-80', '80-85', '85-90',\
                 '>90']
    existing_bins = [b for b in bin_order if b in df['sig0_bin'].values]
    
    fig, ax = plt.subplots(figsize=figsize, facecolor='black')
    ax.set_facecolor('#1a1a1a')
    
    cmap = cm.get_cmap('cool', len(existing_bins))
    bin_colors = {existing_bins[i]: cmap(i) for i in range(len(existing_bins))}
    
    offset = 1  # log(x+offset)
    plotted_bins = []
    
    for bin_label in existing_bins:
        bin_data = df[df['sig0_bin'] == bin_label]['abs_error_node'].dropna()
        if len(bin_data) > 0:
            sorted_data = np.sort(bin_data)
            cdf = np.arange(1, len(sorted_data) + 1) / len(sorted_data)
            color = bin_colors[bin_label]
            
            ax.plot(sorted_data + offset, cdf, color=color, linewidth=1.5, label=bin_label)
            
            p68 = np.percentile(bin_data, 68)
            plotted_bins.append((bin_label, p68, color, len(bin_data)))
    
    # 68th percentile line
    ax.axhline(y=0.68, color='lime', linestyle='--', linewidth=1.5, alpha=0.8)
    ax.text(0.02, 0.70, '68%tile', transform=ax.transAxes, color='lime', fontsize=10, fontweight='bold')
    
    # 68th percentile markers
    for bin_label, p68, color, n in plotted_bins:
        ax.scatter([p68 + offset], [0.68], color=color, s=30, zorder=5)
        ax.axvline(x=p68 + offset, color=color, linestyle=':', linewidth=0.8, alpha=0.4)
    
    # Log scale
    ax.set_xscale('log')
    
    all_errors = df['abs_error_node'].dropna()
    ax.set_xlim((all_errors.min() + offset) * 0.9, (all_errors.max() + offset) * 1.1)
    ax.set_ylim(0, 1)
    
    # X-axis formatter
    def format_func(value, tick_number):
        original = value - offset
        if abs(original) < 0.01:
            return '0'
        elif abs(original) >= 10:
            return f'{original:.0f}'
        elif abs(original) >= 1:
            return f'{original:.1f}'
        else:
            return f'{original:.2f}'
    
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(format_func))
    
    ax.set_xlabel('abs_error_node (m)', color='white', fontsize=12)
    ax.set_ylabel('CDF', color='white', fontsize=12)
    ax.set_title('Parallel Zone: CDF of Error by sig0 Bin', color='white', fontsize=14, fontweight='bold')
    ax.legend(loc='lower right', fontsize=8, facecolor='black', edgecolor='white', 
              labelcolor='white', ncol=2)
    ax.tick_params(colors='white', labelsize=10)
    ax.grid(True, alpha=0.2, color='white')
    for spine in ax.spines.values():
        spine.set_color('white')
    
    plt.tight_layout()
    
    if save:
        filepath = OUTPUT_DIR / "sig0_error_cdf.png"
        fig.savefig(filepath, dpi=150, facecolor='black', bbox_inches='tight')
        print(f"\nSaved: {filepath}")
    
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    # Summary
    print("SUMMARY: 68th percentile error by sig0 bin")
    print(f"{'sig0 Bin':<12} {'N':>8} {'Median':>10} {'68th %':>10}")
    for bin_label, p68, color, n in plotted_bins:
        median = df[df['sig0_bin'] == bin_label]['abs_error_node'].median()
        print(f"{bin_label:<12} {n:>8,} {median:>10.3f} {p68:>10.3f}")
    
    return fig


# Run

fig = plot_sig0_error_cdf(
    analysis_final_df,
    figsize=(10, 7),
    save=True, show=True
)

### CDF Reach-level

In [ ]:
"""
CDF Plot: Reach-Level Error Distribution by Flow Angle Zone
============================================================
Using log(x + offset) transformation
- 3 subplots: Parallel, Perpendicular, Other angles
- X-axis: Error (log scale with offset)
- Lines: Each reach colored by ISA value (cyan -> magenta gradient)
- Filtered: Remove U-shape and oxbow reaches (keep 2,3,5,6,7,9,10,11,12th by ISA rank)
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path
import matplotlib.cm as cm

plt.style.use('dark_background')

OUTPUT_DIR = Path("unit_level_analysis")
OUTPUT_DIR.mkdir(exist_ok=True)


def get_flow_angle_zone(flow_angle):
    """
    Classify flow angle into zones
    """
    fa = flow_angle % 360
    if (fa >= 350 or fa <= 10) or (170 <= fa <= 190):
        return 'Parallel'
    elif (80 <= fa <= 100) or (260 <= fa <= 280):
        return 'Perpendicular'
    else:
        return 'Other'


def create_final_reach_df(final_df):
    """
    Create reach-level DataFrame from node-level DataFrame
    
    Aggregation rules:
    - flow_angle: mean per reach_id
    - near_isa, far_isa: mean per reach_id -> reach_near_isa, reach_far_isa
    - reach_geometry_category: if any node has near_ratio > 0.5 -> Near, etc.
    - error_reach: keep all (multiple dates per reach)
    
    Filtering:
    - Sort reaches by isa_value, keep 2,3,5,6,7,9,10,11,12th (remove U-shape/oxbow)
    """
    
    # Step 1: Get unique reach info and filter by ISA rank
    reach_isa = final_df.groupby('reach_id')['isa_value'].first().reset_index()
    reach_isa_sorted = reach_isa.sort_values('isa_value').reset_index(drop=True)
    reach_isa_sorted['rank'] = range(1, len(reach_isa_sorted) + 1)
    
    # Keep ranks 2,3,5,6,7,9,10,11,12 (1-indexed)
    keep_ranks = [2, 3, 5, 6, 7, 9, 10, 11, 12]
    valid_reaches = reach_isa_sorted[reach_isa_sorted['rank'].isin(keep_ranks)]['reach_id'].tolist()
    
    print(f"Original reaches: {len(reach_isa)}")
    print(f"After filtering (remove U-shape/oxbow): {len(valid_reaches)}")
    
    # Step 2: Filter original data
    df_filtered = final_df[final_df['reach_id'].isin(valid_reaches)].copy()
    
    # Step 3: Calculate reach_geometry_category
    def get_reach_geometry(group):
        has_near = (group['near_ratio'] > 0.5).any()
        has_far = (group['far_ratio'] > 0.5).any()
        
        if has_near and has_far:
            return 'Near/Far'
        elif has_near:
            return 'Near'
        elif has_far:
            return 'Far'
        else:
            return 'NA'
    
    reach_geometry = df_filtered.groupby('reach_id').apply(get_reach_geometry).reset_index()
    reach_geometry.columns = ['reach_id', 'reach_geometry_category']
    
    # Step 4: Aggregate reach-level statistics
    reach_stats = df_filtered.groupby('reach_id').agg({
        'flow_angle': 'mean',
        'near_isa': 'mean',
        'far_isa': 'mean',
        'isa_value': 'first'  # same for all nodes in a reach
    }).reset_index()
    reach_stats.columns = ['reach_id', 'reach_flow_angle', 'reach_near_isa', 'reach_far_isa', 'reach_isa_value']
    
    # Step 5: Get error_reach data (multiple per reach due to different dates)
    reach_errors = df_filtered.groupby(['reach_id', 'time_str']).agg({
        'error_reach': 'first'  # same for all nodes in a reach at same time
    }).reset_index()
    
    # Step 6: Merge all
    final_reach_df = reach_errors.merge(reach_stats, on='reach_id')
    final_reach_df = final_reach_df.merge(reach_geometry, on='reach_id')
    
    # Add absolute error
    final_reach_df['abs_error_reach'] = final_reach_df['error_reach'].abs()
    
    # Add flow zone
    final_reach_df['flow_zone'] = final_reach_df['reach_flow_angle'].apply(get_flow_angle_zone)
    
    print(f"Final reach_df shape: {final_reach_df.shape}")
    print(f"Unique reaches: {final_reach_df['reach_id'].nunique()}")
    print(f"Total observations: {len(final_reach_df)}")
    
    return final_reach_df


def plot_cdf_by_reach_isa(ax, data, y_col, isa_col, title, threshold=0.1832, offset=1):
    """
    CDF plot for each reach, colored by ISA value (cyan -> magenta gradient)
    Using log(x + offset) transformation
    """
    ax.set_facecolor('black')
    
    # Get unique reaches sorted by ISA
    reach_info = data.groupby('reach_id').agg({
        isa_col: 'first',
        'reach_isa_value': 'first'
    }).reset_index().sort_values(isa_col)
    
    # Color map: cyan -> magenta
    n_reaches = len(reach_info)
    cmap = cm.get_cmap('cool', n_reaches)
    
    all_errors = data[y_col].dropna()
    if len(all_errors) > 0:
        x_min_orig = all_errors.min()
        x_max_orig = all_errors.max()
    else:
        x_min_orig, x_max_orig = 0, 100
    
    # Plot CDF for each reach
    plotted_reaches = []
    for i, (_, row) in enumerate(reach_info.iterrows()):
        reach_id = row['reach_id']
        isa_value = row[isa_col]
        
        reach_data = data[data['reach_id'] == reach_id][y_col].dropna()
        if len(reach_data) > 0:
            sorted_data = np.sort(reach_data)
            cdf = np.arange(1, len(sorted_data) + 1) / len(sorted_data)
            color = cmap(i)
            
            # +offset 변환
            ax.plot(sorted_data + offset, cdf, color=color, linewidth=1.5)
            
            p68 = np.percentile(reach_data, 68)
            plotted_reaches.append((reach_id, isa_value, p68, color, len(reach_data)))
    
    # 68th percentile line
    ax.axhline(y=0.68, color='lime', linestyle='--', linewidth=1.5, alpha=0.8)
    ax.text(0.02, 0.70, '68%tile', transform=ax.transAxes, color='lime', fontsize=8, fontweight='bold')
    
    # Reference lines (+offset)
    ax.axvline(x=threshold + offset, color='orange', linestyle='-', linewidth=1.5, alpha=0.8)
    ax.axvline(x=0 + offset, color='white', linestyle='--', linewidth=0.8, alpha=0.5)
    
    # Add markers at 68th percentile - alternate positions
    y_offsets_text = [0.03, 0.08, -0.03, -0.08]
    va_aligns = ['bottom', 'bottom', 'top', 'top']
    
    for i, (reach_id, isa_value, p68, color, n) in enumerate(plotted_reaches):
        ax.scatter([p68 + offset], [0.68], color=color, s=30, zorder=5)
        ax.axvline(x=p68 + offset, color=color, linestyle=':', linewidth=0.8, alpha=0.4)
        
        y_offset = y_offsets_text[i % 4]
        va = va_aligns[i % 4]
        
        ax.text(p68 + offset, 0.68 + y_offset, f'{isa_value*100:.0f}%', color=color, fontsize=8, fontweight='bold',
                ha='center', va=va)
    
    # Add 68th percentile summary text - bottom right
    if plotted_reaches:
        ax.text(0.98, 0.48, f'68%tile by {isa_col}', transform=ax.transAxes,
                fontsize=8, color='lime', va='top', ha='right', family='monospace', fontweight='bold')
        
        y_pos = 0.40
        for reach_id, isa_value, p68, color, n in plotted_reaches:
            text = f'{isa_value*100:5.1f}%: {p68:.2f}m (n={n})'
            ax.text(0.98, y_pos, text, transform=ax.transAxes,
                    fontsize=7, color=color, va='top', ha='right', family='monospace')
            y_pos -= 0.04
    
    # Log scale (symlog 대신)
    ax.set_xscale('log')
    ax.set_xlim((x_min_orig + offset) * 0.9, (x_max_orig + offset) * 1.1)
    ax.set_ylim(0, 1)
    
    # X축 라벨 (offset 빼서 원래 값으로 표시)
    def format_func(value, tick_number):
        original = value - offset
        if abs(original) < 0.01:
            return '0'
        elif abs(original) >= 10:
            return f'{original:.0f}'
        elif abs(original) >= 1:
            return f'{original:.1f}'
        else:
            return f'{original:.2f}'
    
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(format_func))
    
    tick_values_orig = [0, 0.2, 0.5, 1, 2, 5, 10, 20, 50, 100]
    tick_values = [v + offset for v in tick_values_orig 
                   if (x_min_orig + offset) * 0.9 <= v + offset <= (x_max_orig + offset) * 1.1]
    ax.set_xticks(tick_values)
    
    ax.set_xlabel('Error (m)', color='white', fontsize=10)
    ax.set_ylabel('CDF', color='white', fontsize=10)
    ax.set_title(title, color='white', fontsize=12, fontweight='bold')
    ax.tick_params(colors='white', labelsize=9)
    ax.grid(True, alpha=0.2, color='white')
    for spine in ax.spines.values():
        spine.set_color('white')


def plot_reach_level_analysis(final_reach_df, y_col='abs_error_reach', threshold=0.1832, 
                               figsize=(18, 5), save=True, show=False, offset=1):
    """
    Create 1x3 plot for Reach-level analysis by flow angle zone
    
    Parameters
    ----------
    final_reach_df : DataFrame
        Output from create_final_reach_df()
    y_col : str
        Error column (e.g., 'abs_error_reach', 'error_reach')
    offset : float
        Offset for log transformation (default=1)
    """
    
    if len(final_reach_df) == 0:
        print("Warning: No data in final_reach_df")
        return None
    
    # Create figure
    fig, axes = plt.subplots(1, 3, figsize=figsize, facecolor='black')
    
    zones = [
        ('Parallel', 'Parallel (350-10°, 170-190°)'),
        ('Perpendicular', 'Perpendicular (80-100°, 260-280°)'),
        ('Other', 'Other Angles'),
    ]
    
    for idx, (zone_key, zone_title) in enumerate(zones):
        df_zone = final_reach_df[final_reach_df['flow_zone'] == zone_key]
        n_total = len(df_zone)
        n_reaches = df_zone['reach_id'].nunique()
        title = f'{zone_title}\n(n={n_total:,}, reaches={n_reaches})'
        
        if n_total > 0:
            plot_cdf_by_reach_isa(axes[idx], df_zone, y_col, 'reach_isa_value', title, threshold, offset)
        else:
            axes[idx].set_facecolor('black')
            axes[idx].text(0.5, 0.5, 'No Data', transform=axes[idx].transAxes,
                          color='white', fontsize=14, ha='center', va='center')
            axes[idx].set_title(title, color='white', fontsize=12, fontweight='bold')
    
    # Main title
    fig.suptitle(f'Reach-Level Analysis: {y_col} (Filtered: U-shape/Oxbow removed)', 
                 fontsize=14, fontweight='bold', color='white', y=1.02)
    
    plt.tight_layout()
    
    # Save
    if save:
        filename = f"reach_level_analysis_{y_col}_log.png"
        filepath = OUTPUT_DIR / filename
        fig.savefig(filepath, dpi=150, facecolor='black', bbox_inches='tight')
        print(f"Saved: {filepath}")
    
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    return fig

In [ ]:
final_reach_df = create_final_reach_df(final_df)
plot_reach_level_analysis(final_reach_df, y_col='abs_error_reach', offset=0.1)

### Statistical Test for Far ISA and Error relationship in Parallel Zone

In [ ]:
"""
Jonckheere-Terpstra Test: Optimized Implementation
==================================================
- NumPy vectorization for speed
- Ties correction for variance
- Multiple effect sizes: τ, r, Cohen's d
- Effect size interpretation based on Gignac & Szodorai (2016)
- Large-sample p-value warning (Lin et al., 2013; ASA Statement 2016)
"""

import numpy as np
import pandas as pd
from scipy.stats import norm


def r_to_d(r):
    """
    Convert correlation r to Cohen's d
    Formula: d = 2r / sqrt(1 - r²)
    Reference: Cohen (1988), Rosenthal (1994)
    """
    if abs(r) >= 1:
        return np.sign(r) * np.inf
    return 2 * r / np.sqrt(1 - r**2)


def jonckheere_terpstra_manual(groups, alternative='increasing'):
    """
    Jonckheere-Terpstra test for ordered alternatives (with ties correction)
    Optimized with NumPy vectorization
    
    Effect sizes:
    - Kendall's tau-like: (concordant - discordant) / total_pairs
    - r from z: z / sqrt(N) (Rosenthal, 1991; Field, 2005)
    - Cohen's d: converted from r (Cohen, 1988)
    """
    k = len(groups)
    if k < 2:
        return {'statistic': np.nan, 'p_value': np.nan, 'z_score': np.nan, 
                'effect_size': np.nan, 'effect_size_r': np.nan, 'effect_size_d': np.nan}
    
    n = [len(g) for g in groups]
    N = sum(n)
    
    if N < 3:
        return {'statistic': np.nan, 'p_value': np.nan, 'z_score': np.nan,
                'effect_size': np.nan, 'effect_size_r': np.nan, 'effect_size_d': np.nan}
    
    # Step 1: Calculate J statistic using NumPy vectorization
    J = 0
    concordant = 0
    discordant = 0
    ties = 0
    
    for i in range(k - 1):
        for j in range(i + 1, k):
            g_i = np.array(groups[i])
            g_j = np.array(groups[j])
            
            diff_matrix = g_j[:, None] - g_i[None, :]
            
            conc = np.sum(diff_matrix > 0)
            disc = np.sum(diff_matrix < 0)
            tie = np.sum(diff_matrix == 0)
            
            concordant += conc
            discordant += disc
            ties += tie
            J += conc + 0.5 * tie
    
    total_pairs = concordant + discordant + ties
    
    # Step 2: Expected value under H0
    E_J = (N**2 - sum(ni**2 for ni in n)) / 4
    
    # Step 3: Variance with ties correction (Lehmann, 1975)
    all_values = np.concatenate(groups)
    unique, counts = np.unique(all_values, return_counts=True)
    tie_counts = counts[counts > 1]
    
    A = N * (N - 1) * (2 * N + 5)
    B = sum(ni * (ni - 1) * (2 * ni + 5) for ni in n)
    C = sum(t * (t - 1) * (2 * t + 5) for t in tie_counts) if len(tie_counts) > 0 else 0
    
    var_term1 = (A - B - C) / 72
    
    D_groups = sum(ni * (ni - 1) * (ni - 2) for ni in n)
    D_ties = sum(t * (t - 1) * (t - 2) for t in tie_counts) if len(tie_counts) > 0 else 0
    var_term2 = (D_groups * D_ties) / (36 * N * (N - 1) * (N - 2)) if N > 2 else 0
    
    E_groups = sum(ni * (ni - 1) for ni in n)
    E_ties = sum(t * (t - 1) for t in tie_counts) if len(tie_counts) > 0 else 0
    var_term3 = (E_groups * E_ties) / (8 * N * (N - 1)) if N > 1 else 0
    
    var_J = var_term1 + var_term2 + var_term3
    
    if var_J <= 0:
        var_J = (N**2 * (2*N + 3) - sum(ni**2 * (2*ni + 3) for ni in n)) / 72
    
    # Step 4: Z statistic with continuity correction
    if var_J > 0:
        if J > E_J:
            Z = (J - E_J - 0.5) / np.sqrt(var_J)
        elif J < E_J:
            Z = (J - E_J + 0.5) / np.sqrt(var_J)
        else:
            Z = 0
    else:
        Z = 0
    
    # Step 5: p-value
    if alternative == 'increasing':
        p_value = 1 - norm.cdf(Z)
    elif alternative == 'decreasing':
        p_value = norm.cdf(Z)
    else:
        p_value = 2 * (1 - norm.cdf(abs(Z)))
    
    # Step 6: Effect sizes
    # Kendall's tau-like effect size
    tau_effect = (concordant - discordant) / total_pairs if total_pairs > 0 else 0
    
    # r from z (Rosenthal, 1991; Field, 2005)
    r_effect = Z / np.sqrt(N) if N > 0 else 0
    
    # Cohen's d from r (Cohen, 1988)
    d_effect = r_to_d(r_effect)
    
    return {
        'statistic': J,
        'expected': E_J,
        'variance': var_J,
        'z_score': Z,
        'p_value': p_value,
        'effect_size': tau_effect,       # Kendall's tau-like
        'effect_size_r': r_effect,        # r = z/sqrt(N)
        'effect_size_d': d_effect,        # Cohen's d from r
        'concordant': concordant,
        'discordant': discordant,
        'ties': ties,
        'total_pairs': total_pairs,
        'n_total': N
    }


def get_effect_magnitude_r(effect_size):
    """
    Effect size interpretation for r based on Gignac & Szodorai (2016)
    """
    abs_effect = abs(effect_size)
    if abs_effect < 0.10:
        return 'negligible'
    elif abs_effect < 0.20:
        return 'small'
    elif abs_effect < 0.30:
        return 'medium'
    else:
        return 'large'


def get_effect_magnitude_d(effect_size):
    """
    Effect size interpretation for Cohen's d based on Cohen (1988)
    - 0.20: small
    - 0.50: medium
    - 0.80: large
    
    Also considering Gignac-adjusted: 0.20, 0.40, 0.60
    """
    abs_effect = abs(effect_size)
    if abs_effect < 0.20:
        return 'negligible'
    elif abs_effect < 0.50:
        return 'small'
    elif abs_effect < 0.80:
        return 'medium'
    else:
        return 'large'


def get_flow_angle_zone(flow_angle):
    fa = flow_angle % 360
    if (fa >= 350 or fa <= 10) or (170 <= fa <= 190):
        return 'Parallel'
    elif (80 <= fa <= 100) or (260 <= fa <= 280):
        return 'Perpendicular'
    else:
        return 'Other'


def get_isa_bin(isa_value):
    """
    ISA bin classification
    Assumes isa_value is percentage (0~100)
    """
    if isa_value <= 5:
        return '0-5'
    elif isa_value <= 10:
        return '5-10'
    elif isa_value <= 20:
        return '10-20'
    elif isa_value <= 30:
        return '20-30'
    elif isa_value <= 40:
        return '30-40'
    elif isa_value <= 50:
        return '40-50'
    elif isa_value <= 60:
        return '50-60'
    elif isa_value <= 70:
        return '60-70'
    elif isa_value <= 80:
        return '70-80'
    elif isa_value <= 90:
        return '80-90'
    else:
        return '90-100'


def sanity_check():
    """Sanity check: verify implementation with known cases"""
    print("SANITY CHECK")
    
    # Test 1: Perfect increasing
    groups1 = [np.array([1, 2, 3]), np.array([4, 5, 6]), np.array([7, 8, 9])]
    result1 = jonckheere_terpstra_manual(groups1, alternative='increasing')
    pass1 = result1['p_value'] < 0.05 and result1['effect_size'] > 0.9
    print(f"\n[Test 1] Perfect increasing: p={result1['p_value']:.4f}, τ={result1['effect_size']:.3f}, d={result1['effect_size_d']:.3f} -> {'' if pass1 else ''}")
    
    # Test 2: Perfect decreasing
    groups2 = [np.array([7, 8, 9]), np.array([4, 5, 6]), np.array([1, 2, 3])]
    result2 = jonckheere_terpstra_manual(groups2, alternative='increasing')
    pass2 = result2['p_value'] > 0.5 and result2['effect_size'] < -0.9
    print(f"[Test 2] Perfect decreasing: p={result2['p_value']:.4f}, τ={result2['effect_size']:.3f}, d={result2['effect_size_d']:.3f} -> {'' if pass2 else ''}")
    
    # Test 3: No trend
    groups3 = [np.array([1, 5, 9]), np.array([2, 6, 7]), np.array([3, 4, 8])]
    result3 = jonckheere_terpstra_manual(groups3, alternative='increasing')
    pass3 = result3['p_value'] > 0.05
    print(f"[Test 3] No trend: p={result3['p_value']:.4f}, τ={result3['effect_size']:.3f}, d={result3['effect_size_d']:.3f} -> {'' if pass3 else ''}")
    
    # Test 4: With ties
    groups4 = [np.array([1, 1, 2]), np.array([2, 3, 3]), np.array([4, 4, 5])]
    result4 = jonckheere_terpstra_manual(groups4, alternative='increasing')
    pass4 = result4['p_value'] < 0.05
    print(f"[Test 4] With ties: p={result4['p_value']:.4f}, ties={result4['ties']}, d={result4['effect_size_d']:.3f} -> {'' if pass4 else ''}")
    
    all_pass = pass1 and pass2 and pass3 and pass4
    print(f"\nALL TESTS: {'PASSED' if all_pass else 'SOME FAILED'}")
    return all_pass


def run_jt_test(final_df, y_col='abs_error_node', run_sanity=True):
    """
    Jonckheere-Terpstra Test for ISA gradient effect on error
    """
    
    if run_sanity:
        sanity_check()
    
    bin_order = ['0-5', '5-10', '10-20', '20-30', '30-40', '40-50', 
                 '50-60', '60-70', '70-80', '80-90', '90-100']
    
    # Prepare data
    df = final_df.dropna(subset=[y_col, 'flow_angle', 'near_isa', 'far_isa']).copy()
    df['flow_zone'] = df['flow_angle'].apply(get_flow_angle_zone)
    df['near_isa_bin'] = df['near_isa'].apply(get_isa_bin)
    df['far_isa_bin'] = df['far_isa'].apply(get_isa_bin)
    
    results = {}
    
    zones = ['Parallel', 'Perpendicular', 'Other']
    isa_types = [('near_isa_bin', 'Near ISA'), ('far_isa_bin', 'Far ISA')]
    
    print(f"Jonckheere-Terpstra Test: {y_col}")
    print("H0: No monotonic trend (error does not increase with ISA)")
    print("H1: Monotonic increasing trend (error increases with ISA)")
    print("α = 0.05")
    
    for zone in zones:
        df_zone = df[df['flow_zone'] == zone]
        n_total = len(df_zone)
        
        print(f"\n{'━' * 70}")
        print(f"Zone: {zone} (n={n_total:,})")
        print(f"{'━' * 70}")
        
        results[zone] = {}
        
        for isa_col, isa_name in isa_types:
            groups = []
            group_info = []
            
            for bin_label in bin_order:
                bin_data = df_zone[df_zone[isa_col] == bin_label][y_col].values
                if len(bin_data) > 0:
                    groups.append(bin_data)
                    group_info.append({
                        'bin': bin_label,
                        'n': len(bin_data),
                        'median': np.median(bin_data),
                        'mean': np.mean(bin_data)
                    })
            
            if len(groups) >= 2:
                jt_result = jonckheere_terpstra_manual(groups, alternative='increasing')
                
                mag_r = get_effect_magnitude_r(jt_result['effect_size_r'])
                mag_d = get_effect_magnitude_d(jt_result['effect_size_d'])
                
                results[zone][isa_name] = {
                    'statistic': jt_result['statistic'],
                    'z_score': jt_result['z_score'],
                    'p_value': jt_result['p_value'],
                    'effect_size': jt_result['effect_size'],
                    'effect_size_r': jt_result['effect_size_r'],
                    'effect_size_d': jt_result['effect_size_d'],
                    'magnitude_r': mag_r,
                    'magnitude_d': mag_d,
                    'n_groups': len(groups),
                    'n_samples': n_total,
                    'significant': jt_result['p_value'] < 0.05,
                    'group_info': group_info,
                    'concordant': jt_result['concordant'],
                    'discordant': jt_result['discordant'],
                    'ties': jt_result['ties']
                }
                
                sig_marker = "✓ SIGNIFICANT" if jt_result['p_value'] < 0.05 else "✗ Not significant"
                print(f"\n  {isa_name}:")
                print(f"    ISA bins with data: {len(groups)}")
                print(f"    J-T Statistic: {jt_result['statistic']:,.1f}")
                print(f"    Z-score: {jt_result['z_score']:.3f}")
                print(f"    p-value: {jt_result['p_value']:.2e}")
                print(f"    Effect size (τ): {jt_result['effect_size']:.3f}")
                print(f"    Effect size (r): {jt_result['effect_size_r']:.3f} ({mag_r})")
                print(f"    Effect size (d): {jt_result['effect_size_d']:.3f} ({mag_d})")
                print(f"    Pairs: {jt_result['concordant']:,} concordant, {jt_result['discordant']:,} discordant, {jt_result['ties']:,} ties")
                print(f"    -> {sig_marker}")
                
                print(f"\n    Median error by ISA bin:")
                for info in group_info:
                    print(f"      {info['bin']:>6}%: {info['median']:.3f} m (n={info['n']})")
            else:
                print(f"\n  {isa_name}: Not enough bins (need ≥ 2, got {len(groups)})")
                results[zone][isa_name] = None
    
    # Summary Table
    print("SUMMARY TABLE")
    print(f"\n{'Zone':<13} {'ISA Type':<10} {'n':>7} {'z':>8} {'p-value':>12} {'τ':>7} {'r':>7} {'d':>7} {'Mag(r)':<10} {'Result':<15}")
    
    for zone in zones:
        for isa_name in ['Near ISA', 'Far ISA']:
            r = results[zone].get(isa_name)
            if r:
                status = "SIGNIFICANT ↑" if r['significant'] else "not sig."
                print(f"{zone:<13} {isa_name:<10} {r['n_samples']:>7,} {r['z_score']:>8.2f} {r['p_value']:>12.2e} {r['effect_size']:>7.3f} {r['effect_size_r']:>7.3f} {r['effect_size_d']:>7.3f} {r['magnitude_r']:<10} {status:<15}")
            else:
                print(f"{zone:<13} {isa_name:<10} {'N/A':>7} {'N/A':>8} {'N/A':>12} {'N/A':>7} {'N/A':>7} {'N/A':>7} {'N/A':<10} {'insufficient':<15}")
    
    # Large Sample Warning
    print("IMPORTANT: LARGE-SAMPLE P-VALUE CAVEAT")
    print("""
With large sample sizes, p-values can be misleading. As Lin et al. (2013) note,
"p-values approach zero quickly with large samples, leading researchers to claim
support for results of no practical significance." The ASA Statement (2016)
emphasizes: "A p-value does not measure the size of an effect or the importance
of a result."

Therefore, interpretation should focus on EFFECT SIZE, not p-value.
""")
    
    # Effect Size Guide
    print("EFFECT SIZE INTERPRETATION")
    print("""
┌─────────────────────────────────────────────────────────────────────────┐
│  Correlation (r, τ) - Gignac & Szodorai (2016)                          │
│  ─────────────────────────────────────────────────────────────────────  │
│    |r| < 0.10  : negligible                                             │
│    |r| 0.10-0.20: small                                                 │
│    |r| 0.20-0.30: medium                                                │
│    |r| >= 0.30  : large                                                 │
├─────────────────────────────────────────────────────────────────────────┤
│  Cohen's d - Cohen (1988)                                               │
│  ─────────────────────────────────────────────────────────────────────  │
│    |d| < 0.20  : negligible                                             │
│    |d| 0.20-0.50: small                                                 │
│    |d| 0.50-0.80: medium                                                │
│    |d| >= 0.80  : large                                                 │
├─────────────────────────────────────────────────────────────────────────┤
│  Conversion: d = 2r / √(1 - r²)                                         │
│  Note: τ ≈ 0.7 × r (Kendall's tau tends to be smaller than Pearson's r) │
└─────────────────────────────────────────────────────────────────────────┘
""")
    
    # Interpretation
    print("INTERPRETATION")
    
    parallel_far = results.get('Parallel', {}).get('Far ISA')
    parallel_near = results.get('Parallel', {}).get('Near ISA')
    
    if parallel_far and parallel_near:
        print(f"\nParallel Zone Analysis:")
        print(f"  ┌─────────────────────────────────────────────────────┐")
        print(f"  │  Far ISA:  τ={parallel_far['effect_size']:.3f}, r={parallel_far['effect_size_r']:.3f}, d={parallel_far['effect_size_d']:.3f} ({parallel_far['magnitude_r']}) │")
        print(f"  │  Near ISA: τ={parallel_near['effect_size']:.3f}, r={parallel_near['effect_size_r']:.3f}, d={parallel_near['effect_size_d']:.3f} ({parallel_near['magnitude_r']}) │")
        print(f"  └─────────────────────────────────────────────────────┘")
        
        far_r = parallel_far['effect_size_r']
        near_r = parallel_near['effect_size_r']
        
        if far_r > 0.10 and near_r > 0.10:
            ratio = far_r / near_r if near_r != 0 else float('inf')
            print(f"\n  -> Far ISA effect is {ratio:.1f}× stronger than Near ISA effect.")
            print("  -> Urban areas on the FAR side show stronger impact on WSE error,")
            print("    consistent with double-bounce scattering mechanism.")
        elif far_r > 0.10 and near_r < 0.10:
            print("\n  -> Only Far ISA shows meaningful effect (r > 0.10).")
            print("  -> Supports hypothesis: urban effect primarily on FAR side.")
        elif far_r < 0.10 and near_r < 0.10:
            print("\n  -> Both effects are negligible (r < 0.10).")
            print("  -> No clear ISA-error relationship despite statistical significance.")
    
    # References
    print("REFERENCES")
    print("""
Effect size interpretation:
  - Cohen, J. (1988). Statistical Power Analysis for the Behavioral Sciences (2nd ed.). 
    Lawrence Erlbaum Associates.
  - Gignac, G. E., & Szodorai, E. T. (2016). Effect size guidelines for individual 
    differences researchers. Personality and Individual Differences, 102, 74-78.
  - Funder, D. C., & Ozer, D. J. (2019). Evaluating effect size in psychological 
    research: Sense and nonsense. AMPPS, 2(2), 156-168.

Effect size calculation:
  - r = z/√N: Rosenthal (1991), Field (2005)
  - d = 2r/√(1-r²): Cohen (1988)

Large-sample p-value problem:
  - Lin, M., Lucas, H. C., & Shmueli, G. (2013). Too Big to Fail: Large Samples 
    and the p-Value Problem. Information Systems Research, 24(4), 906-917.
  - Wasserstein, R. L., & Lazar, N. A. (2016). The ASA statement on p-values. 
    The American Statistician, 70(2), 129-133.
""")
    
    return results

In [ ]:
results = run_jt_test(final_df, y_col='abs_error_node')

### More focus on Building Height

#### DW near building height

In [ ]:
"""
DW Near Building Analysis: 1x2 Plot
====================================
- CDF by DW Near Building bins
- Boxplot by DW Near Building bins
- Bins: 6m to 11m, 0.5m interval
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from pathlib import Path

plt.style.use('dark_background')

OUTPUT_DIR = Path("unit_level_analysis")
OUTPUT_DIR.mkdir(exist_ok=True)


def get_flow_angle_zone(flow_angle):
    fa = flow_angle % 360
    if (fa >= 350 or fa <= 10) or (170 <= fa <= 190):
        return 'Parallel'
    elif (80 <= fa <= 100) or (260 <= fa <= 280):
        return 'Perpendicular'
    else:
        return 'Other'


def get_building_bin(value):
    """Building height bins: 6-11m, 0.5m interval"""
    if value < 6.0:
        return '<6.0'
    elif value < 6.5:
        return '6.0-6.5'
    elif value < 7.0:
        return '6.5-7.0'
    elif value < 7.5:
        return '7.0-7.5'
    elif value < 8.0:
        return '7.5-8.0'
    elif value < 8.5:
        return '8.0-8.5'
    elif value < 9.0:
        return '8.5-9.0'
    elif value < 9.5:
        return '9.0-9.5'
    elif value < 10.0:
        return '9.5-10.0'
    elif value < 10.5:
        return '10.0-10.5'
    elif value < 11.0:
        return '10.5-11.0'
    else:
        return '≥11.0'


def plot_dw_near_building_analysis(df, y_col='abs_error_node',
                                    threshold=0.1832, offset=1,
                                    figsize=(12, 5),
                                    save=True, show=True):
    """
    1x2 DW Near Building Analysis
    """
    
    print("DW Near Building Analysis: 1x2 Plot")
    
    # Prepare data
    df_plot = df.dropna(subset=[y_col, 'flow_angle', 'dw_near_building']).copy()
    df_plot['flow_zone'] = df_plot['flow_angle'].apply(get_flow_angle_zone)
    df_parallel = df_plot[df_plot['flow_zone'] == 'Parallel'].copy()
    df_parallel['building_bin'] = df_parallel['dw_near_building'].apply(get_building_bin)
    
    print(f"[Data] Parallel zone: {len(df_parallel):,}")
    
    bin_order = ['<6.0', '6.0-6.5', '6.5-7.0', '7.0-7.5', '7.5-8.0', '8.0-8.5',
                 '8.5-9.0', '9.0-9.5', '9.5-10.0', '10.0-10.5', '10.5-11.0', '≥11.0']
    existing_bins = [b for b in bin_order if b in df_parallel['building_bin'].values]
    
    cmap = cm.get_cmap('hot', len(existing_bins))
    bin_colors = {existing_bins[i]: cmap(i) for i in range(len(existing_bins))}
    
    fig, axes = plt.subplots(1, 2, figsize=figsize, facecolor='black')
    
    # Plot 1: CDF
    ax1 = axes[0]
    ax1.set_facecolor('#1a1a1a')
    
    all_errors = df_parallel[y_col].dropna()
    x_min_orig = all_errors.min()
    x_max_orig = all_errors.max()
    
    plotted_bins = []
    for bin_label in existing_bins:
        bin_data = df_parallel[df_parallel['building_bin'] == bin_label][y_col].dropna()
        if len(bin_data) > 0:
            sorted_data = np.sort(bin_data)
            cdf = np.arange(1, len(sorted_data) + 1) / len(sorted_data)
            color = bin_colors[bin_label]
            
            ax1.plot(sorted_data + offset, cdf, color=color, linewidth=1.5)
            
            p68 = np.percentile(bin_data, 68)
            plotted_bins.append((bin_label, p68, color, len(bin_data)))
    
    ax1.axhline(y=0.68, color='lime', linestyle='--', linewidth=1.5, alpha=0.8)
    ax1.text(0.02, 0.70, '68%tile', transform=ax1.transAxes, color='lime', fontsize=8, fontweight='bold')
    ax1.axvline(x=threshold + offset, color='orange', linestyle='-', linewidth=1.5, alpha=0.8)
    
    y_offsets_text = [0.03, 0.08, -0.03, -0.08]
    va_aligns = ['bottom', 'bottom', 'top', 'top']
    
    for i, (bin_label, p68, color, n) in enumerate(plotted_bins):
        ax1.scatter([p68 + offset], [0.68], color=color, s=30, zorder=5)
        ax1.axvline(x=p68 + offset, color=color, linestyle=':', linewidth=0.8, alpha=0.4)
        
        y_offset = y_offsets_text[i % 4]
        va = va_aligns[i % 4]
        ax1.text(p68 + offset, 0.68 + y_offset, f'{bin_label}m', color=color, fontsize=7, fontweight='bold',
                ha='center', va=va)
    
    if plotted_bins:
        ax1.text(0.98, 0.48, '68%tile Error (m)', transform=ax1.transAxes,
                fontsize=7, color='lime', va='top', ha='right', family='monospace', fontweight='bold')
        y_pos = 0.40
        for bin_label, p68, color, n in plotted_bins:
            text = f'{bin_label:>9}m: {p68:.2f}m (n={n})'
            ax1.text(0.98, y_pos, text, transform=ax1.transAxes,
                    fontsize=6, color=color, va='top', ha='right', family='monospace')
            y_pos -= 0.035
    
    ax1.set_xscale('log')
    ax1.set_xlim((x_min_orig + offset) * 0.9, (x_max_orig + offset) * 1.1)
    ax1.set_ylim(0, 1)
    
    def format_func(value, tick_number):
        original = value - offset
        if abs(original) < 0.01:
            return '0'
        elif abs(original) >= 10:
            return f'{original:.0f}'
        elif abs(original) >= 1:
            return f'{original:.1f}'
        else:
            return f'{original:.2f}'
    
    ax1.xaxis.set_major_formatter(ticker.FuncFormatter(format_func))
    tick_values_orig = [0, 0.2, 0.5, 1, 2, 5, 10, 20, 50, 100]
    tick_values = [v + offset for v in tick_values_orig 
                   if (x_min_orig + offset) * 0.9 <= v + offset <= (x_max_orig + offset) * 1.1]
    ax1.set_xticks(tick_values)
    
    ax1.set_xlabel('abs_error_node (m)', color='white', fontsize=10)
    ax1.set_ylabel('CDF', color='white', fontsize=10)
    ax1.set_title(f'CDF by DW Near Building Height (n={len(df_parallel):,})', 
                  color='white', fontsize=11, fontweight='bold')
    ax1.tick_params(colors='white', labelsize=8)
    ax1.grid(True, alpha=0.2, color='white')
    for spine in ax1.spines.values():
        spine.set_color('white')
    
    # Plot 2: Boxplot
    ax2 = axes[1]
    ax2.set_facecolor('#1a1a1a')
    
    box_data = []
    positions = []
    colors = []
    bin_labels = []
    
    for bin_label in existing_bins:
        bin_df = df_parallel[df_parallel['building_bin'] == bin_label]
        if len(bin_df) > 0:
            error_vals = bin_df[y_col].dropna().values
            if len(error_vals) > 0:
                box_data.append(error_vals)
                positions.append(len(positions))
                colors.append(bin_colors[bin_label])
                bin_labels.append(bin_label)
    
    bp = ax2.boxplot(box_data, positions=positions, widths=0.6, patch_artist=True,
                     whis=1.5,
                     medianprops=dict(color='white', linewidth=2),
                     whiskerprops=dict(color='white'),
                     capprops=dict(color='white'),
                     flierprops=dict(marker='o', markersize=3, alpha=0.5))
    
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
        patch.set_edgecolor('white')
    
    for flier, color in zip(bp['fliers'], colors):
        flier.set_markerfacecolor(color)
        flier.set_markeredgecolor(color)
    
    ax2.set_yscale('symlog', linthresh=1)
    error_ticks = [0.1, 0.5, 1, 2, 5, 10, 20]
    ax2.set_yticks(error_ticks)
    ax2.set_yticklabels([str(t) for t in error_ticks], fontsize=8, color='white')
    ax2.set_ylim(0.05, 30)
    
    ax2.set_xticks(positions)
    ax2.set_xticklabels(bin_labels, rotation=45, ha='right', color='white', fontsize=8)
    ax2.set_xlabel('DW Near Building Height (m)', color='white', fontsize=10)
    ax2.set_ylabel('abs_error_node (m)', color='white', fontsize=10)
    ax2.set_title('Boxplot by DW Near Building Height', color='white', fontsize=11, fontweight='bold')
    
    ax2.grid(True, alpha=0.2, color='white')
    ax2.set_xlim(-0.5, len(bin_labels) - 0.5)
    
    for spine in ax2.spines.values():
        spine.set_color('white')
    ax2.tick_params(colors='white', labelsize=8)
    
    fig.suptitle('Parallel Zone: DW Near Building Height Analysis', 
                 fontsize=14, fontweight='bold', color='white', y=0.98)
    
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    
    if save:
        filepath = OUTPUT_DIR / "dw_near_building_analysis_1x2.png"
        fig.savefig(filepath, dpi=150, facecolor='black', bbox_inches='tight')
        print(f"\nSaved: {filepath}")
    
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    return fig


# Run

fig = plot_dw_near_building_analysis(
    final_df,
    y_col='abs_error_node',
    figsize=(12, 5),
    save=True, show=True
)

#### CDFs for all interested vars

In [ ]:
"""
CDF Plot: Parallel Zone Analysis with Multiple Variable Sets
=============================================================
Using log(x + offset) transformation

Variable Sets:
1. ISA (simple): near_isa, far_isa
2. ISA ratio: near_ratio, far_ratio
3. ISA (distance-weighted): dw_near_isa, dw_far_isa
4. ISA ratio (distance-weighted): dw_near_ratio, dw_far_ratio
5. Building (simple): near_building, far_building
6. Building ratio: near_ratio_building, far_ratio_building
7. Building (distance-weighted): dw_near_building, dw_far_building
8. Building ratio (distance-weighted): dw_near_ratio_building, dw_far_ratio_building
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path
import matplotlib.cm as cm

plt.style.use('dark_background')

OUTPUT_DIR = Path("unit_level_analysis")
OUTPUT_DIR.mkdir(exist_ok=True)


# Variable Set Configurations
VARIABLE_SETS = {
    'isa': {
        'near_col': 'near_isa',
        'far_col': 'far_isa',
        'title': 'ISA (%)',
        'bin_type': 'isa',
        'unit': '%',
    },
    'isa_ratio': {
        'near_col': 'near_ratio',
        'far_col': 'far_ratio',
        'title': 'ISA Ratio',
        'bin_type': 'ratio',
        'unit': '',
    },
    'dw_isa': {
        'near_col': 'dw_near_isa',
        'far_col': 'dw_far_isa',
        'title': 'Distance-Weighted ISA (%)',
        'bin_type': 'isa',
        'unit': '%',
    },
    'dw_isa_ratio': {
        'near_col': 'dw_near_ratio',
        'far_col': 'dw_far_ratio',
        'title': 'Distance-Weighted ISA Ratio',
        'bin_type': 'ratio',
        'unit': '',
    },
    'building': {
        'near_col': 'near_building',
        'far_col': 'far_building',
        'title': 'Building Height (m)',
        'bin_type': 'building',
        'unit': 'm',
    },
    'building_ratio': {
        'near_col': 'near_ratio_building',
        'far_col': 'far_ratio_building',
        'title': 'Building Ratio',
        'bin_type': 'ratio',
        'unit': '',
    },
    'dw_building': {
        'near_col': 'dw_near_building',
        'far_col': 'dw_far_building',
        'title': 'Distance-Weighted Building Height (m)',
        'bin_type': 'building',
        'unit': 'm',
    },
    'dw_building_ratio': {
        'near_col': 'dw_near_ratio_building',
        'far_col': 'dw_far_ratio_building',
        'title': 'Distance-Weighted Building Ratio',
        'bin_type': 'ratio',
        'unit': '',
    },
}


# Bin Configurations
BIN_CONFIGS = {
    'isa': {
        'bins': [
            (0, 5, '0-5'),
            (5, 10, '5-10'),
            (10, 20, '10-20'),
            (20, 30, '20-30'),
            (30, 40, '30-40'),
            (40, 50, '40-50'),
            (50, 60, '50-60'),
            (60, 70, '60-70'),
            (70, 80, '70-80'),
            (80, 90, '80-90'),
            (90, 100, '90-100'),
        ],
        'cmap': 'cool',
        'label_suffix': '%',
    },
    'ratio': {
        'bins': [
            (0.0, 0.1, '0-10'),
            (0.1, 0.2, '10-20'),
            (0.2, 0.3, '20-30'),
            (0.3, 0.4, '30-40'),
            (0.4, 0.5, '40-50'),
            (0.5, 0.6, '50-60'),
            (0.6, 0.7, '60-70'),
            (0.7, 0.8, '70-80'),
            (0.8, 0.9, '80-90'),
            (0.9, 1.0, '90-100'),
        ],
        'cmap': 'summer',
        'label_suffix': '%',
    },
    'building': {
        'bins': [
            (0, 0.01, '0'),        # Zero (no building)
            (0.01, 7, '0-7'),      # Low
            (7, 8, '7-8'),         # 
            (8, 9, '8-9'),         # Main concentration
            (9, 10, '9-10'),       #
            (10, 12, '10-12'),     # 
            (12, 15, '12-15'),     # Tail
            (15, 20, '15-20'),     #
            (20, 100, '20+'),      # High
        ],
        'cmap': 'hot',
        'label_suffix': 'm',
    },
}


def get_flow_angle_zone(flow_angle):
    """Classify flow angle into zones"""
    fa = flow_angle % 360
    if (fa >= 350 or fa <= 10) or (170 <= fa <= 190):
        return 'Parallel'
    elif (80 <= fa <= 100) or (260 <= fa <= 280):
        return 'Perpendicular'
    else:
        return 'Other'


def get_bin_label(value, bin_type):
    """Get bin label for a value based on bin type"""
    config = BIN_CONFIGS[bin_type]
    
    for low, high, label in config['bins']:
        if low <= value < high:
            return label
        # Handle last bin (inclusive upper bound)
        if high == config['bins'][-1][1] and value >= high:
            return label
    
    # Default: first or last bin
    if value < config['bins'][0][0]:
        return config['bins'][0][2]
    return config['bins'][-1][2]


def get_bin_order(bin_type):
    """Get ordered list of bin labels"""
    return [label for _, _, label in BIN_CONFIGS[bin_type]['bins']]


def get_bin_colors(bin_type):
    """Get color mapping for bins"""
    config = BIN_CONFIGS[bin_type]
    n_bins = len(config['bins'])
    cmap = cm.get_cmap(config['cmap'], n_bins)
    bin_order = get_bin_order(bin_type)
    return {label: cmap(i) for i, label in enumerate(bin_order)}


def plot_cdf_original(ax, data, y_col, threshold=0.1832, offset=1):
    """
    Original CDF plot by sample_geometry_category
    Using log(x + offset) transformation
    """
    ax.set_facecolor('black')
    
    colors = {
        'NA': 'white',
        'Near': '#3399ff',
        'Far': '#ff6666',
        'Near/Far': '#ffdd00'
    }
    
    geometry_cats = ['NA', 'Near', 'Far', 'Near/Far']
    pct_data = []
    p68_data = []
    
    all_errors = data[y_col].dropna()
    if len(all_errors) > 0:
        x_min_orig = all_errors.min()
        x_max_orig = all_errors.max()
    else:
        x_min_orig, x_max_orig = 0, 100
    
    for cat in geometry_cats:
        cat_data = data[data['sample_geometry_category'] == cat][y_col].dropna()
        if len(cat_data) > 0:
            sorted_data = np.sort(cat_data)
            cdf = np.arange(1, len(sorted_data) + 1) / len(sorted_data)
            
            ax.plot(sorted_data + offset, cdf, color=colors[cat], 
                   label=f'{cat} (n={len(cat_data)})', linewidth=2)
            
            p68 = np.percentile(cat_data, 68)
            p68_data.append((cat, p68, colors[cat]))
            
            pct_below = 100 * (cat_data < threshold).sum() / len(cat_data)
            pct_above = 100 * (cat_data >= threshold).sum() / len(cat_data)
            pct_data.append((cat, pct_below, pct_above, colors[cat]))
    
    # Reference lines (+offset)
    ax.axvline(x=threshold + offset, color='orange', linestyle='-', linewidth=1.5, alpha=0.8)
    ax.axvline(x=0 + offset, color='white', linestyle='--', linewidth=0.8, alpha=0.5)
    ax.axhline(y=0.68, color='lime', linestyle='--', linewidth=1.5, alpha=0.8)
    ax.text(0.02, 0.70, '68%tile', transform=ax.transAxes, color='lime', fontsize=8, fontweight='bold')
    
    # 68th percentile markers
    y_offsets = [0.01, -0.04, 0.01, -0.04]
    va_aligns = ['bottom', 'top', 'bottom', 'top']
    
    for i, (cat, p68, color) in enumerate(p68_data):
        ax.axvline(x=p68 + offset, color=color, linestyle=':', linewidth=1, alpha=0.6)
        ax.scatter([p68 + offset], [0.68], color=color, s=30, zorder=5)
        
        y_pos = y_offsets[i % 4]
        va = va_aligns[i % 4]
        
        ax.text(p68 + offset, y_pos, f'{p68:.2f}', transform=ax.get_xaxis_transform(),
                color=color, fontsize=9, fontweight='bold', ha='center', va=va)
    
    # Threshold percentage text
    if pct_data:
        ax.text(0.98, 0.62, f'{"T=0.1832m":8s} {"<T":>6s} {">T":>6s}', 
                transform=ax.transAxes, fontsize=8, color='lime', va='top', ha='right', 
                family='monospace')
        y_pos = 0.54
        for cat, below, above, color in pct_data:
            text = f'{cat:8s} {below:5.0f}% {above:5.0f}%'
            ax.text(0.98, y_pos, text, transform=ax.transAxes, 
                    fontsize=8, color=color, va='top', ha='right', family='monospace')
            y_pos -= 0.08
    
    # Log scale
    ax.set_xscale('log')
    ax.set_xlim((x_min_orig + offset) * 0.9, (x_max_orig + offset) * 1.1)
    ax.set_ylim(0, 1)
    
    def format_func(value, tick_number):
        original = value - offset
        if abs(original) < 0.01:
            return '0'
        elif abs(original) >= 10:
            return f'{original:.0f}'
        elif abs(original) >= 1:
            return f'{original:.1f}'
        else:
            return f'{original:.2f}'
    
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(format_func))
    
    tick_values_orig = [0, 0.2, 0.5, 1, 2, 5, 10, 20, 50, 100]
    tick_values = [v + offset for v in tick_values_orig 
                   if (x_min_orig + offset) * 0.9 <= v + offset <= (x_max_orig + offset) * 1.1]
    ax.set_xticks(tick_values)
    
    ax.set_xlabel('Error (m)', color='white', fontsize=10)
    ax.set_ylabel('CDF', color='white', fontsize=10)
    ax.set_title(f'Parallel (350-10°, 170-190°)\n(n={len(data):,})', 
                 color='white', fontsize=12, fontweight='bold')
    ax.tick_params(colors='white', labelsize=9)
    ax.legend(loc='lower right', fontsize=8, facecolor='black', edgecolor='white', labelcolor='white')
    ax.grid(True, alpha=0.2, color='white')
    for spine in ax.spines.values():
        spine.set_color('white')


def plot_cdf_by_bin(ax, data, y_col, bin_col, bin_type, title, threshold=0.1832, offset=1):
    """
    CDF plot by bins (variable bin intervals based on bin_type)
    Using log(x + offset) transformation
    """
    ax.set_facecolor('black')
    
    bin_order = get_bin_order(bin_type)
    bin_colors = get_bin_colors(bin_type)
    label_suffix = BIN_CONFIGS[bin_type]['label_suffix']
    
    data = data.copy()
    data['_bin'] = data[bin_col].apply(lambda x: get_bin_label(x, bin_type))
    
    all_errors = data[y_col].dropna()
    if len(all_errors) > 0:
        x_min_orig = all_errors.min()
        x_max_orig = all_errors.max()
    else:
        x_min_orig, x_max_orig = 0, 100
    
    # Plot CDF for each bin
    plotted_bins = []
    for bin_label in bin_order:
        bin_data = data[data['_bin'] == bin_label][y_col].dropna()
        if len(bin_data) > 0:
            sorted_data = np.sort(bin_data)
            cdf = np.arange(1, len(sorted_data) + 1) / len(sorted_data)
            color = bin_colors[bin_label]
            
            ax.plot(sorted_data + offset, cdf, color=color, linewidth=1.5)
            
            p68 = np.percentile(bin_data, 68)
            plotted_bins.append((bin_label, p68, color, len(bin_data)))
    
    # 68th percentile line
    ax.axhline(y=0.68, color='lime', linestyle='--', linewidth=1.5, alpha=0.8)
    ax.text(0.02, 0.70, '68%tile', transform=ax.transAxes, color='lime', fontsize=8, fontweight='bold')
    
    # Reference lines (+offset)
    ax.axvline(x=threshold + offset, color='orange', linestyle='-', linewidth=1.5, alpha=0.8)
    ax.axvline(x=0 + offset, color='white', linestyle='--', linewidth=0.8, alpha=0.5)
    
    # Add text labels - alternate 4 positions
    y_offsets_text = [0.03, 0.08, -0.03, -0.08]
    va_aligns = ['bottom', 'bottom', 'top', 'top']
    
    for i, (bin_label, p68, color, n) in enumerate(plotted_bins):
        ax.scatter([p68 + offset], [0.68], color=color, s=30, zorder=5)
        ax.axvline(x=p68 + offset, color=color, linestyle=':', linewidth=0.8, alpha=0.4)
        
        y_offset = y_offsets_text[i % 4]
        va = va_aligns[i % 4]
        
        ax.text(p68 + offset, 0.68 + y_offset, f'{bin_label}{label_suffix}', color=color, fontsize=8, fontweight='bold',
                ha='center', va=va)
    
    # Add 68th percentile summary text
    if plotted_bins:
        ax.text(0.98, 0.48, '68%tile Error (m)', transform=ax.transAxes,
                fontsize=8, color='lime', va='top', ha='right', family='monospace', fontweight='bold')
        
        y_pos = 0.40
        for bin_label, p68, color, n in plotted_bins:
            text = f'{bin_label:>5}{label_suffix}: {p68:.2f}m (n={n})'
            ax.text(0.98, y_pos, text, transform=ax.transAxes,
                    fontsize=7, color=color, va='top', ha='right', family='monospace')
            y_pos -= 0.04
    
    # Log scale
    ax.set_xscale('log')
    ax.set_xlim((x_min_orig + offset) * 0.9, (x_max_orig + offset) * 1.1)
    ax.set_ylim(0, 1)
    
    def format_func(value, tick_number):
        original = value - offset
        if abs(original) < 0.01:
            return '0'
        elif abs(original) >= 10:
            return f'{original:.0f}'
        elif abs(original) >= 1:
            return f'{original:.1f}'
        else:
            return f'{original:.2f}'
    
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(format_func))
    
    tick_values_orig = [0, 0.2, 0.5, 1, 2, 5, 10, 20, 50, 100]
    tick_values = [v + offset for v in tick_values_orig 
                   if (x_min_orig + offset) * 0.9 <= v + offset <= (x_max_orig + offset) * 1.1]
    ax.set_xticks(tick_values)
    
    ax.set_xlabel('Error (m)', color='white', fontsize=10)
    ax.set_ylabel('CDF', color='white', fontsize=10)
    ax.set_title(title, color='white', fontsize=12, fontweight='bold')
    ax.tick_params(colors='white', labelsize=9)
    ax.grid(True, alpha=0.2, color='white')
    for spine in ax.spines.values():
        spine.set_color('white')


def plot_parallel_analysis_single(df, y_col, var_set_key, threshold=0.1832, 
                                   figsize=(18, 5), save=True, show=False, offset=1):
    """
    Create 1x3 plot for Parallel zone analysis with a specific variable set
    """
    var_config = VARIABLE_SETS[var_set_key]
    near_col = var_config['near_col']
    far_col = var_config['far_col']
    bin_type = var_config['bin_type']
    var_title = var_config['title']
    unit = var_config['unit']
    
    # Check columns exist
    required_cols = [y_col, 'flow_angle', 'sample_geometry_category', near_col, far_col]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        print(f"Warning: Missing columns for {var_set_key}: {missing_cols}")
        return None
    
    df_plot = df.dropna(subset=required_cols).copy()
    df_plot['flow_zone'] = df_plot['flow_angle'].apply(get_flow_angle_zone)
    df_parallel = df_plot[df_plot['flow_zone'] == 'Parallel']
    
    if len(df_parallel) == 0:
        print(f"Warning: No data in Parallel zone for {var_set_key}")
        return None
    
    print(f"[{var_set_key}] Parallel zone: {len(df_parallel):,} samples")
    
    fig, axes = plt.subplots(1, 3, figsize=figsize, facecolor='black')
    
    # Plot 1: Original by geometry category
    plot_cdf_original(axes[0], df_parallel, y_col, threshold, offset)
    
    # Plot 2: By Near bins
    near_title = f'By Near {var_title}'
    plot_cdf_by_bin(axes[1], df_parallel, y_col, near_col, bin_type, near_title, threshold, offset)
    
    # Plot 3: By Far bins
    far_title = f'By Far {var_title}'
    plot_cdf_by_bin(axes[2], df_parallel, y_col, far_col, bin_type, far_title, threshold, offset)
    
    fig.suptitle(f'Parallel Zone Analysis: {y_col} by {var_title}', 
                 fontsize=14, fontweight='bold', color='white', y=1.02)
    
    plt.tight_layout()
    
    if save:
        filename = f"parallel_{var_set_key}_analysis_{y_col}_log.png"
        filepath = OUTPUT_DIR / filename
        fig.savefig(filepath, dpi=150, facecolor='black', bbox_inches='tight')
        print(f"  Saved: {filepath}")
    
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    return fig


def plot_parallel_analysis_all(df, y_col='error_reach', threshold=0.1832,
                                figsize=(18, 5), save=True, show=False, offset=1,
                                var_sets=None):
    """
    Create 1x3 plots for all variable sets
    
    Parameters
    ----------
    df : DataFrame
        Input data with all required columns
    y_col : str
        Error column to analyze
    threshold : float
        Spike threshold
    var_sets : list or None
        List of variable set keys to plot. If None, plot all available.
    """
    
    if var_sets is None:
        var_sets = list(VARIABLE_SETS.keys())
    
    print(f"Parallel Zone Analysis - All Variable Sets")
    print(f"Error column: {y_col}, Threshold: {threshold}")
    
    results = {}
    
    for var_set_key in var_sets:
        print(f"\nProcessing: {var_set_key}...")
        fig = plot_parallel_analysis_single(
            df, y_col, var_set_key, 
            threshold=threshold, figsize=figsize, 
            save=save, show=show, offset=offset
        )
        results[var_set_key] = fig
    
    print("Complete!")
    
    return results


# Legacy function (backward compatibility)
def plot_parallel_isa_analysis(df, y_col, threshold=0.1832, figsize=(18, 5), 
                                save=True, show=False, offset=1):
    """
    Legacy function - calls new function with 'isa' variable set
    """
    return plot_parallel_analysis_single(
        df, y_col, 'isa', 
        threshold=threshold, figsize=figsize, 
        save=save, show=show, offset=offset
    )

In [ ]:
plot_parallel_analysis_all(final_df, y_col='abs_error_node', offset = 0.1, show = True)

#### boxplots and statistical analysis (not working)

In [ ]:
"""
Diagnostic Plot for analysis_final_df
=====================================
8 variables + pixel_count
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use('dark_background')

OUTPUT_DIR = Path("unit_level_analysis")
OUTPUT_DIR.mkdir(exist_ok=True)


def get_flow_angle_zone(flow_angle):
    fa = flow_angle % 360
    if (fa >= 350 or fa <= 10) or (170 <= fa <= 190):
        return 'Parallel'
    elif (80 <= fa <= 100) or (260 <= fa <= 280):
        return 'Perpendicular'
    else:
        return 'Other'


def get_isa_bin(isa_value):
    if isa_value <= 5:
        return '0-5'
    elif isa_value <= 10:
        return '5-10'
    elif isa_value <= 20:
        return '10-20'
    elif isa_value <= 30:
        return '20-30'
    elif isa_value <= 40:
        return '30-40'
    elif isa_value <= 50:
        return '40-50'
    elif isa_value <= 60:
        return '50-60'
    elif isa_value <= 70:
        return '60-70'
    elif isa_value <= 80:
        return '70-80'
    elif isa_value <= 90:
        return '80-90'
    else:
        return '90-100'


def plot_diagnostic(df, 
                    bin_col='far_isa',
                    error_col='abs_error_node',
                    highlight_bins=None,
                    save=True, show=True):
    """
    Diagnostic boxplots: 8 variables + pixel_count
    """
    
    df = df.copy()
    
    # 1. Node별 pixel count 계산
    pixel_counts = df.groupby(['node_id', 'time_str']).size().reset_index(name='pixel_count')
    df = df.merge(pixel_counts, on=['node_id', 'time_str'], how='left')
    
    # 2. Flow zone & ISA bin
    df['flow_zone'] = df['flow_angle'].apply(get_flow_angle_zone)
    df['isa_bin'] = df[bin_col].apply(get_isa_bin)
    
    # Filter to Parallel zone
    df_parallel = df[df['flow_zone'] == 'Parallel'].copy()
    print(f"[Data] Parallel zone: {len(df_parallel):,} / {len(df):,}")
    
    # Variables
    variables = ['near_isa', 'far_isa', 'near_building', 'far_building',
                 'height', 'sig0', 'phase_noise_std', 'dheight_dphase', 
                 'pixel_count']
    
    if highlight_bins is None:
        highlight_bins = ['30-40', '50-60']
    
    # Bin order
    bin_order = ['0-5', '5-10', '10-20', '20-30', '30-40', '40-50', 
                 '50-60', '60-70', '70-80', '80-90', '90-100']
    existing_bins = [b for b in bin_order if b in df_parallel['isa_bin'].values]
    
    # Figure
    n_vars = len(variables)
    n_cols = 3
    n_rows = (n_vars + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 6, n_rows * 4), 
                              facecolor='black')
    axes = axes.flatten()
    
    # Colors
    color_var = '#00bfff'
    color_error = '#ff6b6b'
    
    for idx, var in enumerate(variables):
        ax1 = axes[idx]
        ax1.set_facecolor('#1a1a1a')
        
        if var not in df_parallel.columns:
            ax1.text(0.5, 0.5, f'{var}\nNot Available', 
                    transform=ax1.transAxes, ha='center', va='center', color='gray')
            continue
        
        # Prepare data
        var_data = []
        error_data = []
        positions = []
        bin_labels = []
        
        for i, bin_label in enumerate(existing_bins):
            bin_df = df_parallel[df_parallel['isa_bin'] == bin_label]
            if len(bin_df) > 0:
                var_vals = bin_df[var].dropna().values
                error_vals = bin_df[error_col].dropna().values
                if len(var_vals) > 0 and len(error_vals) > 0:
                    var_data.append(var_vals)
                    error_data.append(error_vals)
                    positions.append(len(positions))
                    bin_labels.append(bin_label)
        
        if len(positions) == 0:
            ax1.text(0.5, 0.5, 'No Data', transform=ax1.transAxes,
                    ha='center', va='center', color='gray')
            continue
        
        # Left y-axis: Variable
        bp1 = ax1.boxplot(var_data, positions=[p - 0.2 for p in positions],
                          widths=0.35, patch_artist=True,
                          boxprops=dict(facecolor=color_var, alpha=0.7),
                          whis=(0, 99),
                          medianprops=dict(color='white', linewidth=2),
                          whiskerprops=dict(color=color_var),
                          capprops=dict(color=color_var),
                          flierprops=dict(marker='o', markerfacecolor=color_var,
                                         markersize=2, alpha=0.3))
        
        ax1.set_ylabel(var, color=color_var, fontsize=10)
        ax1.tick_params(axis='y', labelcolor=color_var, labelsize=8)
        
        # Right y-axis: Error
        ax2 = ax1.twinx()
        bp2 = ax2.boxplot(error_data, positions=[p + 0.2 for p in positions],
                          widths=0.35, patch_artist=True,
                          boxprops=dict(facecolor=color_error, alpha=0.7),
                          whis=(0, 99),
                          medianprops=dict(color='white', linewidth=2),
                          whiskerprops=dict(color=color_error),
                          capprops=dict(color=color_error),
                          flierprops=dict(marker='o', markerfacecolor=color_error,
                                         markersize=2, alpha=0.3))
        
        ax2.set_ylabel(error_col, color=color_error, fontsize=10)
        ax2.tick_params(axis='y', labelcolor=color_error, labelsize=8)
        
        # Highlight bins
        for i, bin_label in enumerate(bin_labels):
            if bin_label in highlight_bins:
                ax1.axvspan(i - 0.5, i + 0.5, alpha=0.15, color='yellow')
        
        # X-axis
        ax1.set_xticks(positions)
        ax1.set_xticklabels(bin_labels, rotation=45, ha='right', color='white', fontsize=8)
        ax1.set_xlabel(f'{bin_col} Bin (%)', color='white', fontsize=9)
        
        ax1.set_title(var, color='white', fontsize=11, fontweight='bold')
        ax1.grid(True, alpha=0.2, color='white')
        ax1.set_xlim(-0.5, len(bin_labels) - 0.5)
        
        for spine in ax1.spines.values():
            spine.set_color('white')
        for spine in ax2.spines.values():
            spine.set_color('white')
    
    # Hide unused
    for idx in range(len(variables), len(axes)):
        axes[idx].set_visible(False)
    
    # Legend
    fig.legend([plt.Rectangle((0,0),1,1, facecolor=color_var, alpha=0.7),
                plt.Rectangle((0,0),1,1, facecolor=color_error, alpha=0.7)],
               ['Variable', error_col],
               loc='upper right', facecolor='black', edgecolor='white',
               labelcolor='white', fontsize=10)
    
    fig.suptitle(f'Diagnostic: Variables vs {error_col} by {bin_col} (Parallel Zone)',
                 fontsize=14, fontweight='bold', color='white', y=1.01)
    
    plt.tight_layout()
    
    if save:
        filepath = OUTPUT_DIR / f"diagnostic_analysis.png"
        fig.savefig(filepath, dpi=150, facecolor='black', bbox_inches='tight')
        print(f"\nSaved: {filepath}")
    
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    return fig


# Run
fig = plot_diagnostic(
    analysis_final_df,
    bin_col='far_isa',
    error_col='abs_error_node',
    highlight_bins=['30-40', '50-60'],
    save=True, show=True
)

In [ ]:
"""
Threshold Analysis: Building Height vs Error (v2)
==================================================
Data-driven parameter selection based on academic literature

References:
-----------
[LOWESS]
- Cleveland, W.S. (1979). "Robust Locally Weighted Regression and Smoothing 
  Scatterplots." JASA, 74(368), 829-836.
- Hurvich, C.M., Simonoff, J.S., & Tsai, C.L. (1998). "Smoothing parameter 
  selection in nonparametric regression using an improved Akaike information 
  criterion." JRSS-B, 60(2), 271-293.

[Kernel Smoothing]
- Silverman, B.W. (1986). "Density Estimation for Statistics and Data Analysis."
- Sheather, S.J. & Jones, M.C. (1991). "A reliable data-based bandwidth selection 
  method for kernel density estimation." JRSS-B, 53(3), 683-690.

[Savitzky-Golay]
- Savitzky, A. & Golay, M.J.E. (1964). "Smoothing and Differentiation of Data by 
  Simplified Least Squares Procedures." Analytical Chemistry, 36(8), 1627-1639.
- Schafer, R.W. (2011). "What Is a Savitzky-Golay Filter?" IEEE Signal Processing 
  Magazine, 28(4), 111-117.

[Quantile Regression]
- Koenker, R. & Bassett, G. (1978). "Regression Quantiles." Econometrica, 46(1), 33-50.
- Koenker, R. & Machado, J.A.F. (1999). "Goodness of Fit and Related Inference 
  Processes for Quantile Regression." JASA, 94(448), 1296-1310.

"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.signal import savgol_filter
from scipy.ndimage import gaussian_filter1d
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')

OUTPUT_DIR = Path("unit_level_analysis")
OUTPUT_DIR.mkdir(exist_ok=True)

ERROR_THRESHOLD = 0.1832


# Utility functions

def get_flow_angle_zone(flow_angle):
    """Classify flow angle into zones"""
    fa = flow_angle % 360
    if (fa >= 350 or fa <= 10) or (170 <= fa <= 190):
        return 'Parallel'
    elif (80 <= fa <= 100) or (260 <= fa <= 280):
        return 'Perpendicular'
    else:
        return 'Other'


# METHOD 1: KERNEL SMOOTHING - Silverman's Rule of Thumb
# Reference: Silverman (1986), "Density Estimation for Statistics and Data Analysis"
# Formula: h = 0.9 * min(σ, IQR/1.34) * n^(-1/5)
# The 0.9 factor (reduced from 1.06) guards against oversmoothing

def silverman_bandwidth(x, scale_factor=0.9):
    """
    Silverman's rule of thumb for bandwidth selection.
    
    h = scale_factor * min(σ, IQR/1.34) * n^(-1/5)
    
    Parameters:
    -----------
    x : array-like
        Input data (used for computing scale)
    scale_factor : float
        Default 0.9 (conservative). Original theory suggests 1.06 for Gaussian.
        
    Returns:
    --------
    h : float
        Optimal bandwidth
        
    Reference:
    ----------
    Silverman, B.W. (1986). Density Estimation for Statistics and Data Analysis.
    Chapman and Hall, London. Section 3.4.2.
    """
    n = len(x)
    std = np.std(x, ddof=1)
    iqr = stats.iqr(x)
    
    # Use min of std and IQR/1.34 to be safe to outliers
    scale = min(std, iqr / 1.34) if iqr > 0 else std
    
    # Silverman's rule
    h = scale_factor * scale * (n ** (-1/5))
    
    return h


def kernel_smooth_with_optimal_bandwidth(x, y, n_bandwidths=3):
    """
    Apply Gaussian kernel smoothing with Silverman-based bandwidth selection.
    
    Returns multiple smoothing levels based on Silverman's rule:
    - Light: h * 0.5 (detail)
    - Optimal: h (Silverman's rule)
    - Heavy: h * 2 (smooth)
    
    The sigma for scipy's gaussian_filter1d is converted from bandwidth h
    using the relationship: sigma ≈ h * n / range(x)
    """
    # Sort by x
    sort_idx = np.argsort(x)
    x_sorted = x[sort_idx]
    y_sorted = y[sort_idx]
    
    n = len(x)
    x_range = x_sorted[-1] - x_sorted[0]
    
    # Compute optimal bandwidth using Silverman's rule
    h_opt = silverman_bandwidth(x_sorted)
    
    # Convert bandwidth to sigma (index-space)
    # sigma = h * n / range gives approximate number of points in bandwidth
    sigma_opt = max(int(h_opt * n / x_range), 5)
    
    # Create multiple smoothing levels
    sigmas = [
        max(sigma_opt // 2, 5),   # Light (50% of optimal)
        sigma_opt,                 # Optimal (Silverman)
        sigma_opt * 2              # Heavy (200% of optimal)
    ]
    
    results = []
    for sigma in sigmas:
        y_smooth = gaussian_filter1d(y_sorted, sigma=sigma)
        results.append({
            'sigma': sigma,
            'x': x_sorted,
            'y_smooth': y_smooth
        })
    
    return results, h_opt, sigma_opt


# METHOD 2: LOWESS - AICc-based Span Selection
# Reference: Hurvich, Simonoff & Tsai (1998), JRSS-B
# AICc = log(RSS/n) + 1 + 2(tr(S) + 1) / (n - tr(S) - 2)
# where S is the smoother matrix and tr(S) ≈ n * span for LOWESS

def compute_lowess_aicc(x, y, frac):
    """
    Compute corrected AIC (AICc) for LOWESS with given span.
    
    AICc = log(RSS/n) + 1 + 2(df + 1) / (n - df - 2)
    
    For LOWESS, effective degrees of freedom ≈ 1/frac
    (approximation based on equivalent kernel theory)
    
    Reference:
    ----------
    Hurvich, C.M., Simonoff, J.S., & Tsai, C.L. (1998). 
    "Smoothing parameter selection in nonparametric regression using 
    an improved Akaike information criterion." JRSS-B, 60(2), 271-293.
    """
    from statsmodels.nonparametric.smoothers_lowess import lowess
    
    n = len(x)
    
    # Fit LOWESS
    smoothed = lowess(y, x, frac=frac, return_sorted=False)
    
    # Residuals
    residuals = y - smoothed
    rss = np.sum(residuals ** 2)
    
    # Effective degrees of freedom (approximation)
    # For local linear regression: df ≈ tr(S) ≈ n * frac / degree_freedom_factor
    # A common approximation for LOWESS: df ≈ 1 + 1/frac
    # More refined: df ≈ n * frac * c, where c depends on kernel and polynomial degree
    # We use the approximation from Cleveland & Devlin (1988)
    df = 1 + 1/frac  # Simplified approximation
    
    # AICc formula
    if n - df - 2 <= 0:
        return np.inf
    
    aicc = np.log(rss / n) + 1 + 2 * (df + 1) / (n - df - 2)
    
    return aicc


def compute_lowess_gcv(x, y, frac):
    """
    Compute Generalized Cross-Validation (GCV) for LOWESS.
    
    GCV = (1/n) * RSS / (1 - df/n)^2
    
    Reference:
    ----------
    Craven, P. & Wahba, G. (1979). "Smoothing noisy data with spline functions."
    Numerische Mathematik, 31, 377-403.
    """
    from statsmodels.nonparametric.smoothers_lowess import lowess
    
    n = len(x)
    
    # Fit LOWESS
    smoothed = lowess(y, x, frac=frac, return_sorted=False)
    
    # Residuals
    residuals = y - smoothed
    rss = np.sum(residuals ** 2)
    
    # Effective degrees of freedom
    df = 1 + 1/frac
    
    # GCV formula
    gcv = (rss / n) / ((1 - df/n) ** 2)
    
    return gcv


def select_lowess_span(x, y, criterion='aicc', span_range=None, n_spans=20):
    """
    Select optimal LOWESS span using AICc or GCV.
    
    Parameters:
    -----------
    x, y : array-like
        Input data
    criterion : str
        'aicc' (default) or 'gcv'
    span_range : tuple
        (min_span, max_span). Default: (0.1, 0.9)
    n_spans : int
        Number of spans to evaluate
        
    Returns:
    --------
    optimal_span : float
    all_spans : array
    all_scores : array
    
    Reference:
    ----------
    Cleveland, W.S. (1979). Recommended span range: 0.2 to 0.8
    Default starting point: 0.5
    """
    if span_range is None:
        # Cleveland (1979) recommends 0.2-0.8
        span_range = (0.1, 0.9)
    
    spans = np.linspace(span_range[0], span_range[1], n_spans)
    scores = []
    
    compute_func = compute_lowess_aicc if criterion == 'aicc' else compute_lowess_gcv
    
    for frac in spans:
        try:
            score = compute_func(x, y, frac)
            scores.append(score)
        except:
            scores.append(np.inf)
    
    scores = np.array(scores)
    optimal_idx = np.argmin(scores)
    optimal_span = spans[optimal_idx]
    
    return optimal_span, spans, scores


# METHOD 3: SAVITZKY-GOLAY - Cross-Validation Parameter Selection
# Reference: Vivó-Truyols et al. (2006), Analytical Chemistry
# Optimal window minimizes prediction error via leave-one-out CV

def compute_savgol_cv_score(x, y, window_length, polyorder):
    """
    Compute leave-one-out cross-validation score for Savitzky-Golay filter.
    
    Uses approximate LOOCV via the formula:
    CV = (1/n) * Σ (y_i - ŷ_i)^2 / (1 - h_ii)^2
    
    For S-G filter, we use standard MSE as approximation since
    exact hat matrix computation is complex.
    
    Reference:
    ----------
    Vivó-Truyols, G. & Schoenmakers, P.J. (2006). "Automatic selection 
    of optimal Savitzky-Golay smoothing." Analytical Chemistry, 78(13), 4598-4608.
    """
    if window_length > len(y):
        return np.inf
    if polyorder >= window_length:
        return np.inf
    if window_length % 2 == 0:
        window_length += 1
    
    try:
        y_smooth = savgol_filter(y, window_length=window_length, polyorder=polyorder)
        
        # MSE (proxy for CV score)
        mse = np.mean((y - y_smooth) ** 2)
        
        # Penalty for overfitting (similar to GCV denominator)
        # df ≈ (polyorder + 1) * n / window_length
        n = len(y)
        df = (polyorder + 1) * n / window_length
        
        # GCV-like adjustment
        gcv_score = mse / ((1 - df/n) ** 2) if df < n else np.inf
        
        return gcv_score
    except:
        return np.inf


def select_savgol_params(x, y, window_range=None, polyorders=[2, 3, 4]):
    """
    Select optimal Savitzky-Golay parameters using cross-validation.
    
    Parameters:
    -----------
    x, y : array-like
        Input data (must be sorted by x)
    window_range : tuple
        (min_window, max_window). Default based on data size.
    polyorders : list
        Polynomial orders to try. Default [2, 3, 4].
        Schafer (2011) recommends 2-4 for most applications.
        
    Returns:
    --------
    optimal_window : int
    optimal_polyorder : int
    cv_results : dict
    
    Reference:
    ----------
    Schafer, R.W. (2011). "What Is a Savitzky-Golay Filter?"
    IEEE Signal Processing Magazine, 28(4), 111-117.
    - Recommends window/polyorder ratio ≈ 3.5
    - Polynomial order 2-3 for typical data
    """
    n = len(y)
    
    if window_range is None:
        # Rule of thumb: window should be 1-10% of data length
        min_window = max(5, n // 100)
        max_window = min(n // 10, 201)
        window_range = (min_window, max_window)
    
    # Generate odd window sizes only
    windows = list(range(
        window_range[0] if window_range[0] % 2 == 1 else window_range[0] + 1,
        window_range[1] + 1,
        2  # Step by 2 to get only odd numbers
    ))
    
    best_score = np.inf
    best_window = windows[len(windows)//2]
    best_polyorder = 3
    cv_results = {'windows': [], 'polyorders': [], 'scores': []}
    
    for window in windows:
        for polyorder in polyorders:
            if polyorder >= window:
                continue
            
            score = compute_savgol_cv_score(x, y, window, polyorder)
            cv_results['windows'].append(window)
            cv_results['polyorders'].append(polyorder)
            cv_results['scores'].append(score)
            
            if score < best_score:
                best_score = score
                best_window = window
                best_polyorder = polyorder
    
    return best_window, best_polyorder, cv_results


# METHOD 4: QUANTILE REGRESSION - BIC-based Model Selection
# Reference: Koenker & Machado (1999), JASA
# BIC_τ = 2 * Σρ_τ(residuals) + k * ln(n)

def compute_quantreg_bic(y, X, q=0.5):
    """
    Compute BIC for quantile regression model.
    
    BIC = 2 * Σρ_τ(y - Xβ) + k * ln(n)
    
    where ρ_τ(u) = u(τ - I(u < 0)) is the check loss function.
    
    Reference:
    ----------
    Koenker, R. & Machado, J.A.F. (1999). "Goodness of Fit and Related 
    Inference Processes for Quantile Regression." JASA, 94(448), 1296-1310.
    """
    import statsmodels.api as sm
    from statsmodels.regression.quantile_regression import QuantReg
    
    try:
        model = QuantReg(y, X)
        result = model.fit(q=q, max_iter=1000)
        
        # Check loss
        residuals = y - result.fittedvalues
        check_loss = np.sum(residuals * (q - (residuals < 0)))
        
        # BIC
        n = len(y)
        k = X.shape[1]  # Number of parameters
        bic = 2 * check_loss + k * np.log(n)
        
        return bic, result
    except:
        return np.inf, None


def select_quantreg_degree(x, y, q=0.5, max_degree=5):
    """
    Select optimal polynomial degree for quantile regression using BIC.
    
    Parameters:
    -----------
    x, y : array-like
        Input data
    q : float
        Quantile (default 0.5 for median)
    max_degree : int
        Maximum polynomial degree to consider
        
    Returns:
    --------
    optimal_degree : int
    bic_scores : dict
    
    Reference:
    ----------
    Standard practice in quantile regression is to use BIC for model selection
    as it provides consistent model selection (Schwarz, 1978).
    For polynomial regression, cubic (degree 3) is often sufficient.
    """
    import statsmodels.api as sm
    
    bic_scores = {}
    best_bic = np.inf
    best_degree = 1
    best_result = None
    
    for degree in range(1, max_degree + 1):
        # Create polynomial features
        X = np.column_stack([x ** i for i in range(1, degree + 1)])
        X = sm.add_constant(X)
        
        bic, result = compute_quantreg_bic(y, X, q)
        bic_scores[degree] = bic
        
        if bic < best_bic:
            best_bic = bic
            best_degree = degree
            best_result = result
    
    return best_degree, bic_scores, best_result


# Main analysis function

def plot_threshold_analysis_v2(df, x_col='far_building', y_col='abs_error_node', 
                                xlabel='Far Building Height (m)', 
                                save=True, show=True, figsize=(14, 12)):
    """
    Threshold analysis with data-driven parameter selection.
    
    All parameters are selected using academically rigorous methods:
    - Method 1 (Kernel): Silverman's rule of thumb
    - Method 2 (LOWESS): AICc minimization
    - Method 3 (Savitzky-Golay): Cross-validation
    
    Edges are trimmed so every point is computed the same way.
    """
    from statsmodels.nonparametric.smoothers_lowess import lowess
    
    print("THRESHOLD ANALYSIS v2 - Data-Driven Parameter Selection")
    
    # Filter to Parallel zone
    df_parallel = df[df['flow_angle'].apply(get_flow_angle_zone) == 'Parallel'].copy()
    print(f"\n[Data] Filtered to Parallel zone: {len(df_parallel):,} / {len(df):,} observations")
    
    # Clean data
    data = df_parallel[[x_col, y_col]].dropna()
    data = data[data[y_col] > 0]  # For log scale
    data = data[data[x_col] > 0]  # Exclude height = 0
    x = data[x_col].values
    y = data[y_col].values
    
    print(f"[Data] Points (excluding height=0): {len(x):,}")
    print(f"[Data] X range: {x.min():.2f} ~ {x.max():.2f}")
    print(f"[Data] Y range: {y.min():.4f} ~ {y.max():.2f}")
    
    # Sort data
    sort_idx = np.argsort(x)
    x_sorted = x[sort_idx]
    y_sorted = y[sort_idx]
    
    # Sample for scatter plots
    if len(x) > 5000:
        scatter_idx = np.random.choice(len(x), 5000, replace=False)
        x_plot, y_plot = x[scatter_idx], y[scatter_idx]
    else:
        x_plot, y_plot = x, y
    
    # Create figure - 3 rows, 1 column, shared x-axis
    fig, axes = plt.subplots(3, 1, figsize=figsize, facecolor='black', sharex=True)
    for ax in axes.flatten():
        ax.set_facecolor('black')
    
    # Store results
    results = {}
    
    # METHOD 1: KERNEL SMOOTHING (Silverman's Rule)
    print("[Method 1] Kernel Smoothing - Silverman's Rule")
    
    ax1 = axes[0]
    ax1.scatter(x_plot, y_plot, c='#3399ff', alpha=0.2, s=5, edgecolors='none')
    
    # Compute optimal bandwidth
    h_opt = silverman_bandwidth(x_sorted)
    n = len(x_sorted)
    x_range = x_sorted[-1] - x_sorted[0]
    sigma_opt = max(int(h_opt * n / x_range), 5)
    
    print(f"  Silverman bandwidth h = 0.9 * min(σ, IQR/1.34) * n^(-1/5)")
    print(f"  Computed h_opt = {h_opt:.4f}")
    print(f"  Converted σ_opt = {sigma_opt} (index units)")
    
    # Edge trimming: remove 3*sigma from each end for consistent computation
    # so every point has a full neighborhood on both sides
    trim_kernel = sigma_opt * 3
    
    # Multiple smoothing levels
    sigma_ratios = [0.5, 1.0, 2.0]
    sigmas = [max(int(sigma_opt * r), 3) for r in sigma_ratios]
    colors_ks = ['#ff6b6b', 'yellow', '#00ff88']
    labels_ks = ['σ×0.5 (detail)', 'σ×1.0 (optimal)', 'σ×2.0 (smooth)']
    
    peak_x_kernel = None
    
    for sigma, ratio, color, label in zip(sigmas, sigma_ratios, colors_ks, labels_ks):
        y_smooth = gaussian_filter1d(y_sorted, sigma=sigma)
        
        # Apply edge trimming for plotting and peak finding
        trim = sigma * 3
        if trim < len(x_sorted) // 2:
            x_trimmed = x_sorted[trim:-trim]
            y_trimmed = y_smooth[trim:-trim]
        else:
            x_trimmed = x_sorted
            y_trimmed = y_smooth
        
        ax1.plot(x_trimmed, y_trimmed, color=color, linewidth=2, 
                 label=f'{label} (σ={sigma})')
        
        # Find peak for optimal sigma (in trimmed region only)
        if ratio == 1.0:
            peak_idx = np.argmax(y_trimmed)
            peak_x_kernel = x_trimmed[peak_idx]
            peak_y_kernel = y_trimmed[peak_idx]
            
            ax1.axvline(x=peak_x_kernel, color='yellow', linestyle='--', linewidth=1.5)
            ax1.scatter([peak_x_kernel], [peak_y_kernel], color='yellow', s=100, 
                       zorder=10, marker='*')
            ax1.text(peak_x_kernel + 0.3, peak_y_kernel * 1.3, 
                    f'Peak: {peak_x_kernel:.1f}m', color='yellow', fontsize=10, fontweight='bold')
    
    results['kernel'] = {'peak': peak_x_kernel, 'sigma': sigma_opt, 'bandwidth': h_opt,
                         'trim': trim_kernel}
    print(f"  Edge trim: {trim_kernel} points from each end")
    print(f"  Valid x range: {x_sorted[trim_kernel]:.2f} ~ {x_sorted[-trim_kernel]:.2f}")
    print(f"  Peak found at: {peak_x_kernel:.1f}m")
    
    ax1.axhline(y=ERROR_THRESHOLD, color='orange', linestyle='-', linewidth=1, alpha=0.7)
    ax1.set_ylabel('Absolute Error (m)', color='white', fontsize=11)
    
    # Show valid range in title
    if trim_kernel < len(x_sorted) // 2:
        valid_min = x_sorted[trim_kernel]
        valid_max = x_sorted[-trim_kernel]
        trim_info = f'Valid range: {valid_min:.1f}~{valid_max:.1f}m (3σ trimmed)'
    else:
        trim_info = 'No trimming'
    
    ax1.set_title(f'Method 1: Kernel Smoothing (Silverman)\n'
                  f'h={h_opt:.3f}, σ={sigma_opt} | {trim_info}', 
                  color='white', fontsize=10, fontweight='bold')
    ax1.legend(loc='upper right', facecolor='black', edgecolor='white', 
               labelcolor='white', fontsize=9)
    ax1.tick_params(colors='white')
    ax1.grid(True, alpha=0.2, color='white')
    ax1.set_yscale('log')
    ax1.set_ylim(bottom=0.01)
    
    # Set xlim to valid (trimmed) range
    if trim_kernel < len(x_sorted) // 2:
        ax1.set_xlim(x_sorted[trim_kernel] - 0.2, x_sorted[-trim_kernel] + 0.2)
    
    for spine in ax1.spines.values():
        spine.set_color('white')
    
    # METHOD 2: LOWESS (AICc-based Span Selection)
    print("[Method 2] LOWESS - AICc Span Selection")
    
    ax2 = axes[1]
    
    # Sample for LOWESS (computational efficiency)
    if len(x) > 5000:
        lowess_idx = np.random.choice(len(x), 5000, replace=False)
        x_lowess, y_lowess = x[lowess_idx], y[lowess_idx]
    else:
        x_lowess, y_lowess = x, y
    
    # Select optimal span using AICc
    print("  Searching for optimal span using AICc...")
    optimal_span, spans, aicc_scores = select_lowess_span(
        x_lowess, y_lowess, criterion='aicc', span_range=(0.05, 0.5), n_spans=30
    )
    
    # Also compute GCV for comparison
    _, _, gcv_scores = select_lowess_span(
        x_lowess, y_lowess, criterion='gcv', span_range=(0.05, 0.5), n_spans=30
    )
    optimal_span_gcv = spans[np.argmin(gcv_scores)]
    
    print(f"  AICc optimal span: {optimal_span:.3f}")
    print(f"  GCV optimal span:  {optimal_span_gcv:.3f}")
    print(f"  Cleveland (1979) recommends: 0.2-0.8, starting at 0.5")
    
    # Sort for LOWESS
    lowess_sort_idx = np.argsort(x_lowess)
    x_lowess_sorted = x_lowess[lowess_sort_idx]
    y_lowess_sorted = y_lowess[lowess_sort_idx]
    
    # Edge trimming for LOWESS: remove span/2 fraction from each end
    # so every point has neighbors on both sides
    trim_lowess_frac = optimal_span / 2
    trim_lowess_n = int(len(x_lowess_sorted) * trim_lowess_frac)
    
    # Plot with multiple spans including optimal
    spans_to_plot = [optimal_span * 0.5, optimal_span, optimal_span * 2]
    spans_to_plot = [max(0.05, min(0.9, s)) for s in spans_to_plot]
    colors_lowess = ['#ff6b6b', 'yellow', '#00ff88']
    
    peak_x_lowess = None
    
    for frac, color in zip(spans_to_plot, colors_lowess):
        smoothed = lowess(y_lowess_sorted, x_lowess_sorted, frac=frac, return_sorted=True)
        
        # Edge trimming
        trim_n = int(len(smoothed) * frac / 2)
        if trim_n > 0 and trim_n < len(smoothed) // 2:
            smoothed_trimmed = smoothed[trim_n:-trim_n]
        else:
            smoothed_trimmed = smoothed
        
        if frac == optimal_span:
            label = f'frac={frac:.3f} (AICc optimal)'
            lw = 3
        else:
            label = f'frac={frac:.3f}'
            lw = 1.5
            
        ax2.plot(smoothed_trimmed[:, 0], smoothed_trimmed[:, 1], linewidth=lw, color=color, label=label)
        
        # Find peak for optimal span (in trimmed region)
        if frac == optimal_span:
            peak_idx = np.argmax(smoothed_trimmed[:, 1])
            peak_x_lowess = smoothed_trimmed[peak_idx, 0]
            peak_y_lowess = smoothed_trimmed[peak_idx, 1]
            
            ax2.axvline(x=peak_x_lowess, color='yellow', linestyle='--', linewidth=1.5)
            ax2.scatter([peak_x_lowess], [peak_y_lowess], color='yellow', s=150, 
                       zorder=10, marker='*')
            ax2.text(peak_x_lowess + 0.3, peak_y_lowess * 1.5, 
                    f'Peak: {peak_x_lowess:.1f}m', color='yellow', fontsize=10, fontweight='bold')
    
    results['lowess'] = {'peak': peak_x_lowess, 'span_aicc': optimal_span, 
                         'span_gcv': optimal_span_gcv, 'aicc_scores': aicc_scores,
                         'trim_frac': trim_lowess_frac}
    print(f"  Edge trim: {trim_lowess_frac*100:.1f}% from each end ({trim_lowess_n} points)")
    print(f"  Peak found at: {peak_x_lowess:.1f}m")
    
    ax2.scatter(x_plot, y_plot, c='#3399ff', alpha=0.1, s=3, edgecolors='none')
    ax2.axhline(y=ERROR_THRESHOLD, color='orange', linestyle='-', linewidth=1, alpha=0.7)
    ax2.set_ylabel('Absolute Error (m)', color='white', fontsize=11)
    
    trim_info_lowess = f'Trimmed {trim_lowess_frac*100:.0f}% from each end'
    ax2.set_title(f'Method 2: LOWESS (AICc Selection)\n'
                  f'span={optimal_span:.3f} | {trim_info_lowess}', 
                  color='white', fontsize=10, fontweight='bold')
    ax2.legend(loc='upper right', facecolor='black', edgecolor='white', 
               labelcolor='white', fontsize=9)
    ax2.tick_params(colors='white')
    ax2.grid(True, alpha=0.2, color='white')
    ax2.set_yscale('log')
    ax2.set_ylim(bottom=0.01)
    for spine in ax2.spines.values():
        spine.set_color('white')
    
    # METHOD 3: SAVITZKY-GOLAY (Cross-Validation)
    print("[Method 3] Savitzky-Golay - Cross-Validation")
    
    ax3 = axes[2]
    
    # Select optimal parameters
    print("  Searching for optimal window and polynomial order...")
    opt_window, opt_polyorder, cv_results = select_savgol_params(
        x_sorted, y_sorted, 
        window_range=(11, min(201, len(x_sorted)//5)),
        polyorders=[2, 3, 4]
    )
    
    # window has to be odd
    if opt_window % 2 == 0:
        opt_window += 1
    
    print(f"  Optimal window: {opt_window}")
    print(f"  Optimal polyorder: {opt_polyorder}")
    print(f"  Schafer (2011) recommends: window/polyorder ≈ 3.5")
    print(f"  Actual ratio: {opt_window/opt_polyorder:.1f}")
    
    # Edge trimming for S-G: remove window//2 from each end
    # so every point gets a full window
    trim_sg = opt_window // 2
    
    # Plot with multiple window sizes around optimal
    windows_to_plot = [
        max(5, opt_window - 20) | 1,  # make it odd
        opt_window,
        min(len(x_sorted) - 1, opt_window + 20) | 1,
        min(len(x_sorted) - 1, opt_window + 40) | 1,
        min(len(x_sorted) - 1, opt_window + 60) | 1
    ]
    # drop duplicates and make them all odd
    windows_to_plot = sorted(list(set([w if w % 2 == 1 else w + 1 for w in windows_to_plot])))
    
    colors_sg = plt.cm.viridis(np.linspace(0.2, 0.8, len(windows_to_plot)))
    
    peak_x_sg = None
    
    for window, color in zip(windows_to_plot, colors_sg):
        if window >= len(x_sorted):
            continue
        polyorder = min(opt_polyorder, window - 1)
        y_smooth = savgol_filter(y_sorted, window_length=window, polyorder=polyorder)
        
        # Edge trimming: remove window//2 from each end
        trim = window // 2
        if trim > 0 and trim < len(x_sorted) // 2:
            x_sg_trimmed = x_sorted[trim:-trim]
            y_sg_trimmed = y_smooth[trim:-trim]
        else:
            x_sg_trimmed = x_sorted
            y_sg_trimmed = y_smooth
        
        if window == opt_window:
            lw = 3
            label = f'window={window} (CV optimal)'
            line_color = 'yellow'
        else:
            lw = 1.5
            label = f'window={window}'
            line_color = color
            
        ax3.plot(x_sg_trimmed, y_sg_trimmed, color=line_color, linewidth=lw, label=label)
        
        # Find peak for optimal window (in trimmed region)
        if window == opt_window:
            peak_idx = np.argmax(y_sg_trimmed)
            peak_x_sg = x_sg_trimmed[peak_idx]
            peak_y_sg = y_sg_trimmed[peak_idx]
            
            ax3.axvline(x=peak_x_sg, color='yellow', linestyle='--', linewidth=1.5)
            ax3.scatter([peak_x_sg], [peak_y_sg], color='yellow', s=150, 
                       zorder=10, marker='*')
            ax3.text(peak_x_sg + 0.3, peak_y_sg + 0.2, 
                    f'Peak: {peak_x_sg:.1f}m', color='yellow', fontsize=10, fontweight='bold')
    
    results['savgol'] = {'peak': peak_x_sg, 'window': opt_window, 'polyorder': opt_polyorder,
                         'trim': trim_sg}
    print(f"  Edge trim: {trim_sg} points from each end (window//2)")
    print(f"  Valid x range: {x_sorted[trim_sg]:.2f} ~ {x_sorted[-trim_sg]:.2f}")
    print(f"  Peak found at: {peak_x_sg:.1f}m")
    
    ax3.axhline(y=ERROR_THRESHOLD, color='orange', linestyle='-', linewidth=1, alpha=0.7)
    ax3.set_xlabel(xlabel, color='white', fontsize=11)
    ax3.set_ylabel('Absolute Error (m)', color='white', fontsize=11)
    
    if trim_sg < len(x_sorted) // 2:
        valid_min_sg = x_sorted[trim_sg]
        valid_max_sg = x_sorted[-trim_sg]
        trim_info_sg = f'Valid: {valid_min_sg:.1f}~{valid_max_sg:.1f}m'
    else:
        trim_info_sg = 'No trimming'
    
    ax3.set_title(f'Method 3: Savitzky-Golay (CV Selection)\n'
                  f'window={opt_window}, poly={opt_polyorder} | {trim_info_sg}', 
                  color='white', fontsize=10, fontweight='bold')
    ax3.legend(loc='upper right', facecolor='black', edgecolor='white', 
               labelcolor='white', fontsize=8)
    ax3.tick_params(colors='white')
    ax3.grid(True, alpha=0.2, color='white')
    for spine in ax3.spines.values():
        spine.set_color('white')
    
    # Finalize
    fig.suptitle(f'Threshold Analysis: {xlabel} vs Absolute Error (Parallel Zone)\n'
                 f'Data-Driven Parameter Selection with Edge Trimming', 
                 fontsize=14, fontweight='bold', color='white', y=0.98)
    
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    
    if save:
        col_name = x_col.replace('_', '-')
        filepath = OUTPUT_DIR / f"threshold_analysis_v2_{col_name}.png"
        fig.savefig(filepath, dpi=150, facecolor='black', bbox_inches='tight')
        print(f"\nSaved: {filepath}")
    
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    # Print summary
    print("SUMMARY: DATA-DRIVEN PARAMETER SELECTION WITH EDGE TRIMMING")
    print(f"\n{'Method':<25} {'Parameter':<35} {'Peak (m)':<10}")
    
    kernel_param = f"σ={results['kernel']['sigma']}, trim={results['kernel']['trim']}pts"
    lowess_param = f"span={results['lowess']['span_aicc']:.3f}, trim={results['lowess']['trim_frac']*100:.0f}%"
    savgol_param = f"win={results['savgol']['window']}, poly={results['savgol']['polyorder']}, trim={results['savgol']['trim']}pts"
    
    print(f"{'1. Kernel Smoothing':<25} {kernel_param:<35} {results['kernel']['peak']:.1f}")
    print(f"{'2. LOWESS':<25} {lowess_param:<35} {results['lowess']['peak']:.1f}")
    print(f"{'3. Savitzky-Golay':<25} {savgol_param:<35} {results['savgol']['peak']:.1f}")
    
    # Consensus estimate
    peaks = [results['kernel']['peak'], results['lowess']['peak'], 
             results['savgol']['peak']]
    peaks = [p for p in peaks if p is not None]
    
    if peaks:
        mean_peak = np.mean(peaks)
        std_peak = np.std(peaks)
        print(f"\n{'Consensus (mean ± std)':<25} {'':<35} {mean_peak:.1f} ± {std_peak:.1f}")
    
    print("REFERENCES & METHODOLOGY")
    print("""
PARAMETER SELECTION:
[1] Silverman, B.W. (1986). Density Estimation for Statistics and Data Analysis.
[2] Cleveland, W.S. (1979). Robust Locally Weighted Regression. JASA, 74(368).
[3] Hurvich, Simonoff & Tsai (1998). AICc for smoothing parameter selection. JRSS-B.
[4] Schafer, R.W. (2011). What Is a Savitzky-Golay Filter? IEEE SPM, 28(4).

EDGE TRIMMING (Boundary Effect Correction):
- Kernel Smoothing: Trim 3σ from each end (99.7% of Gaussian kernel support)
- LOWESS: Trim span/2 from each end (half neighborhood on each side)
- Savitzky-Golay: Trim window/2 from each end (half window coverage)

This ensures all reported values are computed with FULL neighborhood/window
coverage, making comparisons across methods valid and consistent.
""")
    
    return fig, results


# DIAGNOSTIC PLOT: Parameter Selection Curves

def plot_parameter_selection_diagnostics(df, x_col='far_building', y_col='abs_error_node',
                                         save=True, show=True):
    """
    Plot diagnostic curves showing how parameters were selected.
    """
    print("Generating parameter selection diagnostics...")
    
    # Filter and prepare data
    df_parallel = df[df['flow_angle'].apply(get_flow_angle_zone) == 'Parallel'].copy()
    data = df_parallel[[x_col, y_col]].dropna()
    data = data[(data[y_col] > 0) & (data[x_col] > 0)]
    x = data[x_col].values
    y = data[y_col].values
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10), facecolor='black')
    for ax in axes.flatten():
        ax.set_facecolor('black')
    
    # 1. LOWESS AICc curve
    ax1 = axes[0, 0]
    optimal_span, spans, aicc_scores = select_lowess_span(x, y, criterion='aicc', n_spans=30)
    _, _, gcv_scores = select_lowess_span(x, y, criterion='gcv', n_spans=30)
    
    ax1.plot(spans, aicc_scores, 'o-', color='yellow', label='AICc', linewidth=2)
    ax1.plot(spans, gcv_scores, 's--', color='#00ff88', label='GCV', linewidth=1.5, alpha=0.7)
    ax1.axvline(x=optimal_span, color='yellow', linestyle=':', linewidth=2)
    ax1.scatter([optimal_span], [aicc_scores[np.argmin(aicc_scores)]], 
                color='yellow', s=200, marker='*', zorder=10)
    ax1.set_xlabel('LOWESS Span (frac)', color='white')
    ax1.set_ylabel('Information Criterion', color='white')
    ax1.set_title(f'LOWESS: AICc/GCV Selection\nOptimal span = {optimal_span:.3f}', 
                  color='white', fontweight='bold')
    ax1.legend(facecolor='black', edgecolor='white', labelcolor='white')
    ax1.tick_params(colors='white')
    ax1.grid(True, alpha=0.2)
    for spine in ax1.spines.values():
        spine.set_color('white')
    
    # 2. Savitzky-Golay CV surface
    ax2 = axes[0, 1]
    opt_window, opt_polyorder, cv_results = select_savgol_params(x, y)
    
    # Create pivot for visualization
    cv_df = pd.DataFrame(cv_results)
    for po in [2, 3, 4]:
        subset = cv_df[cv_df['polyorders'] == po]
        ax2.plot(subset['windows'], subset['scores'], 'o-', 
                 label=f'polyorder={po}', linewidth=1.5)
    
    ax2.axvline(x=opt_window, color='yellow', linestyle=':', linewidth=2)
    ax2.set_xlabel('Window Size', color='white')
    ax2.set_ylabel('GCV Score', color='white')
    ax2.set_title(f'Savitzky-Golay: CV Selection\nOptimal: window={opt_window}, poly={opt_polyorder}', 
                  color='white', fontweight='bold')
    ax2.legend(facecolor='black', edgecolor='white', labelcolor='white')
    ax2.tick_params(colors='white')
    ax2.grid(True, alpha=0.2)
    for spine in ax2.spines.values():
        spine.set_color('white')
    
    # 3. Kernel bandwidth illustration (Silverman's Rule)
    ax3 = axes[1, 0]
    h_opt = silverman_bandwidth(x)
    
    # Show the formula
    std_x = np.std(x, ddof=1)
    iqr_x = stats.iqr(x)
    n = len(x)
    
    text = f"""Silverman's Rule of Thumb:
    
h = 0.9 × min(σ, IQR/1.34) × n^(-1/5)

For your data:
  n = {n:,}
  σ = {std_x:.3f}
  IQR = {iqr_x:.3f}
  IQR/1.34 = {iqr_x/1.34:.3f}
  min(σ, IQR/1.34) = {min(std_x, iqr_x/1.34):.3f}
  n^(-1/5) = {n**(-1/5):.4f}
  
  h_optimal = {h_opt:.4f}
"""
    ax3.text(0.1, 0.5, text, transform=ax3.transAxes, fontsize=11, 
             color='white', family='monospace', verticalalignment='center')
    ax3.set_xlim(0, 1)
    ax3.set_ylim(0, 1)
    ax3.axis('off')
    ax3.set_title('Kernel: Silverman\'s Rule', color='white', fontweight='bold')
    
    # 4. Summary panel
    ax4 = axes[1, 1]
    
    # Compute trimmed ranges
    trim_kernel = max(int(h_opt * n / (x.max() - x.min())), 5) * 3
    trim_lowess_frac = optimal_span / 2
    trim_sg = opt_window // 2
    
    sort_idx = np.argsort(x)
    x_sorted = x[sort_idx]
    
    summary_text = f"""Edge Trimming Summary:

Method 1 (Kernel):
  σ = {max(int(h_opt * n / (x.max() - x.min())), 5)}
  Trim: 3σ = {trim_kernel} points each end
  
Method 2 (LOWESS):
  span = {optimal_span:.3f}
  Trim: span/2 = {trim_lowess_frac*100:.1f}% each end
  
Method 3 (Savitzky-Golay):
  window = {opt_window}, polyorder = {opt_polyorder}
  Trim: window/2 = {trim_sg} points each end
"""
    ax4.text(0.1, 0.5, summary_text, transform=ax4.transAxes, fontsize=10, 
             color='white', family='monospace', verticalalignment='center')
    ax4.set_xlim(0, 1)
    ax4.set_ylim(0, 1)
    ax4.axis('off')
    ax4.set_title('Edge Trimming Summary', color='white', fontweight='bold')
    
    plt.tight_layout()
    
    if save:
        filepath = OUTPUT_DIR / "parameter_selection_diagnostics.png"
        fig.savefig(filepath, dpi=150, facecolor='black', bbox_inches='tight')
        print(f"Saved: {filepath}")
    
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    return fig


# MAIN

if __name__ == '__main__':
    print("""
╔══════════════════════════════════════════════════════════════════════╗
║  Threshold Analysis v2 - Data-Driven Parameter Selection             ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  Methods & Parameter Selection:                                      ║
║  ─────────────────────────────────────────────────────────────────   ║
║  1. Kernel Smoothing  -> Silverman's rule of thumb (1986)             ║
║  2. LOWESS            -> AICc minimization (Hurvich et al., 1998)     ║
║  3. Savitzky-Golay    -> Cross-validation (Vivó-Truyols, 2006)        ║
║                                                                      ║
║  Edge Trimming: All methods trim boundary regions for consistency    ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝

Usage:
------
from threshold_analysis_v2 import plot_threshold_analysis_v2, plot_parameter_selection_diagnostics

# Main analysis with data-driven parameter selection
fig, results = plot_threshold_analysis_v2(final_df, x_col='far_building')

# Diagnostic plots showing parameter selection curves
plot_parameter_selection_diagnostics(final_df, x_col='far_building')

# For Near building
fig, results = plot_threshold_analysis_v2(final_df, x_col='near_building', 
                                          xlabel='Near Building Height (m)')
""")

In [ ]:
fig, results = plot_threshold_analysis_v2(final_df, x_col='far_building')

In [ ]:
"""
Diagnostic Plot: Building Height Bins (7-10m)
==============================================
Compare variables across building height bins to understand error patterns
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use('dark_background')

OUTPUT_DIR = Path("unit_level_analysis")
OUTPUT_DIR.mkdir(exist_ok=True)


def get_flow_angle_zone(flow_angle):
    fa = flow_angle % 360
    if (fa >= 350 or fa <= 10) or (170 <= fa <= 190):
        return 'Parallel'
    elif (80 <= fa <= 100) or (260 <= fa <= 280):
        return 'Perpendicular'
    else:
        return 'Other'


def get_building_bin(height, bin_min=7.0, bin_max=10.0, bin_step=0.25):
    """
    Create building height bins from bin_min to bin_max with bin_step interval
    
    Example: 7.0-10.0 with 0.25 step creates bins:
    [7.00-7.25), [7.25-7.50), ..., [9.75-10.00]
    """
    if height < bin_min:
        return f'<{bin_min:.2f}'
    if height >= bin_max:
        return f'≥{bin_max:.2f}'
    
    # Find the bin
    bin_start = bin_min + ((height - bin_min) // bin_step) * bin_step
    bin_end = bin_start + bin_step
    
    return f'{bin_start:.2f}-{bin_end:.2f}'


def create_bin_order(bin_min=7.0, bin_max=10.0, bin_step=0.25):
    """Generate ordered list of bin labels"""
    bins = [f'<{bin_min:.2f}']
    
    current = bin_min
    while current < bin_max:
        bins.append(f'{current:.2f}-{current + bin_step:.2f}')
        current += bin_step
    
    bins.append(f'≥{bin_max:.2f}')
    
    return bins


# Variables to compare - organized by group
VARIABLE_GROUPS = {
    'ISA Simple': [
        'near_isa', 
        'far_isa',
        'near_ratio', 
        'far_ratio',
    ],
    'ISA Distance-Weighted': [
        'dw_near_isa', 
        'dw_far_isa', 
        'dw_near_ratio', 
        'dw_far_ratio',
    ],
    'Building Simple': [
        'near_building', 
        'far_building', 
        'near_ratio_building', 
        'far_ratio_building',
    ],
    'Building Distance-Weighted': [
        'dw_near_building', 
        'dw_far_building', 
        'dw_near_ratio_building', 
        'dw_far_ratio_building',
    ],
    'Geometry & Satellite': [
        'dark_frac',
        'xtrk_dist',
        'altitude_diff',  # sat_altitude - alt_va (will be created)
        'sat_altitude',
        'width',
        'flow_angle',
    ],
}

# Flatten for full list
ALL_VARIABLES = []
for group_vars in VARIABLE_GROUPS.values():
    ALL_VARIABLES.extend(group_vars)


def plot_diagnostic_building_bins(df, 
                                   bin_col='far_building',
                                   error_col='abs_error_node',
                                   bin_min=7.0, bin_max=10.0, bin_step=0.25,
                                   variables=None,
                                   highlight_bins=None,
                                   figsize=None,
                                   save=True, show=True):
    """
    Diagnostic boxplots: Compare variables across building height bins
    
    Parameters
    ----------
    df : DataFrame
    bin_col : str
        Column to use for binning (default: 'far_building')
    error_col : str
        Error column (default: 'abs_error_node')
    bin_min, bin_max, bin_step : float
        Bin range and step size
    variables : list or None
        Variables to plot. If None, use ALL_VARIABLES
    highlight_bins : list or None
        Bin labels to highlight
    """
    
    if variables is None:
        variables = ALL_VARIABLES
    
    # Prepare data: Parallel zone only
    required_cols = [error_col, 'flow_angle', bin_col]
    df_plot = df.dropna(subset=required_cols).copy()
    df_plot['flow_zone'] = df_plot['flow_angle'].apply(get_flow_angle_zone)
    df_parallel = df_plot[df_plot['flow_zone'] == 'Parallel'].copy()
    
    # Exclude bin_col = 0 (no building)
    df_parallel = df_parallel[df_parallel[bin_col] > 0].copy()
    
    # Create altitude difference column if possible
    # IMPORTANT: USGS alt_va is in FEET, SWOT sat_altitude is in METERS
    # Convert alt_va from feet to meters: 1 foot = 0.3048 meters
    if 'sat_altitude' in df_parallel.columns and 'alt_va' in df_parallel.columns:
        alt_va_meters = df_parallel['alt_va'] * 0.3048  # Convert feet to meters
        df_parallel['altitude_diff'] = df_parallel['sat_altitude'] - alt_va_meters
        print(f"Note: alt_va converted from feet to meters (×0.3048)")
    elif 'altitude_diff' not in df_parallel.columns:
        print("Warning: Cannot create altitude_diff (sat_altitude or alt_va missing)")
    
    # Create building height bins
    df_parallel['_bin'] = df_parallel[bin_col].apply(
        lambda x: get_building_bin(x, bin_min, bin_max, bin_step)
    )
    
    print(f"Parallel zone (excl. {bin_col}=0): {len(df_parallel):,} samples")
    print(f"Bin column: {bin_col}")
    print(f"Bin range: {bin_min} - {bin_max}m (step: {bin_step}m)")
    
    # Bin order
    bin_order = create_bin_order(bin_min, bin_max, bin_step)
    
    # Filter to bins that exist in data
    existing_bins = [b for b in bin_order if b in df_parallel['_bin'].values]
    print(f"Existing bins: {len(existing_bins)}")
    
    # Print bin counts
    print("\nBin distribution:")
    for b in existing_bins:
        count = (df_parallel['_bin'] == b).sum()
        print(f"  {b}: {count:,}")
    
    # Variables to plot (filter to those that exist)
    variables = [v for v in variables if v in df_parallel.columns]
    print(f"\nVariables to plot: {len(variables)}")
    
    # Figure layout: 2 columns
    n_vars = len(variables)
    n_cols = 2
    n_rows = (n_vars + n_cols - 1) // n_cols
    
    if figsize is None:
        figsize = (18, n_rows * 3.5)
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize, facecolor='black')
    axes = axes.flatten()
    
    # Colors
    color_var = '#00bfff'      # Cyan for variable
    color_error = '#ff6b6b'    # Red for error
    
    for idx, var in enumerate(variables):
        ax1 = axes[idx]
        ax1.set_facecolor('black')
        
        # Prepare data for boxplot
        var_data = []
        error_data = []
        positions = []
        bin_labels = []
        
        for i, bin_label in enumerate(existing_bins):
            bin_df = df_parallel[df_parallel['_bin'] == bin_label]
            if len(bin_df) > 0:
                var_vals = bin_df[var].dropna().values
                error_vals = bin_df[error_col].dropna().values
                if len(var_vals) > 0 and len(error_vals) > 0:
                    var_data.append(var_vals)
                    error_data.append(error_vals)
                    positions.append(len(positions))
                    bin_labels.append(bin_label)
        
        if len(positions) == 0:
            ax1.text(0.5, 0.5, f'{var}\nNo Data', 
                    transform=ax1.transAxes, ha='center', va='center', 
                    color='gray', fontsize=12)
            continue
        
        # Left y-axis: Variable
        bp1 = ax1.boxplot(var_data, positions=[p - 0.2 for p in positions], 
                          widths=0.35, patch_artist=True,
                          boxprops=dict(facecolor=color_var, alpha=0.7),
                          whis=(25, 75),
                          showmeans=True,
                          meanprops=dict(marker='o', markerfacecolor='white', 
                                        markeredgecolor='white', markersize=6),
                          medianprops=dict(linewidth=0),  # Hide median
                          whiskerprops=dict(color=color_var, linewidth=0),
                          capprops=dict(color=color_var, linewidth=0),
                          showfliers=False)
        
        # Calculate means for variable and connect with line
        var_means = [np.mean(d) for d in var_data]
        ax1.plot([p - 0.2 for p in positions], var_means, 
                 color='cyan', linewidth=1.5, linestyle='-', alpha=0.8, zorder=5)
        
        ax1.set_ylabel(var, color=color_var, fontsize=10)
        ax1.tick_params(axis='y', labelcolor=color_var, labelsize=8)
        
        # Right y-axis: Error (25th-75th percentile only, no whiskers)
        ax2 = ax1.twinx()
        bp2 = ax2.boxplot(error_data, positions=[p + 0.2 for p in positions], 
                          widths=0.35, patch_artist=True,
                          boxprops=dict(facecolor=color_error, alpha=0.7),
                          medianprops=dict(linewidth=0),  # Hide median
                          showmeans=True,
                          meanprops=dict(marker='x', markerfacecolor='white',
                                        markeredgecolor='white', markersize=6),
                          whis=(25, 75),
                          whiskerprops=dict(color=color_error, linewidth=0),
                          capprops=dict(color=color_error, linewidth=0),
                          showfliers=False)
        
        # Calculate means for error and connect with line
        error_means = [np.mean(d) for d in error_data]
        ax2.plot([p + 0.2 for p in positions], error_means,
                 color='#ff9999', linewidth=1.5, linestyle='-', alpha=0.8, zorder=5)
        
        # Set error y-axis range based on mean values (with margin)
        error_mean_min = min(error_means)
        error_mean_max = max(error_means)
        error_margin = (error_mean_max - error_mean_min) * 0.3
        if error_margin == 0:
            error_margin = error_mean_max * 0.2
        ax2.set_ylim(max(0, error_mean_min - error_margin), error_mean_max + error_margin)
        
        ax2.set_ylabel(error_col, color=color_error, fontsize=9)
        ax2.tick_params(axis='y', labelcolor=color_error, labelsize=8)
        
        # Highlight bins
        if highlight_bins:
            for i, bin_label in enumerate(bin_labels):
                if bin_label in highlight_bins:
                    ax1.axvspan(i - 0.5, i + 0.5, alpha=0.15, color='yellow')
        
        # X-axis
        ax1.set_xticks(positions)
        ax1.set_xticklabels(bin_labels, rotation=45, ha='right', color='white', fontsize=7)
        ax1.set_xlabel(f'{bin_col} (m)', color='white', fontsize=9)
        
        # Title
        ax1.set_title(var, color='white', fontsize=11, fontweight='bold')
        
        # Grid
        ax1.grid(True, alpha=0.2, color='white')
        ax1.set_xlim(-0.5, len(bin_labels) - 0.5)
        
        # Spines
        for spine in ax1.spines.values():
            spine.set_color('white')
        for spine in ax2.spines.values():
            spine.set_color('white')
    
    # Hide unused subplots
    for idx in range(len(variables), len(axes)):
        axes[idx].set_visible(False)
    
    # Legend
    fig.legend([plt.Rectangle((0,0),1,1, facecolor=color_var, alpha=0.7),
                plt.Rectangle((0,0),1,1, facecolor=color_error, alpha=0.7)],
               ['Variable', error_col],
               loc='upper right', facecolor='black', edgecolor='white',
               labelcolor='white', fontsize=10)
    
    # Main title
    highlight_str = f" | Highlight: {', '.join(highlight_bins)}" if highlight_bins else ""
    fig.suptitle(f'Parallel Zone Diagnostic: {bin_col} Bins ({bin_min}-{bin_max}m, {bin_step}m step){highlight_str}', 
                 fontsize=14, fontweight='bold', color='white', y=0.99)
    
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    
    if save:
        filepath = OUTPUT_DIR / f"diagnostic_{bin_col}_bins_{bin_min}_{bin_max}.png"
        fig.savefig(filepath, dpi=150, facecolor='black', bbox_inches='tight')
        print(f"\nSaved: {filepath}")
    
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    # Print summary statistics
    summary_df = print_summary_stats(df_parallel, variables, existing_bins, error_col)
    
    return fig, summary_df


def print_summary_stats(df_parallel, variables, existing_bins, error_col):
    """Print summary statistics for each bin"""
    
    print("SUMMARY STATISTICS BY BIN")
    
    # Create summary table
    summary_data = []
    
    for bin_label in existing_bins:
        bin_df = df_parallel[df_parallel['_bin'] == bin_label]
        row = {'Bin': bin_label, 'N': len(bin_df)}
        
        # Error
        row[f'{error_col}_median'] = bin_df[error_col].median()
        
        # Variables
        for var in variables:
            if var in bin_df.columns:
                row[f'{var}_median'] = bin_df[var].median()
        
        summary_data.append(row)
    
    summary_df = pd.DataFrame(summary_data)
    
    # Print compact version
    print(f"\n{'Bin':<15} {'N':>6} {'Error':>10}")
    
    for _, row in summary_df.iterrows():
        print(f"{row['Bin']:<15} {row['N']:>6} {row[f'{error_col}_median']:>10.4f}")
    
    # Find peak error bin
    peak_idx = summary_df[f'{error_col}_median'].idxmax()
    peak_bin = summary_df.loc[peak_idx, 'Bin']
    peak_error = summary_df.loc[peak_idx, f'{error_col}_median']
    
    print(f"Peak error bin: {peak_bin} (median error: {peak_error:.4f})")
    
    return summary_df


def plot_diagnostic_by_group(df, group_name, 
                              bin_col='far_building',
                              error_col='abs_error_node',
                              bin_min=7.0, bin_max=10.0, bin_step=0.25,
                              highlight_bins=None,
                              save=True, show=True):
    """
    Plot diagnostic for a specific variable group
    """
    if group_name not in VARIABLE_GROUPS:
        print(f"Unknown group: {group_name}")
        print(f"Available: {list(VARIABLE_GROUPS.keys())}")
        return None
    
    variables = VARIABLE_GROUPS[group_name]
    
    return plot_diagnostic_building_bins(
        df, variables=variables, bin_col=bin_col,
        error_col=error_col, bin_min=bin_min, bin_max=bin_max, bin_step=bin_step,
        highlight_bins=highlight_bins, save=save, show=show
    )


# Main
if __name__ == '__main__':
    print("""
╔══════════════════════════════════════════════════════════════════════╗
║  Diagnostic Plot: Building Height Bins                                ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  Variable Groups:                                                    ║
║  ─────────────────────────────────────────────────────────────────   ║
║  - ISA Simple: near_isa, far_isa, near_ratio, far_ratio              ║
║  - ISA Distance-Weighted: dw_near_isa, dw_far_isa, ...               ║
║  - Building Simple: near_building, far_building, ...                 ║
║  - Building Distance-Weighted: dw_near_building, dw_far_building     ║
║  - Geometry & Satellite: dark_frac, xtrk_dist, altitude_diff, ...    ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝

Usage:
------
from diagnostic_building_bins import plot_diagnostic_building_bins, plot_diagnostic_by_group

# Full diagnostic with all variables (2-column layout)
fig, summary = plot_diagnostic_building_bins(final_df)

# Custom bin range
fig, summary = plot_diagnostic_building_bins(
    final_df, 
    bin_min=6.0, 
    bin_max=12.0, 
    bin_step=0.5
)

# Plot specific variable group
fig, summary = plot_diagnostic_by_group(final_df, 'Geometry & Satellite')
fig, summary = plot_diagnostic_by_group(final_df, 'ISA Simple')

# With highlighting specific bins
fig, summary = plot_diagnostic_building_bins(
    final_df, 
    highlight_bins=['8.00-8.25', '8.25-8.50']
)

Available groups: ISA Simple, ISA Distance-Weighted, Building Simple, 
                  Building Distance-Weighted, Geometry & Satellite
""")

In [ ]:
fig, summary = plot_diagnostic_building_bins(
    final_df, 
    bin_min=6.5, 
    bin_max=10.0, 
    bin_step=0.25
)

In [ ]:
"""
Threshold Analysis v3: Stratified by ISA Bins
==============================================
Same 3 smoothing methods, but with ISA bin-colored lines overlaid

Question: At similar ISA levels, where is the peak building height effect?

"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.signal import savgol_filter
from scipy.ndimage import gaussian_filter1d
from statsmodels.nonparametric.smoothers_lowess import lowess
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')

OUTPUT_DIR = Path("unit_level_analysis")
OUTPUT_DIR.mkdir(exist_ok=True)

ERROR_THRESHOLD = 0.1832


def get_flow_angle_zone(flow_angle):
    fa = flow_angle % 360
    if (fa >= 350 or fa <= 10) or (170 <= fa <= 190):
        return 'Parallel'
    elif (80 <= fa <= 100) or (260 <= fa <= 280):
        return 'Perpendicular'
    else:
        return 'Other'


def silverman_bandwidth(x, scale_factor=0.9):
    """Silverman's rule of thumb for bandwidth selection"""
    n = len(x)
    if n < 5:
        return 0.5
    std = np.std(x, ddof=1)
    iqr = stats.iqr(x)
    scale = min(std, iqr / 1.34) if iqr > 0 else std
    if scale == 0:
        scale = 0.1
    h = scale_factor * scale * (n ** (-1/5))
    return h


def plot_threshold_analysis_by_isa(df, 
                                    x_col='far_building',
                                    y_col='abs_error_node',
                                    isa_col='far_isa',
                                    isa_bins=None,
                                    xlabel='Far Building Height (m)',
                                    min_samples=30,
                                    figsize=(14, 14),
                                    save=True, show=True):
    """
    Threshold analysis with ISA-stratified smoothing curves.
    
    Parameters
    ----------
    df : DataFrame
    x_col : str
        Building height column
    y_col : str
        Error column
    isa_col : str
        ISA column for stratification
    isa_bins : list
        ISA bin edges. Default: [0, 20, 40, 60, 80, 100]
    min_samples : int
        Minimum samples per bin to include
    """
    
    if isa_bins is None:
        isa_bins = [0, 20, 40, 60, 80, 100]
    
    print("THRESHOLD ANALYSIS v3 - ISA-Stratified")
    
    # Filter to Parallel zone
    df_parallel = df[df['flow_angle'].apply(get_flow_angle_zone) == 'Parallel'].copy()
    print(f"\n[Data] Parallel zone: {len(df_parallel):,} / {len(df):,}")
    
    # Clean data
    required = [x_col, y_col, isa_col]
    data = df_parallel[required].dropna()
    data = data[(data[y_col] > 0) & (data[x_col] > 0)]
    print(f"[Data] After cleaning: {len(data):,}")
    
    # Create ISA bins
    bin_labels = []
    for i in range(len(isa_bins) - 1):
        label = f'{int(isa_bins[i])}-{int(isa_bins[i+1])}%'
        mask = (data[isa_col] >= isa_bins[i]) & (data[isa_col] < isa_bins[i+1])
        data.loc[mask, 'isa_bin'] = label
        bin_labels.append(label)
    
    # Handle edge case for max value
    data.loc[data[isa_col] >= isa_bins[-1], 'isa_bin'] = bin_labels[-1]
    data = data.dropna(subset=['isa_bin'])
    
    # Colors for ISA bins (blue=low ISA, red=high ISA)
    n_bins = len(bin_labels)
    colors = plt.cm.RdYlBu_r(np.linspace(0.15, 0.85, n_bins))
    
    # Print bin counts
    print(f"\n[ISA Bins]")
    valid_bins = []
    for label, color in zip(bin_labels, colors):
        count = (data['isa_bin'] == label).sum()
        status = "✓" if count >= min_samples else "✗ (skip)"
        print(f"  {label}: {count:,} {status}")
        if count >= min_samples:
            valid_bins.append((label, color))
    
    # Create figure - 3 rows
    fig, axes = plt.subplots(3, 1, figsize=figsize, facecolor='black', sharex=True)
    for ax in axes:
        ax.set_facecolor('black')
    
    # Store results for each method
    results = {
        'kernel': {'peaks': {}, 'sigma': None},
        'lowess': {'peaks': {}, 'span': None},
        'savgol': {'peaks': {}, 'window': None, 'polyorder': None}
    }
    
    # Note: Parameters are computed per-bin using data-driven methods
    x_all = data[x_col].values
    y_all = data[y_col].values
    
    print(f"\n[Parameter Selection]")
    print(f"  Kernel: Silverman's rule per bin")
    print(f"  LOWESS: Adaptive span based on sample size")
    print(f"  Savitzky-Golay: Window ~20% of bin size (min 5, max 51)")
    
    # METHOD 1: KERNEL SMOOTHING (bin-specific parameters)
    ax1 = axes[0]
    
    # Light scatter of all data
    if len(x_all) > 3000:
        scatter_idx = np.random.choice(len(x_all), 3000, replace=False)
        ax1.scatter(x_all[scatter_idx], y_all[scatter_idx], c='gray', alpha=0.1, s=3, edgecolors='none')
    else:
        ax1.scatter(x_all, y_all, c='gray', alpha=0.1, s=3, edgecolors='none')
    
    for label, color in valid_bins:
        subset = data[data['isa_bin'] == label]
        x = subset[x_col].values
        y = subset[y_col].values
        
        sort_idx = np.argsort(x)
        x_sorted = x[sort_idx]
        y_sorted = y[sort_idx]
        
        n = len(x_sorted)
        x_range = x_sorted[-1] - x_sorted[0] if len(x_sorted) > 1 else 1
        
        # Compute bin-specific bandwidth using Silverman's rule
        h_bin = silverman_bandwidth(x_sorted)
        sigma = max(int(h_bin * n / x_range), 3)
        
        # Store parameter
        results['kernel']['params'] = results['kernel'].get('params', {})
        results['kernel']['params'][label] = {'sigma': sigma, 'h': h_bin, 'n': n}
        
        y_smooth = gaussian_filter1d(y_sorted, sigma=sigma)
        
        # Edge trimming
        trim = min(sigma * 3, n // 3)
        if trim > 0 and trim < n // 2:
            x_trim = x_sorted[trim:-trim]
            y_trim = y_smooth[trim:-trim]
        else:
            x_trim = x_sorted
            y_trim = y_smooth
        
        ax1.plot(x_trim, y_trim, color=color, linewidth=2.5, label=f'{label} (n={n}, σ={sigma})')
        
        # Find and mark peak
        if len(y_trim) > 0:
            peak_idx = np.argmax(y_trim)
            peak_x = x_trim[peak_idx]
            peak_y = y_trim[peak_idx]
            results['kernel']['peaks'][label] = {'x': peak_x, 'y': peak_y, 'sigma': sigma}
            
            ax1.scatter([peak_x], [peak_y], color=color, s=120, marker='*',
                       edgecolors='white', linewidths=0.5, zorder=10)
    
    # Draw mean peak line
    peak_xs = [v['x'] for v in results['kernel']['peaks'].values()]
    if peak_xs:
        mean_peak = np.mean(peak_xs)
        ax1.axvline(x=mean_peak, color='white', linestyle='--', linewidth=3, 
                    label=f'Mean Peak: {mean_peak:.1f}m')
        ax1.axvspan(mean_peak - np.std(peak_xs), mean_peak + np.std(peak_xs),
                   alpha=0.2, color='white')
    
    ax1.axhline(y=ERROR_THRESHOLD, color='orange', linestyle='-', linewidth=1, alpha=0.7)
    ax1.set_ylabel('Absolute Error (m)', color='white', fontsize=11)
    ax1.set_title(f'Method 1: Kernel Smoothing (Silverman per bin)\n'
                  f'ISA-Stratified | Mean Peak = {mean_peak:.1f}m',
                  color='white', fontsize=11, fontweight='bold')
    ax1.legend(loc='upper right', facecolor='black', edgecolor='white',
               labelcolor='white', fontsize=9, ncol=2)
    ax1.set_yscale('log')
    ax1.set_ylim(0.01, None)
    ax1.tick_params(colors='white')
    ax1.grid(True, alpha=0.2, color='white')
    for spine in ax1.spines.values():
        spine.set_color('white')
    
    # METHOD 2: LOWESS (bin-specific span)
    ax2 = axes[1]
    
    # Light scatter
    if len(x_all) > 3000:
        ax2.scatter(x_all[scatter_idx], y_all[scatter_idx], c='gray', alpha=0.1, s=3, edgecolors='none')
    else:
        ax2.scatter(x_all, y_all, c='gray', alpha=0.1, s=3, edgecolors='none')
    
    for label, color in valid_bins:
        subset = data[data['isa_bin'] == label]
        x = subset[x_col].values
        y = subset[y_col].values
        
        sort_idx = np.argsort(x)
        x_sorted = x[sort_idx]
        y_sorted = y[sort_idx]
        
        n = len(x_sorted)
        
        # Bin-specific span: larger span for smaller samples
        # Rule of thumb: span ~ k/n where k is neighborhood size
        # For small n, need larger span to avoid overfitting
        if n < 50:
            span = 0.5  # 50% of data
        elif n < 100:
            span = 0.4
        elif n < 200:
            span = 0.3
        else:
            span = 0.25
        
        # Store parameter
        results['lowess']['params'] = results['lowess'].get('params', {})
        results['lowess']['params'][label] = {'span': span, 'n': n}
        
        # LOWESS
        smoothed = lowess(y_sorted, x_sorted, frac=span, return_sorted=True)
        
        # Edge trimming
        n_smooth = len(smoothed)
        trim = int(n_smooth * span / 2)
        if trim > 0 and trim < n_smooth // 2:
            smoothed_trim = smoothed[trim:-trim]
        else:
            smoothed_trim = smoothed
        
        ax2.plot(smoothed_trim[:, 0], smoothed_trim[:, 1], color=color, 
                linewidth=2.5, label=f'{label} (n={n}, span={span})')
        
        # Find and mark peak
        if len(smoothed_trim) > 0:
            peak_idx = np.argmax(smoothed_trim[:, 1])
            peak_x = smoothed_trim[peak_idx, 0]
            peak_y = smoothed_trim[peak_idx, 1]
            results['lowess']['peaks'][label] = {'x': peak_x, 'y': peak_y, 'span': span}
            
            ax2.scatter([peak_x], [peak_y], color=color, s=120, marker='*',
                       edgecolors='white', linewidths=0.5, zorder=10)
    
    # Draw mean peak line
    peak_xs = [v['x'] for v in results['lowess']['peaks'].values()]
    if peak_xs:
        mean_peak_lowess = np.mean(peak_xs)
        ax2.axvline(x=mean_peak_lowess, color='white', linestyle='--', linewidth=3,
                    label=f'Mean Peak: {mean_peak_lowess:.1f}m')
        ax2.axvspan(mean_peak_lowess - np.std(peak_xs), mean_peak_lowess + np.std(peak_xs),
                   alpha=0.2, color='white')
    
    ax2.axhline(y=ERROR_THRESHOLD, color='orange', linestyle='-', linewidth=1, alpha=0.7)
    ax2.set_ylabel('Absolute Error (m)', color='white', fontsize=11)
    ax2.set_title(f'Method 2: LOWESS (adaptive span per bin)\n'
                  f'ISA-Stratified | Mean Peak = {mean_peak_lowess:.1f}m',
                  color='white', fontsize=11, fontweight='bold')
    ax2.legend(loc='upper right', facecolor='black', edgecolor='white',
               labelcolor='white', fontsize=9, ncol=2)
    ax2.set_yscale('log')
    ax2.set_ylim(0.01, None)
    ax2.tick_params(colors='white')
    ax2.grid(True, alpha=0.2, color='white')
    for spine in ax2.spines.values():
        spine.set_color('white')
    
    # METHOD 3: SAVITZKY-GOLAY (bin-specific window)
    ax3 = axes[2]
    
    # Light scatter
    if len(x_all) > 3000:
        ax3.scatter(x_all[scatter_idx], y_all[scatter_idx], c='gray', alpha=0.1, s=3, edgecolors='none')
    else:
        ax3.scatter(x_all, y_all, c='gray', alpha=0.1, s=3, edgecolors='none')
    
    for label, color in valid_bins:
        subset = data[data['isa_bin'] == label]
        x = subset[x_col].values
        y = subset[y_col].values
        
        sort_idx = np.argsort(x)
        x_sorted = x[sort_idx]
        y_sorted = y[sort_idx]
        
        n = len(x_sorted)
        
        # window per bin: ~10-20% of the data, kept within sane bounds
        # Schafer (2011): window should be odd, and window/polyorder ~ 3.5
        window = max(5, min(n // 5, 51))
        window = window if window % 2 == 1 else window + 1
        polyorder = min(2, window - 1)
        
        # Store parameter
        results['savgol']['params'] = results['savgol'].get('params', {})
        results['savgol']['params'][label] = {'window': window, 'polyorder': polyorder, 'n': n}
        
        if n < window:
            print(f"  Skipping S-G for {label}: n={n} < window={window}")
            continue
        
        y_smooth = savgol_filter(y_sorted, window_length=window, polyorder=polyorder)
        
        # Edge trimming
        trim = window // 2
        if trim > 0 and trim < n // 2:
            x_trim = x_sorted[trim:-trim]
            y_trim = y_smooth[trim:-trim]
        else:
            x_trim = x_sorted
            y_trim = y_smooth
        
        ax3.plot(x_trim, y_trim, color=color, linewidth=2.5, label=f'{label} (n={n}, w={window})')
        
        # Find and mark peak
        if len(y_trim) > 0:
            peak_idx = np.argmax(y_trim)
            peak_x = x_trim[peak_idx]
            peak_y = y_trim[peak_idx]
            results['savgol']['peaks'][label] = {'x': peak_x, 'y': peak_y, 'window': window}
            
            ax3.scatter([peak_x], [peak_y], color=color, s=120, marker='*',
                       edgecolors='white', linewidths=0.5, zorder=10)
    
    # Draw mean peak line
    peak_xs = [v['x'] for v in results['savgol']['peaks'].values()]
    if peak_xs:
        mean_peak_sg = np.mean(peak_xs)
        ax3.axvline(x=mean_peak_sg, color='white', linestyle='--', linewidth=3,
                    label=f'Mean Peak: {mean_peak_sg:.1f}m')
        ax3.axvspan(mean_peak_sg - np.std(peak_xs), mean_peak_sg + np.std(peak_xs),
                   alpha=0.2, color='white')
    
    ax3.axhline(y=ERROR_THRESHOLD, color='orange', linestyle='-', linewidth=1, alpha=0.7)
    ax3.set_xlabel(xlabel, color='white', fontsize=11)
    ax3.set_ylabel('Absolute Error (m)', color='white', fontsize=11)
    ax3.set_title(f'Method 3: Savitzky-Golay (adaptive window per bin)\n'
                  f'ISA-Stratified | Mean Peak = {mean_peak_sg:.1f}m',
                  color='white', fontsize=11, fontweight='bold')
    ax3.legend(loc='upper right', facecolor='black', edgecolor='white',
               labelcolor='white', fontsize=9, ncol=2)
    ax3.tick_params(colors='white')
    ax3.grid(True, alpha=0.2, color='white')
    for spine in ax3.spines.values():
        spine.set_color('white')
    
    # Finalize
    fig.suptitle(f'Threshold Analysis: {xlabel} vs Error by ISA Level (Parallel Zone)\n'
                 f'Colors: Blue=Low ISA → Red=High ISA | White dashed = Mean Peak',
                 fontsize=13, fontweight='bold', color='white', y=0.99)
    
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    
    if save:
        filepath = OUTPUT_DIR / f"threshold_analysis_v3_by_{isa_col}.png"
        fig.savefig(filepath, dpi=150, facecolor='black', bbox_inches='tight')
        print(f"\nSaved: {filepath}")
    
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    # Print Summary
    print("SUMMARY: Peak Building Height by ISA Level (Bin-Specific Parameters)")
    
    print(f"\n{'Method':<12} {'ISA Bin':<12} {'N':>6} {'Param':>12} {'Peak (m)':>10} {'Error (m)':>12}")
    
    for method in ['kernel', 'lowess', 'savgol']:
        method_name = {'kernel': 'Kernel', 'lowess': 'LOWESS', 'savgol': 'S-G'}[method]
        for label in bin_labels:
            if label in results[method]['peaks']:
                peak = results[method]['peaks'][label]
                params = results[method].get('params', {}).get(label, {})
                n = params.get('n', '?')
                
                if method == 'kernel':
                    param_str = f"σ={params.get('sigma', '?')}"
                elif method == 'lowess':
                    param_str = f"span={params.get('span', '?')}"
                else:
                    param_str = f"w={params.get('window', '?')}"
                
                print(f"{method_name:<12} {label:<12} {n:>6} {param_str:>12} {peak['x']:>10.1f} {peak['y']:>12.4f}")
        
        # Mean for this method
        peaks = results[method]['peaks']
        if peaks:
            mean_x = np.mean([v['x'] for v in peaks.values()])
            std_x = np.std([v['x'] for v in peaks.values()])
            print(f"{method_name:<12} {'MEAN±STD':<12} {'':<6} {'':<12} {mean_x:>7.1f}±{std_x:<4.1f}")
        print()
    
    # Overall consensus
    all_peaks = []
    for method in ['kernel', 'lowess', 'savgol']:
        all_peaks.extend([v['x'] for v in results[method]['peaks'].values()])
    
    if all_peaks:
        print(f"OVERALL CONSENSUS: {np.mean(all_peaks):.1f} ± {np.std(all_peaks):.1f} m")
    
    return fig, results


# MAIN

if __name__ == '__main__':
    print("""
╔══════════════════════════════════════════════════════════════════════╗
║  Threshold Analysis v3 - ISA-Stratified                               ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  Same 3 smoothing methods as v2, but with ISA-colored curves         ║
║  overlaid on each plot.                                              ║
║                                                                      ║
║  Key Question: Does the peak building height change with ISA level?  ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝

Usage:
------
from threshold_analysis_v3_stratified import plot_threshold_analysis_by_isa

# Default ISA bins
fig, results = plot_threshold_analysis_by_isa(final_df)

# Custom ISA bins
fig, results = plot_threshold_analysis_by_isa(
    final_df,
    isa_bins=[0, 15, 30, 45, 60, 75, 90],
    min_samples=20
)

# Different columns
fig, results = plot_threshold_analysis_by_isa(
    final_df,
    x_col='near_building',
    isa_col='near_isa',
    xlabel='Near Building Height (m)'
)
""")

In [ ]:
fig, results = plot_threshold_analysis_by_isa(
    final_df, 
    isa_bins=[0, 10, 20, 30, 40, 50, 60, 70, 80, 90],
    min_samples=1
)

#### Heatmap Analysis

In [ ]:
"""
Boxplot Matrix + Heatmap (Side by Side)
========================================
1x2 Layout: Boxplot | Heatmap
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use('dark_background')

OUTPUT_DIR = Path("unit_level_analysis")
OUTPUT_DIR.mkdir(exist_ok=True)


def get_flow_angle_zone(flow_angle):
    fa = flow_angle % 360
    if (fa >= 350 or fa <= 10) or (170 <= fa <= 190):
        return 'Parallel'
    elif (80 <= fa <= 100) or (260 <= fa <= 280):
        return 'Perpendicular'
    else:
        return 'Other'


def plot_boxplot_and_heatmap(df,
                              isa_col='far_isa',
                              building_col='far_building', 
                              error_col='abs_error_node',
                              isa_range=(0, 90),
                              isa_step=10,
                              building_range=(6, 10),
                              building_step=0.5,
                              figsize=(18, 8),
                              save=True, show=True):
    
    print(f"BOXPLOT + HEATMAP: {isa_col} vs {building_col} -> {error_col}")
    
    # Filter to Parallel zone
    df_parallel = df[df['flow_angle'].apply(get_flow_angle_zone) == 'Parallel'].copy()
    print(f"\n[Data] Parallel zone: {len(df_parallel):,} / {len(df):,}")
    
    # Clean data
    data = df_parallel[[isa_col, building_col, error_col]].dropna()
    data = data[(data[building_col] > 0) & (data[error_col] > 0)]
    print(f"[Data] After cleaning: {len(data):,}")
    
    # Create bins
    isa_bins = np.arange(isa_range[0], isa_range[1] + isa_step, isa_step)
    building_bins = np.arange(building_range[0], building_range[1] + building_step, building_step)
    
    n_rows = len(building_bins) - 1
    n_cols = len(isa_bins) - 1
    
    print(f"\n[Grid] {n_rows} rows × {n_cols} cols")
    
    # Calculate median matrix for heatmap
    median_matrix = np.full((n_rows, n_cols), np.nan)
    count_matrix = np.full((n_rows, n_cols), 0)
    
    for i in range(n_rows):
        building_low = building_bins[i]
        building_high = building_bins[i + 1]
        for j in range(n_cols):
            isa_low = isa_bins[j]
            isa_high = isa_bins[j + 1]
            
            mask = ((data[isa_col] >= isa_low) & (data[isa_col] < isa_high) &
                    (data[building_col] >= building_low) & (data[building_col] < building_high))
            cell_data = data[mask][error_col]
            
            if len(cell_data) > 0:
                median_matrix[i, j] = cell_data.median()
                count_matrix[i, j] = len(cell_data)
    
    # Figure: 1 x 2 layout
    fig = plt.figure(figsize=figsize, facecolor='#0d0d0d')
    
    # GridSpec으로 레이아웃 설정
    gs = fig.add_gridspec(1, 2, width_ratios=[1, 1], wspace=0.12,
                          left=0.08, right=0.92, bottom=0.12, top=0.88)
    
    # LEFT: Boxplot Matrix
    gs_box = gs[0].subgridspec(n_rows, n_cols, wspace=0.03, hspace=0.03)
    
    error_ticks = [0.1, 0.5, 1, 2, 5, 10, 20]
    y_lim = (0.05, 30)
    
    box_color = '#4ECDC4'
    median_color = '#FFE66D'
    whisker_color = '#FFFFFF'
    
    axes_box = []
    for i in range(n_rows):
        row_axes = []
        for j in range(n_cols):
            ax = fig.add_subplot(gs_box[n_rows - 1 - i, j])
            row_axes.append(ax)
            
            ax.set_facecolor('#1a1a1a')
            
            building_low = building_bins[i]
            building_high = building_bins[i + 1]
            isa_low = isa_bins[j]
            isa_high = isa_bins[j + 1]
            
            mask = ((data[isa_col] >= isa_low) & (data[isa_col] < isa_high) &
                    (data[building_col] >= building_low) & (data[building_col] < building_high))
            cell_data = data[mask]
            
            if len(cell_data) > 0:
                box_data = cell_data[error_col].values
                
                bp = ax.boxplot([box_data], 
                               positions=[0.5],
                               widths=0.4,
                               patch_artist=True, 
                               manage_ticks=False,
                               whis=1.5,
                               showfliers=True,
                               flierprops=dict(marker='.', markersize=2, 
                                              markerfacecolor=box_color, 
                                              markeredgecolor=box_color,
                                              alpha=0.4))
                
                for patch in bp['boxes']:
                    patch.set_facecolor(box_color)
                    patch.set_alpha(0.7)
                    patch.set_edgecolor(whisker_color)
                    patch.set_linewidth(0.8)
                for whisker in bp['whiskers']:
                    whisker.set_color(whisker_color)
                    whisker.set_linewidth(0.8)
                    whisker.set_alpha(0.8)
                for cap in bp['caps']:
                    cap.set_color(whisker_color)
                    cap.set_linewidth(0.8)
                    cap.set_alpha(0.8)
                for median in bp['medians']:
                    median.set_color(median_color)
                    median.set_linewidth(1.5)
            else:
                ax.text(0.5, 0.5, '—', transform=ax.transAxes,
                       fontsize=10, color='#444444', ha='center', va='center')
            
            ax.set_yscale('symlog', linthresh=1)
            ax.set_ylim(y_lim)
            ax.set_xlim(0, 1)
            ax.set_xticks([])
            
            ax.set_yticks(error_ticks)
            if j == 0:
                ax.set_yticklabels(['0.1', '0.5', '1', '2', '5', '10', '20'])
                ax.tick_params(axis='y', labelsize=7, colors='white', labelleft=True)
            else:
                ax.set_yticklabels([])
                ax.tick_params(axis='y', labelleft=False)
            
            ax.yaxis.grid(True, alpha=0.15, linestyle='-', color='white', linewidth=0.5)
            
            for spine in ax.spines.values():
                spine.set_color('#333333')
                spine.set_linewidth(0.5)
            
            ax.yaxis.set_minor_locator(plt.NullLocator())
            
            # Column labels (하단)
            if i == 0:
                ax.set_xlabel(f'{int(isa_low)}–{int(isa_high)}%', fontsize=8, 
                             color='white', fontweight='bold', labelpad=3)
            
            # Row labels (우측)
            if j == n_cols - 1:
                ax2 = ax.twinx()
                ax2.set_yticks([])
                ax2.set_ylabel(f'{building_low:.1f}–{building_high:.1f}m', 
                              fontsize=8, color='#4ECDC4', fontweight='bold',
                              rotation=0, labelpad=10, va='center')
                for spine in ax2.spines.values():
                    spine.set_visible(False)
        
        axes_box.append(row_axes)
    
    # Boxplot 축 레이블
    fig.text(0.27, 0.04, 'Far ISA (%)', ha='center', fontsize=11, color='white', fontweight='bold')
    fig.text(0.02, 0.5, 'Absolute Error (m)', va='center', rotation='vertical', 
             fontsize=11, color='white', fontweight='bold')
    fig.text(0.27, 0.92, '(a) Boxplot Distribution', ha='center', fontsize=12, 
             color='white', fontweight='bold')
    
    # RIGHT: Heatmap
    ax_heat = fig.add_subplot(gs[1])
    ax_heat.set_facecolor('#1a1a1a')
    
    im = ax_heat.imshow(median_matrix, aspect='auto', cmap='hot', origin='lower',
                        interpolation='nearest')
    
    # Colorbar
    cbar = plt.colorbar(im, ax=ax_heat, shrink=0.8, pad=0.02)
    cbar.set_label('Median Error (m)', color='white', fontsize=10)
    cbar.ax.yaxis.set_tick_params(color='white')
    cbar.outline.set_edgecolor('white')
    plt.setp(plt.getp(cbar.ax.axes, 'yticklabels'), color='white', fontsize=9)
    
    # Annotations
    for i in range(n_rows):
        for j in range(n_cols):
            val = median_matrix[i, j]
            cnt = count_matrix[i, j]
            
            if not np.isnan(val):
                text_color = 'black' if val > np.nanpercentile(median_matrix, 60) else 'white'
                ax_heat.text(j, i, f'{val:.2f}\n({int(cnt)})', 
                           ha='center', va='center', fontsize=7,
                           color=text_color, fontweight='bold')
    
    # X-axis labels - boxplot과 동일한 형식
    isa_labels = [f'{int(isa_bins[j])}–{int(isa_bins[j+1])}%' for j in range(n_cols)]
    ax_heat.set_xticks(np.arange(n_cols))
    ax_heat.set_xticklabels(isa_labels, color='white', fontsize=8, fontweight='bold')
    ax_heat.set_xlabel('Far ISA (%)', color='white', fontsize=11, fontweight='bold')
    
    # Y-axis - tick labels 제거 (왼쪽 boxplot과 공유)
    ax_heat.set_yticks(np.arange(n_rows))
    ax_heat.set_yticklabels([])
    
    # 우측에 Far Building Height (m) 라벨 추가
    # ax_heat.yaxis.set_label_position('right')
    ax_heat.set_ylabel('Far Building Height (m)', color='#4ECDC4', fontsize=11, 
                       fontweight='bold', rotation=-90, labelpad=15)
    
    # Grid
    ax_heat.set_xticks(np.arange(-0.5, n_cols, 1), minor=True)
    ax_heat.set_yticks(np.arange(-0.5, n_rows, 1), minor=True)
    ax_heat.grid(which='minor', color='white', linestyle='-', linewidth=0.5, alpha=0.3)
    
    for spine in ax_heat.spines.values():
        spine.set_color('white')
    
    fig.text(0.70, 0.92, '(b) Median Error Heatmap', ha='center', fontsize=12, 
             color='white', fontweight='bold')
    
    # Main Title
    fig.suptitle('Absolute Error by Far ISA and Building Height (Parallel Flow Zone)',
                 fontsize=14, color='white', fontweight='bold', y=0.98)
    
    if save:
        filepath = OUTPUT_DIR / f"boxplot_heatmap_sidebyside.png"
        fig.savefig(filepath, dpi=300, facecolor='#0d0d0d', bbox_inches='tight')
        print(f"\nSaved: {filepath}")
    
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    return fig


# Run

fig = plot_boxplot_and_heatmap(
    final_df,
    isa_col='far_isa',
    building_col='far_building',
    error_col='abs_error_node',
    isa_range=(0, 90),
    isa_step=10,
    building_range=(6, 10),
    building_step=0.5,
    figsize=(18, 8),
    save=True, show=True
)

#### Grouped Effect (Colin)

In [ ]:
"""
Combined Grouped Boxplot: 2 Rows
================================
Row 1: X = Building Height, Color = Far ISA (Reds)
Row 2: X = Far ISA, Color = Building Height (Greens)
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from pathlib import Path

# White background style
plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

OUTPUT_DIR = Path("figures")
OUTPUT_DIR.mkdir(exist_ok=True)


def get_flow_angle_zone(flow_angle):
    fa = flow_angle % 360
    if (fa >= 350 or fa <= 10) or (170 <= fa <= 190):
        return 'Parallel'
    elif (80 <= fa <= 100) or (260 <= fa <= 280):
        return 'Perpendicular'
    else:
        return 'Other'


def get_isa_bin(isa_value):
    """ISA bins: 0-5, 5-10, 10-20, ..., 60-70, 70-100 (70-80, 80-100 합침)"""
    if isa_value <= 5:
        return '0-5'
    elif isa_value <= 10:
        return '5-10'
    elif isa_value <= 20:
        return '10-20'
    elif isa_value <= 30:
        return '20-30'
    elif isa_value <= 45:
        return '30-45'
    elif isa_value <= 60:
        return '45-60'        
    else:
        return '60+' 

def get_building_bins_dynamic(data, building_col, min_samples=100, start_from=5, interval=2):
    """
    Create building height bins with custom intervals.
    Stop when REMAINING samples (from edge to end) < min_samples.
    
    Parameters
    ----------
    interval : int
        Bin width in meters (1 = 1m intervals, 2 = 2m intervals)
    """
    values = data[building_col].dropna()
    values = values[values >= start_from]
    
    max_val = int(np.ceil(values.max()))
    
    print(f"[Building] Range (>={start_from}m): {values.min():.2f} ~ {values.max():.2f}m")
    print(f"[Building] Total samples: {len(values):,}")
    print(f"[Building] Interval: {interval}m")
    
    valid_edges = [start_from]
    
    for edge in range(start_from + interval, max_val + interval, interval):  # interval씩 증가
        remaining = (values >= edge).sum()
        
        print(f"  {edge}m 이상 남은 샘플: {remaining:,}")
        
        if remaining >= min_samples:
            valid_edges.append(edge)
        else:
            print(f"  -> Stopping: {edge}m 이상 남은 샘플 {remaining} < {min_samples}")
            break
    
    valid_edges.append(np.inf)
    
    # Create labels
    labels = []
    for i in range(len(valid_edges) - 1):
        if valid_edges[i + 1] == np.inf:
            labels.append(f'{valid_edges[i]}+')
        else:
            labels.append(f'{valid_edges[i]}-{valid_edges[i + 1]}')
    
    print(f"[Building] Final bins: {labels}")
    
    return valid_edges, labels


def plot_grouped_boxplot_combined(df,
                                   isa_col='far_isa',
                                   building_col='far_building',
                                   error_col='abs_error_node',
                                   min_samples=50,
                                   interval=2,
                                   offset=0.1,
                                   figsize=(18, 12),
                                   save=True,
                                   show=True):
    """
    Combined grouped boxplot with 2 rows.
    Row 1: X = Building Height, Color = ISA (Reds)
    Row 2: X = ISA, Color = Building Height (Greens)
    """
    
    print("COMBINED GROUPED BOXPLOT")
    
    # Filter to Parallel zone only
    df_parallel = df[df['flow_angle'].apply(get_flow_angle_zone) == 'Parallel'].copy()
    print(f"\n[Data] Parallel zone: {len(df_parallel):,} / {len(df):,}")
    
    # Clean data
    data = df_parallel[[isa_col, building_col, error_col]].dropna()
    print(f"[Data] After dropna: {len(data):,}")
    
    # Define bins
    
    isa_bin_order = ['0-5', '5-10', '10-20', '20-30', '30-45', '45-60', 
                     '60+']
    data['isa_bin'] = data[isa_col].apply(get_isa_bin)
    
    building_edges, building_labels = get_building_bins_dynamic(data, building_col, min_samples, interval=interval)
    data['building_bin'] = pd.cut(data[building_col], bins=building_edges,
                                   labels=building_labels, right=False)
    
    data = data.dropna(subset=['isa_bin', 'building_bin'])
    print(f"[Data] After binning: {len(data):,}")
    
    n_isa_bins = len(isa_bin_order)
    n_building_bins = len(building_labels)
    
    # Sample counts per bin (for x-axis labels)
    building_counts = data.groupby('building_bin', observed=True).size().to_dict()
    isa_counts = data.groupby('isa_bin', observed=True).size().to_dict()
    
    building_labels_with_n = [f'{lbl}\n(n={building_counts.get(lbl, 0):,})' for lbl in building_labels]
    
    existing_isa_bins = [b for b in isa_bin_order if b in data['isa_bin'].values]
    isa_labels_with_n = [f'{lbl}\n(n={isa_counts.get(lbl, 0):,})' for lbl in existing_isa_bins]
    
    # Colors
    import matplotlib.cm as cm
    
    cmap_isa = cm.get_cmap('Oranges', n_isa_bins + 2)
    colors_isa = [cmap_isa(i + 2) for i in range(n_isa_bins)]
    
    cmap_building = cm.get_cmap('Purples', n_building_bins + 2)
    colors_building = [cmap_building(i + 2) for i in range(n_building_bins)]
    
    # Create figure
    fig, axes = plt.subplots(2, 1, figsize=figsize, facecolor='white')
    
    # Row 1: X = Building Height, Color = ISA (Reds)
    ax1 = axes[0]
    ax1.set_facecolor('white')
    
    box_data1 = []
    positions1 = []
    box_colors1 = []
    sample_counts1 = []  # 각 boxplot별 sample 수
    
    group_width = 0.8
    box_width1 = group_width / (n_isa_bins + 1)
    
    for i, building_label in enumerate(building_labels):
        group_center = i + 1
        
        for j, isa_label in enumerate(isa_bin_order):
            mask = (data['building_bin'] == building_label) & (data['isa_bin'] == isa_label)
            subset = data[mask][error_col].values
            
            if len(subset) > 0:
                box_data1.append(subset + offset)  # offset 적용
                sample_counts1.append(len(subset))
            else:
                box_data1.append([np.nan])
                sample_counts1.append(0)
            
            pos_offset = (j - (n_isa_bins - 1) / 2) * box_width1
            positions1.append(group_center + pos_offset)
            box_colors1.append(colors_isa[j])
    
    bp1 = ax1.boxplot(box_data1,
                      positions=positions1,
                      widths=box_width1 * 0.85,
                      patch_artist=True,
                      showfliers=True,
                      flierprops=dict(marker='o', markersize=2, alpha=0.3,
                                     markerfacecolor='gray', markeredgecolor='none'))
    
    for patch, color in zip(bp1['boxes'], box_colors1):
        patch.set_facecolor(color)
        patch.set_edgecolor('black')
        patch.set_linewidth(0.8)
        patch.set_alpha(0.85)
    
    for whisker in bp1['whiskers']:
        whisker.set_color('black')
        whisker.set_linewidth(0.8)
    for cap in bp1['caps']:
        cap.set_color('black')
        cap.set_linewidth(0.8)
    for median in bp1['medians']:
        median.set_color('black')
        median.set_linewidth(1.5)
    
    # Whisker 위에 sample 수 표시 (90도 회전)
    for i, (pos, n) in enumerate(zip(positions1, sample_counts1)):
        if n > 0:
            # Upper whisker의 y값 가져오기 (whiskers는 [lower, upper] 순서로 2개씩)
            upper_whisker = bp1['whiskers'][i * 2 + 1]
            y_top = upper_whisker.get_ydata().max()
            
            ax1.text(pos, y_top * 1.05, f' n={n}', rotation=90, fontsize=11,
                    ha='center', va='bottom', color='black')
    
    # Log scale with offset
    ax1.set_yscale('log')
    ax1.set_ylim(offset * 0.8, 50 + offset)
    
    # Y축 tick labels (원래 값으로 표시)
    tick_values_orig = [0, 0.2, 0.5, 1, 2, 5, 10, 20, 50]
    tick_values = [v + offset for v in tick_values_orig]
    ax1.set_yticks(tick_values)
    ax1.set_yticklabels([str(v) for v in tick_values_orig])
    
    ax1.set_ylabel(r'$|Relative\ WSE\ Error|$ (m)', fontsize=13)
    ax1.set_xlabel('Far Building Height (m)', fontsize=13)
    ax1.set_xticks(range(1, n_building_bins + 1))
    ax1.set_xticklabels(building_labels_with_n, fontsize=10)
    ax1.set_xlim(0.3, n_building_bins + 0.7)
    ax1.tick_params(labelsize=11)
    ax1.yaxis.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)
    ax1.xaxis.grid(False)
    ax1.set_title(f'Bin: Far Building Height & Color: Far ISA (n={len(data):,})', 
                  fontsize=14, fontweight='bold')
    
    for spine in ax1.spines.values():
        spine.set_color('black')
    
    legend_patches1 = [Patch(facecolor=colors_isa[i], edgecolor='black',
                             label=isa_bin_order[i] + '%', alpha=0.85)
                       for i in range(n_isa_bins)]
    ax1.legend(handles=legend_patches1,
               title='Far ISA',
               loc='center left',
               bbox_to_anchor=(1.02, 0.5),
               framealpha=0.95,
               title_fontsize=12,
               fontsize=10)
    
    # Row 2: X = ISA, Color = Building Height (Greens)
    ax2 = axes[1]
    ax2.set_facecolor('white')
    
    box_data2 = []
    positions2 = []
    box_colors2 = []
    sample_counts2 = []
    
    box_width2 = group_width / (n_building_bins + 1)
    
    n_existing_isa = len(existing_isa_bins)
    
    for i, isa_label in enumerate(existing_isa_bins):
        group_center = i + 1
        
        for j, building_label in enumerate(building_labels):
            mask = (data['isa_bin'] == isa_label) & (data['building_bin'] == building_label)
            subset = data[mask][error_col].values
            
            if len(subset) > 0:
                box_data2.append(subset + offset)  # offset 적용
                sample_counts2.append(len(subset))
            else:
                box_data2.append([np.nan])
                sample_counts2.append(0)
            
            pos_offset = (j - (n_building_bins - 1) / 2) * box_width2
            positions2.append(group_center + pos_offset)
            box_colors2.append(colors_building[j])
    
    bp2 = ax2.boxplot(box_data2,
                      positions=positions2,
                      widths=box_width2 * 0.85,
                      patch_artist=True,
                      showfliers=True,
                      flierprops=dict(marker='o', markersize=2, alpha=0.3,
                                     markerfacecolor='gray', markeredgecolor='none'))
    
    for patch, color in zip(bp2['boxes'], box_colors2):
        patch.set_facecolor(color)
        patch.set_edgecolor('black')
        patch.set_linewidth(0.8)
        patch.set_alpha(0.85)
    
    for whisker in bp2['whiskers']:
        whisker.set_color('black')
        whisker.set_linewidth(0.8)
    for cap in bp2['caps']:
        cap.set_color('black')
        cap.set_linewidth(0.8)
    for median in bp2['medians']:
        median.set_color('black')
        median.set_linewidth(1.5)
    
    # Whisker 위에 sample 수 표시 (90도 회전)
    for i, (pos, n) in enumerate(zip(positions2, sample_counts2)):
        if n > 0:
            upper_whisker = bp2['whiskers'][i * 2 + 1]
            y_top = upper_whisker.get_ydata().max()
            
            ax2.text(pos, y_top * 1.05, f' n={n}', rotation=90, fontsize=11,
                    ha='center', va='bottom', color='black')
    
    # Log scale with offset
    ax2.set_yscale('log')
    ax2.set_ylim(offset * 0.8, 50 + offset)
    
    ax2.set_yticks(tick_values)
    ax2.set_yticklabels([str(v) for v in tick_values_orig])
    
    ax2.set_ylabel(r'$|Relative\ WSE\ Error|$ (m)', fontsize=13)
    ax2.set_xlabel('Far ISA (%)', fontsize=13)
    ax2.set_xticks(range(1, n_existing_isa + 1))
    ax2.set_xticklabels(isa_labels_with_n, fontsize=10)
    ax2.set_xlim(0.3, n_existing_isa + 0.7)
    ax2.tick_params(labelsize=11)
    ax2.yaxis.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)
    ax2.xaxis.grid(False)
    ax2.set_title(f'Bin: Far ISA & Color: Far Building Height (n={len(data):,})', 
                  fontsize=14, fontweight='bold')
    
    for spine in ax2.spines.values():
        spine.set_color('black')
    
    legend_patches2 = [Patch(facecolor=colors_building[i], edgecolor='black',
                             label=building_labels[i] + 'm', alpha=0.85)
                       for i in range(n_building_bins)]
    ax2.legend(handles=legend_patches2,
               title='Far Building Height',
               loc='center left',
               bbox_to_anchor=(1.02, 0.5),
               framealpha=0.95,
               title_fontsize=12,
               fontsize=10)
    
    # Main title and layout
    fig.suptitle('Parallel Angle X Far Zone Analysis - ISA & Building Height', 
                 fontsize=18, fontweight='bold', y=0.98)
    
    plt.tight_layout(rect=[0, 0, 0.88, 0.96])
    
    # Save
    if save:
        filepath = OUTPUT_DIR / "figure7_grouped_boxplot_combined.png"
        fig.savefig(filepath, dpi=300, facecolor='white', bbox_inches='tight')
        print(f"\nSaved: {filepath}")
        
        filepath = OUTPUT_DIR / "figure7_grouped_boxplot_combined.pdf"
        fig.savefig(filepath, dpi=300, facecolor='white', bbox_inches='tight')
        print(f"Saved: {filepath}")
    
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    return fig

In [ ]:
fig = plot_grouped_boxplot_combined(
    final_df,
    isa_col='far_isa',
    building_col='far_building',
    error_col='abs_error_node',
    min_samples=500,
    interval = 2,
    # y_max=40,
    save=True,
    show=True
)

In [ ]:
"""
2D Scatter plots with Seaborn
- All individual samples visible
- No aggregation
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

OUTPUT_DIR = Path("figures")
OUTPUT_DIR.mkdir(exist_ok=True)


def get_flow_angle_zone(flow_angle):
    fa = flow_angle % 360
    if (fa >= 350 or fa <= 10) or (170 <= fa <= 190):
        return 'Parallel'
    elif (80 <= fa <= 100) or (260 <= fa <= 280):
        return 'Perpendicular'
    else:
        return 'Other'


def plot_scatter_2d(df, 
                    isa_col='far_isa',
                    building_col='far_building',
                    error_col='abs_error_node',
                    figsize=(14, 12),
                    save=True, show=True):
    """
    2x1 scatter plots:
    - Top: Error vs Building Height (colored by ISA)
    - Bottom: Error vs ISA (colored by Building Height)
    """
    
    # Filter to Parallel zone
    df_parallel = df[df['flow_angle'].apply(get_flow_angle_zone) == 'Parallel'].copy()
    data = df_parallel[[isa_col, building_col, error_col]].dropna()
    data = data[data[building_col] >= 6]  # Building >= 6m
    
    print(f"[Data] Samples: {len(data):,}")
    
    fig, axes = plt.subplots(2, 1, figsize=figsize, facecolor='white')
    
    # Top: Error vs Building Height (colored by ISA)
    ax1 = axes[0]
    scatter1 = ax1.scatter(
        data[building_col], 
        data[error_col],
        c=data[isa_col],
        cmap='Reds',
        alpha=0.4,
        s=10,
        edgecolors='none'
    )
    
    cbar1 = plt.colorbar(scatter1, ax=ax1)
    cbar1.set_label('Far ISA (%)', fontsize=12)
    
    ax1.set_xlabel('Far Building Height (m)', fontsize=13)
    ax1.set_ylabel(r'$|Relative\ Error|$ (m)', fontsize=13)
    ax1.set_title(f'Error vs Building Height (colored by ISA) - n={len(data):,}', 
                  fontsize=14, fontweight='bold')
    ax1.set_ylim(0, 5)
    ax1.grid(True, alpha=0.3)
    ax1.tick_params(labelsize=11)
    
    # Bottom: Error vs ISA (colored by Building Height)
    ax2 = axes[1]
    scatter2 = ax2.scatter(
        data[isa_col], 
        data[error_col],
        c=data[building_col],
        cmap='Greens',
        alpha=0.4,
        s=10,
        edgecolors='none'
    )
    
    cbar2 = plt.colorbar(scatter2, ax=ax2)
    cbar2.set_label('Building Height (m)', fontsize=12)
    
    ax2.set_xlabel('Far ISA (%)', fontsize=13)
    ax2.set_ylabel(r'$|Relative\ Error|$ (m)', fontsize=13)
    ax2.set_title(f'Error vs ISA (colored by Building Height) - n={len(data):,}', 
                  fontsize=14, fontweight='bold')
    ax2.set_ylim(0, 5)
    ax2.grid(True, alpha=0.3)
    ax2.tick_params(labelsize=11)
    
    fig.suptitle('Parallel Zone: Far ISA × Building Height Analysis', 
                 fontsize=18, fontweight='bold', y=0.98)
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    
    if save:
        filepath = OUTPUT_DIR / "figure7_scatter_2d.png"
        fig.savefig(filepath, dpi=300, facecolor='white', bbox_inches='tight')
        print(f"Saved: {filepath}")
        
        filepath = OUTPUT_DIR / "figure7_scatter_2d.pdf"
        fig.savefig(filepath, dpi=300, facecolor='white', bbox_inches='tight')
        print(f"Saved: {filepath}")
    
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    return fig


def plot_scatter_3d(df,
                    isa_col='far_isa',
                    building_col='far_building', 
                    error_col='abs_error_node',
                    figsize=(12, 10),
                    save=True, show=True):
    """
    3D scatter plot: X=Building, Y=ISA, Z=Error
    """
    from mpl_toolkits.mplot3d import Axes3D
    
    # Filter to Parallel zone
    df_parallel = df[df['flow_angle'].apply(get_flow_angle_zone) == 'Parallel'].copy()
    data = df_parallel[[isa_col, building_col, error_col]].dropna()
    data = data[data[building_col] >= 6]
    data = data[data[error_col] <= 5]  # Cap for visibility
    
    print(f"[Data] Samples: {len(data):,}")
    
    fig = plt.figure(figsize=figsize, facecolor='white')
    ax = fig.add_subplot(111, projection='3d')
    
    scatter = ax.scatter(
        data[building_col],
        data[isa_col],
        data[error_col],
        c=data[error_col],
        cmap='Reds',
        alpha=0.4,
        s=10,
        edgecolors='none'
    )
    
    ax.set_xlabel('Building Height (m)', fontsize=12)
    ax.set_ylabel('ISA (%)', fontsize=12)
    ax.set_zlabel('|Relative Error| (m)', fontsize=12)
    ax.set_title(f'3D Scatter: Building × ISA × Error (n={len(data):,})', 
                 fontsize=14, fontweight='bold')
    
    cbar = fig.colorbar(scatter, ax=ax, shrink=0.6, pad=0.1)
    cbar.set_label('|Relative Error| (m)', fontsize=11)
    
    if save:
        filepath = OUTPUT_DIR / "figure7_scatter_3d.png"
        fig.savefig(filepath, dpi=300, facecolor='white', bbox_inches='tight')
        print(f"Saved: {filepath}")
    
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    return fig


# 사용 예시
# plot_scatter_2d(final_df)
# plot_scatter_3d(final_df)

In [ ]:
plot_scatter_2d(final_df)

In [ ]:
plot_scatter_3d(final_df)

In [ ]:
"""
Grouped Boxplot - X: ISA, Color: Building Height
=================================================
- X-axis: Far ISA bins (customizable)
- Y-axis: Relative WSE Error (m) - LINEAR scale
- Color: Building Height bins (customizable)
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from pathlib import Path

# Publication style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'font.size': 14,
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 10,
    'figure.titlesize': 16,
    'font.family': 'sans-serif',
    'axes.linewidth': 1.2,
    'grid.alpha': 0.3,
})

OUTPUT_DIR = Path("figures")


def get_flow_angle_zone(flow_angle):
    """Classify flow angle into zones."""
    fa = flow_angle % 360
    if (fa >= 350 or fa <= 10) or (170 <= fa <= 190):
        return 'Parallel'
    elif (80 <= fa <= 100) or (260 <= fa <= 280):
        return 'Perpendicular'
    else:
        return 'Other'


def create_grouped_boxplot(df,
                           isa_col='far_isa',
                           building_col='far_building',
                           error_col='abs_error_node',
                           # Customizable bins
                           isa_bins=[0, 10, 20, 30, 40, 50, 60, 70, 80, 90],
                           building_bins=[6, 7, 8, 9, 10, 11, np.inf],
                           # Plot settings
                           y_max=None,
                           figsize=(12, 7),
                           title=None,
                           cmap='plasma',
                           filter_parallel=True,
                           save=True,
                           show=True,
                           output_name='grouped_boxplot_building'):
    """
    Grouped boxplot with customizable bins.
    
    Parameters
    ----------
    df : DataFrame
        Input data
    isa_col : str
        Column name for ISA values
    building_col : str
        Column name for building height values
    error_col : str
        Column name for error values
    isa_bins : list
        Bin edges for ISA - THIS IS X-AXIS
    building_bins : list
        Bin edges for building height - THIS IS COLOR
        Use np.inf for last bin like "11+"
    y_max : float or None
        Max Y-axis value. If None, auto-calculated
    figsize : tuple
        Figure size
    title : str or None
        Custom title
    cmap : str
        Colormap name for building height bins
    filter_parallel : bool
        If True, filter to Parallel flow zone only
    save : bool
        Save figure to file
    show : bool
        Display figure
    output_name : str
        Output filename (without extension)
    
    Returns
    -------
    fig : matplotlib Figure
    """
    
    print("GROUPED BOXPLOT: ISA (x) × Building Height (color) -> Error (y)")
    
    # Create bin labels
    
    # ISA labels (X-axis)
    isa_labels = []
    for i in range(len(isa_bins) - 1):
        isa_labels.append(f'{int(isa_bins[i])}-{int(isa_bins[i+1])}%')
    
    # Building height labels (Color)
    building_labels = []
    for i in range(len(building_bins) - 1):
        if building_bins[i+1] == np.inf:
            building_labels.append(f'{building_bins[i]:.0f}+')
        else:
            building_labels.append(f'{building_bins[i]:.0f}-{building_bins[i+1]:.0f}')
    
    n_isa_bins = len(isa_labels)
    n_building_bins = len(building_labels)
    
    print(f"\n[X-axis] ISA bins: {isa_labels}")
    print(f"[Color] Building Height bins: {building_labels}")
    
    # Filter and prepare data
    
    if filter_parallel:
        df_filtered = df[df['flow_angle'].apply(get_flow_angle_zone) == 'Parallel'].copy()
        print(f"\n[Data] Parallel zone: {len(df_filtered):,} / {len(df):,}")
    else:
        df_filtered = df.copy()
        print(f"\n[Data] All data: {len(df_filtered):,}")
    
    # Clean data
    data = df_filtered[[isa_col, building_col, error_col]].dropna()
    data = data[data[error_col] > 0]
    data = data[data[building_col] >= building_bins[0]]
    data = data[(data[isa_col] >= isa_bins[0]) & (data[isa_col] <= isa_bins[-1])]
    print(f"[Data] After cleaning: {len(data):,}")
    
    # Assign bins
    data['isa_bin'] = pd.cut(data[isa_col], bins=isa_bins, 
                              labels=isa_labels, right=False)
    data['building_bin'] = pd.cut(data[building_col], bins=building_bins,
                                   labels=building_labels, right=False)
    
    data = data.dropna(subset=['isa_bin', 'building_bin'])
    print(f"[Data] After binning: {len(data):,}")
    
    # Colors for Building Height bins
    colormap = plt.cm.get_cmap(cmap)
    colors = [colormap(i / (n_building_bins - 1)) for i in range(n_building_bins)]
    
    # Create figure
    fig, ax = plt.subplots(figsize=figsize)
    
    # Prepare grouped boxplot
    # X-axis: ISA, Color: Building Height
    box_data = []
    positions = []
    box_colors = []
    
    group_width = 0.8
    box_width = group_width / (n_building_bins + 1)
    
    for i, isa_label in enumerate(isa_labels):
        group_center = i + 1
        
        for j, building_label in enumerate(building_labels):
            mask = (data['isa_bin'] == isa_label) & (data['building_bin'] == building_label)
            subset = data[mask][error_col].values
            
            if len(subset) > 0:
                box_data.append(subset)
            else:
                box_data.append([np.nan])
            
            offset = (j - (n_building_bins - 1) / 2) * box_width
            positions.append(group_center + offset)
            box_colors.append(colors[j])
    
    # Create boxplot
    bp = ax.boxplot(box_data,
                    positions=positions,
                    widths=box_width * 0.85,
                    patch_artist=True,
                    showfliers=True,
                    flierprops=dict(marker='o', markersize=2, alpha=0.3,
                                   markerfacecolor='gray', markeredgecolor='none'))
    
    # Style boxes
    for patch, color in zip(bp['boxes'], box_colors):
        patch.set_facecolor(color)
        patch.set_edgecolor('black')
        patch.set_linewidth(0.8)
        patch.set_alpha(0.85)
    
    for whisker in bp['whiskers']:
        whisker.set_color('black')
        whisker.set_linewidth(0.8)
    for cap in bp['caps']:
        cap.set_color('black')
        cap.set_linewidth(0.8)
    for median in bp['medians']:
        median.set_color('#8B0000')
        median.set_linewidth(1.5)
    
    # Axes - LINEAR scale
    
    # Y-axis
    if y_max is None:
        y_max = np.nanpercentile(data[error_col], 99) * 1.1
    ax.set_ylim(0, y_max)
    ax.set_ylabel('Relative WSE Error (m)', fontweight='bold', fontsize=14)
    
    # X-axis: ISA
    ax.set_xlabel('Far ISA (%)', fontweight='bold', fontsize=14)
    ax.set_xticks(range(1, n_isa_bins + 1))
    ax.set_xticklabels(isa_labels, fontsize=12)
    ax.set_xlim(0.3, n_isa_bins + 0.7)
    
    # Grid
    ax.yaxis.grid(True, alpha=0.4, linestyle='-', linewidth=0.5)
    ax.xaxis.grid(False)
    ax.set_axisbelow(True)
    
    # Legend for Building Height
    legend_patches = [Patch(facecolor=colors[i], edgecolor='black',
                            label=f'{building_labels[i]}m', alpha=0.85)
                      for i in range(n_building_bins)]
    
    ax.legend(handles=legend_patches,
              title='Building Height',
              loc='lower right',
              framealpha=0.95,
              title_fontsize=12,
              fontsize=10,
              ncol=2 if n_building_bins > 6 else 1)
    
    # Title
    if title is None:
        zone_str = "(Parallel Flow Zone)" if filter_parallel else "(All Data)"
        title = f'Relative WSE Error by ISA and Building Height\n{zone_str}'
    ax.set_title(title, fontweight='bold', fontsize=14, pad=15)
    
    # Sample size
    total_n = len(data)
    ax.text(0.02, 0.98, f'n = {total_n:,}', transform=ax.transAxes,
            fontsize=11, ha='left', va='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    plt.tight_layout()
    
    # Save
    if save:
        filepath_png = OUTPUT_DIR / f"{output_name}.png"
        fig.savefig(filepath_png, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"\nSaved: {filepath_png}")
        
        filepath_pdf = OUTPUT_DIR / f"{output_name}.pdf"
        fig.savefig(filepath_pdf, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"Saved: {filepath_pdf}")
    
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    # Summary
    print("Median Error by ISA:")
    for il in isa_labels:
        mask = data['isa_bin'] == il
        if mask.sum() > 0:
            print(f"  {il}: {data[mask][error_col].median():.3f} m (n={mask.sum():,})")
    
    print("\nMedian Error by Building Height:")
    for bl in building_labels:
        mask = data['building_bin'] == bl
        if mask.sum() > 0:
            print(f"  {bl}m: {data[mask][error_col].median():.3f} m (n={mask.sum():,})")
    
    return fig


# Example usage

print("""
================================================================================
USAGE EXAMPLES
================================================================================

# Basic usage:
fig = create_grouped_boxplot(final_df)

# Custom bins:
fig = create_grouped_boxplot(
    final_df,
    isa_bins=[0, 5, 10, 20, 30, 40, 50, 60, 70, 80, 90],   # X-axis
    building_bins=[6, 7, 8, 9, 10, 11, np.inf],            # Color (np.inf = 11+)
    y_max=5,
    cmap='plasma',
    output_name='my_plot'
)

# Coarse bins:
fig = create_grouped_boxplot(
    final_df,
    isa_bins=[0, 20, 40, 60, 80],
    building_bins=[6, 8, 10, np.inf]
)

# Fine bins:
fig = create_grouped_boxplot(
    final_df,
    isa_bins=[0, 5, 10, 15, 20, 30, 40, 50, 60, 70, 80, 90],
    building_bins=[6, 6.5, 7, 7.5, 8, 8.5, 9, 9.5, 10, 10.5, 11, np.inf]
)

Layout:
- X-axis: Far ISA bins
- Y-axis: Relative WSE Error (m) - linear scale
- Color: Building Height bins

Colormaps: 'plasma', 'viridis', 'cividis', 'coolwarm', 'RdYlBu', 'magma'
================================================================================
""")

In [ ]:
fig = create_grouped_boxplot(
    final_df,
    isa_bins=[0, 10, 20, 30, 40, 50, 60, 70, 80, 90],   # X축
    building_bins=[6, 7, 8, 9, 10, 11, np.inf],            # Color (11+ 포함)
    # y_max=5,
    cmap='plasma'
)